# Qwen3.8 pi 融合：Final + 辅料（10 : 4 : 0.5 : 3 : 8 : 5 : 1）+ 单独搜索的 Coder390 增量 + 跨架构增量 N（Nanbeige4.3-33B）与 N₂（SenseNova-Loop-9B-Preview）

主模型 S = `bytkim/Qwen3.8-27B-pi`（Qwen3.8-27B 的编码微调；commit 和读取的目录在第一次运行时锁定）。

\[
F=S+\lambda_g\big[D-\kappa(S-B)\big]+\mu\,C+\nu\,N+\nu_2\,N_2,\qquad \mu\in\{0.25,0.5,0.75,1.0\}\ \text{（搜索）},\ \ \nu,\nu_2\in\{0.25,0.5,1.0\}\ \text{（在获胜配方上依次评）}
\]

辅料增量 D 按固定比例并入（系数 = 比例 / 10）：

| 辅料 | 仓库 | 比例 | 系数 |
|---|---|---|---|
| Final 供体 M | `YFC-112358/Qwen3.8-27B-TA-Aux-Final-v1` @ `164e6c38…` | 10 | 1 |
| StartLux | V1.0-27B-Preview，**尚未发布**，仓名待填；对 Qwen3.6-27B 取增量 | 4 | 0.4 |
| Winery GrandCru | `WineryLabs/Winery-Qwen3.8-27B-GrandCru-GGUF`（Q8_0 GGUF）@ `9a761951…` | 0.5 | 0.05 |
| DavidAU Turbo-Brilliance-50X | 正式融合时再定：已发布 BF16 safetensors 就读它，否则读 `DavidAU/Qwen3.8-27B-Turbo-Brilliance-50X-Reasoning-Instruct-NEO-MAX-MTP-GGUF`（自动挑 BF16 > F16 > Q8_0 > Q6_K 的单文件；目前仓库里只有 Q6_K，gated，需作者批准） | 3 | 0.3 |
| DELLA 组 | 下面八个模型做 DELLA 合并后的结果 | 8 | 0.8 |
| EfficientThink-Uncensored | `nerkyor/Qwen3.8-27B-EfficientThink-Uncensored-K3-Opus5-Grok4.6-GPT5.6Sol-SFT-SimPO-DFlash2`（Coder390 的基座），对 B 取增量 | 5 | 0.5 |
| DELLA 组 2 | VeriLoop-E2 与 sabi-noema 做 DELLA 合并后的结果 | 1 | 0.1 |

DELLA 组的成员，成员之间权重相同：

| 成员 | 仓库 | 底座 | 备注 |
|---|---|---|---|
| Swift-1.5 | `ukisai/Swift-1.5-Qwen3.8-27b` @ `bc7a1e10…` | B | 也在 Winery 里 |
| ThinkingCap | `bottlecapai/ThinkingCap-Qwen3.8-27B` @ `7891e03d…` | B | gated：先用 HF_TOKEN 对应的账号在模型页同意条款。许可证是 PolyForm Small Business |
| AREX-2 | `BAAI/AREX-2` @ `d4e3502f…` | B | |
| Signal-3.8 | `agentionai/Signal-3.8-27B` @ `54552f73…` | B | 只替换了 `lm_head`，主干与 B 相同，所以在组里只作用于输出头；也是 A3 的来源之一，已经通过 Final 进来过一次 |
| Omnimerge-v6 | `ManniX-ITA/Qwen3.8-27B-Omnimerge-v6` | B | Qwen3.8 底座 + 三个 Qwen3.6 微调的移植增量 |
| Darwin-27B-RSI | `FINAL-Bench/Darwin-27B-RSI` | **Qwen/Qwen3.5-27B** | Qwen3.5 一系（父模型 Darwin-27B-Opus），增量对 Qwen3.5-27B 取，再跨两代移到 3.8 |
| Simplicio-27B | `wesleysimplicio/Simplicio-27B` @ `19330de3…` | B | Unsloth 的 LoRA 微调，读根目录合并好的完整权重；增量是低秩的 |
| Tess-4.5-27B | `migtissera/Tess-4.5-27B` @ `91b23113…` | B | 全量微调 |

DELLA 组 2（参数相同，底座都是 B）：VeriLoop-E2 `tsinghua-sigs-robot-lab/VeriLoop-E2` @ `44b6f230…`（仓库里 `mtp.veriloop_engram_registry.*` 一批整数张量不读取）、sabi-noema `sabicap/sabi-noema` @ `c12b2070…`。

单独搜索的增量 C = Coder390 − EfficientThink-Uncensored：`nerkyor/Qwen3.8-27B-Coder390-EfficientThink-Opus5.5-GPT6Astra-Grok4.7-DSV4Pro-K3-SFT-RLOO-MTP-DFlash2`（`BF16/`）@ `23a3f86a…` 减去它的基座。C 在 λ 括号之外，只加在主干矩阵上，系数 μ 和 λ、κ、输出头系数一起搜索。

本文件只做最后一步。A3 与 Final 已构造、审计并发布，相关代码已删去；要重建供体请用更早的 notebook（`…_f1.ipynb`）。

**跨架构增量 N**（本版新增，`SETTINGS["tm_extra"]`）：`Nanbeige/Nanbeige4.3-33B` 是另一种架构（Llama 结构，52 层、hidden 7168、head_dim 128、词表 166912），不能直接取 A − B。这里用之前 TM notebook（`Qwen38_tm_multisource`）的 Transport-and-Merge：两边在同一份 2000 篇校准文本上的逐篇均值激活 → 通道间的相关代价 + 熵正则 OT → B 每张被改矩阵激活最大的 128 行放入 α·(搬过来的行 − B 的行)。target 的激活直接读 TM notebook 存在 `YFC-112358/tm-acts-qwen38-multi` 里的那份；Nanbeige 的前向在 CPU 上做。go/no-go 判定（与 TM notebook 同一套）一个模块都不过时，成品就不含 N，其余照常。每个供体都必须有贡献（`require_all_donors`，默认开）：λ、μ、ν 都只在正值里选，S 只作对照。N 搬过来的是传输找到的一小份方向，不是 Nanbeige 的 Agent 能力；代理 NLL 也量不到 Agent 任务，ν 选出来之后请用你自己的测试（包括编码 / Agent 类题）判断。

**第二个跨架构增量 N₂**（e15 新增，`SETTINGS["tm_extra2"]`，模块 `qmerge_tm2`）：`sensenova/SenseNova-Loop-9B-Preview` 待遇和 Nanbeige 一样：同一份校准文本、同一个激活仓（同一个 commit）、同一套 go/no-go、CKA 预筛、层对 OT、128 行注入，自己的配置、固定 commit、前向闸门、缓存目录（`DRIVE_DIR/tm_delta2`）和断点续跑；ν₂ 必须 > 0。它是循环深度模型（32 个不同的层，第 [4, 28) 层组成的块执行两遍，一次前向 56 次执行），每次执行各算一个候选供体层，一份供体权重可以搬到多个目标层。供体前向用作者的远程代码：第一次运行会把代码下载到 Drive、打印位置和摘要后**停下**，你读完代码把 `code_sha256` 填进第 1 节才会执行；它在独立子进程里、用单独安装的 transformers 5.18 运行，本内核的 transformers 版本（缓存身份的一部分）不变。模型卡上的结构（执行顺序、层数、宽度、头数、词表、3:1 混合注意力）在前向闸门里按实际 config 和实际前向核对，不符就停下。详见第 1 节“跨架构增量 N₂ 怎么算”。

全部在 CPU（高 RAM）运行时上做，每次都是“从头运行全部单元格”，notebook 自己判断做到了哪一步：

锁定输入 → 算出辅料增量 D 和增量 C，存到 Google Drive → 算跨架构增量 N（供体前向、go/no-go 判定、层对 OT、融合；见下）→ 算 N₂（远程代码审读、前向闸门、逐执行激活、判定、层对 OT、融合）→ λ/κ/输出头/μ 系数搜索（前向 NLL 代理；最多 12 个 GP 主候选 + 可能的 head 补评 + 初始 winner 的 μ=0 消融，`global_trials` 只计主候选）→ 在获胜配方上评 ν（加 N）→ 再评 ν₂（加 N₂）→ 分量筛选（把 F−S 拆成正交分量；主干按逐题一阶梯度定连续系数，输出头只按精确端点差保留或移除；用两折交叉拟合在折外题上做内部代理 NLL 核对，所有题共享教师强制激活协方差分解，不是完全独立验证或终测）→ 构造选中的候选并核对 → 上传 BF16 到 `output_repo`（默认私有）→ CPU 分块导出 FP8 → 额外上传到独立的 `output_repo + "-FP8"` 仓库。

分量筛选以 `failed` / `skipped` 回退时，本次使用全局搜索结果，但不把这个回退选择保存为可复用的选择记录；下次从头运行会重试，已经完成的搜索评分和分量断点仍可复用。

这里不做模型评测：没有冒烟测试、代理验证、生成验证和终测。上传后请用你自己的测试部分评测。

**现在就可以运行**：搜索数据会从公开评测集自动生成；StartLux 未发布时，最后一格先把 D 中已经确定的部分（Final + Winery + DELLA 组 + EfficientThink + DELLA 组 2）和 C 算好存进 Drive，再把 S 下载下来算好它在搜索集和代理验证集上的前向基线，并在真实 S 上核对分量筛选要用的反向，然后算跨架构增量 N（最长的一步，约 20–35 小时，跨会话续跑），再算 N₂（约 10–21 小时；第一次会先停下来让你审读 SenseNova 的远程代码），然后报告在等 StartLux。StartLux 发布后填上仓名再运行：先看 DavidAU 有没有发布 BF16、决定读哪个，再在已有的 D 上补加 StartLux 和 DavidAU 两项。


## 1. 配置

只改下面这一格，然后从上到下运行。

| 待填项 | 说明 |
|---|---|
| `STARTLUX_REPO` | StartLux 正式仓名。留空且 `STARTLUX_REQUIRED=True`：先算 D 已确定的部分、C 和 S 的基线，然后等待 |
| DavidAU 的 `bf16_repos` | 正式融合时依次查这些仓库，哪个已发布完整的 BF16 safetensors 就改读它，否则读 GGUF。仓名是按 DavidAU 的命名习惯猜的，正式发布的名字不同就改这里 |
| `mu_grid` | Coder390 增量 C 的系数 μ 的取值，默认 0.5 / 0.75 / 1.0 |
| `require_all_donors` | 默认 `True`：每个供体都必须有贡献，系数可以很小但不能为 0。λ（D：Final 与各辅料）、μ（C）、ν（N）、ν₂（N₂）在每个可选候选里都 > 0，所以 `lambda_grid` 不能含 0（默认改成 0.025 / 0.05 / 0.10 / 0.20），`mu_grid` 加了 0.25，`nu_grid` 是 0.25 / 0.5 / 1.0；S 只作对照、不能当选；μ=0 消融关闭；ν 必须从 `nu_grid` 里选一个；分量筛选不能把某个供体项涉及的分量全部设成 0；N 的判定一个模块都不过时，仍用通过上界标定的模块构造（结果里记为 `forced`）。输出头 λ_head 和 κ 不是供体项，仍可为 0 |
| `tm_extra2` | 第二个跨架构增量 N₂（SenseNova-Loop-9B-Preview，e15）。`code_sha256` 第一次留空：运行会停下并打印远程代码的位置和摘要，读完代码再填；`expect` 是模型卡上的结构，前向闸门按它核对实际的 config 和执行顺序；`k_proj` 默认 `auto`；`nu_grid` 是 ν₂ 的取值。其余项见“跨架构增量 N₂ 怎么算” |
| `tm_extra` | 跨架构增量 N（Nanbeige4.3-33B）。`enabled=False` 关掉；`nu_grid` 是在获胜配方上评的 ν；`act_repo` 是之前 TM notebook 的激活仓（HF_TOKEN 要能读这个私有 dataset）。其余项见“跨架构增量 N 怎么算” |
| `output_repo` | 上传目标，必须是新的（或空的）仓库，默认私有（`private=True`）。HF_TOKEN 要有这个账号或组织的写权限 |
| `fp8_enabled` / `fp8_repo` | 默认在 BF16 上传完成后额外导出并上传 FP8；仓名留空时为 `output_repo + "-FP8"`，可关闭或改名，隐私设置沿用 `private` |
| `fp8_dir` | 留空：FP8 成品与续跑分片存到 `DRIVE_DIR/fp8/`，建议另外留出约 40 GiB（实际取决于保留原精度的权重比例） |
| 搜索数据 | `AUTO_EVAL_DATA=True`（默认）时由第 4 节从公开评测集自动生成，不用填；想用自己的数据就填 `SETTINGS["search_file"]`，已填的不会被覆盖 |
| HF_TOKEN | 对应的账号要先在 `bottlecapai/ThinkingCap-Qwen3.8-27B` 的模型页同意条款，否则读不到这个 DELLA 成员 |

### 用哪个运行时

全部在 CPU 高 RAM 运行时上做，不需要 GPU。在 GPU 运行时上第 5 节不会开始计算：这些都是 CPU 和磁盘的活，按 GPU 的价格不划算（确实要这样做，设 `allow_prepare_on_gpu=True`）。

| 阶段 | 做什么 | 估计用时 | 估计单元 |
|---|---|---|---|
| StartLux 发布前就能做的部分 | 算 D 已确定的部分（读 Final、B、DELLA 组 8 个成员和 Qwen3.5-27B、EfficientThink、DELLA 组 2 的 2 个成员，各约 55GB，以及 Winery GGUF 约 29GB，共约 800GB）；算 C（读 Coder390 与 EfficientThink，约 110GB）；下载 S 并算它在搜索集和代理验证集上的前向基线；反向自检 | 7–20 小时 | 约 1–3 |
| 跨架构增量 N（等 StartLux 时就做，排在上一行之后） | 前向闸门（4 条短提示走完 Nanbeige 全部层，读一遍权重约 62GB）；Nanbeige 在 2000 篇校准文本上的逐层激活（4 段，每段读一遍权重；26 个记录层）；下载 target 激活约 15GB；判定、CKA 预筛、层对 OT（约 300 个大矩阵 Sinkhorn）、融合 | 约 20–35 小时（激活约 15–29 小时，OT 与融合约 4–7 小时），跨会话自动续跑 | 约 3–5.5 |
| 跨架构增量 N₂（e15，排在 N 之后） | 审读远程代码后放行；下载供体权重约 18GB 到本地盘；为子进程单独安装 transformers 5.18；前向闸门（原生前向与 float32 前向对照、4 条短提示、执行顺序与结构核对）；2000 篇的逐执行激活（8 段，每段一个子进程，每段先装载一次权重约 1–3 分钟；56 次执行里 MLP 记 28 次、注意力记 14 次）；判定、CKA 预筛、层对 OT（约 430 个层对，矩阵比 Nanbeige 的小）、融合 | 约 10–21 小时（激活约 6–13 小时：每 token 约 22 GFLOP，按 150–300 GFLOPS 推算，线性注意力在 CPU 上的实现可能更慢；OT 与融合约 4–8 小时），跨会话自动续跑 | 约 1.5–3.2 |
| StartLux 发布后 | 决定 DavidAU 读 BF16 还是 GGUF；在 D 上补加 StartLux 和 DavidAU（读 StartLux、Qwen3.6 与 DavidAU，约 140–165GB）；全局搜索（最多 12 个 GP 主候选 + 可能的 head 补评 + 初始 winner 的 μ=0 消融，`global_trials` 只计主候选；网格 67 个组合）；分量筛选（96 题的前向、梯度自检、逐题反向，再用交叉拟合评 3 种方案）；有 N 时在获胜配方上再评 3 个 ν，有 N₂ 时再评 3 个 ν₂；构造选中的候选；上传（约 55GB） | 35–86 小时，跨会话自动续跑 | 5.3–13 |

合计约 11–23 单元（N 约占 3–5.5，N₂ 约 1.5–3.2，ν₂ 线搜索约 0.5–0.9；比 e10 多约 1.5–3.5 单元的部分：D 从头重算、多一个 C、全局搜索多 4 次；`component_options["rows"]="search"` 只用 48 题时约少 9–18 小时，`plans` 少一种方案约少 2–3 小时）。费率按 CPU 高 RAM 约 0.15 单元/小时估算，以 Colab 资源面板显示的为准。用时是按模型规模推算的，没有实测，主要花在搜索的前向计算上，长短取决于这台 CPU 的 float32 矩阵乘速度（上表按 150–300 GFLOPS 推算）；每个候选开始评分时会按实测速度打印估计分钟数。

- **搜索怎么在 CPU 上做**：每个候选的 54.7GB 权重都要由 S、D、C、B 逐张量算出（读本地盘约 210GB）。CPU 上前向评分用 float32，51GB 内存装不下整个模型，所以逐层评分：一层的权重算出后，48 条题都过完这一层再换下一层，下一层要读的文件在后台预读。每个候选约 1–2 小时，主要是前向计算本身。
- **分量筛选在 CPU 上的开销**：反向也是逐层做的。前向时每 8 层（按内存自动定）存一次所有题的隐状态，反向时从上往下一段一段地从存下的隐状态重算这一段的输入，再逐题对每一层做反向。整个梯度这一步约为 96 题前向的 5 倍计算量，中间结果在 Drive 上约 5–10GB（算完自动删），各矩阵的分解约 3GB（上传后可删）。反向开始前先做约半小时的梯度自检。之后的交叉拟合最多评 8 次半份（各 48 题）和 1 次全份，约 9–17 小时。
- StartLux 发布前不要把 `STARTLUX_REQUIRED` 改成 `False` 去跑完整流程，除非确定成品不要 StartLux。之后再加 StartLux，搜索和上传都要重做一遍（并换一个新的 `output_repo`）。

### 搜索数据从哪来

第 4 节按固定 commit 下载 6 个公开评测集，每个评测集算一类：

| 类别 | 来源 | 评分 |
|---|---|---|
| math_word | `openai/gsm8k` test | 数值 |
| math_competition | `HuggingFaceH4/MATH-500`（只取整数答案） | 数值 |
| knowledge_mc | `TIGER-Lab/MMLU-Pro` test（10 选 1） | 选项字母 |
| chinese_mc | `ceval/ceval-exam` val（28 个理工、医、法、语文等科目） | 选项字母 |
| code_reasoning | `cruxeval-org/cruxeval`（读代码推断返回值） | Python 字面量 |
| logic | `lukaemon/bbh`（10 个答案封闭的任务） | 选项 / 是否 / 数值 |

- 搜索只用其中的搜索集：每类 8 题，共 48 题。参考回答来自数据集本身：GSM8K、MATH 带解题过程，其余只有答案；都以 `<answer>…</answer>` 结尾。抽题前做一致性检查，不通过的题直接丢掉：GSM8K 每个计算标注都重算一遍，MATH 解答里的 `\boxed{}` 必须等于答案字段，选择题的答案必须是存在的选项。CRUXEval 的代码不在 notebook 里执行。
- 第 4 节仍和以前一样建四份互不重叠的数据（另外三份是代理验证 48 题、生成验证 96 题、终测 96 题），所以搜索集与之前的版本完全相同。分量筛选另外用代理验证集（它因此参与了选模型，不再是独立的测试）；生成验证和终测仍然不用，可以留给你自己的测试。
- 长度用 S 自己的 tokenizer 检查，超出搜索的 1024 token 上限（或另外几份的长度上限）的题不会入选。
- 抽题用固定种子，结果写到 `DRIVE_DIR/eval_data/<指纹>/`。之后的会话直接沿用同一批文件。
- **局限**：这些都是公开评测集，可能出现在模型的训练数据里。搜索的指标只是参考回答部分的平均 NLL（不开思考），不衡量思考长度和生成质量；它只用来在候选之间挑系数和分量，不能当作模型水平。

### 分量筛选怎么做（替代原来的局部搜索）

全局搜索定下 λ、κ、输出头系数和 μ，得到 F。默认最多 12 个 GP 主候选；`global_trials` 只计这些主候选，S 对照、可能的 head 补评和初始 winner 的 μ=0 消融都不占这项预算。非 tied 且 `head_grid` 恰好为二值时，可立即补评同一主候选的另一个 head 值并喂回 GP；当前 `[0, 0.05]` 就是两个端点，不自动生成 0.025，改成三值网格也不会自动补评。原来接着搜四个分组系数（深度二分 × 行内幅度二分）；现在改问：F 相对 S 的增量里，哪些部分真的让代理损失下降，各该留多少？

1. **拆成正交的块**：每个解码层矩阵的增量 Δ = F − S，先用这个矩阵在前向里实际收到的输入（每层抽约 8000 个 token）白化，再做随机化 SVD，按输出方向切成第 1–4、5–16、17–64 个方向和剩下的部分；每块再按 S 自己的微调 S − B 用得最多的 32 个输入方向，切成落在其内和其外两半。这 8 块两两正交，加起来正好等于 Δ。
2. **合成分量**：64 层分成 8 段，同一段、同一类模块（全注意力 / 线性注意力 / MLP）、同一块的合成一个分量，一个矩阵的增量就是它所属的几个分量的和；输出头（λ_head > 0 时）单独一个。一共约 192 个分量。每个分量有一个系数 c：1 是保持 F 里的样子，0 是去掉；只有主干分量在步长方案里使用连续系数，输出头在所有方案里都只做 0/1 取舍。
3. **一阶估计**：对 F 做一次前向和逐题反向，得到每道题的损失对每个主干矩阵的梯度 Gᵢ。主干分量 j 对题 i 的效果 ⟨Gᵢ, Δⱼ⟩ 就是这道题的损失对系数 c_j 的导数：留着它比去掉它，损失大约变化这么多。输出头不用一阶近似，而是在同一最终隐状态上直接把头换回 S，得到 L(1) − L(0) 的精确端点差。
4. **梯度自检**：开始最长的反向之前，先在两道题上把梯度算到最上面两段，挑效果最大的两块，各放大、缩小 10% 真跑一遍；实际变化和一阶预测相差超过 25%，就不再往下算，退回全局搜索的结果。
5. **统计**：按类别分层 bootstrap（2000 次），得到每个分量对目标（各类平均 NLL 的平均，与搜索相同）的效果、标准误、95% 区间、有益概率和有害概率。
6. **方案**：步长 t（默认 0.5、1；可加 2）只对主干分量做连续系数：效果先往 0 收缩 1 个标准误（在噪声以内的分量不动），再令 c = 1 − t × 效果 / 绝对值最大的主干效果，限制在 [0, 1.1]，即有害的缩小乃至去掉，有益的最多放大 10%。输出头的精确端点差不计入这个步长归一化，只按它的符号独立保留或移除，不放大。严格：只留有益概率 ≥ 0.9 的分量，其余去掉（“大概率有增益的才放上去”）。另可加宽松（只去有害概率 ≥ 0.9 的）和点估计（去掉效果为正的）。
7. **交叉拟合挑方案**：96 道题按类别和来源（搜索集 / 代理验证集）均衡地分成两折。每个方案只用一折的统计定系数，在另一折上用真实前向 NLL 评分（权重在内存里逐层算出，不写盘），再反过来；定本折系数时，另一折不提供损失或梯度，因此每道题“比 F 变了多少”都来自没有用该题损失 / 梯度定系数的方案。但所有题在分折前已共同用于教师强制激活协方差分解，所以这不是完全独立的折外验证，更不是未参与选型的最终验证。取折外题上改善最大的方案（bootstrap 下变好的概率不低于 `adopt_probability`，默认 0.8：几个方案里挑最好的那个，它的概率本身偏乐观；任一类别比 S 差超过 `proxy_max_category_drop` 的不算）；没有方案改善就保留 F。选中的方案再用全部 96 道题的统计定一次系数，仍须满足 S 分类门槛且在全部题上比 F 好才采用。
8. **改动上限**：主干分量系数最多 1.1，输出头始终只有 0/1；任何被改动的矩阵，每一行离 S 的改动都不超过这一行范数的 10%（`max_relative_change`），超过的行按比例缩回。全局搜索的候选、各个变体和最后的模型都这样处理。
9. **构造与核对**：改系数只用固定顺序的逐元素运算，结果在任何 CPU 上逐位相同；选中的模型写成文件后，逐张量核对它就是评分时的权重。

借鉴的现有方法（各取一部分，都不是复现）：白化后再 SVD（ASVD、SVD-LLM；RegMean 的输入二阶矩）；以奇异方向的段为单位（Task Singular Vectors、AdaRank）；按 S 自己的微调子空间看合并的干扰（TSV-Merge 的任务干扰视角）；梯度 × 增量作为一阶效果（Sens-Merging、泰勒展开剪枝）；按梯度调分层系数（AdaMerging）；按数据决定留哪些部分、再用真实损失核对（Localize-and-Stitch）。

- **局限**：一阶估计只在 Δ 不大时准，主干一次改很多分量时相互作用不计；所以主干取舍看的是折外题上的真实 NLL，一阶预测只并列作参考，输出头则用精确端点差。折外评分定方案时不用该题的损失 / 梯度，但所有题共享教师强制激活协方差分解，因此交叉拟合只是内部代理估计，不是完全独立验证，也不是未参与选型的最终验证。交叉拟合评的是“用一半题定出的系数”，最后上传的是用全部题定的，两者接近但不完全相同。全局搜索没有用过代理验证集，模型卡仍单独列出 F 和选中模型在代理验证集上相对 S 的变化；其中选中模型一项也是交叉拟合估计，全部 96 题上的数字仍然偏乐观。分量约 192 个、题只有 96 道，大部分分量的效果都在噪声附近，所以步长方案先收缩再走。
- **代理指标可能偏向“撤回 S 的微调”**：搜索和分量筛选打分用的是不开思考时、数据集自带参考答案的 NLL。S（pi）的微调可能让它的输出风格离这种文本更远，于是往 B 或供体方向回拉的改动可能降低这个 NLL，而模型并没有变好；κ(S − B) 这一项本身就是在撤回 S 的微调。所以运行时会打印两项诊断（只提示，不改变选择）：全局搜索后按 κ 分组的最低代理 NLL（随 κ 增大单调下降时提示）；分量筛选后，S 自己微调子空间之内（`.in`）的分量占增量能量和占一阶有益效果的比例（有益效果占比远高于能量占比时提示）。出现提示时，模型卡里会写一段警告；最终以开思考的生成测试为准，也可以把 `kappa_grid` 收窄到 `[0, 0.5]` 再比较。
- **反向自检**（`backward_check`，默认开）：分量筛选的反向要到全局搜索之后才第一次运行。为了提前知道它在这台机器的 transformers 上能不能用，等 StartLux 时（S 已在本地盘上）先在真实 S 上取一道短题，对最后一个线性注意力层和最后一个全注意力层各一个输入投影矩阵求梯度，再把矩阵沿梯度移动 ±0.1% 的范数真跑前向，中心差分要与一阶预测相差不到 25%；同时核对可求导的损失与评分引擎给出的 NLL 一致。另外沿 0.1 × D 的方向（合并本身的改动幅度）也做一次，只记录、不作判据。约 15–40 分钟，结果在 `backward_check.json`，只做一次。不通过时，之后的分量筛选在开始前就跳过，省下十几个小时；确认是这项检查本身有误时设 `backward_check=False`。没有在等待阶段做过时，分量筛选开始前会补做。
- 出错时（例如这台机器的 transformers 不支持对逐层评分做反向，或梯度自检没通过），打印原因并直接用全局搜索选中的候选（`component_on_error="fallback"`）；`"raise"` 则报错停下。

### 融合公式

\[
F=S+\lambda_g\big[D-\kappa(S-B)\big]+\mu\,C+\nu\,N+\nu_2\,N_2,\qquad C=K-E
\]
\[
D=(M-B)+0.4\,(X-R_{3.6})+0.05\,g_W(W-B)+0.3\,g_V(V-B)+0.8\,\mathrm{DELLA}\big(A_1-B,\dots,A_5-B,\ A_6-R_{3.5},\ A_7-B,\ A_8-B\big)+0.5\,(E-B)+0.1\,\mathrm{DELLA}\big(V_1-B,V_2-B\big)
\]

- 比例 10:4:0.5:3:8:5:1 按**系数**解释：Final 的增量系数为 1，其余为 weight/10。λ 只缩放 D 那个括号，比例不变；κ=0 是增量注入，κ=1 是向辅料组合插值；λ=0、λ_head=0、μ=0 精确等于 S（此时 κ 无影响）。
- S 是 pi；X 是 StartLux，R₃.₆ 是它的训练底座 Qwen3.6-27B；W 是 Winery；V 是 DavidAU（BF16 safetensors 时 g_V=1，GGUF 时扣除量化噪声）；A₁…A₅ 是 DELLA 组里的 Swift-1.5、ThinkingCap、AREX-2、Signal、Omnimerge-v6，A₆ 是 Darwin-27B-RSI，R₃.₅ 是 Qwen3.5-27B，A₇ 是 Simplicio-27B，A₈ 是 Tess-4.5-27B；E 是 EfficientThink-Uncensored；V₁、V₂ 是 DELLA 组 2 的成员。
- **C 在 λ 括号外**：K 是 Coder390，E 是它的基座，C = K − E 是 Coder390 自己的微调，直接按 μ 加到 S 上，不被 λ 缩放、不受 κ 影响，只加在主干矩阵上（输出头不加）。μ 和 λ、κ、λ_head 一起由全局搜索决定。E − B 另外以 0.5 进 D（被 λ 缩放），所以 Coder390 一系实际进来的是 λ·0.5·(E−B) + μ·(K−E)。
- **N 也在 λ 括号外**：N 是从 Nanbeige 搬过来的按行稀疏的增量（只有每张被改矩阵的 128 行非零），ν 不进 GP，全局搜索结束后在获胜配方上依次评 `nu_grid` 里的值：`require_all_donors=True`（默认）时取其中代理 NLL 最低的可行值（N 必须有贡献）；关掉时要比不加 N 严格更低才采用。λ=0、λ_head=0、μ=0、ν=0 精确等于 S。
- **N₂ 也在 λ 括号外**：做法与 N 相同。ν₂ 在“获胜配方 + 选中的 ν·N”上依次评 `tm_extra2["nu_grid"]`，取 NLL 最低的可行值（必须 > 0）。ν 和 ν₂ 依次各评 3 个值（共 6 次），而不是 3×3 联合网格（9 次）：每次评分约 1–2 小时，依次评少 3–6 小时；N 和 N₂ 都只改每张矩阵的 128 行、幅度很小，而且多半是不同的行，二者的相互作用是二阶的。λ=0、λ_head=0、μ=0、ν=0、ν₂=0 精确等于 S。
- **行改动上限**（`max_relative_change`，10%）作用在整个 F − S 上，包括 μ·C、ν·N 和 ν₂·N₂。D、C 算好、S 在本地盘上之后（等待阶段就做，正式融合时按完整的 D 再做一次），会打印每行 ‖D‖/‖S‖、‖C‖/‖S‖ 的分布，以及每个 λ、每个 μ、每个 (λ, μ) 组合下有多少行超过上限（κ=0；`row_change_report.json`）；λ·D 在最大的 λ 下超过 1% 时会建议 `lambda_grid` 的上限。
- 有效贡献还取决于各增量本身的大小。D 算完会打印各来源在全模型上的范数（也写在 `aux_delta_report.json`）。DELLA 组的结果大约是一个成员增量的大小，所以 0.8 的 DELLA 组与系数 1 的 Final 是同一量级，D 的范数会明显比 e10 大；先看 `row_change_report.json`，必要时收窄 `lambda_grid`。D 算完还会打印每个 DELLA 成员的主干增量范数，超过组内其他成员中位数 2 倍的成员会被点出来（Darwin 跨了两代底座，要特别看）；主干增量为 0 的成员（Signal）只列为说明。
- 跨底座的增量：StartLux（3.6 → 3.8）和 Darwin（3.5 → 3.8）都是把旧底座上学到的差值原样移到 3.8 上。3.6 → 3.8 在之前的融合里效果不错；3.5 → 3.8 隔了两代，风险更大。
- 重复计入：Signal 既在 A3（因而在 Final）里，又在 DELLA 组里（它只改了 `lm_head`，所以重复的只是输出头）；Winery 是 7 个模型的线性合并，含 Swift-1.5（也在 DELLA 组）和 Qwopus（也在 A3）；Final 经 A3 已含一个 DavidAU 模型，DavidAU 的 TURBO 系列又常混入 Qwen3.6 的 Fable-Fusion。这几个模型的实际份量比表面比例高。
- EfficientThink-Uncensored 和 DavidAU 的 TURBO 系列多是去审查的微调，成品的拒答行为可能和 pi 不同，模型卡里有一句说明。
- 受保护、保持 S 原值的参数：embedding、视觉塔、MTP、各类 norm、A_log、dt_bias、conv1d。输出头单独用 λ_head。

### 跨架构增量 N 怎么算

方法与之前的 TM notebook（`Qwen38_tm_multisource`，v5 注入规则）相同，只是改成在 CPU 上做，并且只有一个供体。

1. **输入**：从 `act_repo` 读校准文本 `calib_2000.json`（2000 篇，各截到 128 token）和 TM notebook 已经算好的 target（Qwen3.8-27B）逐篇均值激活。按元数据核对：B 仓库、校准文本的 SHA-256、T=2000、seq=128、按文档均值池化、所需的张量都在；激活是在 B 的另一个 commit 上算的，就比对两个 commit 的权重文件哈希，不同则报错。target 的层与 TM notebook 一样：注意力只在 16 个全注意力层，MLP 在这 16 层加每 4 层一层（共 32 层）。
2. **前向闸门**：Nanbeige 的流式前向（transformers 自己的解码层，权重按 HTTP 范围读取，float32）先把 4 条短提示（“The capital of France is” → Paris 等）走完全部 52 层，至少答对 2 条才继续。
3. **供体激活**：2000 篇按 500 篇一段，每段把第 0–50 层走一遍（下一层的权重在后台预读），在第 0、2、…、50 层（26 层）记下 v_proj、o_proj、gate_proj、up_proj、down_proj 输入和输出的逐篇均值。同样 token 长度的文档一批、不填充，与 TM notebook 带填充和 mask 的均值相同。每段完成后在 Drive 上留 26 个小文件（全部约 11GB）；段内每 4 层在本地盘存一次隐状态，内核重启后可以接着算，换了机器则从这一段开头重来。
4. **判定**（与 TM notebook 同一套，同一组阈值）：每个模块在 1/4、1/2、3/4 深度各取一对层算 OT，z 倍数 = gap·√T / 零信息期望；同一个 eps 下拿高斯噪声替换供体算锐度，锐度倍数 = 噪声锐度 / 真实锐度；target 与自身配对做上界标定。z 倍数 ≥ 2 且锐度倍数 ≥ 1.5 为 GO，都 ≥ 1.5 为 MARGINAL（也进融合），上界不过为 SKIP，其余 NO-GO。只有 GO / MARGINAL 的模块才往下算。
5. **层对**：TM notebook 对每个 target 层和每个供体层都算 OT（32×26 对），在 CPU 上要 40–50 小时。这里先用线性 CKA（两边激活的 Gram 矩阵）给每个 target 层留 `pair_keep`（默认 2）个最像的供体层，再加上按深度比例对齐的那一层，只对这些层对算两侧（输入、输出）的通道 OT。层级 OT 在至少被一个 target 层留下的供体层上做，没算的层对按最大代价计，与 TM notebook 处理缺失层对的方式相同。
6. **融合**：每张矩阵按层级计划取最多 `top_j`=2 个供体层（计划不集中时取全部留下的层），在 target 激活幅度最大的 128 行上算 Q_out W_供体 Q_in^T；去掉这 128 行的均值，σ₁ 能量占比仍高于 0.35（广播型）就退回不归一化，否则每行缩放到 B 那一行的范数；N 的这 128 行 = 0.05 ×（结果 − B 的行）。只算这 128 行与先搬整张再取这 128 行完全相同。

- **结果**：`DRIVE_DIR/tm_delta/tm_result.record.json`（状态、判定表），`DRIVE_DIR/tm_delta/delta/<指纹>/tm_delta.safetensors`（N，几百 MB）和 `tm_report.json`（每张矩阵的注入/收缩比、σ₁ 占比、退回 raw 与否、|ΔN|/|B|、用了哪些供体层；每个模块的层级计划和 DMR）。上传的仓库里也有一份（`tm_transport_report.json`，ν>0 时）。
- **局限**：TM notebook 实测过，软 OT 计划把通道平均掉，搬进来的主要是方向，幅度靠按行归一化补上；N 不是把 Nanbeige 的能力搬过来。CKA 预筛和“只有一个供体”都是对原方法的改动；判定、注入规则、阈值不变。供体与 B 的词表不同，逐篇均值让这一点不影响统计，但校准文本是通用英文，与 Nanbeige 擅长的 Agent / 编码任务不匹配，零结果有歧义（可能只是语料不对）。
- **许可证**：Nanbeige4.3-33B 是 Apache-2.0。ν>0 时它的仓库会进 `source_licenses.json`。

### 跨架构增量 N₂ 怎么算（e15，SenseNova-Loop-9B-Preview）

判定、阈值、注入规则、α=0.05、每张矩阵 128 行、CKA 预筛都与 N 相同；校准文本和 target 激活就是 N 用的那一份（同一个激活仓 commit，经 `qmerge_tm` 读取，本地已有的 target 激活副本直接复用）。不同之处：

1. **固定 commit 与远程代码审读**：仓库第一次读到时锁定 commit（`revision` 留空时）。这个模型需要 `trust_remote_code`：notebook 先把锁定 commit 的 `.py` 文件下载到 `DRIVE_DIR/tm_delta2/donor_metadata/…/`，列出每个文件的哈希，并把网络、子进程、文件删除、`eval`/`exec` 之类的行单独列出来，然后停下。读完代码确认没问题，把打印出来的 `code_sha256` 填进第 1 节再运行；代码或 commit 一变就对不上，会再停。
2. **供体前向怎么跑**：不手写。线性注意力和 mHC 双残差流手写容易错，所以用作者的远程代码跑完整前向，在每个投影矩阵（`nn.Linear`）上挂 hook 取实际的输入和输出，按篇取均值，这样不用决定取哪条残差流。前向在独立子进程里跑：transformers 5.18（作者的测试版本）用 `pip --target` 装在本地临时目录，只给子进程用；本内核的 transformers 版本是 S 基线等缓存身份的一部分，不变。权重按检查点精度放在内存里（BF16 主干约 18GB，mHC 参数 FP32；若装载器把 FP32 参数转成了 BF16，会从检查点按原值还原），每个解码层只在运行时临时转成 float32，算完转回，所以计算是 float32（与 Nanbeige 相同），内存约 26GB（含转成 float32 的 embedding 和输出头），51GB 的 Colab 放得下。供体权重先完整下载到本地盘（约 18GB，激活算完即删，`keep_checkpoint=False`）；每段 250 篇一个子进程，断开只丢这一段。
3. **前向闸门**（与 Nanbeige 的同一套 4 条短提示，至少答对 2 条）另加两项对照：作者原生精度的前向与 float32 前向在每条提示上的下一 token logits 余弦 ≥ 0.98 且 top-1 相同；实际观察到的解码层执行顺序、层数、hidden、FFN、头数、KV 头数、头维、词表和“3 层线性 + 1 层全注意力”与模型卡（`tm_extra2["expect"]`）一致。任何一项不符都报错停下，并说明差在哪里；不要凭猜测改，读完实际的 `config.json` 确认后才改 `expect`。
4. **共享权重的层**：56 次执行各算一个候选供体层（深度按它在 56 次里的位置算），参加 CKA 预筛、层对 OT 和层级 OT；注意力模块在每次全注意力执行都记录（14 次），MLP 每隔一次执行记录（28 次，与 Nanbeige 每隔一层同样的密度）。一次执行的权重就是它所属物理层的权重，所以同一份权重可以经两次执行搬到不同的目标层。
5. **搬哪些模块**：v_proj、o_proj、gate_proj、up_proj、down_proj 与 N 相同。k_proj 默认 `auto`：供体的头维（模型卡 4096/16 = 256）、旋转维（`partial_rotary_factor`）和 M-RoPE 的分段与 B 完全相同、并且激活仓里有 B 的 k_proj target 激活时才搬（rope_theta 不同只提示：通道配对不变，频率略有差别）；不满足时自动不搬并在结果里写明原因。q_proj 不搬（B 的 q_proj 混着门控通道）。线性注意力层的投影不搬：共用的激活仓里只有 B 全注意力层的注意力激活（以及 MLP），没有 B 线性注意力层的 target 激活可以对齐，它们的递推参数（A_log、dt_bias、conv1d）在 B 里也受保护。
6. **tokenizer**：比较供体与 B 的 `tokenizer.json`（词表、merges、added tokens、normalizer、pre-tokenizer）。一致时校准文本的切分与 B 完全相同；不一致时按供体自己的 tokenizer 切分，按篇取均值的统计不受影响（同 Nanbeige）。结果里记着是哪种。
7. **结果**：`DRIVE_DIR/tm_delta2/tm_result.record.json`（状态、判定表、k_proj 是否搬、tokenizer 是否相同）、`gate.record.json`（闸门与观察到的结构）、`verdict.record.json`、`pairs/`、`donor_acts/`（约 6–7GB）、`delta/<指纹>/tm_delta.safetensors` 和 `tm_report.json`（每张矩阵用了哪些执行、对应哪个物理层）。上传的仓库里有 `tm2_transport_report.json`（ν₂>0 时）。

- **局限**：与 N 相同（搬过来的是方向，不是能力；校准文本是通用英文）。另外：远程代码在 CPU 上跑线性注意力可能用的是较慢的参考实现；子进程里的 transformers 与本内核不同，只用于供体前向。
- **许可证**：模型卡写的是 Apache-2.0，以锁定 commit 里的 LICENSE 为准；ν₂>0 时它的仓库进 `source_licenses.json`。

### DELLA 组怎么算

两个 DELLA 组各自独立地这样算（随机数按成员名区分，互不影响），对每个二维语言权重逐行处理：

1. 每个成员的增量 Aₖ−B（成员填了 `reference_repo` 时是 Aₖ−Rₖ，例如 Darwin 对 Qwen3.5-27B）在行内按绝对值排名。名次最低的元素以 0.63 的概率保留，最高的以 0.77 保留，中间线性过渡，平均 0.7（`density=0.7, epsilon=0.07`）。保留下来的元素除以各自的保留概率，期望不变。
2. 各成员处理后的增量相加，和的符号就是这个位置的当选符号（和为 0 算正）。
3. 取符号与当选符号一致的非零元素的平均值，再乘 `lambda=1`。没有成员一致的位置为 0。

- 输出头不做丢弃和选举，取各成员增量的普通平均，与 A3 那次 DELLA 的 embedding 策略相同。
- DELLA 组 2 只有两个成员：两者符号一致的位置取平均；不一致的位置，丢弃并放大之后绝对值大的那个当选，取它自己的值（不和另一个平均）。
- 随机数由 `seed`、成员名、张量名和行块号决定，与分块方式和线程数无关，重跑逐位相同。参数取 A3 那次 DELLA 的数值；随机流是本 notebook 自己的，与 mergekit 的不逐位相同。
- `aux_delta_report.json` 的 `della` 一项按组记录每个成员的增量范数、保留比例、符号当选比例和合并结果的范数。

### GGUF 来源（Winery、DavidAU）：怎么用

- DavidAU：正式融合时先依次查 `bf16_repos`，有完整 BF16 safetensors 的仓库就改读它（对 B 取增量，没有量化噪声）；都没有就读 GGUF 仓库。`file` 留空时自动挑一个单文件（不读分卷）：BF16 > F16 > Q8_0 > Q6_K，同一类型普通文件优先于 MTP 文件；其他 K 系和 IQ 系量化读不了（文件里个别这类张量会被单独排除并记入报告）。选择和选中的文件记在 `DRIVE_DIR/source_choices.json` 与 D 的身份里，之后的会话不会变；要重新选就删掉 `source_choices.json` 里这一项（D 会重算）。
- GGUF 的层数多于参考模型（扣除 MTP 层之后）时直接报错：那是加层扩展过的模型，按层号对齐会错位。
- 只取二维语言权重：注意力、线性注意力、MLP、输出头。逐张量用 HTTP 范围读取 GGUF，按 Q8_0 / Q6_K 反量化，再按 llama.cpp 的转换规则还原成 Hugging Face 布局。
- 线性注意力的 V 头在新版 llama.cpp 里会被重排。新旧两种顺序都试，选与 B 差得少的那种。与 B 的相对差超过 0.5，或两种顺序分不出高下，这个张量就不用，并记入报告。映射错了，差值会和权重本身一样大，所以能可靠地拦下。
- Q8_0 取整噪声用每块的缩放因子估计，Q6_K 用每个 16 元子块的有效步长 d·scale 估计，方法相同（BF16 / F16 的 GGUF 没有这一项）。增量要乘 \(g_W=\max(0,1-\text{噪声能量}/\text{增量能量})\)，所以没改动的张量 g≈0，纯噪声不会被加进来。
- 输出头只用 token 字符串与 S 完全相同的行。

### 存储与续跑

- `DRIVE_DIR`（Google Drive）：输入锁（`subject_pin.json`、`input_engine/input_commits.json`、`source_subfolders.json`、`source_choices.json`）、GGUF 头部缓存、辅料增量 D（FP16，约 50GB）、增量 C（FP16，约 50GB，`searched_delta/`）、搜索记录、分量筛选的中间结果（`component_store/`）、上传回执。会话断开后重新从头运行，会从断点继续。
- **Drive 需要约 100GB 空间放 D 和 C，另加分量筛选约 13GB、N 约 12GB、N₂ 约 7GB**（供体激活约 11GB，`keep_activations=False` 时算完删；本地盘放不下 target 激活时它也放 Drive，约 15GB）（其中约 10GB 是反向用的隐状态，算完自动删）；补加 StartLux 和 DavidAU 时新旧两份 D 短暂并存，还要再多 50GB。e9/e10 算的旧 D（`aux_delta/` 下的旧目录）这一版用不上（主模型和比例都变了），运行时会打印位置，可以删除。之前版本存在 `DRIVE_DIR/selected_candidate/` 的候选（约 55GB）这一版不再使用，可以删除。Final 已发布，Drive 上的 `qwen38_final_stage`（约 140GB）可以删除。Drive 上删掉的文件（包括 notebook 自动删的旧 D）进了回收站仍占配额，记得清空。
- `SCRATCH_DIR`（本地盘）：S 的副本（54.7GB）、D 的副本（50GB）、C 的副本（约 50GB，放不下就从 Drive 读）；逐层评分时搜索不写候选文件，所以 B（54.7GB）一般也放得下，剩下的空间再用来存中间隐状态（`layer_memo`，每份几百 MB，旧的自动删）。选中的候选（54.7GB）在搜索结束后才构造，空间不够时先删掉中间隐状态和 B 的副本。用分批前向引擎时和原来一样，要留出一个候选的空间。每个新会话要重新下载 S、重新从 Drive 复制 D（算 D 的那个会话不用复制）。
- 从头算 D 要读一遍 Final、B、八个 DELLA 成员、Qwen3.5-27B 和 EfficientThink（各约 55GB）以及 Winery GGUF（约 29GB）；C 另读 Coder390 和 EfficientThink 各一遍。
- **补加来源**：Drive 上已有一份 D、它的来源是这次来源的一部分（其余设置与输入都相同）时，在它上面只补加缺的项。这一版里：等 StartLux 时算出不含 StartLux 和 DavidAU 的 D；正式融合时在它上面补加这两项（读 StartLux、Qwen3.6 和 DavidAU 各一遍）。补加完成后自动删除旧的 D（`aux_delta_drop_partial`）。补加得到的 D 与从头算的只差 FP16 的舍入，`aux_delta_identity.json` 的 `composed_from` 记着它是在哪份 D 上补加的。
- 每个搜索候选只读本地的 S、D、C（μ>0 时）和（κ≠0 时）B，不再每次下载全部来源。
- 选中的候选在本地构造、核对后直接上传，不经过 Drive（`upload=False` 时才存到 `DRIVE_DIR/selected_candidate/`）。上传中断后重新运行会沿用搜索记录；需要继续上传时只重建这一个候选。若上传回执丢失且目标仓库已有内容，只有完整发布清单及逐项哈希全部一致才认定为已上传；不完整或不同则拒绝覆盖。
- 用上一版（e2）算过 D 的话：那是 Final + Winery + Swift-1.5 的 D，这一版用不上。运行时会打印它的位置，可以删除。

### 修订 2026-10-31.e15

- **第二个跨架构增量 N₂（SenseNova-Loop-9B-Preview）**：新模块 `qmerge_tm2`、新配置块 `tm_extra2`、新缓存目录 `DRIVE_DIR/tm_delta2`；做法见“跨架构增量 N₂ 怎么算”。公式多一项 ν₂·N₂，ν₂ 在加了 N 的获胜配方上评（`search_result.json` 的 `nu2_line`），必须 > 0；分量筛选把 N₂ 当成 F − S 的一部分一起拆，不能把 N₂ 的分量全部设成 0。
- **已算好的不受影响**：`qmerge_core`、`qmerge_runtime`、`qmerge_components`、`qmerge_legacy`、`qmerge_gguf`、`qmerge_della`、`qmerge_tm`、`qmerge_evaldata`、`qmerge_fp8` 一字未改（元数据里的代码指纹与 e14 相同），所以 D、C、S 的基线、反向自检和 Nanbeige 的 N 都继续有效。`qmerge_pipeline` 为接上 ν₂ 改了；它的代码哈希只进入候选评分、逐层隐状态缓存、候选构造和选择记录的身份，这些在等待阶段都还没有算过。
- **等待阶段**：N 之后算 N₂；N₂ 出错（包括第一次的远程代码审读停顿和前向闸门不过）只打印原因，不影响前面已算好的。正式融合时 N₂ 在 N 之后、搜索之前；出错默认停下（`on_error="raise"`）。
- `BOOTSTRAP_VERSION` 升到 e15。
- **测试范围**：`1-model-merge/tests/test_tm2.py` 直接从本 notebook 取出内嵌模块的源码，用随机小张量测试新增的纯计算部分：执行顺序与模型卡的核对、每个模块的候选执行层、RoPE 兼容判定、ν 的选择规则、远程代码审读门槛、tokenizer 比较；子进程里的逐层 float32 计算（BF16 存储的小型循环深度模型与整份 float32 前向逐元素一致）、共享层每次执行的激活分开记录、提前停止、FP32 参数还原、等长分批；用合成激活跑通判定 → 层对 → 融合，并与按 `qmerge_tm` 原语手算的结果对照；pipeline 里 F 的 ν₂·N₂ 项、缓存身份和 ν₂ 线搜索。SenseNova 的真实权重、远程代码和 transformers 5.18 没有实测，靠审读、前向闸门和结构核对把关。

### 修订 2026-10-27.e14

- **每个供体都必须有贡献**（`require_all_donors=True`，见第 1 节表格）：搜索网格去掉了 λ=0，μ、ν 都从正值里选；获胜配方不能是 S；μ=0 消融不做；ν 线搜索取 `nu_grid` 里 NLL 最低的那个，不再和“不加 N”比；分量筛选里会让 D/C 或 N 的全部分量变成 0 的方案不考虑；N 判定全不过时强制构造。没有候选比 S 好时，`upload_when_no_gain=True`（默认）仍上传最好的那个，`False` 则不上传。

- **跨架构增量 N**：新模块 `qmerge_tm`（见“跨架构增量 N 怎么算”）。公式多一项 ν·N，ν 在获胜配方上评（`tm_extra["nu_grid"]`），不占 GP 预算；`search_result.json` 的 `nu_line` 记着评过的 ν、NLL 和选择。分量筛选把 N 当成 F − S 的一部分一起拆。
- **等待阶段**：D、C、S 的基线、反向自检之后再算 N；N 出错只打印原因，不影响前面这些。正式融合时 N 在 C 之后、搜索之前；出错默认停下（`on_error="raise"`），修好后重跑从断点继续。
- **缓存**：`qmerge_core` 和 `qmerge_runtime` 一字未改，所以已经算好的 S 基线、反向自检、D、C 都继续有效；`nu` 字段的校验写在 `qmerge_pipeline` 里。N 的指纹进入候选的存储身份，所以有 N 时全局搜索的候选评分会按新身份重算（等待阶段还没有评过候选，不浪费）。
- **测试范围**：N 的数值部分（代价矩阵、Sinkhorn、自适应 eps、锐度、层级 OT、CKA 预筛、只算 128 行的传输、注入规则）在合成数据上与按 TM notebook 公式写的参考实现逐项对照过，并在合成数据上跑通了判定 → 层对 → 融合的流程；Nanbeige 的流式前向、真实权重、真实 transformers 没有实测，靠前向闸门把关。

### 修订 2026-10-24.e13

- **搜索预算与 head 补评**：`global_trials` 仍只计 GP 主候选（默认最多 12 个），S 对照、head 补评和 μ=0 消融都不占这项预算。非 tied 且 `head_grid` 恰好为二值时，默认会在另一侧尚未评过时立即补评，并喂回 GP；其他网格不自动补评。当前 `[0, 0.05]` 就是两个端点，不自动生成 0.025，三值网格不自动补评。没有新增 head-only 候选自动切换为 S 的规则。
- **μ=0 配对消融**：全局搜索的初始 winner 若 μ>0，额外评一次其他参数完全相同的 μ=0 候选。改选要求无 C 候选满足相对 S 的逐类 NLL 回退门槛，且平均 NLL 严格更低；同分不在消融中改选。若无 C 候选就是 S，初始 winner 的排序本已让 S 在同分时优先，不会再进入含 C 的消融。
- **failed/skipped 选择会重试**：分量筛选以 `failed` / `skipped` 回退时，本次仍可用全局搜索 winner 构造和上传，但不把这个回退选择保存为可复用的 `selection_record.json`；下次重跑会重试。已经完成的搜索评分和更底层的分量断点仍按各自身份复用。
- **head 只做精确端点取舍**：只有主干分量在步长方案里使用一阶连续系数；输出头用同一最终隐状态上的精确端点差，只做 0/1 保留或移除，不放大，也不计入主干步长的归一化。
- **layer memo 身份修正**：`λ_backbone=0` 时，解码层 memo 去掉 κ、D/B 和主干分组系数等无关依赖；只有 C 改动时按 μ、C 指纹和行改动上限区分。memo 的共同身份还纳入 `qmerge_pipeline.py` 与 `qmerge_components.py` 的代码哈希，避免跨代码版本错误复用。
- **上传恢复与回执身份**：回执身份纳入分量选择结果，失败后重试成功不会复用旧回执。全部发布文件生成后重新读取远端状态，核对完整发布清单（Hub 自动管理的 `.gitattributes` 除外）及逐项哈希；只有清单和内容都一致才认定为已上传。缺项、内容不同或重试后的选择元数据不同都拒绝静默复用，需要换新的 `output_repo`。
- **crossfit 的含义**：它只是内部代理估计。定本折系数时，另一折不提供损失或梯度，折外再用真实 NLL 比较；但所有题共享教师强制激活协方差分解，所以两折不是完全独立，也不是未参与选型的最终验证。
- **不变项**：Q6_K 读取和 FP8 导出不变，不做 micro-batch。
- **测试范围**：本版只做了最小回归，不是在真实 27B 权重、真实 transformers 环境或 FP8 推理上的验证；这些路径仍未实测。

### 修订 2026-10-22.e12

- **Signal-3.8 留在 DELLA 组里，只作用于输出头**：它只替换了 `lm_head`（模型卡写明其余张量与 Qwen3.8-27B 逐字节相同）。主干上它的增量全为 0，DELLA 中 0 不参与符号投票和均值，结果与不放它逐位相同；输出头上按成员加权均值，它占 1/8（加入 Simplicio、Tess 之后）。
- **DELLA 组加入 Simplicio-27B（A₇）和 Tess-4.5-27B（A₈）**，成员之间权重仍相同，组系数仍是 0.8。两个都是 Qwen3.8-27B 的微调、Apache-2.0、不需要申请访问；commit 已锁定。Simplicio 是 Unsloth 的 LoRA 微调，读根目录合并好的完整权重，它的增量是低秩的，范数可能明显小于其他成员（范数提示里会看到）。
- **全局搜索**：`head_grid` 从 `[0, 0.025, 0.05]` 改为 `[0, 0.05]`。输出头在分量筛选里是单独一个分量，效果精确算出，系数在 [0, 1.1] 里连续可调，0.025 由它覆盖。网格从 103 个组合降到 67 个（都含 S），12 次全局试验的覆盖比例从约 12% 升到约 18%。`global_trials` 没改，注释里写了改成 16 的代价。
- **逐行改动报告覆盖 D**：原来只报告 μ·C 有多少行超过 10% 的行改动上限；现在 `row_change_report` 同时报告 λ·D（每个 λ）、μ·C（每个 μ）、λ·D + μ·C（每个组合）和输出头，κ=0。等待阶段在 S 下载好之后就做一次（D 还不含 StartLux 和 DavidAU），正式融合时按完整的 D 再做。λ·D 在最大的 λ 下超过 1% 的行时，会打印让约 1% 的行超过上限的 λ，供收窄 `lambda_grid` 参考。结果在 `row_change_report.json`，替代 `searched_delta_report.json`。`SETTINGS["row_change_report"]=False` 可关闭。
- **DELLA 成员范数提示**：D 算完后打印每个 DELLA 成员的主干增量范数；超过组内其他成员中位数 2 倍的会提示（主干为 0 的只作说明，如 Signal）。
- **DavidAU 改为读 Q6_K**：它没有发布完整权重（合集里没有 safetensors 仓库，`bf16_repos` 里猜的三个仓名都不存在），GGUF 仓库里只有 `Qwen3.8-27B-TBrilliance-4.41-Q6_K.gguf`（22.4GB），而且是需要作者人工批准的 gated 仓库。e11 的读取器不支持 Q6_K，正式融合会在这一步报错。现在读取器加了 Q6_K 反量化（按 llama.cpp 的 `block_q6_K` 格式），取整噪声按每个 16 元子块的步长 d·scale 估计，和 Q8_0 一样按张量扣除；Q6_K 噪声比 Q8_0 大，扣除后 DavidAU 实际进来的份额会小于 0.3，比例见 `aux_delta_report.json` 的 `gguf` 一项（`median_gain`、`noise_share_of_delta_energy`）。文件里个别不是 Q6_K / Q8_0 / BF16 / F16 的张量会被单独排除并记入报告。GGUF 读取器的版本号没变（F32 / F16 / BF16 / Q8_0 的读法没动），所以已经算好的 D（含 Winery）照常沿用。
- **FP8**：`mtp.layers.*` 的投影也匹配原来的正则，会被量化成 FP8，而 `mtp.fc` 和 norm 仍是 BF16。现在 MTP / NextN 层整体保留原精度，写进 `modules_to_not_convert`。FP8 模块版本号改了，所以旧版本留下的 FP8 分片（如果有）不会被沿用。
- 反馈第 1 条（μ 网格不含 0）没有改：除 S 外每个候选仍至少带 0.5 倍的 C。
- 已有结果：S 的基线、评分引擎的对照、反向自检都沿用（`qmerge_core`、`qmerge_runtime`、`qmerge_components` 没改）。C 沿用。**D 要从头重算**：DELLA 组加了两个成员，旧 D 里的 DELLA 组不能拆开补加（约 800GB 读取）。搜索还没跑，没有别的要重算。
- **这一版的改动没有在真实权重上运行过**。做了语法检查；逐行改动报告和 DELLA 范数提示用 NumPy / torch 替身跑过（与逐行直接计算对照），FP8 的排除规则用一组键名核对过；Q6_K 反量化与按 ggml `dequantize_row_q6_K` 逐元素写的参考实现在随机块上逐位一致。

### 修订 2026-10-18.e11

- **主模型换成 `bytkim/Qwen3.8-27B-pi`**（原来是 Coder390 EfficientThink）。commit 和读取的目录（根目录或 BF16 子目录）第一次运行时自动判断并锁定在 `subject_pin.json`；评测数据的长度检查改用 pi 的 tokenizer，所以评测数据会在新的指纹目录里重建。
- **新增单独搜索的增量 C = Coder390 − EfficientThink-Uncensored**，在 λ 括号之外：F = S + λ[D − κ(S−B)] + μ·C，μ ∈ {0.5, 0.75, 1.0} 和 λ、κ、λ_head 一起搜索，只加在主干矩阵上。C 单独缓存（`searched_delta/`），搜索候选在内存里按 μ 加上。全局搜索从 8 次加到 12 次。C 算好后会打印每行 ‖C‖/‖S‖ 和超过行改动上限的比例。
- **D 的比例**：Winery 0.1 → 0.05；新增 DavidAU Turbo-Brilliance-50X 0.3；DELLA 组 0.2 → 0.8，成员加入 Omnimerge-v6 和 Darwin-27B-RSI；EfficientThink-Uncensored 以 0.5 单独进 D（不在 DELLA 组里）；DELLA 组 2 不变。比例变为 10:4:0.5:3:8:5:1。
- **DELLA 成员可以有自己的底座**：成员填 `reference_repo` 时取 Aₖ−Rₖ。Darwin-27B-RSI 是 Qwen3.5-27B 一系的模型，对 Qwen3.5-27B 取增量。来源与它的底座的核心词表、merges、token ID 必须一致。
- **DavidAU 推迟到正式融合时决定**（`defer`）：那时 `bf16_repos` 里有已发布的 BF16 safetensors 就读它，否则读 GGUF（自动挑 BF16 > F16 > Q8_0 > Q6_K 的单文件）。等待阶段的 D 不含它，正式融合时和 StartLux 一起补加，所以等待期间算好的 D 不会因为 DavidAU 后来发了 BF16 而作废。选择记在 `source_choices.json`。
- **各来源的子目录**：每个 safetensors 来源（及其底座）都可以写 `subfolder`（默认 auto：按文件列表判断，记在 `source_subfolders.json`）。nerkyor 的两个仓库读 `BF16/`。
- 输入 commit 锁改成一个固定文件 `input_engine/input_commits.json`，补加来源（带子目录）时不会让已经锁定的仓库重新锁到当时的 main。
- GGUF：文件名可以留空自动挑；层数多于参考模型（扣除 MTP 层）时报错。
- `output_repo` 默认改为 `YFC-112358/Qwen3.8-27B-pi-AuxMix-v1`。
- 已有结果：e9/e10 的 D、S 的基线、评分引擎的对照和反向自检都不能沿用（主模型变了），全部重算；`inputs.json` 还没写过，可以继续用原来的 `DRIVE_DIR`。
- **这一版的改动没有在真实权重上运行过**：这个环境装不了 torch。做了语法检查；用桩模块和 NumPy 替身跑过：配置解析与比例、配方的 μ（规范化、基线判断、搜索网格 103 个组合）、子目录判断、GGUF 自动选文件和加层检查、DavidAU 的 BF16 / GGUF 选择与持久化、`setup_inputs` 的来源与底座接线（含重复仓库拒绝）、D 中带自己底座的 DELLA 成员与直接计算逐位一致、C（不含 M − B）逐位一致、候选张量 S + λ[D − κ(S−B)] + μC 与直接计算一致。

### 修订 2026-10-17.e10

- **辅料**：新增 DELLA 组 2：VeriLoop-E2（`tsinghua-sigs-robot-lab/VeriLoop-E2` @ `44b6f230…`）与 sabi-noema（`sabicap/sabi-noema` @ `c12b2070…`）做 DELLA，权重 1（与 Winery 相同），比例变为 10:4:1:2:1。DELLA 参数与第一组相同。两个都是 Qwen3.8-27B 的全量 BF16 微调，Apache-2.0，不需要同意条款。
- **D 的补加改为通用规则**：原来只有 StartLux（`late`）能在已有 D 上补加；现在任何“来源是这次来源的一部分”的已有 D 都可以作为底子，选来源最多的那份，只补缺的项。所以 e9 算好的 D 不用重算，只补 DELLA 组 2（约 165GB 读取），之后 StartLux 照样补加。改动只在 `qmerge_pipeline` 的 `existing_aux_delta` / `ensure_aux_delta`（新增 `stored_aux_deltas`）；`AUX_DELTA_VERSION` 没变，所以 e9 的 D 认得出来。
- 已有结果：S 的基线、评分引擎的对照、反向自检都沿用（它们的缓存不含 `qmerge_pipeline` 的代码）。搜索还没跑，没有别的要重算。
- **这一版的改动没有运行过**：这个环境装不了 torch，只做了语法检查，并用 NumPy 替身把补加 D 的选择逻辑（子集判断、选最大底子、精确命中）跑了一遍。

### 修订 2026-10-14.e9

融合公式、D、全局搜索和分量筛选的做法都没有变。改的是默认值、提前核对和诊断。

- **反向自检**：等 StartLux 时就在真实 S 上核对分量筛选要用的梯度（见上面“分量筛选怎么做”下的说明），不必等到全局搜索之后才发现 transformers 跑不了反向。新函数 `qmerge_components.backward_check`，编排在 `qmerge_pipeline.check_backward`；`SETTINGS["backward_check"]` 可关。
- **代理偏差诊断**：全局搜索后打印按 κ 分组的最低代理 NLL（`search_result.json` 的 `kappa_profile`）；分量筛选后打印 S 自己微调子空间之内的分量占增量能量和占有益效果的比例（`component_selection.json` 的 `subject_subspace`）。有迹象时，运行结果的 `proxy_bias_notes` 和模型卡里会写明。只提示，不改变选择。
- **默认值**：`adopt_probability` 0.5 → 0.8（从几个方案里挑交叉拟合最好的那个，赢家的概率本身偏乐观；0.5 在“比 F 好”已是硬条件时几乎不起作用）。`plans` 去掉 `step:2`，剩 `step:0.5`、`step:1`、`strict`：`max_boost=0.1` 时有益分量在 t ≥ 1 就基本顶到 1.1，几种步长的差别主要在对有害分量砍多狠，`step:2` 和 `strict` 都偏激进。约省 2–3 小时。想加回来写进 `component_options["plans"]`。
- 第 1 格开头打印本地盘和记录目录的剩余空间（只提醒，不拦截）。
- 第 5 节的结果格会提示反向自检没通过和代理偏差的提示；`uploaded` 的下一步提示改为要做开思考的生成测试。
- 修正：`qmerge_runtime` 和 `qmerge_components` 两个单元格的 cell id 重复，后者换了新 id。
- 已有结果：D、S 的基线和评分引擎的对照都沿用。分量筛选的梯度和全局搜索的记录如果已有，会因为代码变了而重算（e8 还没有跑到这一步的话没有影响）。
- **这一版的改动没有运行过**：这个环境装不了 torch / transformers，只做了语法、变量名检查，新增的纯计算诊断函数用 NumPy 测过；反向自检本身要第一次在 Colab 上运行才知道。

### 修订 2026-10-12.e8

分量筛选再升级三处，并加了 10% 的改动上限。融合公式、D 和全局搜索都没有变。

- **梯度自检**：前向做完、开始最长的反向之前，先在两道中等长度的题上把梯度算到最上面两段，挑这两段里效果最大的两块，把它们分别放大、缩小 10% 真跑一遍，看损失的实际变化（中心差分）和一阶预测是否吻合（相差 25% 以内）。不吻合就不再往下算，退回全局搜索的结果（`component_on_error="fallback"`），省下十几个小时。约半小时，`component_options["preflight"]` 可关。
- **连续系数**：每个分量一个系数 c（1 = 保持 F 里的样子，0 = 去掉）。分量的一阶效果正好是目标对 c 的导数，所以“步长 t”方案沿梯度走一步：效果先往 0 收缩 `shrink` 个 bootstrap 标准误（在噪声以内的分量不动），再按 c = 1 − t × 效果 / 最大的效果 定系数，限制在 [0, 1.1]。默认比较步长 0.5、1、2 和严格方案（只留大概率有增益的），仍用两折交叉拟合在没见过的题上挑；宽松、点估计方案仍可加进 `plans`。
- **按 S 自己的子空间再切一刀**：对 S 自己的微调 S − B 做 SVD，取它用得最多的 32 个输入方向（`subject_rank`）；每块再分成落在这些方向之内和之外两半。两半仍然和其他所有块正交，一阶效果分开估计，所以和 EfficientThink 自己的改动重叠的部分可以单独保护或去掉。分量约 192 个。要读 B（本地副本，或按范围读）。
- **改动上限 10%**，两层。其一，分量系数最多 1.1：最值得保留的分量也最多放大 10%（缩小和去掉不受限，那是往 S 靠）。其二，任何被改动的矩阵，每一行离 S 的改动不超过这一行范数的 10%（`max_relative_change`），超过的行按比例缩回；全局搜索的候选、各个变体和最后上传的模型都这样处理。按 λ ≤ 0.2 估计，绝大多数行的改动远小于这个界，它主要拦住个别改动特别大的行和放大过头的情况；manifest 里记着每个张量有多少行被缩回。
- 用时：梯度自检约半小时；方案从 3 个变 4 个；合成改了系数的权重稍慢（每个矩阵最多约 96 个秩一项）。合计比 e7 多约 3–7 小时。
- 已有结果：分量梯度要重算（拆法变了）；全局搜索的候选（如果算过）要重算（加了改动上限）。D 和 S 的基线沿用。
- **这一版的改动没有运行过，只做了语法和变量名检查。**

### 修订 2026-10-12.e7

分量筛选改为用**交叉拟合**选方案，并加了一种“点估计”方案。拆分、梯度、全局搜索和融合公式都没有变。

- 要解决的问题：e6 用同一批题既估计每个分量的好坏、又比较去掉分量后的变体，所以选中变体相对 F 的改善偏乐观，而且只能在两个变体里挑。
- 现在：96 道题按类别和来源（搜索集 / 代理验证集）均衡地分成两折。每种方案只用一折的统计挑出要去的分量，在另一折上用真实前向 NLL 评分，再反过来，于是每道题“比 F 变了多少”都来自没见过它的方案。取在没见过的题上改善最大的方案（改善的 bootstrap 概率不低于 `adopt_probability`，默认 0.5；各类别相对 S 的限制与搜索相同）。这个方案再用全部 96 道题的统计挑一次分量、在全部题上评分，也要比 F 好才采用；否则保留 F 的全部增量。
- 新增点估计方案：去掉一阶效果为正的分量，即一阶近似下的最优取舍，介于宽松和严格之间。三种方案由 `component_options["rules"]` 决定。
- 模型卡和 `component_selection.json` 多了两种公平的估计：每种方案在没见过的题上相对 F 的变化（95% 区间、变好的概率，并列一阶预测），以及在代理验证集上 F 和选中模型各自相对 S 的变化（全局搜索没有用过代理验证集，选中模型那一项用的是交叉拟合的结果）。
- 用时：去分量后的评分从 2 次全份（各 96 题）变为最多 6 次半份（各 48 题）加 1 次全份，约多 3–7 小时；相同的方案、去掉全部或一个都不去的方案（就是 S 或 F）不重复评。想省时间可以只留两种方案，或设 `"crossfit": False`（三种方案都在全部题上评，直接取最好的，结果偏乐观）。
- 已有结果：分量梯度和分解如果已经算过，直接沿用（`qmerge_components` 没有改动）；去掉分量后的评分会重做，因为它的缓存与编排代码绑定。
- **这一版的改动没有运行过，只做了语法和变量名检查。**

### 修订 2026-10-12.e6

用**分量筛选**替代原来的 NPS 式局部搜索（深度二分 × 行内幅度二分的四个分组系数）。融合公式、D 和全局搜索都没有变。做法见上面“分量筛选怎么做”。

- 主线是我们讨论的方法：全局搜索选出 F 后，把 F−S 拆成互相正交、加起来正好等于原增量的分量；对 F 做一次前向和逐题反向，得到每道题、每个分量的一阶估计 ⟨∇ℓᵢ(F), Δⱼ⟩；按类别分层 bootstrap 算出每个分量有益 / 有害的概率；去掉没有把握有益（严格）或有把握有害（宽松）的分量，两个变体与 F 一起用真实前向 NLL 比较，取最好的。
- 借鉴的现有方法（各取一部分，都不是复现）：用矩阵实际收到的输入白化后再做 SVD（ASVD、SVD-LLM；RegMean 用的输入二阶矩）；以奇异方向的段为保留 / 去掉的单位（Task Singular Vectors、AdaRank）；梯度 × 增量作为一阶效果（Sens-Merging、泰勒展开剪枝）；按数据决定留哪些部分、再用真实损失核对（Localize-and-Stitch）。
- 新模块 `qmerge_components`：增量的正交分解、逐层评分引擎上的前向 + 逐题反向（分段重算，跨会话续跑）、去掉分量后的权重。去掉分量只用固定顺序的逐元素运算，结果在任何 CPU 上逐位相同，所以选中的变体构造成文件后，照样逐张量核对它就是评分时的权重。
- 用到的题：搜索集 48 题 + 代理验证集 48 题（`component_options["rows"]`；写 `"search"` 只用搜索集，更快但估计更不稳）。代理验证集现在参与了选模型，不要再把它当独立测试；生成验证 96 题和终测 96 题仍然不用，可以留给你自己的测试。等 StartLux 时会顺便算好 S 在代理验证集上的基线。
- 用时：梯度这一步约为 96 题前向的 5 倍计算量（约 10–19 小时），两个变体各一次 96 题前向（共约 3–7 小时）。比原来 8 次局部搜索多 5–10 小时，见上面的阶段表。中间结果存在 `DRIVE_DIR/component_store/`：反向用的隐状态约 5–10GB，算完自动删；各矩阵的分解约 2GB，要留到上传之后。
- 出错时（例如这台机器的 transformers 不支持对逐层评分做反向），打印原因、跳过分量筛选，直接用全局搜索选中的候选（`component_on_error="fallback"`）；写 `"raise"` 则报错停下。内存不够、磁盘或 Drive 出错、权重与搜索时评分的不一致时总是停下，处理好后重跑会接着算。`component_selection=False` 则只做全局搜索。
- 模型卡和 `selection.json` 写明选了哪个变体、去掉了哪些分量；`component_selection.json`（Drive 上和上传的仓库里都有）列出每个分量的一阶估计、95% 区间、有益 / 有害概率，以及每个变体的一阶预测与真实 NLL。代理题同时用来挑模型，模型卡里的 NLL 差是偏乐观的。
- 设置：删去 `local_trials`、`group_values`；新增 `component_selection`、`component_on_error`、`component_options`、`proxy_validation_file`（第 4 节自动填）。
- 已有结果：Drive 上的 D 和 S 在搜索集上的基线都沿用（`qmerge_core`、`qmerge_runtime` 没有改动）。评分引擎的对照会重做一次（约 15–25 分钟；它的记录以后不再随编排代码变化）。之前算过的全局搜索候选（如果有）会重算，因为候选的缓存与编排代码绑定。`inputs.json` 不受影响，不用换 `DRIVE_DIR`。
- **这一版新增的梯度、分解和去分量的代码没有运行过，只做了语法和变量名检查。** 在真实的 transformers 上能不能对逐层评分做反向，要第一次运行才知道；跑不起来时按上面的规则自动跳过。

### 修订 2026-10-12.e5

去掉全部测试，搜索完直接上传。融合公式、D 和搜索方法都没有变。

- 删去冒烟测试（原第 4 节和 `qmerge_smoke` 模块）、代理验证，以及 GPU 上的生成验证和独立终测；相关设置一并删去（`proxy_validation_file`、`validation_file`、`final_test_file`、`generation_options`、`generation_limits`、`generation_policy`、Tailor、`STAGE`、`GPU_MAX_HOURS` 等）。整个流程只在 CPU 运行时上做，不再需要 GPU。原来的第 5–7 节改为第 4–6 节。
- 搜索结束后，构造搜索选中的候选，逐张量核对它就是搜索时评分的权重，然后直接上传到 `output_repo`（`upload=True`，默认私有）。模型卡写明配方、搜索代理的 NLL，以及“没有做生成评测”。`upload=False` 时只把候选存到 `DRIVE_DIR/selected_candidate/`。
- 搜索里没有候选的代理 NLL 比 S 低时，默认仍上传最好的那个候选（`upload_when_no_gain=True`），交给你自己的测试判断；设为 `False` 则不构造也不上传。
- 保留的两项检查都不是模型评测：第一次运行时在 S 上把逐层评分与分批前向对照，决定用哪个评分引擎（只做一次，约 15–25 分钟；`proxy_options["engine"]` 写 `"layerwise"` 或 `"batched"` 可跳过）；上传前核对候选权重。
- 上传中断后重新运行：搜索记录都在，只会重建这一个候选再上传；目标仓库里已经是这份候选时不会重复上传。目标仓库里有别的内容时拒绝上传，不覆盖。
- 已有结果：Drive 上的 D 沿用。S 在搜索集上的基线会重算一次（约 1–2 小时）。e3/e4 的代理验证、生成记录和 `selected_candidate/` 里的候选不再使用，可以删除。`inputs.json` 不受影响，不用换 `DRIVE_DIR`。
- **这一版的改动没有运行过，只做了语法和变量名检查。**

### 修订 2026-10-12.e4

只改速度和稳健性。融合公式、D、候选权重、搜索方法和验收门槛都没有变；D 和候选权重与 e3 逐位相同。

- **搜索改为逐层评分**（`proxy_options["engine"]`，默认 `"auto"`）。e3 在 CPU 上每个候选先写出 54.7GB 的候选文件，再分 7–8 批前向，每批都把整个模型从本地盘换入一遍。现在候选的权重在内存里逐层算出：一层算好，48 条题都过完这一层再换下一层。每个候选只读一遍 S、D、B，不写盘，也不用填充；下一层要读的文件在后台预读。
  - 第一次使用前，先在真实的 S 上用最短的两条题和最长的一条题，与原来的分批前向对照。NLL 差都不超过 2e-3 才启用，否则自动退回分批前向（逐层评分在这台机器的库上根本跑不起来时也一样）。结论记在 `scoring_engine_check.json`。S 的基线和全部候选用同一种引擎评分。
  - 入围候选最后照常写成文件，再逐张量核对：与评分时内存里的权重逐字节相同；没参与前向的视觉塔与 S 逐字节相同。不同就报错。
  - 只改输出头、或只改后半深度分组系数的候选，沿用之前候选存下的中间隐状态（`layer_memo`），只算有变化的层。
- **D 的各来源并行读取**（`parallel_reads`）：Final、B、四个 DELLA 成员和 Winery 各用一个读取线程，比计算提前读几段。HTTP 读取窗口从 32MB 改为 64MB，请求数减半。Hugging Face 返回 429（限速）时，按它要求的时间等待后继续。在模拟网络延迟的测试里，算 D 快约 3.5 倍；实际快多少取决于 Colab 的网络。
- **DELLA 计算重写**：排序键、随机数、符号选举都按 CPU 缓存分块，单线程约快 1.3–1.7 倍，并按行块用满全部 CPU 核（`della_threads="auto"`）。与 e3 逐位相同（7,772 组对照）。
- **行内幅度分组**（局部搜索的 NPS 式分组）用选择算法代替整行排序，结果逐位相同（含大量并列值的对照）。
- **少走 Drive**：D 写到 Drive 时，同时在本地盘留一份，同一会话的搜索不再从 Drive 读回；补加 StartLux 时，旧 D 先整份复制到本地再读；与 Drive 之间复制 D 或入围候选时，4 个文件并行（`copy_workers`）；每个 D 分片写完立即校验。
- **生成自动批大小**（`generation_options["batch_size"]="auto"`）：按显存排尽量少的批次。A100 80GB 上提示都不超过约 900 token 时是 2 批 × 48 条（原来 3 批 × 32 条）；只有少数长题时只把那一批排小。批次组成记在 Drive 上，S 和候选用同一组。解码中途按显存增长推算到最后会不会超出，会超出就提前拆小。显存分配器设为可扩展段（`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`），减少解码缓存增长留下的碎片。
- **已有结果怎么沿用**：Drive 上 e3 算好的 D 直接沿用（身份没变）。S 的两份前向基线会按新引擎重算一次（CPU 上约 2–4 小时，不到 1 单元）。e3 的搜索、选型和生成记录（如果有）不再沿用。`inputs.json` 不受影响，不用换 `DRIVE_DIR`。
- 测试范围：与 e3 相同的离线小型“Hub”（NumPy 代替 torch，玩具模型代替推理）。完整流程在两种设置下跑过（全部缩小；注意力头数和头维度与真实配置相同）：D、搜索中的候选、入围候选都与 e3 逐位相同，HTTP 请求数和字节数也相同；逐层评分与分批前向的 NLL 差在 1e-6 量级，选出的配方相同。冒烟测试新增 7 项检查（其中 1 项只在 GPU 上做），在 Colab 上会用真实的 torch / transformers / accelerate 检查这些新路径；逐层评分在本机的库上跑不起来时只打印说明、不算失败，第 6 节自动改用分批前向。**逐层评分依赖模型内部“每层调用一次、按顺序、隐状态在前”的调用方式，这只在玩具实现上验证过；在真实 transformers 上要靠第 4 节和第 6 节开头的对照确认。所有代码都没有在真实 27B 权重上运行过，上面的用时都是推算的。**

### 修订 2026-10-11.e3

- 辅料：原来单独的 Swift-1.5（权重 1）换成 Swift-1.5、ThinkingCap、AREX-2、Signal-3.8 四个模型的 DELLA 合并，权重 2，比例变为 10:4:1:2。新增模块 `qmerge_della`。
- 流程按运行时分成“准备”（CPU）和“生成验收”（GPU）两段，用保存在 Drive 上的入围候选衔接。`STAGE="all"` 仍可在一个运行时里做完。
- 修正：`apply_chat_template` 在 transformers 5 返回 `BatchEncoding`，它不是 `dict` 的子类，上一版会在第一次评分时报错。
- 推理：评分和生成改为固定分批；显存或内存装得下时权重常驻，不再每次前向都从磁盘换入。生成时显式传入 `use_cache=True`，并在开始前用 3 个解码步确认解码缓存确实生效（这个模型的配置里写的是 `use_cache=false`）。停止符除了 tokenizer 的 EOS，还包括 `generation_config.json` 列出的。生成的分批会考虑提示长度（读入提示时注意力矩阵随长度平方增长）；一批中途显存不足时自动减半重做。
- 默认值变化：`max_new_tokens` 8192 → 4096；评分每批 8 条、生成每批 32 条；CPU 上用 float32 评分。
- S 的基线只按 S 自身缓存，换辅料比例或加入 StartLux 后不用重算；等待 StartLux 时就先算好。
- 续跑更省：每个权重文件每个会话只算一次哈希；D 复制到本地盘再用；已完成的生成报告下次直接读回，不用重新下载模型。
- 搜索代理模型的输入按各自的搜索范围归一化。补加 StartLux 后的报告保留之前各来源的统计。
- S 与 B / Final 的 tokenizer 写法不同时，报错信息会说明差别在哪，并可用 `allow_tokenizer_pipeline_differences` 放行（词表和 token ID 仍然必须一致）。
- 依赖：版本记录按 CPU / GPU 运行时分开（Colab 的 CPU 镜像是 `torch +cpu`，GPU 镜像是 `+cu…`，上一版的单一版本锁会把 CPU 版 torch 装到 GPU 运行时上）；带本地标签的 torch 从 PyTorch 自己的索引安装；没有记录时 transformers 至少 5.8。D 的身份不再包含 torch 版本。
- 准备阶段不会在 GPU 运行时上开始（`allow_prepare_on_gpu`）；机器有 GPU 而 torch 用不了时也不开始。
- 新增第 4 节冒烟测试。
- 验收门槛没有改。96 题下它们很难通过，生成验收开始时会按实际题数打印具体数字。
- 测试范围：不依赖真实推理库的全部代码，在一个离线的小型“Hub”上跑过完整流程（NumPy 代替 torch，玩具模型代替推理）；D（含 DELLA、GGUF、补加 StartLux）与直接计算逐位一致。**权重装载、批量前向、批量生成这些依赖真实 torch / transformers / accelerate 的部分没有运行过，要靠 Colab 上的冒烟测试检查；全部代码都没有在真实 27B 权重上运行过。**

### 修订 2026-10-11.e2

- 四份评测数据改为从 6 个公开评测集按固定 commit 自动生成（现在的第 5 节，`AUTO_EVAL_DATA`）。数据构建用合成样例测过格式、覆盖、互斥和评分往返；没有在真实下载的数据上运行过。

### 修订 2026-10-10.e1

- 删去 A3 / Final 构造与选型代码（已完成）。M 固定为已发布 Final 的 commit `164e6c38…`，S 固定为 `23a3f86a…`，B 从 Final 的构建记录读取。
- 辅料改为固定比例 10:4:1:1，不再搜索 StartLux 的系数 ρ（删去 `late_grid`）。搜索维度剩 λ、κ、输出头系数和四个分组系数。
- 新增 Winery（GGUF）与 Swift-1.5 两个辅料；DavidAU 已通过 A3 包含在 Final 里。
- 辅料增量 D 一次算好并缓存；记录放 Drive，大文件放本地盘。
- 测试范围：GGUF 读取在 llama.cpp 自带 gguf-py 写出的文件上，与它的反量化结果逐位一致；线性注意力还原用 llama.cpp 转换脚本里的重排代码原样对照。D 和候选公式用小张量与直接计算对照。**都没有在真实 27B 权重上运行过。**

In [ ]:
import os, sys, re, math, json, shutil
from pathlib import Path

# ==================== 路径 ====================
PERSIST_TO_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/qwen38_efficientthink"   # 记录、输入锁、辅料增量 D 与 C（各约 50GB）、搜索记录（目录名沿用，主模型已换成 pi）
SCRATCH_DIR = "/content/qwen38_efficientthink_scratch"        # S / D / C / B 的本地副本与候选模型；会话结束即丢
INSTALL_DEPENDENCIES = True
CPU_THREADS = os.cpu_count() or 8

AUTO_EVAL_DATA = True       # 搜索数据留空时，由第 4 节从公开评测集自动生成
EVAL_SIZES = None           # None：默认每类题数。要改就按这个顺序写全四项（用 search_file 和 proxy_validation_file；另外两项决定同时建出的其余数据）：
                            # {"search_file": 8, "proxy_validation_file": 8, "validation_file": 16, "final_test_file": 16}
                            # 改小 search_file 时要同时调低 SETTINGS 里的 search_min_items / search_min_per_category，否则覆盖检查会拦下

# ==================== StartLux ====================
STARTLUX_REPO = ""          # StartLux V1.0-27B-Preview 的正式仓名，发布后填这里
STARTLUX_REQUIRED = True    # True：未发布就等（先把其余辅料的 D 和 S 的基线算好）；False：不等，成品不含 StartLux

if PERSIST_TO_DRIVE:
    if not Path("/content/drive/MyDrive").exists():
        try:
            from google.colab import drive as _colab_drive
        except ImportError as error:
            raise RuntimeError("PERSIST_TO_DRIVE 需要在 Colab 中运行；其他环境请设 PERSIST_TO_DRIVE=False") from error
        _colab_drive.mount("/content/drive")
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
    if Path("/content/drive/MyDrive/qwen38_final_stage").exists():
        print("提示：Final 已发布，Drive 上的 qwen38_final_stage（约 140GB）不再需要，可以删除以腾出空间。")
EFFICIENT_WORK = Path(DRIVE_DIR if PERSIST_TO_DRIVE else SCRATCH_DIR + "/records").expanduser().resolve()
SCRATCH = Path(SCRATCH_DIR).expanduser().resolve()
EFFICIENT_WORK.mkdir(parents=True, exist_ok=True)
SCRATCH.mkdir(parents=True, exist_ok=True)
# 只是提醒，不拦截。本地盘：S、D、C 的本地副本约 160 GiB。记录目录（Drive）：新 D 约 50 GiB、C 约 50 GiB、分量筛选约 13 GiB、N 的供体激活约 11 GiB；
# e9/e10 算的旧 D（aux_delta/ 下的旧目录）这一版用不上，可以删除；Drive 上删掉的文件进回收站后仍占配额。
for _label, _path, _need in (("本地盘", SCRATCH, 170), ("记录目录", EFFICIENT_WORK, 135)):
    _free = shutil.disk_usage(_path).free / 1024**3
    print(f"{_label}剩余 {_free:.0f} GiB（{_path}）" + (f"；低于建议的 {_need} GiB，见第 1 节“存储与续跑”" if _free < _need else ""))
if EFFICIENT_WORK == SCRATCH:
    raise ValueError("记录目录和本地临时目录不能相同")

SETTINGS = {
    "run": True,
    "allow_prepare_on_gpu": False,  # True：允许在 GPU 运行时上运行（全是 CPU 和磁盘的活，按 GPU 的价格不划算）
    "work": str(EFFICIENT_WORK),
    "scratch": str(SCRATCH),
    "S_repo": "bytkim/Qwen3.8-27B-pi",
    "S_subfolder": "auto",     # auto：根目录有 config.json + safetensors 索引就读根目录，否则读唯一的 BF16 子目录；也可写 "" 或子目录名
    "S_revision": "",          # 留空：第一次运行时锁定当时的 commit（记在 DRIVE_DIR/subject_pin.json），之后都用它
    "M_repo": "YFC-112358/Qwen3.8-27B-TA-Aux-Final-v1",
    "M_revision": "164e6c38b67230b9be85768941806bc57ac768a4",   # 已发布并通过审计的 Final
    "B_repo": "Qwen/Qwen3.8-27B",
    "B_revision": "",          # 留空：从 Final 的构建记录读取制作它时用的底座 commit
    # 辅料比例 Final : StartLux : Winery : DavidAU : DELLA 组 : EfficientThink : DELLA 组 2 = 10 : 4 : 0.5 : 3 : 8 : 5 : 1
    # （系数 = weight / donor_weight，即 1 : 0.4 : 0.05 : 0.3 : 0.8 : 0.5 : 0.1）
    "donor_weight": 10,
    "aux_extras": [
        # revision 留空：第一次读到时固定为当时的 commit（记在 DRIVE_DIR/input_engine/input_commits.json），之后各个会话都用它。
        # subfolder：auto（按文件列表判断，结果记在 DRIVE_DIR/source_subfolders.json）、""（根目录）或子目录名。
        {"tag": "StartLux", "repo": STARTLUX_REPO, "revision": "", "format": "safetensors", "subfolder": "auto",
         "reference_repo": "Qwen/Qwen3.6-27B", "reference_revision": "", "reference_subfolder": "",   # StartLux 的训练底座，取增量 X-R
         "weight": 4, "required": STARTLUX_REQUIRED, "late": True},
        {"tag": "Winery", "repo": "WineryLabs/Winery-Qwen3.8-27B-GrandCru-GGUF",
         "revision": "9a76195110b07ffb678dd7f7f66afa884db46a89", "format": "gguf",
         "file": "Winery-Qwen3.8-27B-GrandCru-Q8_0.gguf", "weight": 0.5},
        # DavidAU：正式融合时（StartLux 发布后的那次运行）才决定读哪个、并补加进 D（defer）。
        # 那时 bf16_repos 里有哪个仓库已发布完整的 BF16 safetensors，就改读它；都没有就读下面的 GGUF 仓库，
        # 自动挑单文件的 BF16 > F16 > Q8_0 > Q6_K（普通文件优先于 MTP 文件；目前仓库里只有 Q6_K）。选择记在 DRIVE_DIR/source_choices.json，之后不再变。
        # bf16_repos 里的仓名是按 DavidAU 一贯的命名猜的；正式发布的名字不同就改这里（或加进去）。
        {"tag": "DavidAU", "repo": "DavidAU/Qwen3.8-27B-Turbo-Brilliance-50X-Reasoning-Instruct-NEO-MAX-MTP-GGUF",
         "revision": "", "format": "gguf", "file": "", "weight": 3, "defer": True,
         "bf16_repos": ["DavidAU/Qwen3.8-27B-Turbo-Brilliance-50X-Reasoning-Instruct",
                        "DavidAU/Qwen3.8-27B-Turbo-Brilliance-50X-Reasoning-Instruct-MTP",
                        "DavidAU/Qwen3.8-27B-Turbo-Brilliance-50X-Reasoning-Instruct-NEO-MAX-MTP"]},
        # DELLA 组：八个成员各自取增量，按幅度名次随机丢弃并放大、符号选举、取一致者的均值，结果作为一个辅料，系数 0.8。
        # Signal-3.8 只替换了 lm_head（主干与 B 逐字节相同）：主干增量全为 0，不参与符号投票和均值，只在输出头的加权均值里占一份。
        # Darwin-27B-RSI 是 Qwen3.5-27B 一系的模型，它的增量对 Qwen3.5-27B 取（reference_repo），其余成员对 B。
        {"tag": "DellaMix", "format": "della", "weight": 8,
         "della": {"density": 0.7, "epsilon": 0.07, "lambda": 1.0, "seed": 42, "normalize": True},
         "members": [
             {"tag": "Swift15", "repo": "ukisai/Swift-1.5-Qwen3.8-27b", "subfolder": "",
              "revision": "bc7a1e10b689648585a3ef41494c8d84cf77271a", "weight": 1},
             {"tag": "ThinkingCap", "repo": "bottlecapai/ThinkingCap-Qwen3.8-27B", "subfolder": "",   # gated：先在模型页同意条款
              "revision": "7891e03dbda2e9b63a5b9610f20e83ea441f8e2c", "weight": 1},
             {"tag": "AREX2", "repo": "BAAI/AREX-2", "subfolder": "",
              "revision": "d4e3502f92d9e889031c2d04e387ab2eb3520268", "weight": 1},
             {"tag": "Signal", "repo": "agentionai/Signal-3.8-27B", "subfolder": "",
              "revision": "54552f73a5781d7909d9ae660926ddf409bd8cc0", "weight": 1},
             {"tag": "Omnimerge6", "repo": "ManniX-ITA/Qwen3.8-27B-Omnimerge-v6", "revision": "", "weight": 1},
             {"tag": "DarwinRSI", "repo": "FINAL-Bench/Darwin-27B-RSI", "revision": "", "weight": 1,
              "reference_repo": "Qwen/Qwen3.5-27B", "reference_revision": "", "reference_subfolder": ""},
             # Simplicio-27B：Unsloth 的 LoRA 微调，根目录是合并好的完整 BF16 权重（不读 adapter_model.safetensors）
             {"tag": "Simplicio", "repo": "wesleysimplicio/Simplicio-27B", "subfolder": "",
              "revision": "19330de3c8889de34dcdc73c1c8b6343a11c232c", "weight": 1},
             # Tess-4.5-27B：Qwen3.8-27B 的全量微调
             {"tag": "Tess45", "repo": "migtissera/Tess-4.5-27B", "subfolder": "",
              "revision": "91b231133470bd147aac8036ed94d35e9bf1dc13", "weight": 1},
         ]},
        # EfficientThink-Uncensored（Coder390 的基座）：对 B 取增量，系数 0.5。它同时是下面 C 的减数。
        {"tag": "EfficientThink", "repo": "nerkyor/Qwen3.8-27B-EfficientThink-Uncensored-K3-Opus5-Grok4.6-GPT5.6Sol-SFT-SimPO-DFlash2",
         "revision": "", "format": "safetensors", "subfolder": "auto", "weight": 5},
        # DELLA 组 2：VeriLoop-E2 与 sabi-noema，参数与上一组相同，系数 0.1。
        {"tag": "DellaMix2", "format": "della", "weight": 1,
         "della": {"density": 0.7, "epsilon": 0.07, "lambda": 1.0, "seed": 42, "normalize": True},
         "members": [
             {"tag": "VeriLoopE2", "repo": "tsinghua-sigs-robot-lab/VeriLoop-E2", "subfolder": "",
              "revision": "44b6f230347f9ccf2a5ce96a44bcd74ecaaaa51f", "weight": 1},   # MTP 前缀下的整数数据张量不读取
             {"tag": "SabiNoema", "repo": "sabicap/sabi-noema", "subfolder": "",
              "revision": "c12b20707bb78f811c80bc46561117528ddbd221", "weight": 1},
         ]},
    ],
    # 单独搜索的增量 C（在 λ 括号之外）：F = S + λ[D − κ(S−B)] + μ·C，μ 在 mu_grid 里搜索。只加在主干矩阵上（roles），输出头不加。
    "searched_extras": [
        {"tag": "Coder390", "repo": "nerkyor/Qwen3.8-27B-Coder390-EfficientThink-Opus5.5-GPT6Astra-Grok4.7-DSV4Pro-K3-SFT-RLOO-MTP-DFlash2",
         "revision": "23a3f86a0b41c44b2a54ab99e73169240021ce33", "subfolder": "BF16",
         "reference_repo": "nerkyor/Qwen3.8-27B-EfficientThink-Uncensored-K3-Opus5-Grok4.6-GPT5.6Sol-SFT-SimPO-DFlash2",
         "reference_revision": "", "reference_subfolder": "auto", "roles": ["backbone"]},
    ],
    "mu_grid": [0.25, 0.5, 0.75, 1.0],   # require_all_donors 时不能有 0
    "searched_delta_dir": "",       # 留空：DRIVE_DIR/searched_delta（C 约 50GB）
    # ==================== 跨架构增量 N（Nanbeige4.3-33B，Transport-and-Merge，第 5 节里的一步）====================
    # F = S + λ[D − κ(S−B)] + μC + νN。N 由之前 TM notebook 的方法从另一种架构的模型搬过来：两边在同一份校准文本上的逐篇均值激活
    # → 相关代价 + 熵正则 OT（每对层一个通道计划，层与层之间再一个 OT）→ 在 B 每张被改矩阵激活最大的 topk 行上放入
    # α·(Q_out W_供体 Q_in^T 去均值、按 B 的行范数缩放 − B 的行)。target（Qwen3.8-27B）的激活直接读 TM notebook 存在 act_repo 里的那份；
    # 供体的前向在 CPU 上逐层做（float32，按文档分段，断开后从断点继续）。先做 go/no-go 判定（z 倍数、同 eps 噪声锐度倍数、
    # 自配对上界，和 TM notebook 同一套），只有过关的模块才算层对 OT 并融合；一个都不过就不含 N，流程照常往下走。
    # ν 不进 GP：全局搜索选出获胜配方后，在它上面依次评 nu_grid 里的 ν，代理 NLL 更低才采用。
    "tm_extra": {
        "enabled": True,
        "repo": "Nanbeige/Nanbeige4.3-33B",
        "revision": "",                 # 留空：第一次读到时锁定当时的 commit（记在 DRIVE_DIR/input_engine/input_commits.json）
        "act_repo": "YFC-112358/tm-acts-qwen38-multi",   # 之前 TM notebook 的激活仓（dataset，私有）：校准文本 calib_2000.json 和 target 激活从这里读
        "modules": ["self_attn.v_proj", "self_attn.o_proj", "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"],
                                        # k_proj 默认不搬：供体 head_dim 128 对 B 的 256，RoPE 通道配对对不上（TM notebook 的 ROPE_NOTE）；q_proj 混着门控通道，不搬
        "alpha": 0.05, "topk": 128, "inject": "row",   # 与 TM notebook v5 相同：每张矩阵只改 128 行，注入项去均值后按 B 的行范数缩放
        "pair_keep": 2,                 # 每个 target 层按线性 CKA 留几个供体层算层对 OT（另外总加上深度对齐的那一层）；越大越慢
        "nu_grid": [0.25, 0.5, 1.0],    # N 的系数 ν 在获胜配方上评这几个值（require_all_donors 时必选其一）
        "docs": 2000,                   # 校准文本篇数（target 激活是 2000 篇算的；改小只是更快、噪声更大）
        "chunk_docs": 500,              # 供体前向按这么多篇一段：每段把权重读一遍（约 62GB 下行），段内断开只丢这一段
        "batch_docs": 16, "checkpoint_every": 4,
        "while_waiting": True,          # 等 StartLux 的时候就把 N 算好（排在 D、C、S 基线和反向自检之后）
        "on_error": "raise",            # 正式融合时 N 出错："raise" 停下（修好后重跑会从断点继续）；"skip" 打印原因，本次不含 N（require_all_donors 时无效，总是停下）
        "keep_activations": True,       # 算完保留供体激活（Drive 上约 11GB）；改 alpha / topk 等只需重做 OT 与融合
        "dir": "",                      # 留空：DRIVE_DIR/tm_delta
    },
    # 第二个跨架构增量 N₂（e15）：SenseNova-Loop-9B-Preview，做法与 N 相同（同一个激活仓、同一份校准文本、同一套判定和注入），
    # 自己的配置、固定 commit、前向闸门、缓存目录（DRIVE_DIR/tm_delta2）和断点续跑。ν₂ 在加了 N 的获胜配方上评，必须 > 0。
    # 供体前向用作者的远程代码（trust_remote_code），在独立子进程里、用单独安装的 transformers 5.18 跑，本内核的 transformers 不变。
    "tm_extra2": {
        "enabled": True,
        "repo": "sensenova/SenseNova-Loop-9B-Preview",
        "revision": "",                 # 留空：第一次读到时锁定当时的 commit（与其他来源记在同一个 input_commits.json）
        "code_sha256": "",              # 远程代码包的 SHA-256：第一次运行会打印代码位置和这个值并停下；读完代码确认没问题再填进来
        "transformers": "5.18.*",       # 供体前向子进程用的 transformers（作者的测试环境）；装在本地临时目录，不动本内核的版本
        "modules": ["self_attn.v_proj", "self_attn.o_proj", "mlp.gate_proj", "mlp.up_proj", "mlp.down_proj"],
        "k_proj": "auto",               # 头维 / 旋转维 / M-RoPE 布局与 B 相同、激活仓里也有 B 的 k_proj 时才搬；True 强制（不符就报错），False 不搬
        "module_map": {},               # 供体层里投影的名字与上面不同时在这里对应，例如 {"mlp.gate_proj": "mlp.w1"}
        "layers_modules": [],           # 远程代码把解码层分成几个列表时（例如循环前 / 循环块 / 循环后），按执行前的顺序列出它们的路径
        "mlp_stride": 2,                # 56 次执行里每隔几次记录一次 MLP（与 Nanbeige 每隔一层相同的密度）；注意力在每次全注意力执行都记录
        "nu_grid": [0.25, 0.5, 1.0],    # ν₂ 在“获胜配方 + 选中的 ν·N”上依次评这几个值，必选其一（require_all_donors）
        "chunk_docs": 250, "batch_docs": 16,   # 每段一次子进程（重新装载一次权重，约 1–2 分钟）；段内断开只丢这一段
        "while_waiting": True,          # 等 StartLux 时就算（排在 N 之后）
        "on_error": "raise",
        "keep_activations": True,       # 供体激活约 6–7GB（Drive）
        "dir": "",                      # 留空：DRIVE_DIR/tm_delta2
        # "expect": {...}               # 模型卡上的结构（32 层、[4, 28) 执行两遍共 56 次、hidden 4096、FFN 12288、16/4 头、词表 248320、3:1 混合注意力）。
                                        # 前向闸门按它核对实际的执行顺序和 config，不符就停下；读完实际 config.json 确认无误后才在这里改。
    },
    "della_threads": "auto",        # DELLA 按行块分给多个线程，"auto" 用全部 CPU 核；结果与线程数无关（逐位相同）
    "parallel_reads": True,         # 算 D 时各来源同时读取，并比计算提前读几段；结果相同。排查网络问题时可改 False
    "readahead_mib": 64, "readahead_cache_mib": 2048,   # 每次 HTTP 范围读取的窗口（窗口大，请求就少，不易触发 Hub 限速）与预读缓存上限
    # GGUF：与 B 的相对差上限、两种存储顺序须拉开的倍数、是否扣除 Q8_0 / Q6_K 取整噪声
    "gguf_policy": {"max_rel": 0.5, "min_margin": 2.0, "noise_shrink": True},
    "allow_tokenizer_pipeline_differences": False,   # S 与 B / Final 的 tokenizer 写法不同会报错并说明；确认无妨后改 True
    "aux_delta_dir": "",            # 留空：DRIVE_DIR/aux_delta
    "aux_delta_drop_partial": True, # 在旧 D 上补加来源（StartLux、DavidAU）后删除旧 D
    "aux_delta_local_copy": "auto", # D 在 Drive 上时，算 D 时就在本地盘留一份，搜索直接用；新会话里缺的再从 Drive 复制
    "copy_workers": 4,              # 与 Drive 之间复制 D 或候选时，同时复制的文件数
    "prepare_while_waiting": True,  # 等待 StartLux 或数据时，先算已就绪来源的 D
    "baseline_while_waiting": True, # 等待时顺便下载 S、做架构预检并算好 S 的前向基线
    "backward_check": True,         # 等待时（S 已在本地盘上）先在真实 S 上核对分量筛选要用的反向：一道短题，对最后的一个线性注意力层和一个全注意力层
                                    # 求梯度，再用真实前向的中心差分对照（约 15–40 分钟，只做一次）。不通过时，之后的分量筛选直接跳过，省下十几个小时
    "verify_cache_hashes": False,   # True：每次续跑都重新校验 D 的 SHA-256（约 50GB 读取）
    "input_cache_mode": "auto",     # auto：本地盘依次放 S、D、C、B，放不下的按范围读取 / 从记录目录读取；也可 stream / local
    "search_file": "",         # proxy JSONL（参考回答格式）：系数搜索用
    "proxy_validation_file": "",  # 同样格式：分量筛选另外用的题；留空时由第 4 节自动填上
    "block_mib": 16, "rows_mib": 64, "shard_gib": 1.0,
    "tensor_key_overrides": {},
    "install_inference_dependencies": True,
    "keep_search_candidates": False,
    "max_relative_change": 0.1,
    "row_change_report": True,       # 算好 D、C 且 S 在本地盘上之后，报告 λ·D、μ·C 和二者之和有多少行超过上面的改动上限（等待阶段就做；读一遍 S、D、C）     # 改动上限：任何被改动的矩阵，每一行离 S 最多改这一行范数的 10%（全局候选、变体和最终模型都按此缩回）；None 不限
    # 以下搜索范围是工程起点，不是已证明最优的数值。每个主干候选在 CPU 上大约 1–2 小时（逐层算出权重并前向评分 48 条；推算值）。
    # 每个供体都必须有贡献（系数可以很小，但不能为 0）：λ（D：Final 与各辅料）、μ（C）、ν（N）在每个可选候选里都 > 0；
    # S 只作对照、不能当选；μ=0 消融关闭；分量筛选不能去掉全部主干分量；N 的判定一个模块都不过时，仍用通过上界标定的模块构造。
    "require_all_donors": True,
    "lambda_grid": [0.025, 0.05, 0.10, 0.20],
    "kappa_grid": [0.0, 0.5, 1.0],
    "head_grid": [0.0, 0.05],   # 二值网格可在同一主干上就地补评。分量筛选中头部只做 0/1 取舍；0.025 不再由连续系数覆盖，需要时可自行加回（三值网格不做自动补评）
    "global_trials": 12,        # GP 主候选数，不含 S、head 补评和 μ=0 消融；网格 67 个组合（含 S）。想更密可改 16，主干约多 4–8 小时
    "paired_head_trials": True, # 二值 head_grid：每评一个主候选，立即补评另一个 head 值并喂回 GP；复用最后一层 memo，不占 global_trials
    "mu_ablation": True,        # 初始胜者含 C 时，额外评一次其他参数完全相同的 μ=0 候选；可行且 NLL 严格更低才改选，同分不改选（S 在初始 winner 排序中同分优先）
    "surrogate_exploration": 0.5, "seed": 42,
    "search_min_items": 48, "search_min_per_category": 8,
    "proxy_max_category_drop": 0.05,   # 任何一类的代理 NLL 比 S 高出超过这个数的配方（或分量筛选的变体）不算数
    # ==================== 分量筛选（替代原来的 NPS 式局部搜索；做法见第 1 节说明）====================
    "component_selection": True,       # 全局搜索之后：把 F−S 拆成正交分量，按逐题一阶梯度给每个分量定系数，再用真实 NLL 核对
    "component_on_error": "fallback",  # 分量筛选出错时："fallback" 打印原因并用全局搜索选中的候选；"raise" 报错停下。failed/skipped 回退不冻结选择，下次会重试
    "component_options": {
        "rows": "search+proxy_validation",  # 估计梯度、比较变体用的题：搜索集 + 代理验证集（各 48）；"search" 只用搜索集，更快但更不稳
        "blocks": 8,                  # 深度方向分几段
        "bands": [4, 16, 64],         # 每个矩阵的增量按白化后的奇异方向切段：1–4、5–16、17–64，其余为第四块
        "subject_rank": 32,           # 每块再按 S 自己的微调 S−B 用得最多的 32 个输入方向切成“之内 / 之外”两半；0 不切
        "whiten": True, "input_tokens": 8192,  # 用矩阵实际收到的输入白化；每层最多抽多少个 token
        "plans": ["step:0.5", "step:1", "strict"],  # 步长方案只对主干做连续系数，头部按精确端点差单独 0/1 取舍；严格只留有益概率 ≥ confidence 的分量
        "shrink": 1.0,                # 走一步之前，每个主干分量的效果先往 0 收缩 1 个 bootstrap 标准误（在噪声以内的分量不动）
        "max_boost": 0.1,             # 主干分量系数上限 1.1；头部不放大。缩小和去掉不受限
        "confidence": 0.9,            # 严格 / 宽松方案用的把握
        "crossfit": True,             # 两折交叉拟合：系数只用本折梯度统计，在另一折评分；全量题共享一次白化分解，因此不是完全独立的折外验证
        "adopt_probability": 0.8,     # 交叉拟合下采用一个方案所需的“比 F 好”的 bootstrap 概率。程序在几个方案里挑最好的，这个概率本身偏乐观，所以默认 0.8；0.5 几乎不设门槛
        "preflight": True,            # 长时间反向之前，先在两道题上用真实损失核对梯度（约半小时）；不吻合就退回全局搜索的结果
        "bootstrap": 2000, "seed": 0,
        "segment": "auto",            # 反向时每段几层（按内存自动定，最多 8）
    },
    # device / dtype 写 "auto"：有 GPU 用 GPU + bfloat16，否则 CPU + float32
    #（bfloat16 在没有 AVX512 的 CPU 上会落到很慢的通用实现，float32 在哪种 CPU 上都是全速）。
    "proxy_options": {
        "device": "auto", "dtype": "auto", "attention": "eager",
        "max_tokens": 1024, "max_response_tokens": 512, "token_chunk": 32,
        # 评分引擎。"auto"：先在 S 上把“逐层评分”与普通分批前向对照（最短的两条和最长的一条题，NLL 差都不超过 2e-3 才用），
        # 不通过就用分批前向，结论记在 scoring_engine_check.json。逐层评分：每层权重只进内存一次，候选不写盘。
        # 也可写 "layerwise" / "batched" 强制其中一种。
        "engine": "auto",
        "batch_size": 8, "logits_budget_gib": 2.0,    # 分批前向时每批最多 8 条；一批的 logits 超过 2GiB 时自动拆小
        "activation_reserve_gib": 2, "template_kwargs": {"enable_thinking": False},
    },
    "layer_memo": True,             # 逐层评分时，只换输出头或只改后半深度的候选沿用之前算好的中间隐状态（本地盘有余量时）
    # ==================== 上传 ====================
    "upload": True,                 # 搜索结束后构造选中的候选并上传；False：只存到 DRIVE_DIR/selected_candidate/
    "upload_when_no_gain": True,    # 没有候选的代理 NLL 比 S 低时：True 仍上传最好的候选（由你自己的测试判断）；False 不上传
    "output_repo": "YFC-112358/Qwen3.8-27B-pi-AuxMix-v1",   # 必须是新的（或空的）仓库；HF_TOKEN 要有写权限
    "private": True,
    # ==================== 成品 FP8（第 6 节；不影响融合/搜索缓存）====================
    "fp8_enabled": True,            # BF16 成品上传成功后，额外做 FP8 并上传独立仓库
    "fp8_repo": "",                 # 留空：output_repo + "-FP8"；也可填其他新的/空的仓库
    "fp8_dir": "",                  # 留空：DRIVE_DIR/fp8/，自动按源 commit 分目录续跑
}

_mix = [("Final", SETTINGS["donor_weight"])] + [(x["tag"], x["weight"]) for x in SETTINGS["aux_extras"]]
print("辅料比例:", " : ".join(f"{t} {w:g}" for t, w in _mix))
for _x in SETTINGS["aux_extras"]:
    if _x.get("format") == "della":
        print(f"  {_x['tag']} = DELLA(" + "、".join(m["tag"] + ("（对 " + m["reference_repo"] + "）" if m.get("reference_repo") else "")
                                                for m in _x["members"]) + ")")
    if _x.get("defer"):
        print(f"  {_x['tag']}：正式融合时再决定读 BF16 还是 GGUF，并补加进 D")
for _x in SETTINGS["searched_extras"]:
    print(f"单独搜索：μ × ({_x['tag']} − {_x['reference_repo'].split('/')[-1][:40]}…)，μ ∈ {SETTINGS['mu_grid']}，在 λ 括号外")
if SETTINGS["tm_extra"].get("enabled", True):
    print(f"跨架构增量：ν × N，N = TM({SETTINGS['tm_extra']['repo']} → B)，ν ∈ {SETTINGS['tm_extra'].get('nu_grid', [0.5, 1.0])}，"
          f"在获胜配方上评；模块 {', '.join(m.split('.')[-1] for m in SETTINGS['tm_extra']['modules'])}")
if SETTINGS["tm_extra2"].get("enabled", True):
    print(f"跨架构增量：ν₂ × N₂，N₂ = TM({SETTINGS['tm_extra2']['repo']} → B)，ν₂ ∈ {SETTINGS['tm_extra2'].get('nu_grid', [0.25, 0.5, 1.0])}，"
          f"在加了 N 的获胜配方上评；远程代码"
          + ("已审读（" + SETTINGS['tm_extra2']['code_sha256'][:12] + "…）" if SETTINGS['tm_extra2'].get('code_sha256') else "尚未审读：第一次运行会停下来让你看代码"))
print("等待项:", {"StartLux 仓名": STARTLUX_REQUIRED and not STARTLUX_REPO,
                 "搜索数据": "自动生成" if AUTO_EVAL_DATA else ("已填" if SETTINGS["search_file"] else "未填")})
print("选择:", "全局搜索 + 分量筛选" if SETTINGS["component_selection"] else "只做全局搜索")
print("上传:", SETTINGS["output_repo"] + ("（私有）" if SETTINGS["private"] else "（公开）") if SETTINGS["upload"] else "关闭（只存到 Drive）")
print("FP8:", (SETTINGS["fp8_repo"] or SETTINGS["output_repo"] + "-FP8") if SETTINGS["fp8_enabled"] else "关闭")
print("记录目录:", EFFICIENT_WORK, "；本地临时目录:", SCRATCH, "；CPU 线程:", CPU_THREADS)

## 2. 环境

依赖只在这里处理。第一次在 CPU 运行时上运行，就把版本记到 `DRIVE_DIR/build_runtime_versions.json` 和 `inference_runtime_versions.json`，之后的 CPU 会话会装回同一版本，评分缓存才能接着用。

版本记录按运行时种类分开（Colab 的 CPU 镜像是 CPU 版 torch `+cpu`，GPU 镜像是 CUDA 版 `+cu…`），不会把 CPU 版 torch 装到 GPU 运行时上。这一版的计算都在 CPU 运行时上做。


In [ ]:
"""依赖准备：在导入 torch / numpy 之前执行，只用标准库。"""
import importlib.metadata
import json
from pathlib import Path
import re
import shutil
import subprocess
import sys

BOOTSTRAP_VERSION = "qwen38-auxmix-2026-10-31.e15"
BASE_PACKAGES = ("torch", "numpy", "safetensors", "huggingface_hub", "requests", "psutil", "threadpoolctl")
INFERENCE_PACKAGES = ("transformers", "accelerate")
# 没有版本记录时的最低要求。Qwen3.5 系列的模型类是 transformers 5.8 加入的。
MINIMUM = {"huggingface_hub": ("huggingface_hub>=0.34,<2", (0, 34), 2), "transformers": ("transformers>=5.8", (5, 8), None)}


def gpu_machine():
    """这台机器有没有 NVIDIA GPU。不导入 torch，所以能在安装依赖之前问。"""
    exe = shutil.which("nvidia-smi")
    if not exe:
        return False
    try:
        result = subprocess.run([exe, "-L"], capture_output=True, timeout=60)
    except (OSError, subprocess.SubprocessError):
        return False
    return result.returncode == 0 and b"GPU" in result.stdout


def _version(package):
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return None


def _requirement_satisfied(requirement, installed):
    """True when pip (without --upgrade) would have nothing to do for this requirement."""
    if installed is None:
        return False
    if "==" in requirement:
        return installed == requirement.split("==", 1)[1]
    for text, floor, below_major in MINIMUM.values():
        if requirement == text:
            parts = re.findall(r"\d+", installed)[:2]
            if len(parts) < 2:
                return False
            version = (int(parts[0]), int(parts[1]))
            return floor <= version and (below_major is None or version[0] < below_major)
    return True


def _pip_arguments(requirements):
    """A torch build with a local tag (2.x.y+cu126, +cpu) is not on PyPI; it lives on PyTorch's own index."""
    extra = []
    for requirement in requirements:
        match = re.fullmatch(r"torch==[^+]+\+([A-Za-z0-9_.]+)", requirement)
        if match:
            extra = ["--extra-index-url", "https://download.pytorch.org/whl/" + match[1]]
    return [*extra, *requirements]


def _saved_json(path):
    if not path.exists():
        return None
    value = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(value, dict):
        raise ValueError(f"依赖版本记录不是字典：{path}")
    return value


def _atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(value, sort_keys=True, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    tmp.replace(path)


def prepare_environment(work, *, install, need_inference):
    """Existing version records are authoritative; a disagreement aborts before pip.

    Records are kept per kind of runtime. Colab's CPU image carries a CPU-only torch build (+cpu), its GPU
    image a CUDA build (+cuXXX), and other library versions can differ too, so a record made on one kind
    must never be installed on the other.
      CPU: build_runtime_versions.json is written on the first run, inference_runtime_versions.json by the
           pipeline; later CPU sessions get the same versions back, which keeps the cached scores valid.
      GPU: the image's own versions are used as they are. Only when generation really starts does the
           pipeline write inference_runtime_versions.gpu.json; later GPU sessions follow that record.
    """
    work = Path(work).expanduser().resolve()
    kind = "gpu" if gpu_machine() else "cpu"
    if kind == "gpu" and (_version("torch") or "").endswith("+cpu"):
        raise RuntimeError(
            f"这台机器有 NVIDIA GPU，但装着的是 CPU 版 torch（{_version('torch')}），GPU 用不了。"
            "请“断开连接并删除运行时”，重新连接后从头运行；如果手动装过 torch，请装回带 CUDA 的版本。")
    locks, origins = {}, {}
    build_lock = work / "build_runtime_versions.json"
    saved = _saved_json(build_lock) if kind == "cpu" else None
    if saved is not None:
        for package in BASE_PACKAGES:
            version = saved.get(package)
            if not isinstance(version, str) or not version:
                raise RuntimeError(f"依赖记录缺少 {package}：{build_lock}")
            locks[package], origins[package] = version, str(build_lock)
    if need_inference:
        path = work / ("inference_runtime_versions.gpu.json" if kind == "gpu" else "inference_runtime_versions.json")
        saved = _saved_json(path) or {}
        for package in (*BASE_PACKAGES, *INFERENCE_PACKAGES):
            version = saved.get(package)
            if version is None:
                continue
            if package in locks and locks[package] != version:
                raise RuntimeError(f"{package} 的构建/推理版本锁冲突：{origins[package]} 与 {path}")
            locks[package], origins[package] = version, str(path)
    if kind == "gpu" and locks.get("torch", "").endswith("+cpu"):
        raise RuntimeError(f"{origins['torch']} 记录的是 CPU 版 torch（{locks['torch']}），不能装到有 GPU 的机器上。"
                           "这份记录不是本 notebook 在 GPU 运行时写的；删除它后重新运行本格。")
    requested = BASE_PACKAGES + (INFERENCE_PACKAGES if need_inference else ())
    before = {package: _version(package) for package in requested}
    loaded = {package: sys.modules.get(package) for package in requested}
    requirements = []
    for package in requested:
        target = locks.get(package)
        if target is None and loaded.get(package) is not None:
            target = before[package]
        requirements.append(package + "==" + target if target else MINIMUM[package][0] if package in MINIMUM else package)
    if install and all(_requirement_satisfied(r, before[p]) for r, p in zip(requirements, requested)):
        print("依赖均已满足，跳过 pip。", flush=True)
    elif install:
        changed = [r for r, p in zip(requirements, requested) if not _requirement_satisfied(r, before[p])]
        print("安装依赖：" + "、".join(changed) + ("（按版本记录）" if locks else ""), flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_pip_arguments(requirements)])
    actual = {package: _version(package) for package in requested}
    missing = [package for package, version in actual.items() if version is None]
    if missing:
        raise RuntimeError("缺少依赖：" + ", ".join(missing) + "。开启 INSTALL_DEPENDENCIES，安装后从头运行。")
    for package, version in actual.items():
        if package in locks and locks[package] != version:
            raise RuntimeError(f"{package} 实际版本 {version} 与记录 {locks[package]} 不符（{origins[package]}）")
        if loaded.get(package) is not None and before[package] != version:
            raise RuntimeError(f"安装改变了已导入的 {package}；请重启内核，不能沿用旧模块")
    base_versions = {package: actual[package] for package in BASE_PACKAGES}
    if kind == "cpu" and not build_lock.exists():
        _atomic_json(build_lock, base_versions)
    return {"base_versions": base_versions,
            "inference_versions": {p: actual[p] for p in INFERENCE_PACKAGES if p in actual},
            "kind": kind, "version": BOOTSTRAP_VERSION}

In [ ]:
_DATA_KEYS = ("search_file",)
_NEED_INFERENCE = bool(SETTINGS["run"] and (AUTO_EVAL_DATA or all(SETTINGS.get(k) for k in _DATA_KEYS)))
UNIFIED_ENVIRONMENT = prepare_environment(EFFICIENT_WORK, install=INSTALL_DEPENDENCIES, need_inference=_NEED_INFERENCE)

import torch, requests, psutil, numpy as np, shutil, gc
from huggingface_hub import HfApi
from threadpoolctl import threadpool_limits

torch.set_num_threads(CPU_THREADS)
_CPU_BLAS_LIMIT = threadpool_limits(limits=CPU_THREADS)
SETTINGS["install_inference_dependencies"] = False   # 推理依赖已在上面按版本记录处理
SETTINGS["version_lock_kind"] = UNIFIED_ENVIRONMENT["kind"]   # 版本记录按运行时种类（cpu / gpu）分开

TOK = os.environ.get("HF_TOKEN", "")
if not TOK:
    try:
        from google.colab import userdata
        TOK = userdata.get("HF_TOKEN") or ""
    except Exception:
        pass
if not TOK and SETTINGS["run"]:
    # 下载量很大，匿名访问容易被限速；gated 的 ThinkingCap 和上传（要有写权限）也需要 token。
    from getpass import getpass
    TOK = getpass("Hugging Face token: ")
TOK = TOK.strip()
if TOK and (not TOK.isascii() or any(ch.isspace() for ch in TOK) or len(TOK) > 100):
    # Never echo the value: it may be a real token with something pasted after it.
    raise ValueError(f"Hugging Face token 看起来不对：长度 {len(TOK)}，"
                     f"{'含有非英文字符' if not TOK.isascii() else '含有空白或过长'}。"
                     "token 应是以 hf_ 开头的一串英文字母和数字。请重新运行本格并只粘贴 token，"
                     "或检查 Colab 密钥 / 环境变量 HF_TOKEN 的内容。")
if TOK and not TOK.startswith("hf_"):
    print("注意：token 不是以 hf_ 开头，确认粘贴的是 Hugging Face 访问令牌。")
api = HfApi(token=TOK or None)
print("依赖:", UNIFIED_ENVIRONMENT)
if torch.cuda.is_available():
    _gpu = torch.cuda.get_device_properties(0)
    print(f"运行时：GPU {_gpu.name}，显存 {_gpu.total_memory / 1024**3:.1f} GiB；内存 {psutil.virtual_memory().total / 1024**3:.0f} GiB")
    print("提示：这个 notebook 全部在 CPU 上做，不需要 GPU；在 GPU 运行时上第 5 节不会开始计算。请把运行时换成 CPU（高 RAM）。")
else:
    print(f"运行时：无 GPU；{os.cpu_count()} 核 CPU，内存 {psutil.virtual_memory().total / 1024**3:.0f} GiB")
    if UNIFIED_ENVIRONMENT["kind"] == "gpu":
        print(f"警告：这台机器有 NVIDIA GPU，但当前的 torch（{torch.__version__}）用不了它，第 5 节在这种状态下不会开始计算。"
              "可以“断开连接并删除运行时”，重新连接后再试。")

## 3. 引擎模块

全部代码都内嵌在本文件里，不下载额外脚本。模块文件写到本地临时目录后导入。

| 模块 | 内容 |
|---|---|
| `qmerge_core` | 纯计算：记录读写、评分、配方 |
| `qmerge_runtime` | 权重装载与真实前向（逐层评分、分批评分） |
| `qmerge_components` | 分量筛选：增量的正交分解、逐层的前向与逐题反向、去掉分量后的权重 |
| `qmerge_legacy` | 按 HTTP 范围读取 safetensors、输入对齐 |
| `qmerge_gguf` | GGUF 读取与布局还原 |
| `qmerge_della` | DELLA：按名次丢弃、符号选举、取均值 |
| `qmerge_tm` | 跨架构增量 N：读 target 激活、供体在 CPU 上的流式前向与激活、go/no-go 判定、CKA 预筛、层对与层级 OT、只算 128 行的融合 |
| `qmerge_tm2` | 跨架构增量 N₂（SenseNova-Loop-9B-Preview，e15）：远程代码审读门槛、独立子进程里的供体前向（逐执行的激活）、结构核对、按执行的判定 / 层对 OT / 融合；数值核心复用 `qmerge_tm` |
| `qmerge_pipeline` | 辅料增量 D、候选构造、搜索、分量筛选的编排、上传 |
| `qmerge_evaldata` | 搜索数据构建 |

In [ ]:
import importlib.util, sys, hashlib
from pathlib import Path
EMBEDDED_DIR = SCRATCH / "notebook_modules"
EMBEDDED_DIR.mkdir(parents=True, exist_ok=True)
def load_embedded_module(name, source):
    path = EMBEDDED_DIR / (name + ".py")
    path.write_text(source, encoding="utf-8")
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    globals()[name] = module
    return module

In [ ]:
# 内嵌源代码；独立运行无需额外文件。
MODULE_SOURCE = r'''
"""Pure, CPU-testable primitives. No network calls, model loading, or publishing."""
import hashlib
import json
import math
import re
import unicodedata
from pathlib import Path
import numpy as np
import torch

VERSION = 'auxmix-2026-10-18.e11'


def digest(value):
    return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,separators=(',',':'),allow_nan=False).encode()).hexdigest()


def sha_file(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(8*1024**2),b''): h.update(b)
    return h.hexdigest()


def atomic_json(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(value,ensure_ascii=False,sort_keys=True,indent=2,allow_nan=False)+'\n',encoding='utf-8')
    tmp.replace(path)


def save_record(path,identity,payload):
    doc={'identity':identity,'payload':payload}
    atomic_json(path,{'document':doc,'sha256':digest(doc)})


def load_record(path,identity):
    if not Path(path).exists():return None
    e=json.loads(Path(path).read_text(encoding='utf-8'));d=e.get('document')
    if not isinstance(d,dict) or e.get('sha256')!=digest(d) or d.get('identity')!=identity:
        raise ValueError(f'缓存内容或身份不匹配：{path}。请使用新的工作目录，不要混用结果。')
    return d['payload']


def normal_text(s):return ' '.join(unicodedata.normalize('NFKC',s).split())


def prompt_hash(row):
    # System messages cannot hide exact re-use of the user's evaluation questions.
    return digest([normal_text(m['content']) for m in row['messages'] if m['role']=='user'])


def check_disjoint(splits):
    seen={};count=0
    for split,rows in splits.items():
        for row in rows:
            key=prompt_hash(row)
            if key in seen:
                raise ValueError(f'重复提示：{split}/{row["id"]} 与 {seen[key]}；搜索、校准、验证、终测必须分离。')
            seen[key]=f'{split}/{row["id"]}';count+=1
    return {'checked_items':count,'scope':'exact normalized user prompts only; unknown training contamination not detected'}


def validate_messages(row):
    if not isinstance(row,dict) or not isinstance(row.get('id'),str) or not row['id']:
        raise ValueError('每条样本需要非空字符串 id')
    if not isinstance(row.get('category'),str) or not row['category']:
        raise ValueError('每条样本需要 category')
    ms=row.get('messages')
    if not isinstance(ms,list) or not ms or ms[-1].get('role')!='user':
        raise ValueError('messages 必须以 user 结尾')
    if any(not isinstance(m,dict) or m.get('role') not in {'user','system','assistant'} or not isinstance(m.get('content'),str) for m in ms):
        raise ValueError('当前只支持文本 messages；不把图像/工具占位当真实数据')


def validate_proxy_rows(rows,formal=True):
    if not rows:raise ValueError('前向评分集为空')
    for row in rows:
        validate_messages(row)
        if formal and row.get('smoke_only'):raise ValueError('smoke_only 示例不能用于正式选择')
        if not isinstance(row.get('response'),str) or not row['response'].strip():
            raise ValueError('前向评分必须提供已核验的 response；不会从 expected/题目/模型自答中猜造教师答案')
        if formal and row.get('reference_verified') is not True:
            raise ValueError('正式前向评分须标注 reference_verified=true，并实际核验参考回答')
        if formal and (not isinstance(row.get('source'),str) or not row['source'].strip()):raise ValueError('正式前向评分须记录非空字符串 source')
    if len({r['id'] for r in rows})!=len(rows):raise ValueError('样本 id 重复')
    check_disjoint({'dataset':rows})
    return rows


def read_jsonl(path,kind='proxy',formal=True):
    if not str(path).strip():raise ValueError('数据文件未配置')
    rows=[]
    for i,line in enumerate(Path(path).expanduser().read_text(encoding='utf-8').splitlines(),1):
        if line.strip():
            try:rows.append(json.loads(line))
            except json.JSONDecodeError as e:raise ValueError(f'{path}:{i} JSONL 格式错误') from e
    if kind=='proxy':return validate_proxy_rows(rows,formal)
    if not rows:raise ValueError('生成评测集为空')
    for row in rows:
        validate_messages(row)
        if formal and row.get('smoke_only'):raise ValueError('示例题不能用于正式终测')
        if row.get('metric') not in {'exact','number','json','manual'}:raise ValueError('metric 只支持 exact/number/json/manual')
        if row['metric']=='manual':
            if not str(row.get('rubric','')).strip():raise ValueError('manual 题须提供 rubric')
        elif 'expected' not in row:raise ValueError('自动评分须提供 expected')
    if len({r['id'] for r in rows})!=len(rows):raise ValueError('id 重复')
    check_disjoint({'dataset':rows})
    return rows


def check_coverage(rows,min_items,min_per_category):
    counts={c:sum(r['category']==c for r in rows) for c in sorted({r['category'] for r in rows})}
    if len(rows)<min_items or not counts or min(counts.values())<min_per_category:
        raise ValueError(f'样本覆盖不足：总数 {len(rows)}，各类 {counts}；至少 {min_items} 条/{min_per_category} 条每类')
    return counts


def category_means(rows,values):
    if len(values)!=len(rows) or not all(math.isfinite(float(x)) for x in values):raise ValueError('分数长度或有限性错误')
    return {c:float(np.mean([v for r,v in zip(rows,values) if r['category']==c])) for c in sorted({r['category'] for r in rows})}


def weighted_mean(rows,values,weights=None):
    means=category_means(rows,values);weights=weights or {}
    if set(weights)-set(means):raise ValueError('分类权重包含未知分类')
    w={k:float(weights.get(k,1.)) for k in means}
    if any(not math.isfinite(x) or x<=0 for x in w.values()):raise ValueError('分类权重必须为有限正数')
    return sum(means[c]*w[c] for c in means)/sum(w.values())


def paired_interval(rows,diffs,weights=None,repeats=2000,seed=42):
    """Fixed-sample, stratified paired bootstrap; no optional-stopping interpretation."""
    if repeats<100:raise ValueError('bootstrap 至少 100 次')
    point=weighted_mean(rows,diffs,weights);rng=np.random.default_rng(seed)
    cats=sorted({r['category'] for r in rows});w=np.array([(weights or {}).get(c,1.) for c in cats],float);w/=w.sum()
    arr=np.asarray(diffs,float);out=np.zeros(repeats)
    for c,weight in zip(cats,w):
        ids=np.array([i for i,r in enumerate(rows) if r['category']==c])
        # Bounded scratch even for large user-provided datasets.
        for start in range(0,repeats,64):
            n=min(64,repeats-start);samples=rng.choice(ids,size=(n,len(ids)),replace=True)
            out[start:start+n]+=weight*arr[samples].mean(1)
    return {'mean':point,'ci95':np.quantile(out,[.025,.975]).tolist()}


def choose_proxy(rows,losses,policy):
    p={'min_items':96,'min_per_category':16,'min_gain':.01,'max_category_regression':.05,
       'max_base_regression':.05,'repeats':2000,'seed':42,'category_weights':{},**policy}
    for key in ('min_items','min_per_category','repeats','seed'):
        if type(p[key]) is not int or p[key] < (100 if key=='repeats' else 0 if key=='seed' else 1):raise ValueError('代理计数/种子配置非法：'+key)
    for key in ('min_gain','max_category_regression','max_base_regression'):
        if not math.isfinite(float(p[key])) or p[key]<0:raise ValueError('代理阈值必须为有限非负数：'+key)
    if any(float(v)<0 for values in losses.values() for v in values):raise ValueError('NLL 不能为负数')
    if not {'DELLA','ISO'}<=set(losses):raise ValueError('必须同时评测 DELLA 和 ISO')
    stats={tag:{'mean_nll':weighted_mean(rows,v,p['category_weights']),'category_nll':category_means(rows,v)} for tag,v in losses.items()}
    result={'decision':'undecided','winner':None,'reasons':[],'stats':stats,'metric':'mean response-token NLL; lower is better','scope':'forward_proxy_not_generation_quality'}
    try:check_coverage(rows,p['min_items'],p['min_per_category'])
    except ValueError as e:result['reasons'].append(str(e));return result
    diff=[a-b for a,b in zip(losses['DELLA'],losses['ISO'])]
    interval=paired_interval(rows,diff,p['category_weights'],p['repeats'],p['seed']);result['della_minus_iso_nll']=interval
    tag='ISO' if interval['mean']>0 else 'DELLA';other='DELLA' if tag=='ISO' else 'ISO'
    if abs(interval['mean'])<p['min_gain'] or (interval['ci95'][0]<=0<=interval['ci95'][1]):
        result['reasons'].append('代理分差太小或 95% 配对区间含零')
    for c,loss in stats[tag]['category_nll'].items():
        if loss-stats[other]['category_nll'][c]>p['max_category_regression']:result['reasons'].append(f'{c}: 相对另一候选 NLL 回退过大')
        if 'BASE' in stats and loss-stats['BASE']['category_nll'][c]>p['max_base_regression']:result['reasons'].append(f'{c}: 相对 BASE NLL 回退过大')
    if not result['reasons']:result.update(decision='selected',winner=tag)
    return result


@torch.no_grad()
def response_nll(logits,input_ids,prompt_length,token_chunk=32,logits_start=0):
    """Correct causal shift. prompt_length = first reference token in full sequence.

    logits_start is the original position represented by logits[:,0]. Supports
    models returning only suffix logits without scoring prompt tokens.
    """
    if logits.ndim!=3 or input_ids.ndim!=2 or logits.shape[0]!=1 or input_ids.shape[0]!=1:raise ValueError('当前评分器使用 batch=1')
    length=input_ids.shape[1]
    if not 1<=prompt_length<length or token_chunk<1:raise ValueError('参考回答必须非空，且必须有非空上下文')
    first=prompt_length-1-logits_start;stop=length-1-logits_start
    if first<0 or stop>logits.shape[1]:raise ValueError('logits 不覆盖所需的参考回答预测位置')
    total=0.;count=0
    for a in range(first,stop,token_chunk):
        b=min(a+token_chunk,stop);block=logits[:,a:b].float()
        if not torch.isfinite(block).all():raise ValueError('评分 logits 含 NaN/Inf')
        target=input_ids[:,a+logits_start+1:b+logits_start+1].to(block.device)
        if int(target.min())<0 or int(target.max())>=block.shape[-1]:raise ValueError('目标 token 超出模型词表')
        total+=float(torch.nn.functional.cross_entropy(block.reshape(-1,block.shape[-1]),target.reshape(-1),reduction='sum'))
        count+=b-a
    return total/count,count



def chat_token_ids(tokenizer,messages,template_kwargs=None):
    """Token ids of the generation prefix, as a flat list of ints.

    transformers >= 5 returns a BatchEncoding from apply_chat_template. That class is a UserDict,
    not a dict, so it has to be recognised as a Mapping; 4.x returned the ids themselves.
    """
    from collections.abc import Mapping
    kwargs=dict(template_kwargs or {})
    if {'tokenize','add_generation_prompt','return_tensors','return_dict'} & set(kwargs):raise ValueError('不能覆盖基础模板参数')
    ids=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,**kwargs)
    if isinstance(ids,Mapping):ids=ids['input_ids']
    if hasattr(ids,'tolist'):ids=ids.tolist()
    if ids and isinstance(ids[0],(list,tuple)):
        if len(ids)!=1:raise ValueError('聊天模板返回了多条序列')
        ids=ids[0]
    ids=[int(x) for x in ids]
    if not ids:raise ValueError('提示无 token')
    return ids


def pack_reference(tokenizer,row,template_kwargs=None,max_tokens=1024,max_response_tokens=512):
    """Build exactly the same generation prefix for every candidate.

    The supplied `response` is the literal continuation of that prefix, not a
    full assistant chat turn. Separate tokenization is deliberate and recorded;
    it avoids scoring end-of-turn/prompt wrappers as answers. No silent truncation.
    """
    ids=chat_token_ids(tokenizer,row['messages'],template_kwargs)
    tail=tokenizer.encode(row['response'],add_special_tokens=False)
    if hasattr(tail,'tolist'):tail=tail.tolist()
    tail=[int(x) for x in tail]
    if not tail:raise ValueError('参考回答无 token')
    if len(ids)+len(tail)>max_tokens or len(tail)>max_response_tokens:raise ValueError(f'{row["id"]}: 超出统一 token 预算；不截断参考答案')
    full=ids+tail
    return torch.tensor([full],dtype=torch.long),len(ids),{'token_sha256':digest(full),'prompt_tokens':len(ids),'response_tokens':len(tail),'encoding':'generation_prefix + literal continuation tokens; no automatic EOS'}


def fuse_block(subject,donor,base,scale,kappa):
    if subject.shape!=donor.shape or subject.shape!=base.shape:raise ValueError('融合张量形状不一致')
    if not all(math.isfinite(float(x)) for x in (scale,kappa)) or scale<0 or not 0<=kappa<=1:raise ValueError('非法融合系数')
    if scale==0:return subject.clone()
    delta=(donor-base)-float(kappa)*(subject-base)
    result=subject+float(scale)*delta
    if not torch.isfinite(result).all():raise ValueError('融合出现 NaN/Inf')
    return result


def top_mask(magnitude,k):
    """True for the k largest entries of every row of a 2-D tensor; among equal values the lower column wins.

    That is exactly the first k columns of a stable descending argsort, found by selection instead of sorting:
    every entry above the k-th largest value, plus the leftmost entries equal to it until there are k.
    """
    rows,n=magnitude.shape
    if k<=0:return torch.zeros((rows,n),dtype=torch.bool,device=magnitude.device)
    if k>=n:return torch.ones((rows,n),dtype=torch.bool,device=magnitude.device)
    threshold=torch.kthvalue(magnitude,n-k+1,dim=1,keepdim=True).values      # the k-th largest of each row
    above=magnitude>threshold
    equal=magnitude==threshold
    need=k-above.sum(dim=1,keepdim=True)                                    # equal entries still to take, >= 1
    return above|(equal&(equal.cumsum(dim=1)<=need))


def magnitude_scale(delta,high_fraction,low_scale,high_scale):
    """NPS-inspired row-magnitude grouping, NOT official NPS-Pruning reproduction.
    Stable ties use column index; each row is complete, so chunking is irrelevant.
    """
    if delta.ndim!=2 or not 0<=high_fraction<=1 or any(not math.isfinite(v) or not 0<=v<=1 for v in (low_scale,high_scale)):
        raise ValueError('非法行幅度分组参数')
    if low_scale==high_scale:return delta*low_scale
    k=int(math.ceil(delta.shape[1]*high_fraction))
    return delta*torch.where(top_mask(delta.abs(),k),high_scale,low_scale)


def magnitude_scale_by_sort(delta,high_fraction,low_scale,high_scale):
    """The original formulation with a full stable argsort; top_mask gives exactly this mask. Kept as the reference."""
    if low_scale==high_scale:return delta*low_scale
    k=int(math.ceil(delta.shape[1]*high_fraction))
    mask=torch.zeros_like(delta,dtype=torch.bool)
    if k:mask.scatter_(1,torch.argsort(delta.abs(),dim=1,descending=True,stable=True)[:,:k],True)
    return delta*torch.where(mask,high_scale,low_scale)


def parameter_role(name,shape):
    if re.search(r'(^|\.)(visual|vision_tower|vision_model|mtp|nextn)(\.|$)',name):return 'protected'
    if name.endswith('embed_tokens.weight'):return 'protected'
    if name.endswith('lm_head.weight'):return 'head'
    if len(shape)!=2 or any(x in name.lower() for x in ('norm','a_log','dt_bias','conv1d')):return 'protected'
    return 'backbone'


def validate_recipe(recipe):
    required={'lambda_backbone','lambda_head','kappa','group_scales','high_fraction'}
    # rho_late: coefficient of the second donor's delta inside the bracket. Absent means 0.
    # mu: coefficient of the separately searched delta C, added outside the lambda bracket. Absent means 0.
    if not required<=set(recipe) or set(recipe)-required-{'rho_late','mu'}:raise ValueError(f'配方字段必须是 {sorted(required)}（可另含 rho_late、mu）；未实现的 eta/校正增量不能静默忽略')
    for key in ('rho_late','mu'):
        if key in recipe and (isinstance(recipe[key],bool) or not math.isfinite(float(recipe[key])) or not 0<=recipe[key]<=2):raise ValueError(key+' 必须在 [0,2]')
    for k in ('lambda_backbone','lambda_head'):
        if not math.isfinite(float(recipe[k])) or not 0<=recipe[k]<=1:raise ValueError(k+' 必须在 [0,1]')
    if not math.isfinite(float(recipe['kappa'])) or not 0<=recipe['kappa']<=1:raise ValueError('kappa 必须在 [0,1]')
    if not 0<=recipe['high_fraction']<=1:raise ValueError('high_fraction 必须在 [0,1]')
    if len(recipe['group_scales'])!=4 or any(not math.isfinite(v) or not 0<=v<=1 for v in recipe['group_scales']):raise ValueError('group_scales 需要四个 [0,1] 系数')
    return recipe


def canonical_recipe(recipe):
    r=json.loads(json.dumps(validate_recipe(recipe)))
    if r['lambda_backbone']==0:r.update(kappa=0.,group_scales=[1.,1.,1.,1.],high_fraction=.2) if r['lambda_head']==0 else None
    # A zero coefficient, or a recipe that changes nothing, is the same recipe as one without the field.
    if r.get('rho_late',0)==0 or (r['lambda_backbone']==0 and r['lambda_head']==0):r.pop('rho_late',None)
    if r.get('mu',0)==0:r.pop('mu',None)
    return r


def recipe_vector(recipe):
    return [recipe['lambda_backbone'],recipe['lambda_head'],recipe['kappa'],*recipe['group_scales'],float(recipe.get('rho_late',0.)),float(recipe.get('mu',0.))]


def gp_predict(x,y,candidates,length=.5,noise=1e-5):
    """Small deterministic RBF surrogate; posterior spread is a search heuristic."""
    x=np.asarray(x,float);y=np.asarray(y,float);z=np.asarray(candidates,float)
    if x.ndim!=2 or z.ndim!=2 or x.shape[1]!=z.shape[1] or len(x)!=len(y) or not len(x):raise ValueError('代理输入维度不匹配')
    if not all(np.isfinite(v).all() for v in (x,y,z)):raise ValueError('代理输入非有限')
    kernel=lambda a,b:np.exp(-((a[:,None]-b[None,:])**2).sum(-1)/(2*length**2))
    k=kernel(x,x)+np.eye(len(x))*noise;kz=kernel(z,x)
    center=y.mean();scale=max(y.std(),1e-6);normalized=(y-center)/scale
    chol=np.linalg.cholesky(k)
    alpha=np.linalg.solve(chol.T,np.linalg.solve(chol,normalized))
    v=np.linalg.solve(chol,kz.T)
    return center+scale*(kz@alpha),scale*np.sqrt(np.maximum(0,1-(v*v).sum(0)))


@torch.no_grad()
def tailor_block(subject,candidate,hessian,keep_fraction=.8,damp=.01):
    """Block-diagonal, Hessian-compensated sparse patch, inspired by Model Tailor.

    For each row, discarded entries revert exactly to S; free entries compensate
    their local least-squares error under H=X^T X/N + ridge. Not a reproduction
    of official Model Tailor's full mask selection/global Hessian method.
    """
    if subject.ndim!=2 or subject.shape!=candidate.shape or hessian.shape!=(subject.shape[1],)*2:raise ValueError('修复形状不匹配')
    if not 0<keep_fraction<=1 or not math.isfinite(damp) or damp<=0:raise ValueError('修复参数非法')
    if any(not torch.isfinite(t).all() for t in (subject,candidate,hessian)):raise ValueError('修复输入非有限')
    d=(candidate-subject).double();h=hessian.double()
    if not torch.allclose(h,h.T,rtol=1e-5,atol=1e-8):raise ValueError('Hessian 必须对称')
    ridge=max(float(h.diag().mean()),1e-12)*damp;h=h+torch.eye(h.shape[0],dtype=h.dtype,device=h.device)*ridge
    if torch.linalg.cholesky_ex(h).info.item()!=0:raise ValueError('Hessian 加阻尼后仍非正定')
    k=max(1,math.ceil(d.shape[1]*keep_fraction))
    mask=torch.zeros_like(d,dtype=torch.bool)
    ids=torch.argsort(d.abs()*h.diag().sqrt(),dim=1,descending=True,stable=True)[:,:k];mask.scatter_(1,ids,True)
    if k==d.shape[1]:return candidate.clone(),mask
    repaired=torch.zeros_like(d)
    for row in range(d.shape[0]):
        keep=ids[row];removed=torch.where(~mask[row])[0]
        rhs=h[keep[:,None],removed[None,:]]@d[row,removed]
        correction=torch.linalg.solve(h[keep[:,None],keep[None,:]],rhs)
        repaired[row,keep]=d[row,keep]+correction
    result=(subject.double()+repaired).to(candidate.dtype)
    if not torch.isfinite(result).all():raise ValueError('局部修复溢出')
    return result,mask


def final_answer(text):
    if text.count('<think>')!=text.count('</think>'):return None
    text=re.sub(r'<think>.*?</think>','',text,flags=re.S).strip()
    if '<answer>' in text or '</answer>' in text:
        m=re.fullmatch(r'<answer>(.*?)</answer>',text,re.S);return m[1].strip() if m else None
    return text


def _decimal_sum_sign(values):
    """Exact sign of a short finite-decimal sum, without expanding huge exponent gaps."""
    import decimal
    terms=[]
    for value in values:
        if not value:continue
        sign,digits,exponent=value.as_tuple()
        terms.append((int(decimal.Decimal((sign,digits,0))),exponent,len(digits)))
    if not terms:return 0
    terms.sort(key=lambda t:t[1],reverse=True)
    coefficient,exponent,_=terms[0]
    for i,(other,next_exponent,_) in enumerate(terms[1:],1):
        if not coefficient:coefficient,exponent=other,next_exponent;continue
        # Remaining magnitudes sum to less than n * 10**max(exponent+digits).
        # A larger nonzero accumulated term determines the sign immediately.
        bound=max(e+n for _,e,n in terms[i:])+len(str(len(terms)-i))
        if exponent>=bound:return 1 if coefficient>0 else -1
        coefficient=coefficient*10**(exponent-next_exponent)+other
        exponent=next_exponent
    return (coefficient>0)-(coefficient<0)


def grade_generation(row,record):
    import decimal
    if record.get('truncated') or record.get('error'):return 0.
    text=final_answer(record['text'])
    if text is None or not text:return 0.
    if row['metric']=='manual':return None
    if row['metric']=='exact':
        vals=row['expected'] if isinstance(row['expected'],list) else [row['expected']]
        return float(normal_text(text) in [normal_text(str(x)) for x in vals])
    if row['metric']=='json':
        try:return float(digest(json.loads(text))==digest(row['expected']))
        except (ValueError,TypeError):return 0.
    if row['metric']=='number':
        try:
            x=decimal.Decimal(text.replace(',',''));expected=decimal.Decimal(str(row['expected']))
            rtol=decimal.Decimal(str(row.get('rtol',0)));atol=decimal.Decimal(str(row.get('atol',0)))
        except (ValueError,decimal.InvalidOperation):return 0.
        if any(not v.is_finite() for v in (x,expected,rtol,atol)):return 0.
        if rtol<0 or atol<0:raise ValueError('数值评分容差不能为负数')
        if x==expected:return 1.
        if rtol==0 and atol==0:return 0.
        # Preserve decimal/integer precision; float coercion can turn distinct
        # answers above 2**53 (or long decimals) into identical values.
        try:
            with decimal.localcontext() as ctx:
                ctx.prec=max(ctx.prec,sum(len(v.as_tuple().digits) for v in (x,expected,rtol,atol))+2)
                ctx.Emin=decimal.MIN_EMIN;ctx.Emax=decimal.MAX_EMAX
                ctx.traps[decimal.Underflow]=True
                tolerance=max(atol,rtol*max(x.copy_abs(),expected.copy_abs()))
                hi,lo=(x,expected) if x>=expected else (expected,x)
                return float(_decimal_sum_sign((hi,lo.copy_negate(),tolerance.copy_negate()))<=0)
        except decimal.DecimalException:return 0.
    raise ValueError('未知生成评分方式')


def generation_gate(rows,baseline,candidate,policy):
    """Paired capability gate, measured efficiency, no proxy substituted for scores."""
    p={'min_items':96,'min_per_category':16,'max_category_drop':.02,'max_overall_drop':.01,
       'max_truncation_rate':.05,'max_token_increase':.10,'min_score_gain':.01,
       'min_token_reduction':.05,'repeats':2000,'seed':42,'category_weights':{},**policy}
    for key in ('min_items','min_per_category','repeats','seed'):
        if type(p[key]) is not int or p[key] < (100 if key=='repeats' else 0 if key=='seed' else 1):raise ValueError('生成评测计数配置非法：'+key)
    for key in ('max_category_drop','max_overall_drop','max_truncation_rate','min_score_gain','min_token_reduction'):
        if not math.isfinite(float(p[key])) or not 0<=p[key]<=1:raise ValueError('生成评测阈值须为 [0,1] 内的有限数：'+key)
    if not math.isfinite(float(p['max_token_increase'])) or p['max_token_increase']<0:raise ValueError('token 增长上限须为有限非负数')
    check_coverage(rows,p['min_items'],p['min_per_category'])
    if len(baseline)!=len(rows) or len(candidate)!=len(rows):raise ValueError('生成记录不完整')
    a=[x.get('score') for x in baseline];b=[x.get('score') for x in candidate]
    if any(v is None for v in a+b):return {'passed':False,'pending_manual':True,'reasons':['人工评分未齐全；不会跳过']}
    if any(not math.isfinite(float(v)) or not 0<=v<=1 for v in a+b):raise ValueError('生成分数不在 [0,1]')
    diff=paired_interval(rows,[y-x for x,y in zip(a,b)],p['category_weights'],p['repeats'],p['seed'])
    cats_a=category_means(rows,a);cats_b=category_means(rows,b);reasons=[]
    if diff['ci95'][0]<-p['max_overall_drop']:reasons.append('不能建立设定余量内的总体非劣性')
    for c in cats_a:
        if cats_a[c]-cats_b[c]>p['max_category_drop']:reasons.append(f'{c}: 分类分数回退超过上限')
    nt_a=float(np.mean([x['output_tokens'] for x in baseline]));nt_b=float(np.mean([x['output_tokens'] for x in candidate]))
    trunc=sum(bool(x.get('truncated')) for x in candidate)/len(rows)
    if trunc>p['max_truncation_rate']:reasons.append('截断率过高')
    if nt_b>nt_a*(1+p['max_token_increase']):reasons.append('平均输出 token 增长超出预算')
    gain=diff['mean']>=p['min_score_gain'] and diff['ci95'][0]>0
    efficiency=nt_a>0 and nt_b<=nt_a*(1-p['min_token_reduction'])
    if not (gain or efficiency):reasons.append('未证明设定的分数收益，也未达到效率收益门槛')
    return {'passed':not reasons,'pending_manual':False,'reasons':reasons,'paired_score_difference':diff,
            'baseline_category_scores':cats_a,'candidate_category_scores':cats_b,'baseline_mean_tokens':nt_a,
            'candidate_mean_tokens':nt_b,'candidate_truncation_rate':trunc,
            'scope':'only this fixed dataset and protocol; token reduction is not a wall-clock throughput benchmark'}
'''
load_embedded_module('qmerge_core', MODULE_SOURCE)
del MODULE_SOURCE

In [ ]:
# 内嵌源代码；独立运行无需额外文件。
MODULE_SOURCE = r'''
"""Pinned safetensors snapshots and actual forward/generation inference.

Weights are placed by capacity. "resident": every weight is loaded once and stays on the inference
device (a GPU with enough memory, or RAM). "offload": accelerate reads each module's weights from the
original safetensors files for every forward pass (CPU with less RAM than the model). Scoring and
generation run in fixed batches. LayerwiseScorer scores many sequences with each decoder layer's weights
in memory once per model (no padding, no per-batch weight reads). Generation with batch_size 'auto' plans
as few batches as the GPU holds and splits a batch early when its memory growth would not fit at its end.
This is not a weight-only quality estimator, quantized substitute, or guarantee of Colab resource fit.
"""
import gc
import importlib.metadata
import inspect
import json
import math
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import time
from contextlib import contextmanager
import torch
from safetensors import safe_open
import qmerge_core as core

METADATA_EXACT={'config.json','generation_config.json','tokenizer.json','tokenizer_config.json',
 'special_tokens_map.json','added_tokens.json','vocab.json','merges.txt','chat_template.jinja',
 'preprocessor_config.json','processor_config.json','video_preprocessor_config.json'}
DTYPES={'BF16':torch.bfloat16,'F32':torch.float32,'F16':torch.float16,'F64':torch.float64,
        'I64':torch.int64,'I32':torch.int32,'I16':torch.int16,'I8':torch.int8,'U8':torch.uint8,'BOOL':torch.bool}


def safe_relative(name):
    p=Path(name)
    if p.is_absolute() or '..' in p.parts or '\\' in name or not name:raise ValueError('不安全的相对文件名')
    return name



def runtime_versions():
    # Only what can change inference results. The Python patch level and packages the pipeline never
    # imports are left out, so a refreshed Colab image does not invalidate every cached score.
    out={'python':'.'.join(sys.version.split()[0].split('.')[:2]),'implementation':core.VERSION}
    for p in ('torch','numpy','safetensors','huggingface_hub','transformers','accelerate'):
        try:out[p]=importlib.metadata.version(p)
        except importlib.metadata.PackageNotFoundError:out[p]=None
    for name in ('qmerge_core','qmerge_runtime'):
        module=sys.modules.get(name);file=getattr(module,'__file__',None)
        if file and Path(file).is_file():out[name+'_code_sha256']=core.sha_file(file)
    return out



LIBRARIES=('torch','numpy','safetensors','huggingface_hub','transformers','accelerate')


def version_record(work,kind=None):
    """Where the library versions of this kind of runtime are recorded.

    Colab's CPU image carries a CPU-only torch build (+cpu), its GPU image a CUDA build (+cuXXX), and other
    versions can differ as well. Each kind therefore keeps its own record; nothing that one kind of runtime
    hands to the other (D, the selection record, the saved candidate, finished reports) is keyed on versions.
    """
    if kind is None:kind='gpu' if torch.cuda.is_available() else 'cpu'
    if kind not in {'cpu','gpu'}:raise ValueError('version_lock_kind 只能是 cpu / gpu')
    return Path(work)/('inference_runtime_versions.gpu.json' if kind=='gpu' else 'inference_runtime_versions.json')


def ensure_inference_dependencies(work,install=True,kind=None):
    """Library versions are bound to the record directory, per kind of runtime (version_record).
    Module code hashes are part of each cache identity instead."""
    path=version_record(work,kind)
    old=json.loads(path.read_text()) if path.exists() else None
    before=runtime_versions()
    if install and (old or before['transformers'] is None or before['accelerate'] is None):
        req=[f'{p}=={old[p]}' for p in ('transformers','accelerate') if old.get(p)] if old else ['transformers','accelerate']
        if req:subprocess.check_call([sys.executable,'-m','pip','install','-q',*req])
    actual=runtime_versions()
    if any(actual[p] is None for p in ('transformers','accelerate')):raise RuntimeError('请先安装 transformers 和 accelerate')
    for p in ('torch','numpy','safetensors','huggingface_hub'):
        if before[p] is not None and actual[p]!=before[p]:
            raise RuntimeError(f'安装推理依赖改变了 {p}；重启内核并核对融合环境版本后再运行，不能沿用旧的已导入模块')
    changed=[p for p in LIBRARIES if old and old.get(p) and old[p]!=actual[p]]
    if changed:raise RuntimeError(f'推理依赖版本与记录（{path.name}）不一致：'+'，'.join(f'{p} {old[p]} -> {actual[p]}' for p in changed)+'；恢复版本或更换独立记录目录')
    if not old:core.atomic_json(path,{p:actual[p] for p in LIBRARIES})
    return actual


def describe_runtime():
    """What this session runs on. The pipeline uses it to decide which stage belongs here."""
    import psutil
    info={'cores':os.cpu_count() or 1,'ram_gib':round(psutil.virtual_memory().total/1024**3,1),'gpu':None,'gpu_gib':0.}
    if torch.cuda.is_available():
        props=torch.cuda.get_device_properties(0)
        info.update(gpu=str(props.name),gpu_gib=round(props.total_memory/1024**3,1))
    info['kind']='gpu' if info['gpu'] else 'cpu'
    # A GPU machine whose torch cannot use the GPU must not be mistaken for a CPU runtime: it is billed as a GPU.
    info['gpu_unusable']=bool(not info['gpu'] and shutil.which('nvidia-smi'))
    return info


def cpu_gflops(repeats=3):
    """Measured float32 matrix-product speed of this CPU in GFLOPS (one 2048x5120 linear layer on 256 rows)."""
    weight=torch.randn(2048,5120);x=torch.randn(256,5120)
    torch.nn.functional.linear(x,weight);started=time.perf_counter()
    for _ in range(repeats):torch.nn.functional.linear(x,weight)
    return 2*256*2048*5120*repeats/max(time.perf_counter()-started,1e-9)/1e9


_TUNED=None


def tune_cpu_threads():
    """(threads, GFLOPS): the torch thread count with the fastest float32 matrix product on this CPU.

    The notebook asks for one thread per logical core. On a machine with hyper-threading, one per physical
    core is often faster for this workload, so both are measured once per session and the better one is kept.
    Fewer threads win unless more are clearly faster.
    """
    global _TUNED
    if _TUNED is None:
        logical=os.cpu_count() or 1;best=None
        for n in sorted({max(1,logical//2),logical,torch.get_num_threads()}):
            torch.set_num_threads(n);speed=cpu_gflops()
            if best is None or speed>best[1]*1.05:best=(n,speed)
        torch.set_num_threads(best[0]);_TUNED=best
    return _TUNED


_HASH_MEMO={}


def file_sha256(path,strict=False):
    """sha256 of a file. Within one session it is reused while the file's size and mtime are unchanged."""
    path=Path(path);stat=path.stat();key=str(path.resolve());known=_HASH_MEMO.get(key)
    if not strict and known is not None and known[0]==(stat.st_size,stat.st_mtime_ns):return known[1]
    value=core.sha_file(path);_HASH_MEMO[key]=((stat.st_size,stat.st_mtime_ns),value)
    return value


def remember_sha256(path,value):
    """Register a hash that was just computed or verified for exactly this file."""
    path=Path(path);stat=path.stat();_HASH_MEMO[str(path.resolve())]=((stat.st_size,stat.st_mtime_ns),value)


def _hub_sha256(engine,repo,name):
    """The sha256 the Hub recorded for an LFS file, or None when it gives none."""
    prefix=getattr(engine,'subfolders',{}).get(repo,'');remote=(prefix+'/' if prefix else '')+name
    for sibling in getattr(engine._infos.get(repo),'siblings',None) or []:
        if sibling.rfilename==remote:
            lfs=getattr(sibling,'lfs',None)
            value=lfs.get('sha256') if isinstance(lfs,dict) else getattr(lfs,'sha256',None)
            return value if isinstance(value,str) and len(value)==64 else None
    return None


def metadata_names(engine,repo):
    return [n for n in engine.files(repo) if n in METADATA_EXACT or n.startswith('chat_templates/')]


def source_identity(engine,repo):
    return {'repo':repo,'commit':engine.pin(repo),'subfolder':getattr(engine,'subfolders',{}).get(repo,'')}


def download_metadata(engine,repo,root):
    identity=source_identity(engine,repo);directory=Path(root)/('meta-'+core.digest(identity)[:24]);directory.mkdir(parents=True,exist_ok=True)
    for name in metadata_names(engine,repo):
        path=directory/safe_relative(name);path.parent.mkdir(parents=True,exist_ok=True)
        content=engine._small_bytes(repo,name)
        if not path.exists() or path.read_bytes()!=content:path.write_bytes(content)
    if not (directory/'config.json').exists() or not (directory/'tokenizer_config.json').exists():
        raise ValueError('源缺少配置或 tokenizer_config.json')
    return directory


def download_checkpoint(engine,repo,root,verify_hashes=True):
    """Only files referenced by the full-model index; no nested draft/quantized copies."""
    from huggingface_hub import hf_hub_download
    identity=source_identity(engine,repo);directory=Path(root)/('checkpoint-'+core.digest(identity)[:24]);directory.mkdir(parents=True,exist_ok=True)
    cat=engine._catalogue(repo)
    required=sorted({v['file'] for v in cat.values()}|set(metadata_names(engine,repo)))
    if 'model.safetensors.index.json' in engine.files(repo):required.append('model.safetensors.index.json')
    old=core.load_record(directory/'_download_receipt.json',identity)
    records=old or {};missing=[n for n in required if n not in records or not (directory/n).exists()]
    need=sum(engine._sizes[repo].get(n) or 0 for n in missing)+512*1024**2
    if shutil.disk_usage(directory).free<need:raise RuntimeError(f'下载 {repo} 的完整 BF16 权重磁盘不足；预计还需 {need/1024**3:.1f} GiB')
    prefix=identity['subfolder'].strip('/')
    for name in required:
        path=directory/safe_relative(name)
        if name in records and path.exists():
            rec=records[name]
            if path.stat().st_size!=rec['size'] or (verify_hashes and file_sha256(path)!=rec['sha256']):raise ValueError(f'已下载文件改变：{path}')
            remember_sha256(path,rec['sha256'])
            continue
        # Use a dedicated local_dir and move only the requested subfolder file.
        remote=(prefix+'/' if prefix else '')+name
        cache=directory/'_download_workspace'
        fetched=Path(hf_hub_download(repo,name if not prefix else remote,revision=identity['commit'],
                        token=engine.token or None,local_dir=str(cache)))
        path.parent.mkdir(parents=True,exist_ok=True);shutil.move(str(fetched),path)
        digest=core.sha_file(path);expected=_hub_sha256(engine,repo,name)
        if expected and digest!=expected:
            path.unlink();raise ValueError(f'下载的文件与 Hub 记录的 SHA-256 不符，已删除，请重跑：{repo}/{name}')
        remember_sha256(path,digest)
        records[name]={'size':path.stat().st_size,'sha256':digest,'hub_sha256_checked':bool(expected)}
        core.save_record(directory/'_download_receipt.json',identity,records)
        print('下载并锁定:',repo,name,flush=True)
    return directory


def checkpoint_catalogue(directory):
    directory=Path(directory).resolve()
    cfg=json.loads((directory/'config.json').read_text())
    if cfg.get('quantization_config') or cfg.get('compression_config'):raise ValueError('不允许用量化 checkpoint 偷换 BF16 比较')
    ip=directory/'model.safetensors.index.json'
    if ip.exists():mapping=json.loads(ip.read_text())['weight_map']
    elif (directory/'model.safetensors').exists():
        with safe_open(str(directory/'model.safetensors'),framework='pt',device='cpu') as f:mapping={k:'model.safetensors' for k in f.keys()}
    else:raise ValueError('缺少完整 safetensors 索引/文件')
    if not mapping:raise ValueError('空权重索引')
    result={}
    for filename in sorted(set(mapping.values())):
        safe_relative(filename);path=directory/filename
        with path.open('rb') as f:
            size=int.from_bytes(f.read(8),'little')
            if not 2<=size<=100*1024**2:raise ValueError('异常 safetensors 头部')
            header=json.loads(f.read(size))
        for key,fn in mapping.items():
            if fn!=filename:continue
            if key not in header:raise ValueError('索引引用了不存在的权重')
            info=header[key]
            if info['dtype'] not in DTYPES:raise ValueError('不支持/量化的权重 dtype: '+info['dtype'])
            if re.search(r'(^|\.)(qweight|qzeros|weight_scale|weight_scale_inv)(\.|$)',key):raise ValueError('发现量化权重')
            result[key]={'file':str(path),'dtype':info['dtype'],'shape':info['shape'],
                         'offset':8+size+int(info['data_offsets'][0]),'nbytes':int(info['data_offsets'][1])-int(info['data_offsets'][0])}
    return result


def safetensors_layout(path):
    """{tensor name: (byte offset in the file, byte length)} of a local safetensors file, from its header."""
    with open(path,'rb') as f:
        size=int.from_bytes(f.read(8),'little')
        if not 2<=size<=100*1024**2:raise ValueError('异常 safetensors 头部：'+str(path))
        header=json.loads(f.read(size))
    return {k:(8+size+int(v['data_offsets'][0]),int(v['data_offsets'][1])-int(v['data_offsets'][0])) for k,v in header.items() if k!='__metadata__'}


class ReadAhead:
    """Reads byte ranges of local files on a background thread, so that they are in the page cache by the time
    the main thread reads them (through safetensors, which holds the GIL while it copies). Only a hint: it
    changes no value, and an error just means no read-ahead."""
    def __init__(self,chunk=16*1024**2):
        import queue,threading
        self._queue=queue.Queue();self._chunk=int(chunk);self._stop=False
        self._thread=threading.Thread(target=self._work,name='read-ahead',daemon=True);self._thread.start()
    def request(self,ranges):
        ranges=[r for r in ranges if r and r[2]>0]
        if ranges and not self._stop:self._queue.put(ranges)
    def _work(self):
        while True:
            ranges=self._queue.get()
            if ranges is None:return
            for path,offset,length in ranges:
                if self._stop:break
                try:
                    fd=os.open(path,os.O_RDONLY)
                    try:
                        if hasattr(os,'posix_fadvise'):os.posix_fadvise(fd,offset,length,os.POSIX_FADV_WILLNEED)
                        end=offset+length
                        while offset<end and not self._stop:          # os.pread releases the GIL while it waits
                            n=len(os.pread(fd,min(self._chunk,end-offset),offset))
                            if not n:break
                            offset+=n
                    finally:os.close(fd)
                except Exception:pass                       # a hint only: never let it stop the thread
    def close(self):
        self._stop=True;self._queue.put(None);self._thread.join(timeout=120)


def aliases(name):
    out=[name]
    if name.startswith('model.language_model.'):
        out.extend(['model.'+name[len('model.language_model.'):], 'language_model.'+name[len('model.language_model.'):]])
    elif name.startswith('model.') and not any(x in name for x in ('visual','mtp')):out.append('model.language_model.'+name[len('model.'):])
    if name.startswith('language_model.'):
        out.extend(['model.'+name,'model.'+name[len('language_model.'):]])
    if name.startswith('model.mtp.'):out.append(name[len('model.'):])
    elif name.startswith('mtp.'):out.append('model.'+name)
    return list(dict.fromkeys(out))


def match_model_weights(model,catalogue):
    state=model.state_dict();mapping={}
    for name,value in state.items():
        hits=[k for k in aliases(name) if k in catalogue]
        if name in hits:key=name
        elif len(hits)==1:key=hits[0]
        elif not hits:
            # Only explicit tied embedding/head equivalence; no shape-based matching.
            cfg=getattr(model,'config',None)
            tied=bool(getattr(getattr(cfg,'text_config',cfg),'tie_word_embeddings',False))
            candidates=[k for k in catalogue if k.endswith('embed_tokens.weight')] if tied and name.endswith('lm_head.weight') else []
            if len(candidates)==1:key=candidates[0]
            else:raise ValueError(f'checkpoint 缺少模型所需权重 {name}；拒绝随机初始化/静默丢参数')
        else:raise ValueError('权重前缀映射不唯一：'+name)
        if tuple(catalogue[key]['shape'])!=tuple(value.shape):raise ValueError('权重维度与运行时架构不符：'+name)
        if value.is_floating_point()!=DTYPES[catalogue[key]['dtype']].is_floating_point:raise ValueError('权重浮点/整数角色不符：'+name)
        mapping[name]=key
    extras=set(catalogue)-set(mapping.values())
    # Some compatible runtimes omit official auxiliary MTP inference heads. Preserve
    # these in export, record them here; never permit missing/extra text trunk keys.
    allowed=[k for k in extras if re.search(r'(^|\.)(mtp|nextn)(\.|$)',k)]
    if extras-set(allowed):raise ValueError('运行时无法解释额外权重：'+str(sorted(extras-set(allowed))[:5]))
    return mapping,{'matched':len(mapping),'unused_mtp_keys':sorted(allowed)}


def effective_ram():
    import psutil
    available=psutil.virtual_memory().available
    try:
        val=Path('/sys/fs/cgroup/memory.max').read_text().strip()
        if val!='max':available=min(available,max(0,int(val)-int(Path('/sys/fs/cgroup/memory.current').read_text())))
    except (OSError,ValueError):pass
    return available


def make_empty_model(directory,options):
    from transformers import AutoConfig,AutoModelForCausalLM,AutoModelForImageTextToText
    from accelerate import init_empty_weights
    dtype_name=options.get('dtype','bfloat16')
    if dtype_name not in {'bfloat16','float32'}:raise ValueError('推理 dtype 仅支持 bfloat16/float32')
    dtype=getattr(torch,dtype_name)
    try:
        cfg=AutoConfig.from_pretrained(str(directory),local_files_only=True,trust_remote_code=False)
        if getattr(cfg,'quantization_config',None):raise ValueError('推理配置不能量化')
        loader=AutoModelForImageTextToText if getattr(cfg,'vision_config',None) is not None else AutoModelForCausalLM
        # Keep real CPU buffers, particularly nonpersistent rotary frequencies.
        with init_empty_weights(include_buffers=False):
            model=loader.from_config(cfg,torch_dtype=dtype,attn_implementation=options.get('attention','eager'),trust_remote_code=False)
    except (KeyError,ValueError,AttributeError) as exc:
        raise RuntimeError('当前 Transformers/模型配置未通过空模型预检。先在独立环境升级到支持此架构的版本并重启，不要绕过布局检查。') from exc
    if hasattr(model,'tie_weights'):model.tie_weights()
    return model


def preflight_checkpoint(engine,repo,metadata_dir,options):
    """Validate library architecture and exact keys from headers before full downloads."""
    model=make_empty_model(metadata_dir,options)
    try:
        _,report=match_model_weights(model,engine._catalogue(repo))
        return report
    finally:
        del model;gc.collect()



def weights_bytes(model,dtype):
    """Bytes the checkpoint-backed weights take in the inference dtype."""
    size=torch.empty((),dtype=dtype).element_size()
    return sum(v.numel()*(size if v.is_floating_point() else v.element_size()) for v in model.state_dict().values())


def choose_placement(model,options,dtype,device):
    """resident: every weight is loaded once and stays on the inference device.
    offload: accelerate reads each module's weights from the safetensors files for every forward pass."""
    choice=options.get('placement','auto')
    if choice not in {'auto','resident','offload'}:raise ValueError('placement 只能是 auto / resident / offload')
    if choice!='auto':return choice
    need=weights_bytes(model,dtype)
    if device=='cpu':
        return 'resident' if need*1.05+int(options.get('activation_reserve_gib',2)*1024**3)<=effective_ram()*.85 else 'offload'
    reserve=int(options.get('gpu_reserve_bytes',float(options.get('gpu_reserve_gib',4))*1024**3))
    return 'resident' if need+reserve<=torch.cuda.mem_get_info(torch.device(device))[0] else 'offload'


def _place_resident(model,mapping,cat,device):
    """Every tensor is read once and assigned; nothing is read again during inference."""
    from contextlib import ExitStack
    from accelerate.utils import set_module_tensor_to_device
    extra={'clear_cache':False} if 'clear_cache' in inspect.signature(set_module_tensor_to_device).parameters else {}
    with ExitStack() as stack:
        handles={}
        for name,key in mapping.items():
            path=cat[key]['file'];handle=handles.get(path)
            if handle is None:handle=handles[path]=stack.enter_context(safe_open(path,framework='pt',device='cpu'))
            # No dtype argument: the value is converted to the dtype the model created this tensor with,
            # the same rule load_state_dict applies.
            set_module_tensor_to_device(model,name,device,value=handle.get_tensor(key),**extra)
    if device!='cpu':model.to(torch.device(device))        # buffers that are not stored in the checkpoint
    return model


def _place_offloaded(model,mapping,cat,dtype,device,classes,options):
    from accelerate import dispatch_model
    # Preload decoder blocks: hybrid/recurrent layers sometimes use a submodule's
    # .weight directly instead of calling that submodule's forward.
    largest=max((p.numel()*torch.empty((),dtype=dtype).element_size() for p in model.parameters()),default=0)
    for module in model.modules():
        if type(module).__name__ in classes:
            largest=max(largest,sum(p.numel()*torch.empty((),dtype=dtype).element_size() for p in module.parameters()))
    estimate=largest*2+int(options.get('activation_reserve_gib',2)*1024**3)
    if device.startswith('cuda') and estimate>torch.cuda.mem_get_info(torch.device(device))[0]*.85:
        raise RuntimeError('最大预加载模块与激活预留超过可用 GPU 显存')
    if estimate>effective_ram()*.85:
        raise RuntimeError(f'最大预加载模块与激活预留约 {estimate/1024**3:.1f} GiB，当前 CPU RAM 不足；不是仅调小下载块就能解决')
    weights={name:{'safetensors_file':cat[key]['file'],'weight_name':key,
                  'dtype':str(dtype).split('.')[-1] if DTYPES[cat[key]['dtype']].is_floating_point else str(DTYPES[cat[key]['dtype']]).split('.')[-1]}
             for name,key in mapping.items()}
    # Load checkpoint-backed buffers too. Accelerate excludes nonpersistent
    # buffers from offloading, keeping initialized rotary frequencies real.
    return dispatch_model(model,device_map={'':'disk'},main_device=device,offload_index=weights,
                          offload_buffers=True,force_hooks=True,preload_module_classes=classes)


@contextmanager
def load_model(directory,options):
    directory=Path(directory).resolve();device=str(options.get('device','cpu'))
    if device!='cpu' and not (device.startswith('cuda') and torch.cuda.is_available()):raise ValueError('所选推理设备不可用')
    # A model loaded before this one is only released once its caller has returned. Collect it and hand the
    # cached blocks back to the driver now, or the free memory measured below would still count it as taken.
    gc.collect()
    if torch.cuda.is_available():torch.cuda.empty_cache()
    cat=checkpoint_catalogue(directory);dtype=getattr(torch,options.get('dtype','bfloat16'))
    model=make_empty_model(directory,options)
    mapping,report=match_model_weights(model,cat)
    classes=sorted(set(getattr(model,'_no_split_modules',None) or [])|set(options.get('preload_module_classes',[])))
    try:
        mode=choose_placement(model,options,dtype,device);size=weights_bytes(model,dtype)/1024**3;started=time.perf_counter()
        model=_place_resident(model,mapping,cat,device) if mode=='resident' else _place_offloaded(model,mapping,cat,dtype,device,classes,options)
        model.eval()
        for p in model.parameters():p.requires_grad_(False)
        model._qmerge_loading_report={**report,'placement':mode,'weights_gib':round(size,2)}
        print(f'实际推理: {device} {options.get("dtype","bfloat16")}；匹配权重 {report["matched"]}；权重 {size:.1f} GiB，'
              +('常驻' if mode=='resident' else '每次前向按模块从磁盘换入')+f'（装载 {time.perf_counter()-started:.0f}s）；不加载 DFlash draft',flush=True)
        yield model
    finally:
        del model;gc.collect()
        if torch.cuda.is_available():torch.cuda.empty_cache()


def load_tokenizer(directory):
    from transformers import AutoTokenizer
    tok=AutoTokenizer.from_pretrained(str(directory),local_files_only=True,trust_remote_code=False)
    if not tok.chat_template:raise ValueError('缺少明确的 chat_template')
    return tok


@torch.inference_mode()
def score_forward(model,ids,prompt_length,device='cpu',token_chunk=32):
    ids=ids.to(device);kwargs={'input_ids':ids,'attention_mask':torch.ones_like(ids),'use_cache':False,'return_dict':True}
    signature=inspect.signature(model.forward).parameters
    suffix=ids.shape[1]-prompt_length+1;start=0
    if 'logits_to_keep' in signature:kwargs['logits_to_keep']=suffix;start=prompt_length-1
    elif 'num_logits_to_keep' in signature:kwargs['num_logits_to_keep']=suffix;start=prompt_length-1
    result=model(**kwargs)
    loss,n=core.response_nll(result.logits,ids,prompt_length,token_chunk,start)
    del result,kwargs,ids
    return {'nll':loss,'tokens':n,'suffix_logits':start!=0}


@torch.inference_mode()
def score_forward_batch(model,items,device='cpu',token_chunk=32,pad_id=0):
    """Several (ids, prompt_length) pairs in one right-padded forward pass.

    The model is causal, so padding on the right cannot change the logits of earlier positions; each
    row is then scored by the same response_nll as a single sequence. Returns one result per item.
    """
    width=max(ids.shape[1] for ids,_ in items)
    batch=torch.full((len(items),width),int(pad_id),dtype=torch.long);mask=torch.zeros((len(items),width),dtype=torch.long)
    for r,(ids,_) in enumerate(items):
        n=ids.shape[1];batch[r,:n]=ids[0];mask[r,:n]=1
    first=min(p for _,p in items)-1                    # earliest position whose logits are needed
    kwargs={'input_ids':batch.to(device),'attention_mask':mask.to(device),'use_cache':False,'return_dict':True}
    signature=inspect.signature(model.forward).parameters;start=0
    if 'logits_to_keep' in signature:kwargs['logits_to_keep']=width-first;start=first
    elif 'num_logits_to_keep' in signature:kwargs['num_logits_to_keep']=width-first;start=first
    result=model(**kwargs);out=[]
    for r,(ids,p) in enumerate(items):
        loss,n=core.response_nll(result.logits[r:r+1],ids,p,token_chunk,start)
        out.append({'nll':loss,'tokens':n,'suffix_logits':start!=0})
    del result,kwargs,batch,mask
    return out


def plan_score_groups(lengths,batch_size,vocab,element_bytes,budget_gib=2.0):
    """Fixed batches for forward scoring. lengths: {id: (total_tokens, prompt_tokens)}.

    Longest first. A batch is closed when it is full, or when its block of logits
    (rows x positions from the earliest response to the end x vocabulary) would exceed the budget.
    The plan depends only on the dataset and the options, never on what happens to be cached.
    """
    order=sorted(lengths,key=lambda k:(-lengths[k][0],k));groups=[];group=[];limit=int(budget_gib*1024**3)
    def cost(keys):
        width=max(lengths[k][0] for k in keys);first=min(lengths[k][1] for k in keys)-1
        return len(keys)*(width-first)*vocab*element_bytes
    for key in order:
        if group and (len(group)>=batch_size or cost(group+[key])>limit):groups.append(group);group=[]
        group.append(key)
    if group:groups.append(group)
    return groups


def _text_config(config_dir):
    cfg=json.loads((Path(config_dir)/'config.json').read_text())
    return cfg.get('text_config',cfg)


def metadata_hashes(directory):
    directory=Path(directory)
    files={n:directory/n for n in METADATA_EXACT if (directory/n).is_file()}
    templates=directory/'chat_templates'
    if templates.is_dir():
        files.update({p.relative_to(directory).as_posix():p for p in templates.rglob('*') if p.is_file()})
    return {name:core.sha_file(path) for name,path in sorted(files.items())}



def checkpoint_fingerprint(directory,strict=False):
    """Content fingerprint of a checkpoint directory.

    Each weight file is hashed once per session (file_sha256); later calls reuse that while the file's
    size and mtime are unchanged. strict=True reads everything again.
    """
    directory=Path(directory).resolve();cat=checkpoint_catalogue(directory)
    hashes={str(Path(v['file']).relative_to(directory)):None for v in cat.values()}
    for name in hashes:hashes[name]=file_sha256(directory/name,strict)
    # Tailor patches may duplicate original tensor keys. The index selects
    # which copy is loaded, even when the set of referenced shards is unchanged.
    index=directory/'model.safetensors.index.json'
    if index.is_file():hashes[index.name]=core.sha_file(index)
    hashes.update(metadata_hashes(directory))
    return core.digest(hashes),hashes



def score_checkpoint(directory,rows,cache_root,model_identity,tokenizer_dir,options,model_loader=None,tokenizer=None):
    options={'device':'cpu','dtype':'bfloat16','max_tokens':1024,'max_response_tokens':512,
             'token_chunk':32,'template_kwargs':{'enable_thinking':False},'batch_size':1,'logits_budget_gib':2.0,**options}
    loader=model_loader or load_model
    fingerprint,_=checkpoint_fingerprint(Path(directory))
    identity={'kind':'response_nll','version':core.VERSION,'model':model_identity,'weights':fingerprint,
              'dataset':core.digest(rows),'options':options,'runtime':runtime_versions(),
              'tokenizer_files':metadata_hashes(tokenizer_dir)}
    root=Path(cache_root)/core.digest(identity);root.mkdir(parents=True,exist_ok=True)
    records={}
    for row in rows:
        value=core.load_record(root/(core.digest(row['id'])+'.json'),{'context':identity,'item':row['id']})
        if value is not None:records[row['id']]=value
    missing=[r for r in rows if r['id'] not in records]
    if missing:
        tok=tokenizer or load_tokenizer(tokenizer_dir);by_id={r['id']:r for r in rows};label=model_identity.get('tag','model')
        # Validate all token budgets before loading large weights. Every row is packed, cached or not,
        # so that the batches below are the same on every run.
        packed={r['id']:core.pack_reference(tok,r,options['template_kwargs'],options['max_tokens'],options['max_response_tokens']) for r in rows}
        for row in rows:
            ids,p,meta=packed[row['id']]
            prefix=tok.decode(ids[0,max(0,p-8):p].tolist(),skip_special_tokens=False)
            if prefix.rstrip().endswith('<think>') and '</think>' not in row['response']:
                raise ValueError('模板开启了推理但参考 continuation 未闭合思考标签；显式使用 enable_thinking=False 或提供完整已核验推理续写')
        size=max(1,int(options['batch_size']));element=4 if options['dtype']=='float32' else 2
        if size==1:groups=[[r['id']] for r in missing]
        else:
            plan=plan_score_groups({k:(v[0].shape[1],v[1]) for k,v in packed.items()},size,int(_text_config(tokenizer_dir).get('vocab_size',262144)),
                                   element,float(options['logits_budget_gib']))
            groups=[g for g in plan if any(k not in records for k in g)]
        total=sum(len(g) for g in groups)
        if options['device']=='cpu' and options.get('estimate_time',True):
            backbone=sum(math.prod(v['shape']) for k,v in checkpoint_catalogue(directory).items() if core.parameter_role(k,tuple(v['shape']))=='backbone')
            tokens=sum(packed[k][0].shape[1] for g in groups for k in g);threads,speed=tune_cpu_threads()
            print(f'前向评分 {label}：{total} 条、{tokens} token、{len(groups)} 批；{threads} 线程下 float32 约 {speed:.0f} GFLOPS，'
                  f'估计 {2*backbone*tokens/speed/1e9/60:.0f} 分钟（另加每批读一遍权重的时间）',flush=True)
        pad=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
        reserve=int(float(options['logits_budget_gib'])*1024**3)+2*1024**3
        with loader(directory,{**options,'gpu_reserve_bytes':reserve}) as model:
            done=0
            for group in groups:
                started=time.perf_counter()
                if len(group)==1:results=[score_forward(model,packed[group[0]][0],packed[group[0]][1],options['device'],options['token_chunk'])]
                else:results=score_forward_batch(model,[packed[k][:2] for k in group],options['device'],options['token_chunk'],0 if pad is None else pad)
                seconds=(time.perf_counter()-started)/len(group)
                for key,result in zip(group,results):
                    rec={'id':key,'category':by_id[key]['category'],**result,**packed[key][2],'seconds':seconds,'batch':len(group)}
                    core.save_record(root/(core.digest(key)+'.json'),{'context':identity,'item':key},rec)
                    records[key]=rec
                done+=len(group)
                print(f'前向评分 {label}: {done}/{total}，本批 {len(group)} 条，每条 {seconds:.1f}s，NLL={sum(r["nll"] for r in results)/len(results):.5f}',flush=True)
    if missing and checkpoint_fingerprint(Path(directory))[0]!=fingerprint:raise ValueError('评分期间 checkpoint 内容改变，拒绝缓存/选型')
    ordered=[records[r['id']] for r in rows]
    report={'context':identity,'records':ordered,'mean_nll':core.weighted_mean(rows,[r['nll'] for r in ordered],options.get('category_weights'))}
    core.atomic_json(root/'report.json',report)
    return report



# ---------------------------------------------------------------- layer-by-layer scoring
LAYERWISE_VERSION='layerwise-2026-10-12.1'
_VISION_NAME=re.compile(r'(^|\.)(visual|vision_tower|vision_model)(\.|$)')


class EngineUnsupported(RuntimeError):
    """The layer-by-layer engine does not fit this model or library version; the batched forward is used instead."""


class _StopForward(Exception):
    """Raised by the stand-in of the last decoder layer to end a forward pass there."""


def tensor_digest(t):
    """blake2b of dtype, shape and the raw bytes of a tensor (bfloat16 included), as hex."""
    import hashlib
    t=t.detach()
    if t.device.type!='cpu':t=t.cpu()
    t=t.contiguous()
    raw=t.view({1:torch.uint8,2:torch.int16,4:torch.int32,8:torch.int64}[t.element_size()]).numpy()
    h=hashlib.blake2b(digest_size=32);h.update(f'{t.dtype}|{tuple(t.shape)}|'.encode());h.update(raw)
    return h.hexdigest()


class CheckpointWeights:
    """The tensors of a checkpoint directory, one at a time (the files stay open while it is in use)."""
    def __init__(self,directory):
        from contextlib import ExitStack
        self.directory=Path(directory).resolve();self.catalogue=checkpoint_catalogue(self.directory)
        self._stack=ExitStack();self._handles={}
    def tensor(self,key):
        path=self.catalogue[key]['file'];handle=self._handles.get(path)
        if handle is None:handle=self._handles[path]=self._stack.enter_context(safe_open(path,framework='pt',device='cpu'))
        return handle.get_tensor(key)
    def ranges(self,keys):
        """(file, byte offset, byte length) of each of these tensors: read-ahead hints."""
        return [(self.catalogue[k]['file'],self.catalogue[k]['offset'],self.catalogue[k]['nbytes']) for k in keys if 'offset' in self.catalogue.get(k,{})]
    def close(self):
        self._handles={};self._stack.close()


class LayerwiseScorer:
    """Forward scoring of many sequences with every decoder layer's weights in memory once per model.

    The batched path reads all weights again for every batch (on a CPU with less RAM than the model they come
    from disk each time). Here the model's own forward still does everything outside the decoder layers, in
    three kinds of passes:
      trace   every decoder layer is replaced by a recorder that keeps the arguments it is called with (hidden
              states, rotary values, the mask of its layer type, ...) and passes the hidden states on; the last
              one ends the pass;
      layers  the real layers run one after the other, each over all sequences, with exactly those arguments;
              only the weights of the current layer (and of the parts outside the layers) are in memory;
      finish  the final hidden states are handed back to the model, whose own norm and output head give the
              logits; NLL as in score_forward.
    Sequences run one by one, without padding. Assumed, and checked as far as possible: the decoder layers are
    one ModuleList; each is called once per forward, in order, with the hidden states first; a layer's output
    depends only on its arguments and weights. The pipeline compares the result with the batched forward on
    the real model before using it (choose_scoring_engine).
    """
    def __init__(self,config_dir,options,catalogue):
        self.options=dict(options);self.device=str(options.get('device','cpu'))
        self.model=make_empty_model(config_dir,options)
        for p in self.model.parameters():p.requires_grad_(False)
        self.model.eval()
        cfg=_text_config(config_dir);self.num_layers=int(cfg['num_hidden_layers']);self.hidden_size=int(cfg['hidden_size'])
        found=[(n,m) for n,m in self.model.named_modules() if isinstance(m,torch.nn.ModuleList) and re.search(r'(^|\.)layers$',n)
               and not _VISION_NAME.search(n) and not re.search(r'(^|\.)(mtp|nextn)(\.|$)',n) and len(m)==self.num_layers]
        if len(found)!=1:raise EngineUnsupported(f'找不到唯一的解码层列表（{len(found)} 个候选）')
        self.prefix,self.layers=found[0]
        self.mapping,_=match_model_weights(self.model,catalogue)
        pattern=re.compile(re.escape(self.prefix)+r'\.(\d+)\.')
        self.units={i:[] for i in range(self.num_layers)};self.outer=[]
        for name in self.mapping:
            m=pattern.match(name)
            if m:self.units[int(m[1])].append(name)
            elif not _VISION_NAME.search(name):self.outer.append(name)     # the vision tower is never called for text
        if any(not names for names in self.units.values()):raise EngineUnsupported('有解码层没有任何权重')
        from accelerate.utils import set_module_tensor_to_device
        self._set=set_module_tensor_to_device
        self._extra={'clear_cache':False} if 'clear_cache' in inspect.signature(set_module_tensor_to_device).parameters else {}
        self.tuple_output=None

    # -- weights
    def _load(self,names,weights,digests):
        for name in names:
            key=self.mapping[name];value=weights.tensor(key)
            if digests is not None:digests[key]=tensor_digest(value)
            self._set(self.model,name,self.device,value=value,**self._extra)
            del value

    def _release(self,names):
        for name in names:self._set(self.model,name,'meta',**self._extra)

    # -- the stand-ins for the decoder layers
    def _stubbed(self,make):
        from contextlib import contextmanager
        @contextmanager
        def scope():
            saved=[layer.__dict__.get('forward') for layer in self.layers]
            for i,layer in enumerate(self.layers):layer.forward=make(i)
            try:yield
            finally:
                for layer,old in zip(self.layers,saved):
                    if old is None:layer.__dict__.pop('forward',None)
                    else:layer.forward=old
        return scope()

    def _wrap(self,hidden):
        return (hidden,) if self.tuple_output else hidden

    def _forward_kwargs(self,ids):
        return {'input_ids':ids,'attention_mask':torch.ones_like(ids),'use_cache':False,'return_dict':True}

    def _trace(self,ids):
        """The arguments of every decoder layer for one sequence: [(positional rest, keyword args, hidden as keyword)]."""
        last=self.num_layers-1
        for tuple_output in ([False,True] if self.tuple_output is None else [self.tuple_output]):
            calls=[];args_of=[None]*self.num_layers;shapes=[]
            def make(i):
                def stub(*args,**kwargs):
                    calls.append(i);kwargs=dict(kwargs)
                    if args:hidden,rest,named=args[0],tuple(args[1:]),False
                    else:hidden,rest,named=kwargs.pop('hidden_states'),(),True
                    args_of[i]=(rest,kwargs,named,hidden);shapes.append(tuple(hidden.shape))
                    if i==last:raise _StopForward()
                    return (hidden,) if tuple_output else hidden
                return stub
            with self._stubbed(make):
                try:self.model(**self._forward_kwargs(ids))
                except _StopForward:pass
            expected=(1,ids.shape[1],self.hidden_size)
            if calls==list(range(self.num_layers)) and all(s==expected for s in shapes):
                first=args_of[0][3]
                if first.device.type=='meta' or not first.is_floating_point():raise EngineUnsupported('第一层的输入不是实际的浮点张量')
                if self.tuple_output is None:self.tuple_output=tuple_output
                return [a[:3] for a in args_of],first
        raise EngineUnsupported('解码层的调用方式不是“每层一次、按顺序、隐状态在前”')

    def _layer(self,i,call,hidden):
        rest,kwargs,named=call
        out=self.layers[i](*rest,hidden_states=hidden,**kwargs) if named else self.layers[i](hidden,*rest,**kwargs)
        out=out[0] if isinstance(out,(tuple,list)) else out
        if tuple(out.shape)!=tuple(hidden.shape):raise EngineUnsupported('解码层输出的形状与输入不同')
        return out

    def _finish(self,hidden,ids,prompt_length):
        last=self.num_layers-1
        def make(i):
            def stub(*args,**kwargs):
                if i==last:return self._wrap(hidden)
                return self._wrap(args[0] if args else kwargs['hidden_states'])
            return stub
        with self._stubbed(make):
            return score_forward(self.model,ids,prompt_length,self.device,self.options.get('token_chunk',32))

    @torch.inference_mode()
    def run(self,weights,items,label='model',memo=None,digests=None,progress=True):
        """items: [(key, ids [1, L], prompt_length)]. Returns {key: {'nll', 'tokens'}}.

        digests (a dict) receives the digest of every tensor used; the layers resumed from the memo bring the
        digests stored with their hidden states. While a layer runs, the files behind the next one are read
        ahead (weights.ranges) on a background thread.
        """
        started=time.perf_counter();ids={k:x.to(self.device) for k,x,_ in items}
        ahead=ReadAhead() if hasattr(weights,'ranges') else None
        def hint(i):
            if ahead is not None and i<self.num_layers:
                try:ahead.request(weights.ranges([self.mapping[n] for n in self.units[i]]))
                except Exception:pass                       # only a hint
        layer_digests={} if digests is not None else None
        try:
            self._load(self.outer,weights,digests)
            start=0;saved=None
            if memo is not None:
                resumed=memo.resume(self.device,need_digests=layer_digests is not None)
                if resumed is not None:
                    start,saved,saved_digests=resumed
                    if layer_digests is not None:layer_digests.update(saved_digests)
            hint(start)
            traces={};states={}
            for key,x,_ in items:traces[key],states[key]=self._trace(ids[key])
            if saved is not None:
                if set(saved)!=set(states):raise ValueError('分层评分缓存的序列与本次不符')
                states=saved
                if progress:print(f'  分层评分 {label}：前 {start} 层沿用已算好的结果',flush=True)
            for i in range(start,self.num_layers):
                self._load(self.units[i],weights,layer_digests)
                hint(i+1)
                try:
                    for key in states:states[key]=self._layer(i,traces[key][i],states[key])
                finally:self._release(self.units[i])
                if memo is not None:memo.after_layer(i,states,layer_digests)
                done=i+1-start;total=self.num_layers-start
                if progress and (done in (1,4) or done%8==0 or done==total):
                    elapsed=time.perf_counter()-started
                    print(f'  分层评分 {label}：第 {i+1}/{self.num_layers} 层，已用 {elapsed/60:.1f} 分钟，'
                          f'估计还需 {elapsed/done*(total-done)/60:.0f} 分钟',flush=True)
            results={}
            for key,x,p in items:
                results[key]=self._finish(states[key],ids[key],p)
            if digests is not None:digests.update(layer_digests)
            return results
        finally:
            if ahead is not None:ahead.close()
            self._release(self.outer)


class LayerMemo:
    """Hidden states after chosen layers, kept on the local disk for later runs whose first layers are the same.

    keys: {layer index: key}. A key must name everything the hidden states after that layer depend on (data,
    options, library versions, and the weights of every layer up to it); the pipeline builds it.
    It only saves time, so it never takes the disk the stage needs: the oldest files go first, a file is
    written only while `reserve` bytes stay free, and a failed write turns it off for the run.
    """
    def __init__(self,directory,keys,limit=24,reserve=8*1024**3):
        self.directory=Path(directory);self.keys=dict(keys);self.limit=int(limit);self.reserve=int(reserve);self.disabled=False
    def _path(self,key):return self.directory/(key[:40]+'.safetensors')
    def resume(self,device,need_digests=False):
        """(layers done, {sequence: hidden states}, {tensor: digest}) from the deepest stored layer, or None.
        need_digests: only states that were stored with the digests of their weights."""
        for i in sorted(self.keys,reverse=True):
            path=self._path(self.keys[i])
            if path.is_file():
                try:
                    with safe_open(str(path),framework='pt',device='cpu') as f:
                        meta=f.metadata() or {}
                        if meta.get('key')!=self.keys[i] or (need_digests and not meta.get('digests')):continue
                        names=json.loads(meta['names'])
                        return i+1,{names[k]:f.get_tensor(k).to(device) for k in f.keys()},json.loads(meta.get('digests') or '{}')
                except (OSError,ValueError,KeyError):continue
        return None
    def after_layer(self,i,states,digests=None):
        """Keep the hidden states after layer i (if i is a chosen layer), with the digests of the weights of the
        layers up to it when given."""
        if self.disabled or i not in self.keys:return
        from safetensors.torch import save_file
        path=self._path(self.keys[i])
        if path.is_file():return
        self.directory.mkdir(parents=True,exist_ok=True)
        size=sum(t.numel()*t.element_size() for t in states.values())+(1<<20)
        old=sorted(self.directory.glob('*.safetensors'),key=lambda p:p.stat().st_mtime)
        while old and (len(old)>=self.limit or shutil.disk_usage(self.directory).free<size+self.reserve):
            old.pop(0).unlink(missing_ok=True)
        if shutil.disk_usage(self.directory).free<size+self.reserve:return
        names={f'h{n}':key for n,key in enumerate(sorted(states))}
        tmp=path.with_name(path.name+'.tmp')
        try:
            meta={'key':self.keys[i],'names':json.dumps(names,ensure_ascii=False)}
            if digests:meta['digests']=json.dumps(digests,sort_keys=True)
            save_file({k:states[key].detach().to('cpu').contiguous() for k,key in names.items()},str(tmp),metadata=meta)
            tmp.replace(path)
        except OSError as error:
            tmp.unlink(missing_ok=True);self.disabled=True
            print(f'注意：分层评分的中间隐状态写不进本地盘（{error}），本次运行不再保存；只影响速度，不影响结果',flush=True)


def score_layerwise(scorer,weights,rows,cache_root,model_identity,weights_identity,tokenizer_dir,options,memo=None,
                    want_digests=False,label=None):
    """score_checkpoint with the layer-by-layer engine; the report has the same shape. Cached under its identity."""
    options={'device':'cpu','dtype':'bfloat16','max_tokens':1024,'max_response_tokens':512,
             'token_chunk':32,'template_kwargs':{'enable_thinking':False},'batch_size':1,'logits_budget_gib':2.0,**options}
    identity={'kind':'response_nll_layerwise','version':core.VERSION,'engine':LAYERWISE_VERSION,'model':model_identity,
              'weights':weights_identity,'dataset':core.digest(rows),'options':options,'runtime':runtime_versions(),
              'tokenizer_files':metadata_hashes(tokenizer_dir),'digests':bool(want_digests)}
    root=Path(cache_root)/core.digest(identity);root.mkdir(parents=True,exist_ok=True)
    try:cached=core.load_record(root/'report.record.json',identity)
    except ValueError:cached=None
    if cached is not None:return cached
    label=label or model_identity.get('tag','model')
    tok=load_tokenizer(tokenizer_dir)
    packed={r['id']:core.pack_reference(tok,r,options['template_kwargs'],options['max_tokens'],options['max_response_tokens']) for r in rows}
    for row in rows:
        ids,p,_=packed[row['id']]
        prefix=tok.decode(ids[0,max(0,p-8):p].tolist(),skip_special_tokens=False)
        if prefix.rstrip().endswith('<think>') and '</think>' not in row['response']:
            raise ValueError('模板开启了推理但参考 continuation 未闭合思考标签；显式使用 enable_thinking=False 或提供完整已核验推理续写')
    tokens=sum(packed[r['id']][0].shape[1] for r in rows)
    if options['device']=='cpu' and options.get('estimate_time',True):
        threads,speed=tune_cpu_threads()
        params=sum(math.prod(v['shape']) for k,v in weights_catalogue(weights).items() if core.parameter_role(k,tuple(v['shape']))=='backbone')
        print(f'分层评分 {label}：{len(rows)} 条、{tokens} token（不填充）；{threads} 线程下 float32 约 {speed:.0f} GFLOPS，'
              f'估计 {2*params*tokens/speed/1e9/60:.0f} 分钟（另加读一遍权重的时间）',flush=True)
    digests={} if want_digests else None;started=time.perf_counter()
    results=scorer.run(weights,[(r['id'],packed[r['id']][0],packed[r['id']][1]) for r in rows],label,memo,digests)
    seconds=time.perf_counter()-started
    records=[{'id':r['id'],'category':r['category'],**results[r['id']],**packed[r['id']][2],'seconds':seconds/len(rows),'engine':'layerwise'} for r in rows]
    report={'context':identity,'records':records,'mean_nll':core.weighted_mean(rows,[x['nll'] for x in records],options.get('category_weights')),
            'tensor_digests':digests,'weights_digest':core.digest(digests) if digests else None,'seconds':seconds}
    core.save_record(root/'report.record.json',identity,report)
    print(f'分层评分 {label}：完成，平均 NLL={report["mean_nll"]:.5f}，用时 {seconds/60:.1f} 分钟',flush=True)
    return report


def weights_catalogue(weights):
    """Shapes of the tensors behind a weights object (a checkpoint, or a recipe over the subject's catalogue)."""
    return getattr(weights,'catalogue',None) or {}



class GenerationBudgetExceeded(RuntimeError):
    """A wall-clock limit set by the caller was reached. Finished batches are already saved."""


def stop_token_ids(tokenizer,config_dir):
    """The tokenizer's EOS plus the stop ids listed in the model's own generation_config.json."""
    ids=[] if tokenizer.eos_token_id is None else [int(tokenizer.eos_token_id)]
    try:
        extra=json.loads((Path(config_dir)/'generation_config.json').read_text()).get('eos_token_id')
        for value in (extra if isinstance(extra,list) else [extra]):
            if isinstance(value,int) and not isinstance(value,bool) and value not in ids:ids.append(value)
    except (OSError,ValueError):pass
    if not ids:raise ValueError('共同 tokenizer 缺少 EOS')
    return ids


def generation_cache_bytes(config_dir,batch,tokens,element_bytes=2):
    """Rough size of the decoding state of `batch` sequences of `tokens` tokens: the float32 recurrent
    states of the linear-attention layers plus the key/value cache of the full-attention layers."""
    tc=_text_config(config_dir)
    kinds=tc.get('layer_types') or ['full_attention']*int(tc['num_hidden_layers'])
    linear=sum(k=='linear_attention' for k in kinds);full=len(kinds)-linear
    state=linear*int(tc.get('linear_num_value_heads') or 0)*int(tc.get('linear_key_head_dim') or 0)*int(tc.get('linear_value_head_dim') or 0)*4
    heads=int(tc.get('num_key_value_heads') or tc.get('num_attention_heads') or 1)
    head_dim=int(tc.get('head_dim') or int(tc['hidden_size'])//max(1,int(tc.get('num_attention_heads') or 1)))
    return int(batch)*(state+full*2*heads*head_dim*element_bytes*int(tokens))


def prefill_pair_bytes(config_dir):
    """Bytes one full-attention layer needs per row and per (query, key) pair of positions while the prompts
    are read in one pass. With eager attention the scores, the masked scores and the float32 softmax of
    every head are alive together: at most 10 bytes per head in bfloat16 and 8 in float32; 12 is used."""
    return int(_text_config(config_dir).get('num_attention_heads') or 1)*12


def plan_generation_groups(lengths,batch_size,order_keys=None,pair_bytes=0,budget_bytes=0):
    """Fixed batches for generation: rows of the same category and similar prompt length together.

    lengths: {id: prompt_tokens}; order_keys: {id: category}. A batch runs until its longest answer ends,
    so mixing short-answer and long-answer categories would make every batch as slow as the slowest item.
    A batch is closed when it is full, or (pair_bytes > 0) when reading its prompts would need more than
    budget_bytes in one attention layer: rows x longest prompt squared x pair_bytes. The plan depends only
    on the dataset and the options.
    """
    order_keys=order_keys or {}
    order=sorted(lengths,key=lambda k:(str(order_keys.get(k,'')),lengths[k],k));groups=[];group=[]
    for key in order:
        wide=pair_bytes and (len(group)+1)*max(lengths[k] for k in group+[key])**2*pair_bytes>budget_bytes
        if group and (len(group)>=batch_size or wide):groups.append(group);group=[]
        group.append(key)
    if group:groups.append(group)
    return groups


def generation_margin(capacity):
    """GPU memory kept free besides weights and decoding state: CUDA context, cuBLAS workspace, logits, and the
    blocks the caching allocator holds while the key/value cache grows."""
    return int(0.04*capacity)+int(1.5*1024**3)


def checkpoint_weight_bytes(directory,dtype):
    """Bytes the checkpoint's weights take once loaded in `dtype` (integer tensors keep their own size)."""
    size=torch.empty((),dtype=dtype).element_size()
    return sum(math.prod(v['shape'])*(size if DTYPES[v['dtype']].is_floating_point else torch.empty((),dtype=DTYPES[v['dtype']]).element_size())
               for v in checkpoint_catalogue(directory).values())


def generation_batch_need(config_dir,options,capacity,weights_bytes):
    """need(rows, longest prompt): GPU bytes a batch takes. It fits when weights + 1.1 x the decoding state of all
    its rows at the end (longest prompt + max_new_tokens) + a margin <= capacity, and reading its prompts (the
    attention scores of the longest prompt) fits as well."""
    element=4 if options['dtype']=='float32' else 2;new=int(options['max_new_tokens'])
    pair=prefill_pair_bytes(config_dir);margin=generation_margin(capacity)
    state=generation_cache_bytes(config_dir,1,0,element);token=generation_cache_bytes(config_dir,1,1,element)-state
    def need(rows,longest):
        final=rows*(state+token*(longest+new))*1.10
        prefill=rows*(state+token*longest)+rows*longest*longest*pair
        return weights_bytes+max(final,prefill)+margin
    return need


def plan_generation_auto(lengths,order_keys,config_dir,options,capacity,weights_bytes):
    """batch_size='auto': as few batches as the GPU holds, in the same (category, prompt length) order as the
    fixed plan, and as equal in size as the prompt lengths allow.

    A batch runs until its longest answer ends, and on a GPU that holds the whole model a decoding step costs
    nearly the same for 48 rows as for 32 (the weights are read once per step either way), so fewer, larger
    batches finish sooner. The batches are consecutive runs of that order: the fewest is found by filling each
    batch as far as it fits (generation_batch_need); then the smallest row cap that keeps that number evens
    them out, so a few long prompts make only their own batch smaller. The plan depends only on the dataset,
    the options and the GPU's size. Returns (groups, bytes beyond the weights the largest batch needs).
    """
    order=sorted(lengths,key=lambda k:(str(order_keys.get(k,'')),lengths[k],k))
    need=generation_batch_need(config_dir,options,capacity,weights_bytes)
    def split(cap):
        groups=[];i=0
        while i<len(order):
            j=i+1;longest=lengths[order[i]]
            while j<len(order) and j-i<cap:
                widest=max(longest,lengths[order[j]])
                if need(j-i+1,widest)>capacity:break
                longest=widest;j+=1
            groups.append(order[i:j]);i=j
        return groups
    if not order:return [],0
    fewest=len(split(len(order)))
    for cap in range(-(-len(order)//fewest),len(order)+1):
        groups=split(cap)
        if len(groups)<=fewest:break
    return groups,max(need(len(g),max(lengths[k] for k in g)) for g in groups)-weights_bytes


class _ProjectedOutOfMemory(RuntimeError):
    """The decoding state of this batch, extrapolated to max_new_tokens, would not fit: split it now, not at the end."""
    def __init__(self,rows,projected,capacity):
        super().__init__(f'projected GPU memory {projected/1024**3:.1f} GiB of {capacity/1024**3:.1f} GiB; {rows} rows fit')
        self.rows=rows


_BATCH_CAP={}      # device -> rows per generate() call that are still tried after an out-of-memory error in this session


def _out_of_memory(error):
    """An allocation failure of the inference device. The exception class differs between torch versions."""
    kinds=tuple(k for k in (getattr(torch,'OutOfMemoryError',None),getattr(getattr(torch,'cuda',None),'OutOfMemoryError',None)) if isinstance(k,type))
    return bool(kinds and isinstance(error,kinds)) or 'out of memory' in str(error).lower()


def _gpu_peak(device,reset=False):
    """Text for progress lines: the peak of allocated GPU memory since the last reset; '' when unknown."""
    if device=='cpu' or not torch.cuda.is_available():return ''
    try:
        if reset:
            torch.cuda.reset_peak_memory_stats(torch.device(device));return ''
        return f'，显存峰值 {torch.cuda.max_memory_allocated(torch.device(device))/1024**3:.1f} GiB'
    except Exception:return ''


def _generation_monitor(label,rows,stop_ids,limits,guard=None):
    """Progress lines and the wall-clock limit; generate() calls it once per decoding step.

    guard (GPU only): {'device', 'capacity', 'base', 'max_new'}. The memory in use grows by the same amount every
    step (the key/value cache); after a few dozen steps that growth is extrapolated to max_new_tokens. If the batch
    would not fit at its end, it is stopped now and split (_ProjectedOutOfMemory), instead of running out of memory
    near its end with most of its time already spent.
    """
    try:from transformers.generation.streamers import BaseStreamer
    except Exception:return None
    stops=set(stop_ids);deadline=limits.get('deadline');rate=float(limits.get('units_per_hour') or 0)
    if guard is not None and rows>1:
        first=max(2,min(32,guard['max_new']//4));second=max(first+1,min(128,guard['max_new']//2))
    else:guard=None
    class Monitor(BaseStreamer):
        def __init__(self):
            self.step=-1;self.started=time.perf_counter();self.done=[False]*rows;self.mark=None
        def watch(self):
            used=torch.cuda.memory_allocated(torch.device(guard['device']))
            if self.step==first:self.mark=(self.step,used);return
            if self.mark is None or not (self.step==second or (self.step>second and self.step%256==0)):return
            slope=max(0.,(used-self.mark[1])/(self.step-self.mark[0]))
            projected=used+slope*max(0,guard['max_new']-self.step)
            margin=guard.get('margin');limit=guard['capacity']-(generation_margin(guard['capacity']) if margin is None else int(margin))
            if projected>limit:
                per_row=max(1.,(projected-guard['base'])/rows)
                fit=int(max(1,min(rows-1,(limit-guard['base'])//per_row)))
                raise _ProjectedOutOfMemory(fit,projected,guard['capacity'])
        def put(self,value):
            self.step+=1
            if self.step==0:return                      # the prompt itself
            for r,t in enumerate(value.reshape(-1).tolist()[:rows]):
                if int(t) in stops:self.done[r]=True
            if deadline is not None and time.time()>deadline:
                raise GenerationBudgetExceeded(f'{label}: 到达设定的时间上限，在第 {self.step} 步中止本批（已完成的批次已保存）')
            if guard is not None and not all(self.done):
                try:self.watch()
                except _ProjectedOutOfMemory:raise
                except Exception:pass                   # the guard is a convenience: never let it break generation
            if self.step in (20,100) or self.step%500==0:
                elapsed=time.perf_counter()-self.started
                cost=f'，本批已用约 {elapsed/3600*rate:.2f} 单元' if rate else ''
                print(f'  {label}: 第 {self.step} 步，{elapsed/self.step:.2f}s/步，已结束 {sum(self.done)}/{rows}{cost}',flush=True)
        def end(self):pass
    return Monitor()


@torch.inference_mode()
def generate_batch(model,id_lists,gc_config,device,pad_id,seed,streamer=None):
    """Greedy generation for several prompts at once (left padded). Returns the new tokens of each row."""
    width=max(len(x) for x in id_lists)
    ids=torch.full((len(id_lists),width),int(pad_id),dtype=torch.long);mask=torch.zeros((len(id_lists),width),dtype=torch.long)
    for r,x in enumerate(id_lists):
        ids[r,width-len(x):]=torch.tensor(x,dtype=torch.long);mask[r,width-len(x):]=1
    torch.manual_seed(seed)
    # use_cache / do_sample / num_beams are passed as arguments as well: arguments outrank every config. The model
    # config of this family says use_cache=false, and some transformers versions let such model-derived defaults
    # replace values of a passed generation_config that equal the library defaults.
    kwargs={'input_ids':ids.to(device),'attention_mask':mask.to(device),'generation_config':gc_config,
            'use_cache':True,'do_sample':False,'num_beams':1}
    if streamer is not None:kwargs['streamer']=streamer
    out=model.generate(**kwargs)
    new=out[:,width:].cpu()
    return [[int(t) for t in new[r].tolist()] for r in range(len(id_lists))]


def decoding_feeds_one_token(model,ids,gc_config,device,pad_id):
    """Whether generate() feeds only the newest token after the first step; None when it cannot be observed.

    Without the decoding cache every step would run the whole sequence again: the same output, orders of
    magnitude slower. Three forced steps on one prompt show which it is.
    """
    if not hasattr(model,'register_forward_pre_hook'):return None
    import copy
    widths=[]
    def record(module,args,kwargs):
        value=kwargs.get('input_ids')
        if value is None:value=kwargs.get('inputs_embeds')
        if value is not None:widths.append(int(value.shape[1]))
    probe=copy.deepcopy(gc_config);probe.max_new_tokens=3;probe.min_new_tokens=3
    handle=model.register_forward_pre_hook(record,with_kwargs=True)
    try:generate_batch(model,[list(ids)],probe,device,pad_id,0)
    finally:handle.remove()
    if len(widths)<3:return None
    return all(w==1 for w in widths[1:])


def _auto_plan(cache_root,rows,inputs,lengths,categories,config_dir,options,capacity,weights_bytes):
    """The 'auto' batches of this dataset, kept in the phase directory so that S and the candidate, in any later
    session, use the same batches (a different GPU size would otherwise give different ones). Returns
    (groups, bytes to keep free for the decoding state)."""
    if capacity is None:                                   # CPU (allow_slow): small fixed batches
        groups=plan_generation_groups(lengths,8,categories)
        return groups,0
    key=core.digest({'dataset':core.digest(rows),'options':options,'inputs':core.digest(inputs)})
    path=Path(cache_root)/('generation_plan-'+key[:24]+'.json')
    groups,reserve=plan_generation_auto(lengths,categories,config_dir,options,capacity,weights_bytes)
    try:saved=core.load_record(path,{'kind':'generation_plan','key':key})
    except (ValueError,OSError):saved=None
    if saved is not None and sorted(k for g in saved['groups'] for k in g)==sorted(lengths):
        fits=generation_batch_need(config_dir,options,capacity,weights_bytes)
        need=max(fits(len(g),max(lengths[k] for k in g)) for g in saved['groups'])
        if need<=capacity:return saved['groups'],max(reserve,need-weights_bytes)
        print(f'注意：这份数据先前在更大的 GPU 上排的批次（每批最多 {max(len(g) for g in saved["groups"])} 条）放不下，改为每批最多 '
              f'{max(len(g) for g in groups)} 条；S 和候选的批次组成可能因此不同。',flush=True)
    core.save_record(path,{'kind':'generation_plan','key':key},{'groups':groups,'capacity':capacity,'sizes':[len(g) for g in groups]})
    print(f'生成批次（batch_size=auto）：{len(groups)} 批，每批 {"/".join(str(len(g)) for g in groups)} 条'
          f'（GPU {capacity/1024**3:.0f} GiB，权重 {weights_bytes/1024**3:.1f} GiB，最长提示 {max(lengths.values())} token）',flush=True)
    return groups,reserve


def generate_checkpoint(directory,rows,cache_root,model_identity,tokenizer_dir,options,manual_scores_file='',model_loader=None,limits=None):
    """limits is not part of the cache identity: deadline (epoch seconds), units_per_hour (for progress lines),
    allow_slow (permit CPU generation, or a GPU that cannot hold the whole model)."""
    from transformers import GenerationConfig
    options={'device':'cpu','dtype':'bfloat16','max_input_tokens':2048,'max_new_tokens':4096,
             'template_kwargs':{'enable_thinking':True},'seed':42,'batch_size':1,'prefill_budget_gib':8.0,**options}
    loader=model_loader or load_model;limits=dict(limits or {});label=model_identity.get('tag','model');device=options['device']
    fingerprint,_=checkpoint_fingerprint(Path(directory))
    identity={'kind':'generation','version':core.VERSION,'model':model_identity,'weights':fingerprint,
              'dataset':core.digest(rows),'options':options,'runtime':runtime_versions(),
              'tokenizer_files':metadata_hashes(tokenizer_dir)}
    root=Path(cache_root)/core.digest(identity);root.mkdir(parents=True,exist_ok=True)
    records={r['id']:core.load_record(root/(core.digest(r['id'])+'.json'),{'context':identity,'item':r['id']}) for r in rows}
    missing=[r for r in rows if records[r['id']] is None];cached_decoding=None
    if missing:
        if options['device']=='cpu' and not limits.get('allow_slow'):
            raise RuntimeError('生成评测需要 GPU：27B 模型在 CPU 上每个 token 要几十秒。换到 GPU 运行时再运行；'
                               '确实要在 CPU 上跑（例如很小的模型），把 generation_limits 的 allow_slow 设为 True')
        tok=load_tokenizer(tokenizer_dir);stops=stop_token_ids(tok,tokenizer_dir)
        pad=tok.pad_token_id if tok.pad_token_id is not None else stops[0]
        gc_config=GenerationConfig(do_sample=False,num_beams=1,max_new_tokens=options['max_new_tokens'],
            eos_token_id=stops if len(stops)>1 else stops[0],pad_token_id=pad,bos_token_id=tok.bos_token_id,repetition_penalty=1.,use_cache=True)
        inputs={}
        for row in rows:                                 # every row, so that the batches never depend on the cache
            ids=core.chat_token_ids(tok,row['messages'],options['template_kwargs'])
            if len(ids)>options['max_input_tokens']:raise ValueError('生成评测输入超过统一上限：'+row['id'])
            inputs[row['id']]=ids
        lengths={k:len(v) for k,v in inputs.items()};categories={r['id']:r['category'] for r in rows}
        element=4 if options['dtype']=='float32' else 2
        capacity=torch.cuda.get_device_properties(torch.device(device)).total_memory if device.startswith('cuda') else None
        capacity=limits.get('memory_capacity') or capacity                      # a smaller pretend GPU (smoke test)
        if options['batch_size']=='auto':
            plan,reserve=_auto_plan(Path(cache_root),rows,inputs,lengths,categories,tokenizer_dir,options,capacity,
                                    checkpoint_weight_bytes(directory,getattr(torch,options['dtype'])))
        else:
            size=max(1,int(options['batch_size']))
            plan=plan_generation_groups(lengths,size,categories,prefill_pair_bytes(tokenizer_dir),int(float(options['prefill_budget_gib'])*1024**3))
            # room for the decoding state of the actual prompts (not the max_input_tokens cap) plus working memory
            reserve=max(generation_cache_bytes(tokenizer_dir,len(g),max(lengths[k] for k in g)+options['max_new_tokens'],element) for g in plan)+4*1024**3
        groups=[g for g in plan if any(records[k] is None for k in g)]
        skip={pad,tok.bos_token_id,*stops}
        with loader(directory,{**options,'gpu_reserve_bytes':reserve}) as model:
            own=getattr(model,'generation_config',None)
            if own is not None and getattr(own,'do_sample',False):own.do_sample=False   # greedy only: never inherit sampling defaults
            placement=(getattr(model,'_qmerge_loading_report',None) or {}).get('placement')
            if options['device']!='cpu' and placement=='offload' and not limits.get('allow_slow'):
                raise RuntimeError('显存装不下整个模型和解码缓存：每个解码步都要把全部权重从磁盘换入一遍。'
                                   '请把运行时换成 A100 并打开“高 RAM”（80GB 显存），或调小 batch_size / max_new_tokens；'
                                   '确实要这样跑，把 generation_limits 的 allow_slow 设为 True')
            cached_decoding=decoding_feeds_one_token(model,inputs[groups[0][0]],gc_config,options['device'],pad)
            if cached_decoding is False:
                raise RuntimeError('generate() 没有使用解码缓存：每一步都在重算整段序列，结果相同但会慢几个数量级。'
                                   '这是当前 transformers 版本与本模型配置（use_cache=false）的组合问题，已在生成开始前停下。')
            for number,group in enumerate(groups,1):
                queue=[group]
                while queue:
                    part=queue.pop(0);cap=_BATCH_CAP.get(device)
                    if cap and len(part)>cap:
                        # as few pieces as the cap allows, of equal size (a piece runs as long as its longest answer)
                        size=-(-len(part)//-(-len(part)//cap))
                        queue[:0]=[part[i:i+size] for i in range(0,len(part),size)];continue
                    tag=f'生成评测 {label} 批 {number}/{len(groups)}'+('' if len(part)==len(group) else f'（其中 {len(part)} 条）')
                    started=time.perf_counter();failure=None;_gpu_peak(device,reset=True)
                    guard=None
                    if capacity and limits.get('memory_guard',True):
                        guard={'device':device,'capacity':capacity,'max_new':int(options['max_new_tokens']),
                               'base':torch.cuda.memory_allocated(torch.device(device)),'margin':limits.get('memory_margin')}
                    try:
                        tails=generate_batch(model,[inputs[k] for k in part],gc_config,device,pad,options['seed'],
                                             _generation_monitor(tag,len(part),stops,limits,guard))
                    except _ProjectedOutOfMemory as error:
                        tails=None;gc.collect()
                        if torch.cuda.is_available():torch.cuda.empty_cache()
                        _BATCH_CAP[device]=error.rows
                        print(f'{tag}: 按前面各步的显存增长推算，这批到最后会超出显存（{error}）。已提前停下，'
                              f'这 {len(part)} 条改为每次 {error.rows} 条，本次会话后面的批次也按这个上限拆分。',flush=True)
                        queue.insert(0,part);continue
                    except RuntimeError as error:
                        if not _out_of_memory(error):raise
                        failure=(str(error).strip().splitlines() or [''])[0][:160]
                        if len(part)==1:
                            raise RuntimeError(f'{tag}: 单独一条也显存不足（{failure}）。调小 generation_options 的 max_new_tokens，'
                                               '或换显存更大的 GPU。已完成的批次都已保存。') from error
                    if failure is not None:
                        # Handled after the except block has ended: inside it the traceback still holds the
                        # tensors of the failed step, and nothing could be released.
                        tails=None;gc.collect()
                        if torch.cuda.is_available():torch.cuda.empty_cache()
                        _BATCH_CAP[device]=max(1,len(part)//2)
                        print(f'{tag}: 显存不足（{failure}）。这 {len(part)} 条改为每次 {_BATCH_CAP[device]} 条重做，'
                              '本次会话后面的批次也按这个上限拆分；已保存的结果不受影响。',flush=True)
                        queue.insert(0,part);continue
                    seconds=time.perf_counter()-started
                    for key,tail in zip(part,tails):
                        cut=next((i for i,t in enumerate(tail) if t in stops),None)
                        produced=tail if cut is None else tail[:cut+1]
                        if cut is None and len(tail)<options['max_new_tokens']:raise ValueError('生成在停止符之前意外结束：'+key)
                        raw=tok.decode([t for t in produced if t not in skip],skip_special_tokens=False)
                        prefix=tok.decode(inputs[key][-8:],skip_special_tokens=False)
                        text=('<think>\n' if prefix.rstrip().endswith('<think>') else '')+raw
                        rec={'id':key,'text':text,'output_tokens':len(produced),'truncated':cut is None,
                             'seconds':seconds/len(part),'token_sha256':core.digest(produced),'batch':len(part)}
                        core.save_record(root/(core.digest(key)+'.json'),{'context':identity,'item':key},rec);records[key]=rec
                    steps=max(len(t) for t in tails)
                    print(f'{tag}: {len(part)} 条，提示最长 {max(len(inputs[k]) for k in part)} token，{steps} 步，{seconds/60:.1f} 分钟'
                          f'（{seconds/max(steps,1):.2f}s/步），截断 {sum(records[k]["truncated"] for k in part)} 条{_gpu_peak(device)}',flush=True)
                    if limits.get('deadline') is not None and time.time()>limits['deadline'] and (queue or number<len(groups)):
                        raise GenerationBudgetExceeded(f'{tag}: 到达设定的时间上限，剩余批次未运行（已完成的批次已保存）')
    if missing and checkpoint_fingerprint(Path(directory))[0]!=fingerprint:raise ValueError('生成期间 checkpoint 内容改变，拒绝验收')
    manual={}
    if manual_scores_file:
        for line in Path(manual_scores_file).read_text().splitlines():
            if not line.strip():continue
            r=json.loads(line);key=r['request_sha256']
            if key in manual:raise ValueError('重复人工评分请求')
            manual[key]=r
    pending=[];ordered=[]
    for row in rows:
        rec=records[row['id']];request={'identity':identity,'id':row['id'],'output':rec,'rubric':row.get('rubric')}
        request_sha=core.digest(request);score=core.grade_generation(row,rec)
        if score is None:
            if request_sha in manual:
                score=float(manual[request_sha]['score'])
                if not math.isfinite(score) or not 0<=score<=1:raise ValueError('人工分数须在 [0,1]')
            else:pending.append({'request_sha256':request_sha,'model':model_identity,'id':row['id'],
                'messages':row['messages'],'text':rec['text'],'rubric':row['rubric'],'score':None})
        ordered.append({**rec,'score':score,'request_sha256':request_sha})
    # JSONL out and JSONL in use the exact same format; edit only score.
    (root/'pending_manual_scores.jsonl').write_text(''.join(json.dumps(x,ensure_ascii=False)+'\n' for x in pending),encoding='utf-8')
    report={'identity':identity,'records':ordered,'pending_manual':len(pending),'manual_requests':str(root/'pending_manual_scores.jsonl'),
            'decoding_cache_observed':cached_decoding,'batch_cap_after_out_of_memory':_BATCH_CAP.get(device)}
    core.atomic_json(root/'report.json',report)
    return report
'''
load_embedded_module('qmerge_runtime', MODULE_SOURCE)
del MODULE_SOURCE

In [ ]:
# 内嵌源代码；独立运行无需额外文件。
MODULE_SOURCE = r'''
"""Orthogonal pieces of a merge delta and their first-order effects on every question's loss.

Used by the component stage of the pipeline, which replaces the old NPS-style local search. The delta of the
merged model F against the subject S is cut, matrix by matrix, into pieces that are orthogonal to each other and
add up to it exactly (decompose_delta); the pipeline pools the pieces of many matrices into components (depth
block x module type x band x inside / outside S's own fine-tune subspace). One forward and one backward pass of F
over the proxy questions (gradient_components) give, for every question i and piece P, <grad l_i(F), P>: to first
order, how much keeping P instead of dropping it raises l_i, i.e. the derivative of l_i with respect to the
coefficient of P. The output head is one piece whose effect is measured exactly instead. combine() builds a matrix
with any coefficient on each of its pieces, bit for bit the same in every session. Before the long backward pass,
the gradient path is checked against the real loss on two questions (central finite differences).

Ideas taken from existing methods and adapted here (none of them is reproduced):
  * the delta is whitened by the inputs its matrix actually sees before its SVD (ASVD, SVD-LLM; the input
    second moment of RegMean): a direction counts by how much it changes the layer's outputs on real inputs;
  * bands of the delta's singular directions are the units that are kept or dropped (Task Singular Vectors,
    AdaRank);
  * gradient x delta is each part's first-order effect on the loss (Sens-Merging; Taylor-expansion pruning);
  * parts of the delta are kept or dropped by data and the result is checked with the real loss
    (Localize-and-Stitch);
  * each band is split by the input subspace S's own fine-tune S - B uses most, where a merge interferes with what
    S learned (the task-interference view of TSV-Merge);
  * a coefficient per component, set by one step along the first-order gradient (AdaMerging learns layer-wise
    coefficients by gradient steps).
"""
import hashlib
import inspect
import json
import math
import time
from pathlib import Path
import torch
from safetensors import safe_open
import qmerge_core as core
import qmerge_runtime as runtime

COMPONENTS_VERSION='components-2026-10-12.2'


class WeightsChanged(ValueError):
    """The weights read now are not the ones that were scored before."""


# ---------------------------------------------------------------- pieces of one matrix
def check_bands(bands):
    bands=[int(b) for b in bands]
    if not bands or bands[0]<1 or any(a>=b for a,b in zip(bands,bands[1:])):raise ValueError('bands 必须是递增的正整数，例如 [4, 16, 64]')
    return bands


def band_names(bands):
    """Names of the pieces of one matrix: each band of singular directions, then the rest."""
    names=[];low=0
    for high in bands:
        names.append(f'd{low+1}-{high}');low=high
    return names+['rest']


def band_slices(bands,rank):
    """Column ranges of U for each band; a matrix with fewer directions than the last band has shorter ones."""
    out=[];low=0
    for high in bands:
        out.append((min(low,rank),min(high,rank)));low=high
    return out


def piece_names(bands,split):
    """Names of the pieces of one matrix: each band and the rest, each cut into the part inside S's own fine-tune
    subspace (.in) and the part outside it (.out) when split."""
    names=band_names(bands)
    return [f'{n}.{side}' for n in names for side in ('in','out')] if split else names


def decompose_delta(delta,inputs=None,rank=64,seed=0,oversample=16,power_iters=2,damp=1e-2):
    """Orthonormal output directions U (out x r) in which `delta` acts most strongly, and C = U^T delta (r x in).

    With inputs (rows of the inputs this matrix actually receives), "strongly" means ||delta x|| over those
    inputs, plus a small ridge (damp times their mean square) for directions they barely reach: U holds the
    leading left singular vectors of delta A^(1/2), A = inputs^T inputs / rows + ridge. Without inputs, those of
    delta itself. Randomized range finder with power iterations (Halko, Martinsson & Tropp 2011); A is never
    formed. The pieces U_b C_b of disjoint column bands b and the rest, delta - U C, are orthogonal to each
    other, both plainly and weighted by A, and add up to delta exactly.
    """
    D=delta.float();out_features,in_features=D.shape
    r=max(0,min(int(rank),out_features,in_features))
    if r==0:return torch.zeros((out_features,0)),torch.zeros((0,in_features))
    k=min(r+int(oversample),out_features,in_features)
    gen=torch.Generator().manual_seed(int(seed))
    if inputs is not None and inputs.shape[0]>0:
        X=inputs.float();n=X.shape[0]
        mu=float(damp)*float(torch.dot(X.reshape(-1),X.reshape(-1)))/(n*in_features)
        mu=mu if mu>0 and math.isfinite(mu) else 1e-12
        def apply(T):return X.T@(X@T)/n+mu*T
        W=X.T@torch.randn((n,k),generator=gen)/math.sqrt(n)+math.sqrt(mu)*torch.randn((in_features,k),generator=gen)
    else:
        def apply(T):return T
        W=torch.randn((in_features,k),generator=gen)
    Q=torch.linalg.qr(D@W).Q
    for _ in range(int(power_iters)):
        Q=torch.linalg.qr(D@apply(D.T@Q)).Q
    T=D.T@Q
    K=(T.T@apply(T)).double();K=(K+K.T)/2
    values,vectors=torch.linalg.eigh(K)
    order=torch.argsort(values,descending=True)[:r]
    # QR keeps the span of every leading set of columns, so the bands stay what they are; it only makes the
    # columns orthonormal to working precision.
    U=torch.linalg.qr((Q.double()@vectors[:,order]).float()).Q.contiguous()
    return U,(U.T@D).contiguous()


def subject_directions(tau,rank,seed):
    """Orthonormal input directions V (in x r) that S's own fine-tune tau = S - B uses most: its leading right singular
    vectors. Cutting a piece P into P V V^T (inside) and P - P V V^T (outside) keeps every piece of a matrix
    orthogonal to every other one, because the bands are cut on the output side and this split on the input side."""
    return decompose_delta(tau.T.contiguous(),None,rank,seed)[0]


def piece_scores(G,delta,part,bands,names):
    """<G, piece> for every piece of one matrix (G: the loss's gradient with respect to that matrix; part: its
    decomposition, U and C, with V, CV = C V and R = (delta - U C) V when split)."""
    G=G.float()
    if not G.is_contiguous():G=G.contiguous()
    total=float(torch.dot(G.reshape(-1),delta.reshape(-1)))
    labels=band_names(bands)[:-1];split=names!=band_names(bands)
    U=part.get('U') if part else None
    if U is None or not U.shape[1]:
        band=[0.]*len(labels);inside=[0.]*len(labels);rest_in=0.
        slices=[]
    else:
        slices=band_slices(bands,U.shape[1]);C=part['C'];UG=U.T@G
        band=[float((UG[a:b].double()*C[a:b].double()).sum()) if b>a else 0. for a,b in slices]
        if split:
            UGV=(UG@part['V']).double();CV=part['CV'].double()
            inside=[float((UGV[a:b]*CV[a:b]).sum()) if b>a else 0. for a,b in slices]
            rest_in=float(((G@part['V']).double()*part['R'].double()).sum())
    rest=total-sum(band)
    if not split:return dict(zip(names,band+[rest]))
    out={}
    for label,b,i in zip(labels,band,inside):
        out[label+'.in']=i;out[label+'.out']=b-i
    out['rest.in']=rest_in;out['rest.out']=rest-rest_in
    return out


def _accumulate(coefficients,rows):
    """sum_s coefficients[:, s] (x) rows[s], term by term in a fixed order with exactly rounded operations."""
    out=torch.zeros((coefficients.shape[0],rows.shape[1]),dtype=torch.float32);t=torch.empty_like(out)
    for s in range(coefficients.shape[1]):
        torch.mul(coefficients[:,s:s+1],rows[s:s+1],out=t);out.add_(t)
    return out


def combine(F,S,part,bands,coefficients,chunk_elements=1<<20):
    """W' - S in float32 for the matrix whose pieces (of F - S) carry these coefficients (missing ones are 1), or None
    when they are all 1 (W' is F itself).

    With r for the rest and b for the bands, W' - S = c_r.out D + sum_b (c_b.out - c_r.out) P_b
    + sum_b (c_b.in - c_b.out) P_b.in + (c_r.in - c_r.out) P_r.in, where D = F - S, P_b = U_b C_b,
    P_b.in = U_b (C V)_b V^T and P_r.in = R V^T (without the split, only the first two sums, with c_b and c_rest). Every
    term is a multiple of D or a sum of rank-1 terms, each formed by exactly rounded element-wise operations in a fixed
    order, with no fused multiply-add and no matrix product, so the result has the same bits on any CPU and any
    number of threads.
    """
    split='V' in part;names=piece_names(bands,split)
    if set(coefficients)-set(names):raise ValueError('未知的分量：'+str(sorted(set(coefficients)-set(names))))
    c={n:float(coefficients.get(n,1.)) for n in names}
    if all(v==1. for v in c.values()):return None
    U,C=part['U'],part['C'];rank=U.shape[1];labels=band_names(bands)[:-1];slices=band_slices(bands,rank)
    if F.shape!=S.shape or F.ndim!=2 or U.shape[0]!=F.shape[0] or C.shape[1]!=F.shape[1]:raise ValueError('分量与权重的形状不符')
    base=c['rest.out'] if split else c['rest']
    out=F.float()-S.float()
    if base!=1.:out=out*base
    width=out.shape[1];Vt=part['V'].T.contiguous() if split else None
    z=torch.zeros((rank,width),dtype=torch.float32);active=[False]*rank
    for label,(a,b) in zip(labels,slices):
        if b<=a:continue
        alpha=(c[label+'.out'] if split else c[label])-base
        beta=c[label+'.in']-c[label+'.out'] if split else 0.
        if alpha:torch.mul(C[a:b],alpha,out=z[a:b])
        if beta:z[a:b].add_(_accumulate(part['CV'][a:b],Vt)*beta)
        if alpha or beta:active[a:b]=[True]*(b-a)
    gamma=c['rest.in']-c['rest.out'] if split else 0.
    Rg=(part['R']*gamma).contiguous() if gamma else None
    rows=out.shape[0];step=max(1,int(chunk_elements)//max(1,width));t=torch.empty((min(step,rows),width),dtype=torch.float32)
    for i in range(0,rows,step):
        j=min(i+step,rows);block=out[i:j];tb=t[:j-i]
        for r in range(rank):
            if active[r]:
                torch.mul(U[i:j,r:r+1],z[r:r+1],out=tb);block.add_(tb)
        if Rg is not None:
            for s in range(Vt.shape[0]):
                torch.mul(Rg[i:j,s:s+1],Vt[s:s+1],out=tb);block.add_(tb)
    return out


def cap_rows(update,reference,cap):
    """Scale down every row of `update` whose norm exceeds cap times the norm of the same row of `reference`, to that
    limit. The scale is rounded down to a multiple of 2^-20 and applied by one exactly rounded multiplication, so the
    same inputs give the same bits on any CPU. Returns (update, number of rows scaled)."""
    u=torch.linalg.vector_norm(update.double(),dim=1);r=torch.linalg.vector_norm(reference.double(),dim=1)
    over=u>float(cap)*r
    count=int(over.sum())
    if not count:return update,0
    scale=torch.floor(float(cap)*r[over]/u[over]*2.**20)/2.**20
    update=update.clone();update[over]=update[over]*scale.to(update.dtype).unsqueeze(1)
    return update,count


# ---------------------------------------------------------------- files of the gradient pass
def write_states(path,states):
    """Hidden states (or their gradients), one tensor per question, as one safetensors file."""
    from safetensors.torch import save_file
    path=Path(path);names={f'h{n}':key for n,key in enumerate(sorted(states))}
    tmp=path.with_name(path.name+'.tmp')
    save_file({k:states[key].detach().to('cpu').contiguous() for k,key in names.items()},str(tmp),
              metadata={'names':json.dumps(names,ensure_ascii=False)})
    tmp.replace(path)


def read_states(path,keys,device='cpu'):
    with safe_open(str(path),framework='pt',device='cpu') as f:
        names=json.loads((f.metadata() or {}).get('names','{}'))
        states={names[k]:f.get_tensor(k).to(device) for k in f.keys()}
    if set(states)!=set(keys):raise ValueError('分量筛选的中间结果与本次的题目不符：'+str(path))
    return states


def write_decomposition(path,layer,parts,info):
    """One layer's decomposition: for every matrix, its tensors (U, C and, when split, V, CV, R) and their sizes."""
    from safetensors.torch import save_file
    path=Path(path);tensors={}
    for key,part in parts.items():
        for name,value in part.items():tensors[f'{key}::{name}']=value.contiguous()
    if not tensors:tensors['__none__']=torch.zeros(1)
    tmp=path.with_name(path.name+'.tmp')
    save_file(tensors,str(tmp),metadata={'layer':str(int(layer)),'info':json.dumps(info,sort_keys=True)})
    tmp.replace(path)


def read_decomposition(path,keys=None):
    """{key: {'U', 'C' and, when split, 'V', 'CV', 'R'}} from one layer's decomposition file (only `keys` when given)."""
    out={}
    with safe_open(str(path),framework='pt',device='cpu') as f:
        for name in f.keys():
            key,sep,part=name.rpartition('::')
            if sep and (keys is None or key in keys):out.setdefault(key,{})[part]=f.get_tensor(name)
    return out


def decomposition_info(path):
    """(layer, {key: {'rank', 'inputs', 'energy'}}) of one layer's decomposition file."""
    with safe_open(str(path),framework='pt',device='cpu') as f:
        meta=f.metadata() or {}
    return int(meta['layer']),json.loads(meta.get('info','{}'))


def _progress(path,value):
    core.atomic_json(path,value)
    return value


def _seed_of(key,seed):
    return int(hashlib.sha256(f'{int(seed)}|{key}'.encode()).hexdigest()[:12],16)


# ---------------------------------------------------------------- the gradient pass on the layer-by-layer scorer
class InputSampler:
    """Forward hooks that keep every stride-th row of the inputs of the given linear layers.

    Layers that read the same tensor (q/k/v, gate/up) share one copy of its rows. Each sequence starts at its own
    offset, so that the chat template's opening tokens, the same in every question, are not taken from all of them."""
    def __init__(self,model,names,mapping,stride):
        self.stride=max(1,int(stride));self.offset=0;self.rows={};self._handles=[];self._seen=[]
        for name in names:
            path,_,leaf=name.rpartition('.')
            try:module=model.get_submodule(path)
            except AttributeError:continue
            if leaf!='weight' or not isinstance(module,torch.nn.Linear):continue
            key=mapping[name];self.rows[key]=[]
            self._handles.append(module.register_forward_hook(self._hook(key)))

    def _hook(self,key):
        def hook(module,args,output):
            if not args or not torch.is_tensor(args[0]):return
            x=args[0]
            for seen,rows in self._seen:
                if seen is x:
                    self.rows[key].append(rows);return
            rows=x.detach().reshape(-1,x.shape[-1])[self.offset::self.stride].to('cpu',torch.float32,copy=True)
            self._seen.append((x,rows));self.rows[key].append(rows)
        return hook

    def next_sequence(self,index=0):
        self._seen=[];self.offset=int(index)%self.stride

    def inputs(self,key):
        rows=self.rows.get(key)
        return torch.cat(rows) if rows else None

    def close(self):
        for handle in self._handles:handle.remove()
        self._handles=[];self._seen=[]


def _assign(sc,values,trainable=()):
    """Put these values into the scorer's model (converted to its dtype); the names in `trainable` collect gradients."""
    with torch.no_grad():
        for name,value in values.items():
            sc._set(sc.model,name,sc.device,value=value,**sc._extra)
    for name in values:
        try:sc.model.get_parameter(name).requires_grad_(name in trainable)
        except AttributeError:pass


def _freeze(sc,names):
    for name in names:
        try:sc.model.get_parameter(name).requires_grad_(False)
        except AttributeError:pass


def response_nll_tensor(logits,input_ids,prompt_length,logits_start=0):
    """core.response_nll as a differentiable tensor: the mean cross-entropy over the reference answer's tokens."""
    length=input_ids.shape[1];first=prompt_length-1-logits_start;stop=length-1-logits_start
    if logits.ndim!=3 or logits.shape[0]!=1 or first<0 or stop>logits.shape[1] or stop<=first:raise ValueError('logits 不覆盖参考回答')
    target=input_ids[0,first+logits_start+1:stop+logits_start+1].to(logits.device)
    return torch.nn.functional.cross_entropy(logits[0,first:stop].float(),target,reduction='mean')


def _loss_and_grad(sc,hidden,ids,prompt_length):
    """One question's response NLL from the hidden states after the last decoder layer (the model's own final norm
    and output head, as in the scorer's finish), and its gradient with respect to those hidden states.
    Returns (NLL computed exactly as the scorer does, the same as a differentiable tensor, the gradient)."""
    h=hidden.detach().clone().requires_grad_(True);last=sc.num_layers-1
    def make(i):
        def stub(*args,**kwargs):
            if i==last:return sc._wrap(h)
            return sc._wrap(args[0] if args else kwargs['hidden_states'])
        return stub
    kwargs=sc._forward_kwargs(ids);start=0;suffix=ids.shape[1]-prompt_length+1
    parameters=inspect.signature(sc.model.forward).parameters
    for name in ('logits_to_keep','num_logits_to_keep'):
        if name in parameters:
            kwargs[name]=suffix;start=prompt_length-1;break
    with torch.enable_grad(),sc._stubbed(make):
        logits=sc.model(**kwargs).logits
        loss=response_nll_tensor(logits,ids,prompt_length,start)
        loss.backward()
    value,_=core.response_nll(logits.detach(),ids,prompt_length,int(sc.options.get('token_chunk',32)),start)
    if h.grad is None:raise RuntimeError('最后一层的隐状态没有得到梯度')
    return value,float(loss.detach()),h.grad.detach()


def _layer_backward(sc,i,call,x,grad_out):
    """Gradient of the loss with respect to decoder layer i's input; the layer's trainable weights collect theirs."""
    x=x.detach().requires_grad_(True)
    with torch.enable_grad():
        out=sc._layer(i,call,x)
        out.backward(grad_out)
    if x.grad is None:raise RuntimeError(f'第 {i} 层的输入没有得到梯度')
    return x.grad.detach()


BACKWARD_CHECK_VERSION='backward-check-2026-10-14.1'
_PREFERRED=('linear_attn.in_proj_qkv.weight','self_attn.q_proj.weight','linear_attn.out_proj.weight','self_attn.o_proj.weight','mlp.down_proj.weight')


def layer_kind(names):
    """'linear_attention', 'full_attention' or 'other' for a decoder layer, from its parameter names."""
    if any('.linear_attn.' in n for n in names):return 'linear_attention'
    if any('.self_attn.' in n for n in names):return 'full_attention'
    return 'other'


def backward_check(sc,weights,item,directions=None,rel_step=1e-3,tolerance=0.25,floor=2e-6,expected_nll=None,progress=True):
    """The gradient path of gradient_components, checked against the real loss on the real model, before anything long.

    One question (item: (question, ids [1, L], prompt length)) runs forward through every layer of `weights` (in
    practice S). For the last decoder layer of each kind (linear attention, full attention), one weight matrix (the
    input projection) collects its gradient G through exactly the calls gradient_components makes: _loss_and_grad at
    the top, then _layer_backward layer by layer down to it. Then that matrix is moved by +P and -P and the layers
    from it to the top are run again for real; half the difference of the two losses must equal <G, P> within
    `tolerance` (relative) plus `floor`. P is a step of rel_step x ||W|| along G, and each direction(key) in
    `directions` (for instance a multiple of the matrix's aux delta, as the merge would move it).
    Also: the differentiable loss must equal the scorer's NLL, and (expected_nll) the NLL the scoring engine gave
    this question. Returns a record with 'ok'; an exception means the path does not work here.
    """
    started=time.perf_counter();key,ids,plen=item;ids=ids.to(sc.device);L=sc.num_layers;directions=dict(directions or {})
    def say(text):
        if progress:print(f'  反向自检：{text}，已用 {(time.perf_counter()-started)/60:.0f} 分钟',flush=True)
    catalogue=runtime.weights_catalogue(weights)
    def matrix(i):
        names=[n for n in sc.units[i] if len(catalogue.get(sc.mapping[n],{}).get('shape',()))==2 and 'norm' not in n.lower()]
        for suffix in _PREFERRED:
            for n in names:
                if n.endswith(suffix):return n
        return names[0] if names else None
    picks={}
    for i in range(L):
        kind=layer_kind(sc.units[i])
        if matrix(i) is not None:picks[kind]=i                 # the last layer of each kind
    if not picks:raise ValueError('解码层里没有可以检查的二维权重')
    chosen={i:(kind,matrix(i)) for kind,i in picks.items()};lo=min(chosen)
    def read(names):
        out={}
        for name in names:
            value=weights.tensor(sc.mapping[name])
            out[name]=value.clone() if value.is_inference() else value
        return out
    held=list(sc.outer)+[n for i in range(lo,L) for n in sc.units[i]]
    try:
        with torch.no_grad():
            _assign(sc,read(sc.outer))
            traces,x=sc._trace(ids)
            for i in range(lo):
                _assign(sc,read(sc.units[i]))
                try:x=sc._layer(i,traces[i],x)
                finally:sc._release(sc.units[i])
            say(f'前 {lo} 层的前向完成（{int(ids.shape[1])} token）')
            base={i:read(sc.units[i]) for i in range(lo,L)}
            for i in range(lo,L):_assign(sc,base[i])
            inputs={lo:x}
            for i in range(lo,L):inputs[i+1]=sc._layer(i,traces[i],inputs[i])
        nll,check,grad=_loss_and_grad(sc,inputs[L],ids,plen)
        record={'version':BACKWARD_CHECK_VERSION,'question':key,'tokens':int(ids.shape[1]),'nll':nll,'nll_differentiable':check,
                'nll_scoring_engine':expected_nll,'checks':[],'problems':[]}
        if not abs(nll-check)<=1e-3:record['problems'].append(f'可求导的损失 {check:.6f} 与评分用的 NLL {nll:.6f} 不一致')
        if expected_nll is not None and not abs(nll-expected_nll)<=1e-3:
            record['problems'].append(f'逐层重算的 NLL {nll:.6f} 与评分引擎给出的 {expected_nll:.6f} 不一致')
        grads={}
        for i in range(L-1,lo-1,-1):
            names=[chosen[i][1]] if i in chosen else []
            if names:_assign(sc,{n:base[i][n] for n in names},trainable=set(names))
            try:
                grad=_layer_backward(sc,i,traces[i],inputs[i],grad)
                for n in names:
                    p=sc.model.get_parameter(n)
                    grads[n]=None if p.grad is None else p.grad.detach().float().clone()
                    p.grad=None
            finally:_freeze(sc,names)
        say('反向完成，开始用真实前向核对')
        def loss_with(i,name,value):
            """The loss with matrix `name` of layer i replaced by value (float32), all else unchanged."""
            with torch.no_grad():
                _assign(sc,{name:value})
                try:
                    h=inputs[i]
                    for j in range(i,L):h=sc._layer(j,traces[j],h)
                    return sc._finish(h,ids,plen)['nll']
                finally:_assign(sc,{name:base[i][name]})
        for i in sorted(chosen,reverse=True):
            kind,name=chosen[i];G=grads.get(name);W=base[i][name].float()
            if G is None or not bool(torch.isfinite(G).all()) or float(G.norm())==0:
                record['problems'].append(f'第 {i+1} 层 {name} 没有得到有效的梯度')
                record['checks'].append({'layer':i,'kind':kind,'tensor':sc.mapping[name],'direction':'gradient','ok':False,'decisive':True});continue
            steps={'gradient':G*(rel_step*float(W.norm())/float(G.norm()))}
            for label,make in directions.items():
                P=make(sc.mapping[name])
                if P is not None:steps[label]=P.float().reshape(W.shape)
            for label,P in steps.items():
                predicted=float((G.double()*P.double()).sum())
                actual=(loss_with(i,name,W+P)-loss_with(i,name,W-P))/2
                ok=bool(math.isfinite(actual) and abs(actual-predicted)<=tolerance*abs(predicted)+floor)
                record['checks'].append({'layer':i,'kind':kind,'tensor':sc.mapping[name],'direction':label,'step_norm':float(P.norm()),
                                         'weight_norm':float(W.norm()),'predicted':predicted,'actual':actual,'ok':ok,
                                         'decisive':label=='gradient'})
                say(f'第 {i+1} 层（{kind}）{name.rsplit(".",2)[-2]} 沿 {label}：实际 {actual:+.3e}，一阶预测 {predicted:+.3e}'+('' if ok else '，不吻合' if label=='gradient' else '，相差较大（只记录：说明一阶估计在合并本身的改动幅度上只是近似）'))
        # Only the small step along G decides: it tests that the gradient is right. A step the size of the merge's own
        # change (directions) tests how far the first-order estimate carries; that is recorded, not required.
        record['ok']=not record['problems'] and all(c['ok'] for c in record['checks'] if c.get('decisive',True))
        record['seconds']=time.perf_counter()-started
        return record
    finally:
        _freeze(sc,held);sc._release(held)


def plan_segment(sc,tokens,limit=8):
    """Layers per backward segment: as many as the hidden states of all questions fit in RAM for, at most `limit`."""
    elem=torch.empty((),dtype=getattr(torch,sc.options.get('dtype','bfloat16'))).element_size()
    states=max(1,tokens*sc.hidden_size*elem)
    def numel(name):
        try:return sc.model.get_parameter(name).numel()
        except AttributeError:return 0
    layer=max(sum(numel(n) for n in names) for names in sc.units.values())*4
    budget=runtime.effective_ram()-4*layer-2*states-6*1024**3
    return max(1,min(int(limit),int(budget//states)))


def gradient_components(sc,weights,subject,items,targets,store,options,label='F',expected=None,progress=True,base=None):
    """First-order effects of the pieces of F - S on every question's loss, with F on the layer-by-layer scorer sc.

    weights: F; subject: S; base: B (all give tensor(key) by checkpoint key). items: [(question, ids [1, L], prompt
    length)]. targets: {key: 'matrix' | 'head'}. A decoder-layer matrix is cut into bands of output directions plus the
    rest (decompose_delta on the inputs it actually receives); with B and options['subject_rank'] > 0 each of those is
    cut again into the part inside and outside the input subspace of S's own fine-tune S - B (subject_directions).
    The output head is one piece, measured exactly: the loss with S's head on F's final hidden states.
    expected: {key: tensor digest} of F as it was scored before; every tensor read is checked against it
    (WeightsChanged). With options['preflight'] the gradient path is checked on two questions before the backward
    pass (_preflight).

    Phases: forward (every layer over all questions: its inputs are sampled and its deltas decomposed; the hidden
    states are kept at the start of each segment), top (each question's loss and its gradient at the last hidden
    states), backward (segment by segment from the top: the segment's inputs again from its kept states, then
    layer by layer each question's backward, which gives the gradient of every target matrix). Everything lives
    in `store`; an interrupted pass resumes there at the last finished layer, segment or phase.

    Returns {'questions', 'nll': [l_i(F) as the scorer computes it], 'check_nll', 'scores': {key: {piece: [value
    per question]}}, 'pieces', 'bands', 'split', 'decomposition': {key: {'layer', 'file', 'rank', 'subject_rank',
    'inputs', 'energy'}}, 'preflight'}.
    """
    store=Path(store);store.mkdir(parents=True,exist_ok=True)
    bands=check_bands(options.get('bands',[4,16,64]))
    subject_rank=int(options.get('subject_rank',32)) if base is not None else 0;split=subject_rank>0
    pieces=piece_names(bands,split)
    whiten=bool(options.get('whiten',True));seed=int(options.get('seed',0));damp=float(options.get('damp',1e-2))
    questions=[k for k,_,_ in items]
    if len(set(questions))!=len(questions):raise ValueError('题目 id 重复')
    ids={k:x.to(sc.device) for k,x,_ in items};plen={k:int(p) for k,_,p in items}
    L=sc.num_layers
    matrices={i:[n for n in sc.units[i] if targets.get(sc.mapping[n])=='matrix'] for i in range(L)}
    heads=[n for n in sc.outer if targets.get(sc.mapping[n])=='head']
    if len(heads)>1:raise ValueError('输出头不止一个张量：'+', '.join(heads))
    found={sc.mapping[n] for names in matrices.values() for n in names}|{sc.mapping[n] for n in heads}
    if set(targets)-found:raise ValueError('有要拆分的张量不在逐层评分的模型里：'+str(sorted(set(targets)-found)[:3]))
    tokens=sum(int(x.shape[1]) for x in ids.values())
    stride=max(1,math.ceil(tokens/max(1,int(options.get('input_tokens',8192)))))
    progress_path=store/'progress.json'
    state=json.loads(progress_path.read_text()) if progress_path.is_file() else None
    if state is None:
        segment=options.get('segment','auto')
        segment=plan_segment(sc,tokens) if segment in (None,'auto') else max(1,int(segment))
        state=_progress(progress_path,{'phase':'forward','next_layer':0,'segment':segment})
    segment=int(state['segment']);starts=list(range(0,L,segment))
    if state['phase']!='done' and progress:
        elem=torch.empty((),dtype=getattr(torch,sc.options.get('dtype','bfloat16'))).element_size()
        size=(len(starts)+3)*tokens*sc.hidden_size*elem
        print(f'  分量筛选 {label}：{len(questions)} 条题、{tokens} token；反向每段 {segment} 层；中间结果存在 {store}'
              f'（隐状态约 {size/1024**3:.1f} GB，算完删掉；另有各矩阵的分解，要留到上传之后）',flush=True)
    ahead=runtime.ReadAhead() if hasattr(weights,'ranges') else None
    def hint(i):
        if ahead is not None and 0<=i<L:
            try:ahead.request(weights.ranges([sc.mapping[n] for n in sc.units[i]]))
            except Exception:pass                       # only a hint
    def read(names):
        """F's values of these parameters, checked against `expected`, as ordinary (not inference-mode) tensors."""
        out={}
        for name in names:
            key=sc.mapping[name];value=weights.tensor(key)
            if expected is not None and runtime.tensor_digest(value)!=expected.get(key):
                raise WeightsChanged('分量筛选读到的权重与搜索时评分的不同：'+key)
            out[name]=value.clone() if value.is_inference() else value
        return out
    started=time.perf_counter()
    def say(text):
        if progress:print(f'  分量筛选 {label}：{text}，已用 {(time.perf_counter()-started)/60:.0f} 分钟',flush=True)
    try:
        if state['phase']=='done':return _result(store,L,pieces,bands,segment,split)
        # The arguments every decoder layer is called with, per question (the model's own forward up to the layers).
        with torch.no_grad():
            _assign(sc,read(sc.outer))
            traces={};first={}
            for k in questions:
                traces[k],x=sc._trace(ids[k])
                if state['phase']=='forward':first[k]=x
                del x
        sc._release(sc.outer)
        # ---- forward: inputs sampled, deltas decomposed, hidden states kept at each segment's start
        if state['phase']=='forward':
            start=int(state['next_layer'])
            if start and not (store/f'ckpt-{start:03d}.safetensors').is_file():start=0
            states=first if start==0 else read_states(store/f'ckpt-{start:03d}.safetensors',questions,sc.device)
            first=None
            if start:say(f'从第 {start+1} 层接着做前向')
            hint(start);begun=time.perf_counter()
            with torch.no_grad():
                for i in range(start,L):
                    if i in starts:
                        path=store/f'ckpt-{i:03d}.safetensors'
                        if not path.is_file():write_states(path,states)
                        state=_progress(progress_path,{'phase':'forward','next_layer':i,'segment':segment})
                    values=read(sc.units[i]);_assign(sc,values);hint(i+1)
                    sampler=InputSampler(sc.model,matrices[i],sc.mapping,stride) if whiten and matrices[i] else None
                    try:
                        for n,k in enumerate(questions):
                            if sampler is not None:sampler.next_sequence(n)
                            states[k]=sc._layer(i,traces[k][i],states[k])
                    finally:
                        if sampler is not None:sampler.close()
                        sc._release(sc.units[i])
                    _decompose_layer(sc,i,matrices[i],values,subject,base,sampler,store/f'decomp-{i:03d}.safetensors',bands,seed,damp,subject_rank)
                    del values,sampler
                    done=i+1-start
                    if done in (1,4) or (i+1)%8==0 or i+1==L:
                        rate=(time.perf_counter()-begun)/done
                        say(f'前向与分解 第 {i+1}/{L} 层，这一步估计还需 {rate*(L-i-1)/60:.0f} 分钟')
            write_states(store/'final.safetensors',states)
            del states
            state=_progress(progress_path,{'phase':'top','segment':segment})
        first=None
        # ---- each question's loss, and its gradient at the last hidden states
        if state['phase']=='top':
            final=read_states(store/'final.safetensors',questions,sc.device)
            _assign(sc,read(sc.outer))
            nll={};check={};grads={}
            for k in questions:nll[k],check[k],grads[k]=_loss_and_grad(sc,final[k],ids[k],plen[k])
            worst=max(abs(nll[k]-check[k]) for k in questions)
            if not worst<=1e-3:raise RuntimeError(f'可求导的损失与评分用的 NLL 不一致（最大差 {worst:.2e}）')
            scores={}
            if heads:
                name=heads[0];key=sc.mapping[name]
                _assign(sc,{name:subject.tensor(key)})            # S's head on F's final hidden states
                swapped={k:sc._finish(final[k],ids[k],plen[k])['nll'] for k in questions}
                scores[key]={'all':[nll[k]-swapped[k] for k in questions]}
            sc._release(sc.outer)
            if not all(math.isfinite(v) for v in [*nll.values(),*(x for s in scores.values() for x in s['all'])]):
                raise RuntimeError('损失含 NaN/Inf')
            core.atomic_json(store/'nll.json',{'questions':questions,'nll':[nll[k] for k in questions],'check':[check[k] for k in questions]})
            core.atomic_json(store/'scores.json',scores)
            write_states(store/f'grad-{L:03d}.safetensors',grads)
            del final,grads
            # Without decoder-layer matrices to cut (a recipe that changes only the output head) there is nothing to go back for.
            state=_progress(progress_path,{'phase':'backward','next_segment':len(starts)-1,'segment':segment} if any(matrices.values())
                            else {'phase':'done','segment':segment})
            say('每道题的损失和它对最后一层隐状态的梯度已算好'+('；输出头换回 S 的效果已直接算出' if heads else ''))
        # ---- check the gradient path on two questions before the long backward pass
        if state['phase']=='backward' and options.get('preflight',True) and any(matrices.values()):
            check=store/'preflight.json'
            record=json.loads(check.read_text()) if check.is_file() else None
            fresh=int(state['next_segment'])==len(starts)-1 and (store/f'grad-{L:03d}.safetensors').is_file()
            if record is None and not fresh:
                record={'ok':True,'checks':[],'skipped':'反向已经开始，没有再做自检','reported':True}
            if record is None:
                say('先在两道题上用真实损失核对梯度（约半小时）')
                record=_preflight(sc,read,subject,store,questions,ids,plen,traces,matrices,starts,L,bands,pieces)
                core.atomic_json(check,record)
            if not record['ok']:
                raise RuntimeError('梯度自检没有通过：'+'；'.join(f'{x["tensor"]} {x["piece"]} 实际 {x["actual"]} / 一阶预测 {x["predicted"]}'
                                                       for x in record['checks'] if not x['ok'])[:400])
            if record.get('reported') is None:
                say('梯度自检通过：'+'；'.join(f'{x["piece"]}（第 {x["layer"]+1} 层）实际 {sum(x["actual"]):+.2e}，一阶预测 {sum(x["predicted"]):+.2e}'
                                          for x in record['checks']))
                record['reported']=True;core.atomic_json(check,record)
        # ---- backward, one segment at a time from the top
        if state['phase']=='backward':
            scores=json.loads((store/'scores.json').read_text());begun=time.perf_counter();finished=0
            for si in range(int(state['next_segment']),-1,-1):
                s=starts[si];e=starts[si+1] if si+1<len(starts) else L
                for i in range(s,e):                                  # a segment interrupted earlier starts over
                    for n in matrices[i]:scores.pop(sc.mapping[n],None)
                grads=read_states(store/f'grad-{e:03d}.safetensors',questions,sc.device)
                inputs={s:read_states(store/f'ckpt-{s:03d}.safetensors',questions,sc.device)}
                hint(s)
                with torch.no_grad():
                    for i in range(s,e-1):                            # the inputs of the segment's layers, again
                        _assign(sc,read(sc.units[i]));hint(i+1)
                        try:inputs[i+1]={k:sc._layer(i,traces[k][i],inputs[i][k]) for k in questions}
                        finally:sc._release(sc.units[i])
                for i in range(e-1,s-1,-1):
                    names=matrices[i];values=read(sc.units[i])
                    deltas={n:values[n].float()-subject.tensor(sc.mapping[n]).float() for n in names}
                    parts=read_decomposition(store/f'decomp-{i:03d}.safetensors')
                    _assign(sc,values,trainable=set(names));del values
                    hint(i-1)
                    try:
                        for k in questions:
                            grads[k]=_layer_backward(sc,i,traces[k][i],inputs[i][k],grads[k])
                            for n in names:
                                p=sc.model.get_parameter(n);key=sc.mapping[n]
                                G=p.grad if p.grad is not None else torch.zeros_like(deltas[n])
                                for piece,value in piece_scores(G,deltas[n],parts.get(key),bands,pieces).items():
                                    if not math.isfinite(value):raise RuntimeError('梯度含 NaN/Inf：'+key)
                                    scores.setdefault(key,{}).setdefault(piece,[]).append(value)
                                p.grad=None;del G
                    finally:
                        _freeze(sc,names);sc._release(sc.units[i])
                    del deltas,parts;inputs.pop(i,None)
                    say(f'反向 第 {i+1}/{L} 层完成')
                write_states(store/f'grad-{s:03d}.safetensors',grads)
                core.atomic_json(store/'scores.json',scores)
                state=_progress(progress_path,{'phase':'backward','next_segment':si-1,'segment':segment})
                (store/f'grad-{e:03d}.safetensors').unlink(missing_ok=True)
                del grads,inputs
                finished+=1;rate=(time.perf_counter()-begun)/finished
                say(f'反向 第 {s+1}–{e} 层完成（从上往下，还剩 {si} 段），这一步估计还需 {rate*si/60:.0f} 分钟')
            state=_progress(progress_path,{'phase':'done','segment':segment})
        return _result(store,L,pieces,bands,segment,split)
    finally:
        if ahead is not None:ahead.close()
        _freeze(sc,sc.outer);sc._release(sc.outer)
        for names in sc.units.values():
            _freeze(sc,names);sc._release(names)


def _decompose_layer(sc,i,names,values,subject,base,sampler,path,bands,seed,damp,subject_rank):
    """Decompose the delta F - S of every target matrix of layer i and keep, in one file, U and C (bands of the
    activation-whitened output directions) and, when subject_rank > 0, V (the input directions S's own fine-tune S - B
    uses most), CV = C V and R = (delta - U C) V; also the squared Frobenius norm of every piece (its share of the
    delta), in the order of piece_names."""
    parts={};info={}
    for name in names:
        key=sc.mapping[name]
        s=subject.tensor(key).float()
        delta=values[name].float()-s
        if delta.ndim!=2 or not bool(torch.isfinite(delta).all()):raise ValueError('增量不是有限值的矩阵：'+key)
        inputs=sampler.inputs(key) if sampler is not None else None
        U,C=decompose_delta(delta,inputs,bands[-1],_seed_of(key,seed),damp=damp)
        part={'U':U,'C':C};slices=band_slices(bands,U.shape[1])
        flat=delta.reshape(-1);total=float(torch.dot(flat,flat))
        band=[float(C[a:b].double().square().sum()) for a,b in slices];rest=max(0.,total-sum(band))
        if subject_rank>0:
            tau=s-base.tensor(key).float()
            if not bool(torch.isfinite(tau).all()):raise ValueError('S - B 不是有限值：'+key)
            V=subject_directions(tau,subject_rank,_seed_of(key,seed+1));del tau
            CV=(C@V).contiguous();R=(delta@V-U@CV).contiguous()
            part.update(V=V,CV=CV,R=R);energy=[]
            for (a,b),e in zip(slices,band):
                inside=float(CV[a:b].double().square().sum());energy+=[inside,max(0.,e-inside)]
            inside=float(R.double().square().sum());energy+=[inside,max(0.,rest-inside)]
        else:energy=band+[rest]
        parts[key]=part
        info[key]={'rank':int(U.shape[1]),'subject_rank':int(part['V'].shape[1]) if 'V' in part else 0,
                   'inputs':0 if inputs is None else int(inputs.shape[0]),'energy':energy}
        del delta,inputs,flat,s
    write_decomposition(path,i,parts,info)


def _preflight(sc,read,subject,store,questions,ids,plen,traces,matrices,starts,L,bands,pieces,eps=0.1):
    """Check the gradient path against the real loss before the long backward pass.

    Two questions of middling length go backward through the top two segments exactly as in the backward pass. For the
    piece with the largest effect on them there, and for the largest one in the lower of the two segments, F with that
    piece scaled by 1 + eps and by 1 - eps is run for real from the hidden states kept at its segment (the changed
    matrix in float32, as the scorer holds it: the change is far below bfloat16 resolution). Half the difference of
    the two losses must match eps x <G, piece> for each question within 25% (or 2e-5). Returns the record."""
    by_length=sorted(questions,key=lambda k:(int(ids[k].shape[1]),k))
    picks=list(dict.fromkeys([by_length[len(by_length)//4],by_length[len(by_length)//2]]))
    every=read_states(store/f'grad-{L:03d}.safetensors',questions,sc.device);grads={k:every[k] for k in picks};del every
    low=starts[-2] if len(starts)>1 else starts[-1]
    found={}
    for si in range(len(starts)-1,-1,-1):
        s=starts[si];e=starts[si+1] if si+1<len(starts) else L
        if s<low:break
        every=read_states(store/f'ckpt-{s:03d}.safetensors',questions,sc.device);inputs={s:{k:every[k] for k in picks}};del every
        with torch.no_grad():
            for i in range(s,e-1):
                _assign(sc,read(sc.units[i]))
                try:inputs[i+1]={k:sc._layer(i,traces[k][i],inputs[i][k]) for k in picks}
                finally:sc._release(sc.units[i])
        for i in range(e-1,s-1,-1):
            names=matrices[i];values=read(sc.units[i])
            deltas={n:values[n].float()-subject.tensor(sc.mapping[n]).float() for n in names}
            parts=read_decomposition(store/f'decomp-{i:03d}.safetensors') if names else {}
            _assign(sc,values,trainable=set(names));del values
            try:
                for k in picks:
                    grads[k]=_layer_backward(sc,i,traces[k][i],inputs[i][k],grads[k])
                    for n in names:
                        p=sc.model.get_parameter(n);key=sc.mapping[n]
                        G=p.grad if p.grad is not None else torch.zeros_like(deltas[n])
                        for piece,value in piece_scores(G,deltas[n],parts.get(key),bands,pieces).items():
                            found.setdefault((i,n,piece),[]).append(value)
                        p.grad=None;del G
            finally:
                _freeze(sc,names);sc._release(sc.units[i])
            del deltas,parts;inputs.pop(i,None)
    if not found:return {'questions':picks,'eps':eps,'checks':[],'ok':True}
    ranked=sorted(found.items(),key=lambda kv:-abs(sum(kv[1])))
    chosen=[ranked[0]]+[kv for kv in ranked if kv[0][0]<(starts[-1] if len(starts)>1 else L) and kv[0][0]!=ranked[0][0][0]][:1]
    checks=[]
    for (i,name,piece),scores in chosen:
        key=sc.mapping[name];s0=max(x for x in starts if x<=i)
        every=read_states(store/f'ckpt-{s0:03d}.safetensors',questions,sc.device);begin={k:every[k] for k in picks};del every
        part=read_decomposition(store/f'decomp-{i:03d}.safetensors',{key})[key]
        losses={}
        for sign in (1,-1):
            states=dict(begin)
            with torch.no_grad():
                for layer in range(s0,L):
                    values=read(sc.units[layer])
                    if layer==i:
                        S=subject.tensor(key)
                        values[name]=S.float()+combine(values[name],S,part,bands,{piece:1.+sign*eps})
                    _assign(sc,values);del values
                    try:states={k:sc._layer(layer,traces[k][layer],states[k]) for k in picks}
                    finally:sc._release(sc.units[layer])
                _assign(sc,read(sc.outer))
                try:losses[sign]={k:sc._finish(states[k],ids[k],plen[k])['nll'] for k in picks}
                finally:sc._release(sc.outer)
        actual=[(losses[1][k]-losses[-1][k])/2 for k in picks];predicted=[eps*v for v in scores]
        checks.append({'tensor':key,'layer':i,'piece':piece,'actual':actual,'predicted':predicted,
                       'ok':all(abs(a-b)<=.25*abs(b)+2e-5 for a,b in zip(actual,predicted))})
    return {'questions':picks,'eps':eps,'checks':checks,'ok':all(x['ok'] for x in checks)}


def _result(store,L,pieces,bands,segment,split):
    """What the finished pass gives back; the large intermediate files go."""
    for pattern in ('ckpt-*.safetensors','grad-*.safetensors','final.safetensors','*.tmp'):
        for path in store.glob(pattern):path.unlink(missing_ok=True)
    nll=json.loads((store/'nll.json').read_text())
    info={}
    for i in range(L):
        path=store/f'decomp-{i:03d}.safetensors'
        if path.is_file():
            layer,entries=decomposition_info(path)
            for key,value in entries.items():info[key]={'layer':layer,'file':path.name,**value}
    check=store/'preflight.json'
    return {'questions':nll['questions'],'nll':nll['nll'],'check_nll':nll['check'],'scores':json.loads((store/'scores.json').read_text()),
            'pieces':pieces,'bands':bands,'split':split,'decomposition':info,'segment':segment,
            'preflight':json.loads(check.read_text()) if check.is_file() else None}
'''
load_embedded_module('qmerge_components', MODULE_SOURCE)
del MODULE_SOURCE

In [ ]:
# 内嵌源代码；独立运行无需额外文件。
MODULE_SOURCE = r'''
"""Read pinned safetensors ranges and publish resumable model builds."""
import gc
import hashlib
import io
import json
import math
import os
from pathlib import Path
import re
import threading
import time

import requests
import torch
from huggingface_hub import CommitOperationAdd, CommitOperationDelete, hf_hub_url
from safetensors.torch import save_file


_ENGINE_VERSION = "2026-10-02.remaining-cpu.2"
_DTYPES = {
    "F64": (torch.float64, 8), "F32": (torch.float32, 4),
    "F16": (torch.float16, 2), "BF16": (torch.bfloat16, 2),
    "I64": (torch.int64, 8), "I32": (torch.int32, 4),
    "I16": (torch.int16, 2), "I8": (torch.int8, 1),
    "U8": (torch.uint8, 1), "BOOL": (torch.bool, 1),
}
_FLOATS = {"F64", "F32", "F16", "BF16"}


def _canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"), ensure_ascii=False,
                      allow_nan=False).encode("utf-8")


def _digest(value):
    return hashlib.sha256(_canonical(value)).hexdigest()


def _json_bytes(value):
    return json.dumps(value, sort_keys=True, indent=2, ensure_ascii=False,
                      allow_nan=False).encode("utf-8") + b"\n"


def _safe_name(name):
    path = Path(name)
    if path.is_absolute() or ".." in path.parts or "\\" in name:
        raise ValueError(f"Unsafe repository filename: {name}")
    return name


def _access_message(repo, name, status):
    return (f"{repo}: 无权读取 {name}（HTTP {status}）。这是需要先在模型页面同意条款的仓库（gated），"
            "或者 HF_TOKEN 没有读权限。用与 HF_TOKEN 相同的账号打开 "
            f"https://huggingface.co/{repo} 点同意后再运行。")


class Engine:
    """Each input repository stays at one commit throughout this engine session."""

    def __init__(self, api, token, work, shard_gb=1.0, block_mb=16, revisions=None,
                 readahead_mb=None, readahead_cache_mb=None):
        self.api = api
        self.token = token
        self.work = Path(work)
        self.work.mkdir(parents=True, exist_ok=True)
        self.shard_bytes = int(float(shard_gb) * 1024 ** 3)
        self.block_bytes = int(float(block_mb) * 1024 ** 2)
        if self.shard_bytes < 1 or self.block_bytes < 8:
            raise ValueError("shard_gb must be positive. block_mb must permit at least eight bytes.")
        self.revisions = dict(revisions or {})
        self.pins = {}
        self._infos = {}
        self._names = {}
        self._sizes = {}
        self._headers = {}
        self._catalogues = {}
        self._json_cache = {}
        self.bytes_read = 0
        self.range_requests = 0
        # Read-ahead changes how many requests carry the bytes, never the bytes.
        # It is not part of any recipe, fingerprint or cache identity.
        ahead = globals().get("READAHEAD_MB", 32) if readahead_mb is None else readahead_mb
        cap = globals().get("READAHEAD_CACHE_MB", 768) if readahead_cache_mb is None else readahead_cache_mb
        self.readahead_bytes = max(0, int(float(ahead) * 1024 ** 2))
        self.readahead_cache_bytes = max(self.readahead_bytes, int(float(cap) * 1024 ** 2))
        self._windows = {}       # (repo, file, first byte) -> bytes; dict order is recency
        self._window_bytes = 0
        self._tails = {}         # (repo, file) -> last byte of recent requests
        self._shape_cache = {}
        self.session = requests.Session()
        self.session.headers.update({"Accept-Encoding": "identity"})
        if token:
            self.session.headers["Authorization"] = f"Bearer {token}"
        # Several repositories may be read at the same time, one thread each (the aux-delta pass does). Every
        # thread gets its own HTTP session; one lock per repository keeps the reads of a repository in order;
        # one lock guards the shared read-ahead bookkeeping. Single-threaded use is unchanged.
        self._state = threading.RLock()
        self._repo_locks = {}
        self._local = threading.local()
        self._local.session = self.session
        self.rate_limit_waits = 0

    def pin(self, repo):
        if repo not in self.pins:
            info = self.api.model_info(repo, revision=self.revisions.get(repo, "main"),
                                       files_metadata=True, token=self.token)
            sha = info.sha
            if not isinstance(sha, str) or not re.fullmatch(r"[0-9a-fA-F]{40,64}", sha):
                raise ValueError(f"The Hub did not return a commit SHA for {repo}.")
            self.pins[repo] = sha
            self._remember_info(repo, info)
            print(f"Pinned {repo} @ {sha}", flush=True)
        return self.pins[repo]

    def _remember_info(self, repo, info):
        self._infos[repo] = info
        self._names[repo] = sorted(s.rfilename for s in info.siblings)
        self._sizes[repo] = {s.rfilename: s.size for s in info.siblings}

    def files(self, repo):
        self.pin(repo)
        return list(self._names[repo])

    def _url(self, repo, name):
        return hf_hub_url(repo_id=repo, filename=_safe_name(name), revision=self.pin(repo))

    def _session(self):
        """The HTTP session of the calling thread (the engine's own one in the thread that made the engine)."""
        session = getattr(self._local, "session", None)
        if session is None:
            session = requests.Session()
            session.headers.update(dict(self.session.headers))
            self._local.session = session
        return session

    def _repo_lock(self, repo):
        with self._state:
            lock = self._repo_locks.get(repo)
            if lock is None:
                lock = self._repo_locks[repo] = threading.Lock()
            return lock

    def _small_bytes(self, repo, name, limit=128 * 1024 ** 2):
        with self._session().get(self._url(repo, name), stream=True, timeout=(30, 180)) as response:
            if response.status_code in (401, 403):
                raise InputNotReady(_access_message(repo, name, response.status_code))
            response.raise_for_status()
            declared = response.headers.get("Content-Length")
            if declared is not None and int(declared) > limit:
                raise ValueError(f"Metadata file exceeds the size limit: {repo}/{name}")
            data = bytearray()
            for chunk in response.iter_content(1024 ** 2):
                data.extend(chunk)
                if len(data) > limit:
                    raise ValueError(f"Metadata file exceeds the size limit: {repo}/{name}")
            with self._state:
                self.bytes_read += len(data)
            return bytes(data)

    def json(self, repo, name):
        key = (repo, self.pin(repo), name)
        if key not in self._json_cache:
            self._json_cache[key] = json.loads(self._small_bytes(repo, name))
        return self._json_cache[key]

    def manifest(self, repo):
        if "build_manifest.json" not in self.files(repo):
            raise ValueError(f"No build_manifest.json in {repo}.")
        return self.json(repo, "build_manifest.json")

    def _range(self, repo, name, start, end):
        from urllib3.exceptions import HTTPError as TransportHTTPError
        if not (0 <= start <= end):
            raise ValueError("Invalid byte range.")
        expected = end - start + 1
        error = None
        attempt = throttled = 0
        while True:
            try:
                with self._session().get(self._url(repo, name),
                                         headers={"Range": f"bytes={start}-{end}"},
                                         stream=True, timeout=(30, 180)) as response:
                    # Reject a full-shard response before reading its body.
                    if response.status_code != 206:
                        if response.status_code in (401, 403):
                            raise InputNotReady(_access_message(repo, name, response.status_code))
                        if response.status_code == 429:
                            raise _Throttled(_retry_after(getattr(response, "headers", None) or {}))
                        if response.status_code >= 500:
                            response.raise_for_status()
                        raise ValueError(f"The server ignored the byte range ({response.status_code}) "
                                         f"for {repo}/{name}.")
                    content_range = response.headers.get("Content-Range", "")
                    match = re.fullmatch(r"bytes (\d+)-(\d+)/(\d+)", content_range)
                    if not match or (int(match[1]), int(match[2])) != (start, end):
                        raise ValueError(f"Invalid Content-Range for {repo}/{name}: {content_range}")
                    total = int(match[3])
                    known = self._sizes[repo].get(name)
                    if end >= total or (known is not None and total != known):
                        raise ValueError(f"The file size changed for {repo}/{name}.")
                    self._sizes[repo][name] = total
                    encoding = response.headers.get("Content-Encoding", "identity").lower()
                    if encoding not in ("", "identity"):
                        raise ValueError("Compressed byte-range responses cannot be used.")
                    length = response.headers.get("Content-Length")
                    if length is not None and int(length) != expected:
                        raise ValueError(f"Invalid Content-Length for {repo}/{name}.")
                    data = response.raw.read(expected + 1, decode_content=False)
                    if len(data) < expected:
                        raise OSError(f"Short range read for {repo}/{name} bytes {start}-{end}: "
                                      f"expected {expected}, received {len(data)}.")
                    if len(data) > expected:
                        raise ValueError(f"Oversized range for {repo}/{name} bytes {start}-{end}: "
                                         f"expected {expected}, received {len(data)}.")
                    with self._state:
                        self.bytes_read += len(data)
                        self.range_requests += 1
                    return data
            except _Throttled as exc:
                # The Hub's request quota (HTTP 429): wait as long as it asks, then carry on. This does not
                # count against the retries for broken transfers below.
                throttled += 1
                if throttled > 12:
                    raise RuntimeError(f"{repo}/{name}: Hugging Face 连续 {throttled} 次限速（HTTP 429），已停止。"
                                       "稍后从头运行即可从断点继续。") from exc
                wait = min(max(exc.wait, 5.0), 300.0)
                with self._state:
                    self.rate_limit_waits += 1
                print(f"Hugging Face 限速（HTTP 429）：{repo}/{name} 等待 {wait:.0f} 秒后继续（第 {throttled} 次）", flush=True)
                time.sleep(wait)
            except (requests.RequestException, OSError, TransportHTTPError) as exc:
                error = exc
                if attempt == 3:
                    raise
                print(f"Range read retry {attempt + 1}/3: {repo}/{name} "
                      f"bytes {start}-{end} ({type(exc).__name__})", flush=True)
                time.sleep(min(2 ** attempt, 8))
                attempt += 1
        raise RuntimeError("Byte-range read failed.") from error

    def _drop_windows(self, repo=None):
        with self._state:
            for key in [k for k in self._windows if repo is None or k[0] == repo]:
                self._window_bytes -= len(self._windows.pop(key))
            for key in [k for k in self._tails if repo is None or k[0] == repo]:
                del self._tails[key]

    def io_stats(self):
        return {"range_requests": self.range_requests, "bytes_read": self.bytes_read,
                "readahead_windows": len(self._windows), "readahead_bytes_held": self._window_bytes}

    def _read(self, repo, name, start, end, limit):
        """Bytes start..end (inclusive) of one pinned file, with sequential read-ahead.

        A request that begins where an earlier request on the same file stopped,
        or inside a held window, is treated as a sequential scan: the Range
        request is extended towards ``limit`` (the last byte of the tensor being
        read) and the surplus stays in memory for the next calls. Every byte
        still arrives through _range with its 206, Content-Range and length
        checks, and a pinned commit is immutable, so a window cannot go stale.
        Any other request is fetched exactly as before and is not kept, so
        random access never downloads more than it asked for.

        Thread-safe: reads of one repository are serialised by its lock (so a
        scan stays sequential); different repositories are read concurrently.
        """
        if self.readahead_bytes <= 0:
            return self._range(repo, name, start, end)
        with self._repo_lock(repo):
            with self._state:
                tails = self._tails.setdefault((repo, name), [])
                hit = None
                for key, held in self._windows.items():
                    if key[0] == repo and key[1] == name and key[2] <= start < key[2] + len(held):
                        hit = key
                        break
                prefix, fetch_start, sequential = b"", start, (start - 1) in tails
                if hit is not None:
                    held = self._windows.pop(hit)
                    self._window_bytes -= len(held)
                    offset = start - hit[2]
                    if end < hit[2] + len(held):
                        self._windows[hit] = held  # re-inserted last: most recently used
                        self._window_bytes += len(held)
                        self._note_tail(tails, end)
                        return held[offset:end - hit[2] + 1]
                    prefix, fetch_start, sequential = held[offset:], hit[2] + len(held), True
                    del held
                self._note_tail(tails, end)
            if not sequential:
                return self._range(repo, name, start, end)
            fetch_end = max(end, min(limit, fetch_start + self.readahead_bytes - 1))
            fetched = self._range(repo, name, fetch_start, fetch_end)
            data = prefix + fetched if prefix else fetched
            del prefix, fetched
            if fetch_end <= end:
                return data
            with self._state:
                replaced = self._windows.pop((repo, name, start), None)  # an overlapping older window
                if replaced is not None:
                    self._window_bytes -= len(replaced)
                self._windows[(repo, name, start)] = data
                self._window_bytes += len(data)
                while self._window_bytes > self.readahead_cache_bytes and len(self._windows) > 1:
                    oldest = next(iter(self._windows))
                    self._window_bytes -= len(self._windows.pop(oldest))
            return data[:end - start + 1]

    @staticmethod
    def _note_tail(tails, end):
        tails.append(end)
        if len(tails) > 16:
            del tails[0]

    def _header(self, repo, filename):
        sha = self.pin(repo)
        key = (repo, sha, filename)
        if key in self._headers:
            return self._headers[key]
        cache = self.work / "headers" / (_digest(list(key)) + ".json")
        if cache.exists():
            envelope = json.loads(cache.read_text())
            if envelope.get("identity") == list(key):
                header, base = envelope["header"], envelope["base"]
                known = self._sizes[repo].get(filename)
                if known is None or known == envelope["file_size"]:
                    self._sizes[repo][filename] = envelope["file_size"]
                    self._validate_header(repo, filename, header, base)
                    self._headers[key] = (header, base)
                    return header, base
        head = self._range(repo, filename, 0, 7)
        size = int.from_bytes(head, "little")
        if not 2 <= size <= 100 * 1024 ** 2:
            raise ValueError(f"Invalid safetensors header size in {repo}/{filename}.")
        header = json.loads(self._range(repo, filename, 8, 8 + size - 1))
        base = 8 + size
        self._validate_header(repo, filename, header, base)
        cache.parent.mkdir(parents=True, exist_ok=True)
        cache.write_bytes(_json_bytes({"identity": list(key), "header": header, "base": base,
                                     "file_size": self._sizes[repo][filename]}))
        self._headers[key] = (header, base)
        return header, base

    def _validate_header(self, repo, filename, header, base):
        if not isinstance(header, dict):
            raise ValueError("A safetensors header must be an object.")
        size = self._sizes[repo].get(filename)
        intervals = []
        for name, entry in header.items():
            if name == "__metadata__":
                continue
            dtype = entry.get("dtype")
            if dtype not in _DTYPES:
                raise ValueError(f"Unsupported or quantized dtype {dtype} in {repo}/{name}.")
            shape = entry.get("shape")
            offsets = entry.get("data_offsets")
            if not isinstance(shape, list) or any(type(d) is not int or d < 0 for d in shape):
                raise ValueError(f"Invalid shape for {repo}/{name}.")
            if (not isinstance(offsets, list) or len(offsets) != 2
                    or any(type(d) is not int for d in offsets)):
                raise ValueError(f"Invalid offsets for {repo}/{name}.")
            first, last = offsets
            if first < 0 or last - first != math.prod(shape) * _DTYPES[dtype][1]:
                raise ValueError(f"Tensor size does not match its offsets for {repo}/{name}.")
            if size is not None and base + last > size:
                raise ValueError(f"Tensor exceeds its shard size for {repo}/{name}.")
            if last > first:
                intervals.append((first, last))
        intervals.sort()
        if any(a[1] > b[0] for a, b in zip(intervals, intervals[1:])):
            raise ValueError(f"Overlapping tensors in {repo}/{filename}.")

    def _catalogue(self, repo):
        if repo in self._catalogues:
            return self._catalogues[repo]
        names = self.files(repo)
        if "config.json" not in names:
            raise ValueError(f"{repo} has no full-model config.json.")
        config = self.json(repo, "config.json")
        if config.get("quantization_config") is not None or config.get("compression_config"):
            raise ValueError(f"Quantized input is not supported: {repo}")
        index_name = "model.safetensors.index.json"
        if index_name in names:
            weight_map = self.json(repo, index_name).get("weight_map")
            if not isinstance(weight_map, dict) or not weight_map:
                raise ValueError(f"Invalid safetensors index in {repo}.")
        elif "model.safetensors" in names:
            header, _ = self._header(repo, "model.safetensors")
            weight_map = {name: "model.safetensors" for name in header if name != "__metadata__"}
        else:
            raise ValueError(f"{repo} requires a full safetensors model or a safetensors index.")
        catalogue = {}
        for filename in sorted(set(weight_map.values())):
            if filename not in names:
                raise ValueError(f"The index references a missing file: {repo}/{filename}")
            header, base = self._header(repo, filename)
            actual = {name for name in header if name != "__metadata__"}
            mapped = {name for name, shard in weight_map.items() if shard == filename}
            if not mapped <= actual:
                raise ValueError(f"The index references absent tensor keys: {repo}/{filename}")
            # Treat the index as authoritative. A replaced MTP tensor may remain
            # physically present in an old shard but be unreferenced by the index.
            for name in sorted(mapped):
                if re.search(r"(?:^|\.)(?:qweight|qzeros|weight_scale|weight_scale_inv)(?:$|\.)", name):
                    raise ValueError(f"Quantized tensor found: {repo}/{name}")
                entry = header[name]
                catalogue[name] = {**entry, "file": filename, "base": base}
        self._catalogues[repo] = catalogue
        return catalogue

    def shapes(self, repo):
        cached = self._shape_cache.get(repo)
        if cached is None or cached[0] is not self._catalogue(repo):
            catalogue = self._catalogue(repo)
            cached = self._shape_cache[repo] = (catalogue, {name: tuple(entry["shape"]) for name, entry in catalogue.items()})
        return dict(cached[1])

    def info(self, repo, name):
        entry = self._catalogue(repo)[name]
        return {**entry, "shape": list(entry["shape"]), "data_offsets": list(entry["data_offsets"])}

    def assert_compatible(self, repos, metadata_repo=None):
        repos = list(dict.fromkeys(repos))
        if not repos:
            raise ValueError("At least one input repository is required.")
        metadata_repo = metadata_repo or repos[0]
        if metadata_repo not in repos:
            repos.append(metadata_repo)
        base = self._catalogue(metadata_repo)
        reference_shapes = self.shapes(metadata_repo)
        if not base:
            raise ValueError("The model has no tensors.")
        for repo in repos:
            current = self._catalogue(repo)
            if self.shapes(repo) != reference_shapes:
                missing = sorted(set(base) - set(current))[:5]
                extra = sorted(set(current) - set(base))[:5]
                changed = sorted(name for name in set(base) & set(current)
                                 if base[name]["shape"] != current[name]["shape"])[:5]
                raise ValueError(f"Tensor structure differs in {repo}. Missing {missing}, "
                                 f"extra {extra}, shape changes {changed}.")
            for name in base:
                if self.is_float(repo, name) != self.is_float(metadata_repo, name):
                    raise ValueError(f"Float and integer tensor roles differ for {repo}/{name}.")
                if not self.is_float(repo, name) and self.dtype(repo, name) != self.dtype(metadata_repo, name):
                    raise ValueError(f"Integer tensor dtypes differ for {repo}/{name}.")
        return reference_shapes

    def dtype(self, repo, name):
        return _DTYPES[self._catalogue(repo)[name]["dtype"]][0]

    def is_float(self, repo, name):
        return self._catalogue(repo)[name]["dtype"] in _FLOATS

    @torch.no_grad()
    def _flat(self, repo, name, first, last, output_dtype=None):
        entry = self._catalogue(repo)[name]
        dtype, item_size = _DTYPES[entry["dtype"]]
        count = math.prod(entry["shape"])
        if not 0 <= first <= last <= count:
            raise ValueError(f"Invalid tensor slice for {repo}/{name}.")
        result = torch.empty(last - first, dtype=output_dtype or dtype, device="cpu")
        block_elements = max(1, self.block_bytes // item_size)
        data_start = entry["base"] + entry["data_offsets"][0]
        tensor_end = data_start + count * item_size - 1
        for index in range(first, last, block_elements):
            stop = min(index + block_elements, last)
            data = bytearray(self._read(repo, entry["file"], data_start + index * item_size,
                                        data_start + stop * item_size - 1, tensor_end))
            chunk = torch.frombuffer(data, dtype=dtype)
            result[index - first:stop - first].copy_(chunk)
        return result

    def _shape(self, repo, name):
        return tuple(self._catalogue(repo)[name]["shape"])

    @torch.no_grad()
    def tensor(self, repo, name):
        shape = self._shape(repo, name)
        return self._flat(repo, name, 0, math.prod(shape)).reshape(shape)

    @torch.no_grad()
    def rows(self, repo, name, i, j):
        shape = self._shape(repo, name)
        if not shape:
            if (i, j) != (0, 1):
                raise ValueError("A scalar supports only rows(0, 1).")
            return self._flat(repo, name, 0, 1, torch.float32).reshape(())
        if not 0 <= i <= j <= shape[0]:
            raise ValueError(f"Invalid row interval for {repo}/{name}.")
        width = math.prod(shape[1:])
        return self._flat(repo, name, i * width, j * width, torch.float32).reshape((j - i, *shape[1:]))

    def token_metadata(self, repo):
        assets = {}
        for name in self.files(repo):
            leaf = Path(name).name
            if (leaf.startswith(("tokenizer", "vocab", "merges", "special_tokens", "added_tokens", "chat_template"))
                    and not name.endswith(".safetensors")):
                assets[name] = hashlib.sha256(self._small_bytes(repo, name)).hexdigest()
        return assets

    def _output_info(self, repo):
        info = self.api.model_info(repo, revision="main", files_metadata=True, token=self.token)
        return info, {s.rfilename: s.size for s in info.siblings}

    def _output_bytes(self, repo, name, sha, limit=128 * 1024 ** 2):
        url = hf_hub_url(repo_id=repo, filename=name, revision=sha)
        with self._session().get(url, stream=True, timeout=(30, 180)) as response:
            response.raise_for_status()
            data = bytearray()
            for chunk in response.iter_content(1024 ** 2):
                data.extend(chunk)
                if len(data) > limit:
                    raise ValueError("Output metadata exceeds the size limit.")
            self.bytes_read += len(data)
            return bytes(data)

    def _output_json(self, repo, name, sha):
        return json.loads(self._output_bytes(repo, name, sha))

    def _pin_output(self, repo, sha):
        info = self.api.model_info(repo, revision=sha, files_metadata=True, token=self.token)
        if info.sha != sha:
            raise ValueError("The Hub returned a different publication commit.")
        self.pins[repo] = sha
        self._remember_info(repo, info)
        self._drop_windows(repo)
        self._catalogues.pop(repo, None)
        self._shape_cache.pop(repo, None)
        self._headers = {key: value for key, value in self._headers.items() if key[0] != repo}
        self._json_cache = {key: value for key, value in self._json_cache.items() if key[0] != repo}

    def _commit(self, repo, operations, message, parent_commit):
        return self.api.create_commit(repo_id=repo, repo_type="model", operations=operations,
                                      commit_message=message, parent_commit=parent_commit,
                                      token=self.token)

    def _metadata_files(self, repo, directory, output_dtype):
        names = self.files(repo)
        selected = []
        exact = {"config.json", "generation_config.json", "preprocessor_config.json",
                 "processor_config.json", "video_preprocessor_config.json", "chat_template.jinja"}
        prefixes = ("tokenizer", "special_tokens", "added_tokens", "vocab", "merges",
                    "chat_template", "configuration_", "modeling_", "processing_", "tokenization_")
        for name in names:
            leaf = Path(name).name
            if name in exact or leaf.startswith(prefixes) or name.startswith("chat_templates/"):
                if name.endswith((".safetensors", ".bin", ".pt", ".h5", ".onnx")):
                    continue
                _safe_name(name)
                path = directory / name
                path.parent.mkdir(parents=True, exist_ok=True)
                if name == "config.json":
                    config = json.loads(_json_bytes(self.json(repo, name)))
                    if output_dtype == "bfloat16":
                        config["torch_dtype"] = "bfloat16"
                        if "dtype" in config:
                            config["dtype"] = "bfloat16"
                    path.write_bytes(_json_bytes(config))
                else:
                    path.write_bytes(self._small_bytes(repo, name))
                selected.append((name, path))
        for name in names:
            if Path(name).name.lower().startswith(("license", "notice")):
                destination = "source_notices/base/" + _safe_name(name)
                path = directory / destination
                path.parent.mkdir(parents=True, exist_ok=True)
                path.write_bytes(self._small_bytes(repo, name))
                selected.append((destination, path))
        import shutil
        for destination, source in getattr(self, "extra_metadata", {}).items():
            destination = _safe_name(destination)
            path = directory / destination
            path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copyfile(source, path)
            selected.append((destination, path))
        return selected


from concurrent.futures import ThreadPoolExecutor


class InputNotReady(RuntimeError):
    """缺少可访问输入或真实底座；不允许减配继续发布。"""


class _Throttled(OSError):
    """HTTP 429 from the Hub; `wait` is how many seconds it asked for."""
    def __init__(self, wait):
        super().__init__(f"rate limited, retry after {wait:.0f}s")
        self.wait = float(wait)


def _retry_after(headers):
    """Seconds to wait after a 429: Retry-After, else the reset time of a RateLimit header (t=...), else 30."""
    try:
        value = headers.get("Retry-After") or headers.get("retry-after")
        if value is not None:
            return float(value)
        limit = headers.get("RateLimit") or headers.get("ratelimit") or ""
        match = re.search(r"(?:^|;)\s*t=(\d+(?:\.\d+)?)", str(limit))
        if match:
            return float(match[1])
    except (TypeError, ValueError, AttributeError):
        pass
    return 30.0


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_bytes(_json_bytes(value))
    tmp.replace(path)


class LockedEngine(Engine):
    def __init__(self, *args, **kwargs):
        work = Path(kwargs.get("work") or args[2])
        self.lock_path = work / "input_commits.json"
        old = json.loads(self.lock_path.read_text()) if self.lock_path.exists() else {}
        self.saved_pins = dict(old.get("pins", {}))
        supplied = dict(kwargs.pop("revisions", {}) or {})
        kwargs["revisions"] = {**self.saved_pins, **supplied}
        self._lock_mutex = threading.RLock()
        super().__init__(*args, **kwargs)

    def pin(self, repo):
        with self._lock_mutex:
            sha = super().pin(repo)
            if self.saved_pins.get(repo) != sha:
                self.saved_pins[repo] = sha
                atomic_json(self.lock_path, {"pins": self.saved_pins})
            return sha


def is_visual(name):
    return name.startswith("visual.") or ".visual." in name


def is_mtp(name):
    return name.startswith("mtp.") or ".mtp." in name or ".nextn." in name


def is_embedding(name):
    return name.endswith("embed_tokens.weight") or name.endswith("lm_head.weight")


def module_kind(name, shape):
    if is_embedding(name):
        return "emb"
    if len(shape) <= 1 or any(k in name.lower() for k in ("norm", "a_log", "dt_bias", "conv1d")):
        return "norm"
    if any(k in name for k in ("attn", "in_proj", "out_proj", "q_proj", "k_proj", "v_proj", "o_proj")):
        return "attn"
    return "ffn"


def row_ranges(shape, workspace_mb=None, arrays=12):
    # Rows() uses a flattened trailing dimension, including scalar and vector cases.
    rows, width = (shape[0], math.prod(shape[1:])) if shape else (1, 1)
    budget = ROWS_MB if workspace_mb is None else workspace_mb
    step = max(1, int(budget * 1024**2 / max(4 * arrays * width, 1)))
    return [(i, min(i + step, rows)) for i in range(0, rows, step)]


def known_key_aliases(name):
    # Explicit wrapper equivalence only; never pair unrelated tensors by shape.
    out = [name]
    if name.startswith("model.language_model."):
        out.append("model." + name[len("model.language_model."):])
        out.append("language_model." + name[len("model.language_model."):])
    elif name.startswith("model.") and not is_visual(name) and not is_mtp(name):
        out.append("model.language_model." + name[len("model."):])
    if name.startswith("model.mtp."):
        out.append(name[len("model."):])
    elif name.startswith("mtp."):
        out.append("model." + name)
    return list(dict.fromkeys(out))


def _tokenizer_signature(engine, repo):
    if "tokenizer.json" not in engine.files(repo):
        raise InputNotReady(f"{repo}: 缺少 tokenizer.json，无法验证 token ID")
    data = engine.json(repo, "tokenizer.json")
    model = data.get("model", {})
    core = model.get("vocab")
    if not isinstance(core, dict) or not core:
        raise InputNotReady(f"{repo}: 只支持显式 BPE token→ID 词表")
    core = {str(k): int(v) for k, v in core.items()}
    tokens = dict(core)
    for entry in data.get("added_tokens", []):
        token, index = entry["content"], int(entry["id"])
        if token in tokens and tokens[token] != index:
            raise InputNotReady(f"{repo}: added token 的 ID 与核心词表冲突")
        tokens[token] = index
    if len(set(tokens.values())) != len(tokens):
        raise InputNotReady(f"{repo}: 多个 token 占用同一 ID")
    merges = []
    for item in model.get("merges", []):
        parts = item.split(" ", 1) if isinstance(item, str) else list(item)
        if len(parts) != 2:
            raise InputNotReady(f"{repo}: 不认识的 BPE merge 编码")
        merges.append(tuple(parts))
    return {"core": core, "tokens": tokens, "merges": merges,
            "type": model.get("type"), "normalizer": data.get("normalizer"),
            "pre_tokenizer": data.get("pre_tokenizer"),
            "post_processor": data.get("post_processor"), "decoder": data.get("decoder")}


def check_tokenizer_compatible(reference, current, repo):
    # Qwen3.6→3.8 additive special tokens are supported; token reassignments are not.
    if reference["type"] != current["type"] or reference["core"] != current["core"]:
        raise InputNotReady(f"{repo}: 核心词表或核心 token ID 不一致；必须先显式对齐，不能直接混权")
    if reference["merges"] != current["merges"]:
        raise InputNotReady(f"{repo}: BPE merges 的内容/顺序不同，不能仅按张量形状混合")
    a, b = reference["tokens"], current["tokens"]
    reassigned = [s for s in a.keys() & b.keys() if a[s] != b[s]]
    ar, br = {v: k for k, v in a.items()}, {v: k for k, v in b.items()}
    collisions = [i for i in ar.keys() & br.keys() if ar[i] != br[i]]
    if reassigned or collisions:
        raise InputNotReady(f"{repo}: special-token ID 重分配或碰撞；不能当成简单增补词表")
    pipeline_diff = [k for k in ("normalizer", "pre_tokenizer", "post_processor", "decoder")
                     if reference[k] != current[k]]
    if pipeline_diff and not ALLOW_TOKENIZER_PIPELINE_DIFFERENCES:
        raise InputNotReady(f"{repo}: tokenizer pipeline 不同：{pipeline_diff}")
    return {"missing_tokens": len(a.keys() - b.keys()),
            "extra_tokens": len(b.keys() - a.keys()), "pipeline_differences": pipeline_diff}


def check_text_config(anchor_cfg, cfg, repo):
    a, b = anchor_cfg.get("text_config", anchor_cfg), cfg.get("text_config", cfg)
    fields = ("hidden_size", "intermediate_size", "num_hidden_layers", "num_attention_heads",
              "num_key_value_heads", "head_dim", "layer_types", "linear_num_key_heads",
              "linear_num_value_heads", "linear_key_head_dim", "linear_value_head_dim",
              "linear_conv_kernel_dim", "full_attention_interval")
    for key in fields:
        if key in a and key in b and a[key] != b[key]:
            raise InputNotReady(f"{repo}: text_config.{key} 不同：{b[key]} != {a[key]}")
    for key in ("hidden_size", "num_hidden_layers"):
        if key not in b:
            raise InputNotReady(f"{repo}: config 缺少结构字段 {key}")


class TensorNode:
    """rows: CPU FP32 matrix [row, flattened trailing dimensions]."""
    def tensor(self, name, dtype=torch.float32):
        out = torch.empty(self.shapes[name], dtype=dtype)
        view = out.reshape(self.shapes[name][0] if self.shapes[name] else 1, -1)
        for i, j in row_ranges(self.shapes[name]):
            view[i:j] = self.rows(name, i, j).to(dtype)
        return out

    def integer(self, name):
        raise NotImplementedError

    def has(self, name):
        return name in self.shapes

    def support(self, name, i, j):
        return torch.ones(j - i, 1, dtype=torch.bool)


class RepoNode(TensorNode):
    def __init__(self, engine, repo, anchor=None, strict=True):
        self.engine, self.repo, self.anchor = engine, repo, anchor
        self.config = engine.json(repo, "config.json")
        self.commit = engine.pin(repo)
        actual = engine.shapes(repo)
        self.shapes = dict(actual if anchor is None else anchor.shapes)
        self.real_shapes = actual
        self.key_map, self.embedding_masks = {}, {}
        self.tokenizer = _tokenizer_signature(engine, repo)
        self.audit = {"repo": repo, "commit": self.commit, "aliases": {}, "missing": [], "extra": []}
        if anchor is not None:
            check_text_config(anchor.config, self.config, repo)
            self.audit["tokenizer"] = check_tokenizer_compatible(anchor.tokenizer, self.tokenizer, repo)
            if self.audit["tokenizer"]["pipeline_differences"]:
                print("注意：输出保留 BASE tokenizer；输入 pipeline 有差异:", repo,
                      self.audit["tokenizer"]["pipeline_differences"])
        overrides = TENSOR_KEY_OVERRIDES.get(repo, {})
        for name, shape in self.shapes.items():
            candidates = [overrides[name]] if name in overrides else known_key_aliases(name)
            hits = [key for key in candidates if key in actual]
            if name in hits:
                key = name
            elif len(hits) == 1:
                key = hits[0]
            elif len(hits) > 1:
                raise InputNotReady(f"{repo}: {name} 的包装前缀映射不唯一")
            else:
                # A tied lm_head may be absent on disk. It is not an absent task delta.
                tied = self.config.get("text_config", self.config).get("tie_word_embeddings",
                       self.config.get("tie_word_embeddings", False))
                tied_hits = [k for k in actual if k.endswith("embed_tokens.weight")] if tied and name.endswith("lm_head.weight") else []
                if len(tied_hits) == 1:
                    key = tied_hits[0]
                else:
                    self.audit["missing"].append(name)
                    continue
            source_shape = actual[key]
            if source_shape != shape and not (is_embedding(name) and len(shape) == 2
                                              and len(source_shape) == 2 and source_shape[1] == shape[1]):
                raise InputNotReady(f"{repo}: 张量 {name} 的形状不兼容：{source_shape} != {shape}")
            if anchor is not None:
                if engine.is_float(repo, key) != anchor.engine.is_float(anchor.repo, name):
                    raise InputNotReady(f"{repo}: {name} 的浮点/整数角色不一致")
            self.key_map[name] = key
            if key != name:
                self.audit["aliases"][name] = key
            if is_embedding(name):
                ref_tokens = self.tokenizer["tokens"] if anchor is None else anchor.tokenizer["tokens"]
                src_tokens = self.tokenizer["tokens"]
                mask = torch.zeros(shape[0], dtype=torch.bool)
                if ref_tokens == src_tokens and source_shape == shape:
                    mask[:] = True  # Same tensor layout, including unused padding rows.
                else:
                    for token, index in ref_tokens.items():
                        if index < shape[0] and index < source_shape[0] and src_tokens.get(token) == index:
                            mask[index] = True
                self.embedding_masks[name] = mask
        missing_core = [n for n in self.audit["missing"] if not is_visual(n) and not is_mtp(n)]
        if strict and missing_core:
            raise InputNotReady(f"{repo}: 缺少语言主干张量（不允许静默丢模型）：{missing_core[:5]}")
        core = [n for n in self.shapes if not is_visual(n) and not is_mtp(n)]
        coverage = sum(n in self.key_map for n in core) / max(len(core), 1)
        if coverage < 0.80:
            raise InputNotReady(f"{repo}: 语言主干覆盖率 {coverage:.1%}，不足80%")
        self.audit["core_coverage"] = coverage
        self.audit["extra"] = sorted(set(actual) - set(self.key_map.values()))
        extra_core = [n for n in self.audit["extra"] if not is_visual(n) and not is_mtp(n)]
        if extra_core:
            raise InputNotReady(f"{repo}: 发现无法解释的额外语言张量：{extra_core[:5]}")
        self.identity = {"kind": "repo", "repo": repo, "commit": self.commit,
                         "fallback_anchor": anchor.identity if anchor is not None else None,
                         "alignment": self.audit, "embedding_masks": {
                             n: hashlib.sha256(m.numpy().tobytes()).hexdigest()
                             for n, m in self.embedding_masks.items()}}
        print(repo, f"语言主干覆盖 {coverage:.1%}", f"包装映射 {len(self.audit['aliases'])}",
              f"缺失视觉/MTP等 {len(self.audit['missing'])}")

    def has(self, name):
        return name in self.key_map

    def support(self, name, i, j):
        if name not in self.key_map:
            return torch.zeros(j - i, 1, dtype=torch.bool)
        if name in self.embedding_masks:
            return self.embedding_masks[name][i:j, None].clone()
        return torch.ones(j - i, 1, dtype=torch.bool)

    def rows(self, name, i, j):
        if name not in self.key_map:
            if self.anchor is None:
                raise KeyError(name)
            return self.anchor.rows(name, i, j)
        key = self.key_map[name]
        shape = self.shapes[name]
        width = math.prod(shape[1:]) if shape else 1
        if is_embedding(name):
            mask = self.embedding_masks[name][i:j]
            if bool(mask.all()):
                return self.engine.rows(self.repo, key, i, j).reshape(j - i, width)
            if self.anchor is None:
                raise InputNotReady(f"锚的词表行不完整：{name}")
            result = self.anchor.rows(name, i, j).clone()
            end = min(j, self.real_shapes[key][0])
            if end > i:
                chunk = self.engine.rows(self.repo, key, i, end).reshape(end - i, width)
                m = mask[:end - i]
                result[:end - i][m] = chunk[m]
            return result
        value = self.engine.rows(self.repo, key, i, j) if shape else self.engine.tensor(self.repo, key).float()
        return value.reshape(j - i, width)

    def integer(self, name):
        if name not in self.key_map:
            return self.anchor.integer(name)
        return self.engine.tensor(self.repo, self.key_map[name])


class RebasedModel(TensorNode):
    """Virtual full model: BASE_target + (source - source_reference). No intermediate repository."""
    def __init__(self, source, reference, anchor, tag="rebase"):
        self.source,self.reference,self.anchor=source,reference,anchor
        self.children=[source,reference,anchor]
        self.shapes=anchor.shapes
        self.tag=tag
        self.identity={"kind":"rebase","tag":tag,"formula":"target_base + source - source_reference",
                       "target_base":anchor.identity,"source":source.identity,"source_reference":reference.identity}
    def has(self,name): return self.source.has(name) and self.reference.has(name)
    def support(self,name,i,j): return self.source.support(name,i,j) & self.reference.support(name,i,j)
    def rows(self,name,i,j):
        base=self.anchor.rows(name,i,j)
        if not self.has(name): return base
        delta=self.source.rows(name,i,j).sub(self.reference.rows(name,i,j))
        mask=self.support(name,i,j)
        return base.add(delta.masked_fill(~mask,0))
    def integer(self,name): return self.anchor.integer(name)


def union_sources(nodes):
    result = []
    def walk(node):
        if isinstance(node, RepoNode):
            result.append(node.repo)
            if node.anchor is not None:
                result.append(node.anchor.repo)
        else:
            for child in node.children:
                walk(child)
            if hasattr(node, "adapter_repo"):
                result.append(node.adapter_repo)
    for node in nodes:
        walk(node)
    return list(dict.fromkeys(result))


def equal_integer(nodes, name):
    first = nodes[0].integer(name)
    for node in nodes[1:]:
        other = node.integer(name)
        if other.dtype != first.dtype or not torch.equal(first, other):
            raise InputNotReady(f"整数/布尔张量不一致：{name}")
    return first.clone()


# Follow-up module defaults; pipeline supplies settings explicitly.
ROWS_MB=64
READAHEAD_MB=32
READAHEAD_CACHE_MB=768
TENSOR_KEY_OVERRIDES={}
ALLOW_TOKENIZER_PIPELINE_DIFFERENCES=False
'''
load_embedded_module('qmerge_legacy', MODULE_SOURCE)
del MODULE_SOURCE

In [ ]:
# 内嵌源代码；独立运行无需额外文件。
MODULE_SOURCE = r'''
"""Read one pinned GGUF file by HTTP byte ranges and give 2-D language weights in Hugging Face layout.

Scope is what this merge needs: F32 / F16 / BF16 / Q8_0 / Q6_K tensors, the llama.cpp names of the
Qwen3.5-family ("qwen35") 2-D language weights, and llama.cpp's linear-attention V-head
reordering. The layout of every tensor is checked against the reference model's real weights
before use (resolve_tensor): a wrong guess shows up as a delta as large as the weights
themselves, so a mapping error excludes that tensor instead of being merged silently.
"""
import hashlib
import json
import math
from pathlib import Path
import re
import struct

import numpy as np
import torch

# VERSION names how F32 / F16 / BF16 / Q8_0 are read. Q6_K (added in e12) changes none of those results, so the
# version is kept and an aux delta built from a Q8_0 source before e12 stays valid.
VERSION = 'gguf-reader-2026-10-18.2'
# ggml type id -> (name, elements per block, bytes per block)
GGML_TYPES = {0: ('F32', 1, 4), 1: ('F16', 1, 2), 8: ('Q8_0', 32, 34), 14: ('Q6_K', 256, 210), 30: ('BF16', 1, 2)}
_SCALAR = {0: '<B', 1: '<b', 2: '<H', 3: '<h', 4: '<I', 5: '<i', 6: '<f', 7: '<?', 10: '<Q', 11: '<q', 12: '<d'}
_KEEP_STRING_ARRAYS = {'tokenizer.ggml.tokens'}
MAX_HEADER_BYTES = 512 * 1024 ** 2


class GGUFError(ValueError):
    pass


class _Cursor:
    """Sequential reader over the start of a remote file; fetches more bytes only when needed."""
    def __init__(self, fetch, size, chunk=8 * 1024 ** 2):
        self.fetch, self.size, self.chunk = fetch, int(size), int(chunk)
        self.buf = bytearray()
        self.pos = 0

    def _need(self, n):
        while len(self.buf) < self.pos + n:
            start = len(self.buf)
            if start >= self.size:
                raise GGUFError('GGUF 头部超出文件末尾')
            if start >= MAX_HEADER_BYTES:
                raise GGUFError('GGUF 头部异常大')
            self.buf += self.fetch(start, min(self.size, start + self.chunk) - 1)

    def take(self, n):
        if n < 0:
            raise GGUFError('负长度')
        self._need(n)
        out = bytes(self.buf[self.pos:self.pos + n])
        self.pos += n
        return out

    def skip(self, n):
        self._need(n)
        self.pos += n

    def unpack(self, fmt):
        return struct.unpack(fmt, self.take(struct.calcsize(fmt)))[0]

    def string(self):
        n = self.unpack('<Q')
        if n > 64 * 1024 ** 2:
            raise GGUFError('GGUF 字符串长度异常')
        return self.take(n).decode('utf-8')


def _value(cur, vtype, key):
    if vtype in _SCALAR:
        return cur.unpack(_SCALAR[vtype])
    if vtype == 8:
        text = cur.string()
        return text if len(text) <= 4096 else {'omitted_string_length': len(text)}
    if vtype == 9:
        etype = cur.unpack('<I')
        count = cur.unpack('<Q')
        if count > 50_000_000:
            raise GGUFError('GGUF 数组长度异常')
        if etype == 8:
            if key in _KEEP_STRING_ARRAYS:
                return [cur.string() for _ in range(count)]
            for _ in range(count):
                cur.skip(cur.unpack('<Q'))
            return {'omitted_array': 'string', 'count': count}
        if etype in _SCALAR:
            cur.skip(struct.calcsize(_SCALAR[etype]) * count)
            return {'omitted_array': etype, 'count': count}
        if etype == 9:
            for _ in range(count):
                _value(cur, 9, '')
            return {'omitted_array': 'array', 'count': count}
        raise GGUFError(f'未知 GGUF 数组元素类型 {etype}')
    raise GGUFError(f'未知 GGUF 值类型 {vtype}')


_SPLIT = re.compile(r'-\d{5}-of-\d{5}\.gguf$')
_QUANT_RANK = (('BF16', 0), ('F16', 1), ('Q8_0', 2), ('Q6_K', 3))


def pick_file(names):
    """The GGUF file of a repository to read when none is named: a single-file (not split) model GGUF in a type
    this reader can dequantize, preferring BF16 > F16 > Q8_0 > Q6_K and a regular file over an MTP one; ties go to the
    shorter, then alphabetically first name. Returns (file, the candidates considered)."""
    found = []
    for name in names:
        leaf = name.rsplit('/', 1)[-1]
        if not leaf.lower().endswith('.gguf') or 'mmproj' in leaf.lower() or _SPLIT.search(leaf):
            continue
        for label, rank in _QUANT_RANK:
            if re.search(r'(?i)(?:^|[-_.])' + label + r'(?:[-_.]|$)', leaf[:-5] + '.'):
                found.append(((rank, 'mtp' in leaf.lower(), len(name), name), name))
                break
    if not found:
        raise GGUFError('仓库里没有单文件的 BF16 / F16 / Q8_0 / Q6_K GGUF（这个读取器不支持其他 K 系、IQ 系量化和分卷文件）；'
                        '请在 SETTINGS 里用 file 指定文件名。现有的 GGUF：' + ', '.join(sorted(n for n in names if n.lower().endswith('.gguf')))[:600])
    found.sort()
    return found[0][1], [n for _, n in found]


def tensor_nbytes(ne, ggml_type):
    name, block, size = GGML_TYPES[ggml_type]
    count = math.prod(ne) if ne else 1
    if not ne or ne[0] % block:
        raise GGUFError(f'{name} 张量的首维 {ne[:1]} 不是块大小 {block} 的倍数')
    return count // block * size


def parse_header(fetch, size):
    """Parse a little-endian GGUF v2/v3 header. Returns metadata, tensors and the data offset."""
    cur = _Cursor(fetch, size)
    if cur.take(4) != b'GGUF':
        raise GGUFError('不是 GGUF 文件')
    version = cur.unpack('<I')
    if version not in (2, 3):
        raise GGUFError(f'不支持的 GGUF 版本 {version}（大端文件和 v1 文件不支持）')
    n_tensors = cur.unpack('<Q')
    n_kv = cur.unpack('<Q')
    if n_tensors > 1_000_000 or n_kv > 1_000_000:
        raise GGUFError('GGUF 计数异常')
    kv = {}
    for _ in range(n_kv):
        key = cur.string()
        kv[key] = _value(cur, cur.unpack('<I'), key)
    tensors = {}
    for _ in range(n_tensors):
        name = cur.string()
        n_dims = cur.unpack('<I')
        if not 1 <= n_dims <= 4:
            raise GGUFError(f'{name}: 维数 {n_dims} 异常')
        ne = [cur.unpack('<Q') for _ in range(n_dims)]
        ggml_type = cur.unpack('<I')
        offset = cur.unpack('<Q')
        if name in tensors:
            raise GGUFError(f'重复张量名 {name}')
        tensors[name] = {'ne': ne, 'type': ggml_type, 'offset': offset}
    alignment = kv.get('general.alignment', 32)
    if not isinstance(alignment, int) or alignment <= 0 or alignment & (alignment - 1):
        raise GGUFError('general.alignment 必须是 2 的幂')
    data_start = -(-cur.pos // alignment) * alignment
    spans = []
    for name, info in tensors.items():
        if info['type'] not in GGML_TYPES:
            info['nbytes'] = None  # unsupported type: listed, never read
            continue
        info['nbytes'] = tensor_nbytes(info['ne'], info['type'])
        start = data_start + info['offset']
        if info['offset'] % alignment or start + info['nbytes'] > size:
            raise GGUFError(f'{name}: 数据偏移越界或未对齐')
        spans.append((start, start + info['nbytes'], name))
    spans.sort()
    for a, b in zip(spans, spans[1:]):
        if a[1] > b[0]:
            raise GGUFError(f'张量数据重叠：{a[2]} / {b[2]}')
    return {'version': version, 'kv': kv, 'tensors': tensors, 'data_start': data_start,
            'header_bytes': cur.pos, 'file_size': int(size)}


def dequantize(data, ggml_type, rows, cols):
    """Raw bytes -> (float32 [rows, cols], per-row quantization noise energy, float64 [rows]).

    The noise of Q8_0 rounding is modelled as uniform on [-d/2, d/2] for each of the 32 values
    of a block with scale d, i.e. 32*d^2/12 per block. Q6_K (llama.cpp block_q6_K: 128 bytes of low nibbles,
    64 bytes of high 2-bit pairs, 16 int8 sub-block scales, one float16 super-block scale d; value
    = d * scale_j * (q - 32) for the 16 values of sub-block j) is modelled the same way with step d * scale_j:
    16*(d*scale_j)^2/12 per sub-block. Float types carry no extra noise here.
    """
    name = GGML_TYPES[ggml_type][0]
    raw = np.frombuffer(data, dtype=np.uint8)
    if name == 'Q8_0':
        blocks = raw.reshape(-1, 34)
        scale = blocks[:, :2].copy().view('<f2').astype(np.float32).reshape(-1)
        values = (blocks[:, 2:].view(np.int8).astype(np.float32) * scale[:, None]).reshape(rows, cols)
        noise = (scale.astype(np.float64) ** 2).reshape(rows, cols // 32).sum(axis=1) * (32.0 / 12.0)
    elif name == 'Q6_K':
        blocks = raw.reshape(-1, 210)
        n = blocks.shape[0]
        ql = blocks[:, :128].reshape(n, 2, 64)
        qh = blocks[:, 128:192].reshape(n, 2, 32)
        lo, hi = ql[:, :, :32], ql[:, :, 32:]
        # The 128 values of each half: (lo & 15, hi & 15, lo >> 4, hi >> 4) with the high bits qh >> 0, 2, 4, 6.
        q = np.concatenate([(lo & 15) | (((qh >> 0) & 3) << 4), (hi & 15) | (((qh >> 2) & 3) << 4),
                            (lo >> 4) | (((qh >> 4) & 3) << 4), (hi >> 4) | (((qh >> 6) & 3) << 4)], axis=2)
        q = q.reshape(n, 16, 16).astype(np.float32) - np.float32(32)
        d = blocks[:, 208:210].copy().view('<f2').astype(np.float32).reshape(n, 1)
        step = d * blocks[:, 192:208].copy().view(np.int8).astype(np.float32)          # d * scale, as ggml forms it
        values = (step[:, :, None] * q).reshape(rows, cols)
        noise = (step.astype(np.float64) ** 2).reshape(rows, -1).sum(axis=1) * (16.0 / 12.0)
    elif name == 'F32':
        values = raw.view('<f4').astype(np.float32).reshape(rows, cols)
        noise = np.zeros(rows)
    elif name == 'F16':
        values = raw.view('<f2').astype(np.float32).reshape(rows, cols)
        noise = np.zeros(rows)
    elif name == 'BF16':
        values = (raw.view('<u2').astype(np.uint32) << 16).view(np.float32).reshape(rows, cols)
        noise = np.zeros(rows)
    else:
        raise GGUFError(f'不支持的 GGUF 类型 {name}')
    if not np.isfinite(values).all():
        raise GGUFError('GGUF 数据含 NaN/Inf')
    return values, noise


class GGUFFile:
    """One GGUF file at one pinned commit, read with the engine's checked HTTP byte ranges."""
    def __init__(self, engine, repo, filename, cache_dir):
        self.engine, self.repo, self.filename = engine, repo, filename
        self.commit = engine.pin(repo)
        if filename not in engine.files(repo):
            raise GGUFError(f'{repo}@{self.commit[:12]} 中没有 {filename}')
        size = engine._sizes[repo][filename]
        identity = [repo, self.commit, filename, size, VERSION]
        path = Path(cache_dir) / (hashlib.sha256(json.dumps(identity).encode()).hexdigest()[:32] + '.json')
        header = None
        if path.exists():
            try:
                saved = json.loads(path.read_text(encoding='utf-8'))
                # A header cached before a type became readable lists that type's tensors as unreadable: parse again.
                if saved.get('identity') == identity and not any(
                        t['type'] in GGML_TYPES and t.get('nbytes') is None for t in saved['header']['tensors'].values()):
                    header = saved['header']
            except (OSError, ValueError, KeyError):
                header = None  # an interrupted Drive write is re-parsed, not trusted
        if header is None:
            header = parse_header(lambda a, b: engine._range(repo, filename, a, b), size)
            path.parent.mkdir(parents=True, exist_ok=True)
            tmp = path.with_name(path.name + '.tmp')
            tmp.write_text(json.dumps({'identity': identity, 'header': header}, ensure_ascii=False), encoding='utf-8')
            tmp.replace(path)
        self.header = header
        self.kv = header['kv']
        self.tensors = header['tensors']
        self.arch = self.kv.get('general.architecture', '')
        self.identity = {'kind': 'gguf', 'repo': repo, 'commit': self.commit, 'file': filename,
                         'file_size': size, 'architecture': self.arch, 'reader': VERSION}

    def type_name(self, name):
        return GGML_TYPES.get(self.tensors[name]['type'], ('type%d' % self.tensors[name]['type'],))[0]

    def rows_of(self, name):
        ne = self.tensors[name]['ne']
        return (ne[1] if len(ne) > 1 else 1), ne[0]

    def read_rows(self, name, first, last):
        info = self.tensors[name]
        if info.get('nbytes') is None:
            raise GGUFError(f'{name}: 不支持的 GGUF 类型 {self.type_name(name)}')
        if len(info['ne']) > 2:
            raise GGUFError(f'{name}: 只读取二维张量')
        rows, cols = self.rows_of(name)
        if not 0 <= first <= last <= rows:
            raise GGUFError(f'{name}: 行区间越界')
        _, block, size = GGML_TYPES[info['type']]
        row_bytes = cols // block * size
        start = self.header['data_start'] + info['offset']
        limit = start + info['nbytes'] - 1
        out = np.empty((last - first, cols), dtype=np.float32)
        noise = np.empty(last - first, dtype=np.float64)
        step = max(1, int(getattr(self.engine, 'block_bytes', 16 * 1024 ** 2)) // row_bytes)
        for i in range(first, last, step):
            j = min(i + step, last)
            data = self.engine._read(self.repo, self.filename, start + i * row_bytes, start + j * row_bytes - 1, limit)
            out[i - first:j - first], noise[i - first:j - first] = dequantize(data, info['type'], j - i, cols)
        return out, noise


# Hugging Face language weight suffix -> llama.cpp tensor base name (qwen35 / qwen3next).
HF_TO_GGUF = {
    'self_attn.q_proj': 'attn_q', 'self_attn.k_proj': 'attn_k', 'self_attn.v_proj': 'attn_v',
    'self_attn.o_proj': 'attn_output',
    'linear_attn.in_proj_qkv': 'attn_qkv', 'linear_attn.in_proj_z': 'attn_gate',
    'linear_attn.in_proj_a': 'ssm_alpha', 'linear_attn.in_proj_b': 'ssm_beta', 'linear_attn.out_proj': 'ssm_out',
    'mlp.gate_proj': 'ffn_gate', 'mlp.up_proj': 'ffn_up', 'mlp.down_proj': 'ffn_down',
}
_LAYER = re.compile(r'^(?:model\.language_model\.|model\.|language_model\.)layers\.(\d+)\.(.+)\.weight$')


def grouped_from_tiled(num_k_heads, per_k, head_dim):
    """Index array p with hf = gguf[p] for llama.cpp's grouped->tiled V-head reorder.

    llama.cpp (_LinearAttentionVReorderBase._reorder_v_heads) writes
    gguf[(v*num_k + k)*head_dim + d] = hf[(k*per_k + v)*head_dim + d].
    """
    n = num_k_heads * per_k * head_dim
    return np.arange(n).reshape(per_k, num_k_heads, head_dim).transpose(1, 0, 2).reshape(n)


def text_dims(config, gfile=None):
    """Linear-attention sizes from the reference config; GGUF metadata fills a key the config lacks."""
    tc = config.get('text_config', config)
    kv, arch = (gfile.kv, gfile.arch) if gfile is not None else ({}, '')
    def pick(key, gguf_key, scale=1):
        if tc.get(key) is not None:
            return int(tc[key])
        value = kv.get(f'{arch}.{gguf_key}')
        if value is None:
            raise GGUFError(f'参考 config 与 GGUF 都没有 {key}')
        return int(value) // scale
    nk = pick('linear_num_key_heads', 'ssm.group_count')
    nv = pick('linear_num_value_heads', 'ssm.time_step_rank')
    dk = pick('linear_key_head_dim', 'ssm.state_size')
    dv = int(tc['linear_value_head_dim']) if tc.get('linear_value_head_dim') is not None else pick('linear_value_head_dim', 'ssm.inner_size', nv)
    if nv % nk:
        raise GGUFError('linear_num_value_heads 不是 linear_num_key_heads 的倍数')
    return {'nk': nk, 'nv': nv, 'dk': dk, 'dv': dv, 'hidden': int(tc['hidden_size']),
            'layers': int(tc['num_hidden_layers'])}


def check_dimensions(gfile, dims):
    """The GGUF's own hyper-parameters must describe the same network as the reference config."""
    arch, kv, problems = gfile.arch, gfile.kv, []
    expect = {'embedding_length': dims['hidden'], 'ssm.group_count': dims['nk'], 'ssm.time_step_rank': dims['nv'],
              'ssm.state_size': dims['dk'], 'ssm.inner_size': dims['nv'] * dims['dv']}
    for key, value in expect.items():
        got = kv.get(f'{arch}.{key}')
        if got is not None and int(got) != value:
            problems.append(f'{arch}.{key}={got}，参考模型为 {value}')
    blocks = kv.get(f'{arch}.block_count')
    if blocks is not None and int(blocks) < dims['layers']:
        problems.append(f'{arch}.block_count={blocks} 少于 {dims["layers"]} 层')
    # An MTP file counts its next-token-prediction blocks in block_count; anything beyond those means
    # inserted (expanded) layers, whose blk.N no longer lines up with layer N of the reference.
    nextn = int(kv.get(f'{arch}.nextn_predict_layers') or 0)
    if blocks is not None and int(blocks) > dims['layers'] + nextn:
        problems.append(f'{arch}.block_count={blocks}，多于参考模型的 {dims["layers"]} 层（另有 MTP {nextn} 层）：'
                        '这像是加层扩展过的模型，按层号对齐会错位')
    if problems:
        raise GGUFError('GGUF 与参考模型结构不同：' + '；'.join(problems))


def candidate_layouts(name, shape, shapes, gfile, dims):
    """All plausible ways the HF tensor `name` may be stored in this GGUF (usually exactly one)."""
    if len(shape) != 2:
        return []
    if name.endswith('lm_head.weight'):
        return [{'tensor': 'output.weight', 'offset': 0, 'label': 'plain'}] if 'output.weight' in gfile.tensors else []
    m = _LAYER.match(name)
    if m is None or m[2] not in HF_TO_GGUF:
        return []
    layer, suffix = int(m[1]), m[2]
    base = f'blk.{layer}.{HF_TO_GGUF[suffix]}.weight'
    nk, nv, dk, dv = dims['nk'], dims['nv'], dims['dk'], dims['dv']
    per_k = nv // nk
    if suffix.startswith('self_attn.') and suffix != 'self_attn.o_proj' and base not in gfile.tensors:
        fused = f'blk.{layer}.attn_qkv.weight'  # llama.cpp --fuse-qkv: rows are [q; k; v]
        prefix = name[:-len(suffix) - len('.weight')]
        parts = [prefix + p + '.weight' for p in ('self_attn.q_proj', 'self_attn.k_proj', 'self_attn.v_proj')]
        if fused in gfile.tensors and all(p in shapes for p in parts):
            index = parts.index(name)
            return [{'tensor': fused, 'offset': sum(shapes[p][0] for p in parts[:index]), 'label': 'fused_qkv'}]
        return []
    if base not in gfile.tensors:
        return []
    plain = {'tensor': base, 'offset': 0, 'label': 'plain'}
    if nk == nv or not suffix.startswith('linear_attn.'):
        return [plain]
    tiled = dict(tensor=base, offset=0, label='tiled_v_heads')
    if suffix == 'linear_attn.in_proj_qkv':
        tiled['row_perm'] = (2 * nk * dk, nk, per_k, dv)
    elif suffix == 'linear_attn.in_proj_z':
        tiled['row_perm'] = (0, nk, per_k, dv)
    elif suffix in ('linear_attn.in_proj_a', 'linear_attn.in_proj_b'):
        tiled['row_perm'] = (0, nk, per_k, 1)
    elif suffix == 'linear_attn.out_proj':
        tiled['col_perm'] = (nk, per_k, dv)
    else:
        return [plain]
    # Converters since llama.cpp PR 19468 write tiled V heads; older files keep the grouped order.
    return [tiled, {**plain, 'label': 'grouped_v_heads'}]


def _row_index(layout, rows):
    if 'row_perm' not in layout:
        return None
    start, nk, per_k, hd = layout['row_perm']
    index = np.arange(rows)
    index[start:start + nk * per_k * hd] = start + grouped_from_tiled(nk, per_k, hd)
    return index


def _col_index(layout):
    if 'col_perm' not in layout:
        return None
    nk, per_k, hd = layout['col_perm']
    return grouped_from_tiled(nk, per_k, hd)


def vocabulary_mask(gfile, subject_tokens, rows):
    """Head rows whose token string is identical at the same id in the GGUF and in S."""
    tokens = gfile.kv.get('tokenizer.ggml.tokens')
    mask = np.zeros(rows, dtype=bool)
    if not isinstance(tokens, list):
        return mask, 'GGUF 没有 tokenizer.ggml.tokens；输出头不参与'
    by_id = {index: token for token, index in subject_tokens.items()}
    for index in range(min(rows, len(tokens))):
        if by_id.get(index) == tokens[index]:
            mask[index] = True
    return mask, None


def resolve_tensor(gfile, name, shape, layouts, reference_rows, intervals, row_mask=None,
                   max_rel=0.5, min_margin=2.0, noise_shrink=True):
    """Load the GGUF tensor, choose its layout against the reference, and size its noise.

    Returns (values or None, record). `values` is float32 in Hugging Face layout. The chosen
    layout must give ||W - ref|| / ||ref|| <= max_rel and, when several layouts are possible,
    be at least `min_margin` times better than the next one. The shrink factor
    max(0, 1 - noise / ||W - ref||^2) removes the expected Q8_0 rounding energy from the delta.
    """
    rows, cols = shape
    record = {'tensor': name, 'status': 'excluded'}
    if not layouts:
        record['reason'] = 'GGUF 中没有对应张量'
        return None, record
    first = layouts[0]
    record.update(gguf=first['tensor'], type=gfile.type_name(first['tensor']))
    g_rows, g_cols = gfile.rows_of(first['tensor'])
    if g_cols != cols or g_rows < first['offset'] + rows:
        record['reason'] = f'形状不符：GGUF {[g_rows, g_cols]}，需要 {[rows, cols]}'
        return None, record
    if gfile.tensors[first['tensor']].get('nbytes') is None:
        record['reason'] = f'GGUF 类型 {record["type"]} 未实现'
        return None, record
    raw, noise = gfile.read_rows(first['tensor'], first['offset'], first['offset'] + rows)
    mask = np.ones(rows, dtype=bool) if row_mask is None else np.asarray(row_mask, dtype=bool)
    if not mask.any():
        record['reason'] = '没有可用的行'
        return None, record
    indices = [(_row_index(l, rows), _col_index(l)) for l in layouts]
    err = [0.0] * len(layouts)
    ref_sq = 0.0
    for i, j in intervals:
        ref = np.asarray(reference_rows(i, j), dtype=np.float32).reshape(j - i, cols)
        keep = mask[i:j]
        ref_sq += float(np.square(ref[keep], dtype=np.float64).sum())
        for k, (ri, ci) in enumerate(indices):
            block = raw[i:j] if ri is None else raw[ri[i:j]]
            if ci is not None:
                block = block[:, ci]
            err[k] += float(np.square(block[keep] - ref[keep], dtype=np.float64).sum())
    rel = [math.sqrt(e / ref_sq) if ref_sq > 0 else math.inf for e in err]
    order = sorted(range(len(layouts)), key=lambda k: rel[k])
    best = order[0]
    record['relative_difference'] = {layouts[k]['label']: rel[k] for k in range(len(layouts))}
    record['layout'] = layouts[best]['label']
    if not rel[best] <= max_rel:
        record['reason'] = f'与参考模型的相对差 {rel[best]:.3f} 超过 {max_rel}：映射或来源不对'
        return None, record
    if len(order) > 1 and not rel[order[1]] >= min_margin * rel[best]:
        record['reason'] = '几种存储布局的差异不够明显，无法确定哪一种正确'
        return None, record
    ri, ci = indices[best]
    values = raw if ri is None else raw[ri]
    if ci is not None:
        values = values[:, ci]
    row_noise = noise if ri is None else noise[ri]
    noise_sq = float(row_noise[mask].sum())
    delta_sq = err[best]
    gain = max(0.0, 1.0 - noise_sq / delta_sq) if noise_shrink and delta_sq > 0 else (1.0 if delta_sq > 0 else 0.0)
    record.update(status='ok', delta_sq=delta_sq, noise_sq=noise_sq, reference_sq=ref_sq, gain=gain,
                  rows_used=int(mask.sum()))
    return np.ascontiguousarray(values, dtype=np.float32), record


class GGUFSource:
    """Tensor source for the aux-delta pass: HF-named 2-D weights of one GGUF, layout-checked."""
    def __init__(self, gfile, subject, config, policy):
        self.gfile, self.subject = gfile, subject
        self.dims = text_dims(config, gfile)
        check_dimensions(gfile, self.dims)
        self.policy = {'max_rel': float(policy.get('max_rel', 0.5)), 'min_margin': float(policy.get('min_margin', 2.0)),
                       'noise_shrink': bool(policy.get('noise_shrink', True))}
        self.layouts = {n: candidate_layouts(n, s, subject.shapes, gfile, self.dims) for n, s in subject.shapes.items()}
        self.head_mask, self.head_note = None, None
        self.identity = {**gfile.identity, 'policy': self.policy, 'dims': self.dims}
        mapped = sum(bool(v) for v in self.layouts.values())
        print(f'{gfile.repo} ({gfile.filename}, {gfile.arch}): 可对应的二维张量 {mapped} 个', flush=True)

    def has(self, name):
        return bool(self.layouts.get(name))

    def head_rows(self, name):
        if self.head_mask is None:
            rows = self.subject.shapes[name][0]
            self.head_mask, self.head_note = vocabulary_mask(self.gfile, self.subject.tokenizer['tokens'], rows)
        return self.head_mask

    def resolve(self, name, reference_rows, intervals, is_head=False):
        """(float32 tensor in HF layout or None, shrink factor, row support or None, record)."""
        mask = self.head_rows(name) if is_head else None
        values, record = resolve_tensor(self.gfile, name, tuple(self.subject.shapes[name]), self.layouts.get(name, []),
                                        lambda i, j: reference_rows(i, j).numpy(), intervals, mask,
                                        self.policy['max_rel'], self.policy['min_margin'], self.policy['noise_shrink'])
        if is_head and self.head_note:
            record['note'] = self.head_note
        if values is None:
            return None, 0.0, None, record
        support = torch.from_numpy(np.ones(values.shape[0], dtype=bool) if mask is None else mask.copy())
        return torch.from_numpy(values), float(record['gain']), support, record
'''
load_embedded_module('qmerge_gguf', MODULE_SOURCE)
del MODULE_SOURCE

In [ ]:
# 内嵌源代码；独立运行无需额外文件。
MODULE_SOURCE = r'''
"""DELLA for row blocks of task deltas, in NumPy: drop by magnitude rank, rescale, elect signs, fuse.

Same conventions as the DELLA build of the A3 donor (mergekit-style):
  drop     within each row, rank the entries of one source's delta by magnitude; the entry of rank r
           (0 = smallest, n-1 = largest) is kept with probability (density - epsilon) + 2*epsilon*r/(n-1)
           and a kept entry is divided by that probability;
  elect    the sign of the weighted sum of the pruned deltas wins (a zero sum counts as positive);
  fuse     the weighted mean of the pruned entries that have the elected sign (entries that are zero
           or have the other sign do not count), times lambda. normalize=False gives the weighted sum.

Everything is a function of whole rows, and the random draws are generated per fixed block of
BLOCK_ROWS rows from a counter-free PCG64 stream seeded by (seed, source, tensor, block). The result
therefore does not depend on how the caller cuts a tensor into row intervals, on the NumPy version
(only raw bit-generator output is used) or on the sort algorithm (sort keys are unique).

The kernels below are written for memory traffic (few passes, no masked ufuncs, no mixed-type ufuncs) and
are cut into row chunks that run on a thread pool. Every value they produce is bit for bit the value of the
plain formulation above; VERSION names that formulation and therefore does not change with the kernels.
"""
import hashlib
import math
import sys
from concurrent.futures import ThreadPoolExecutor

import numpy as np

VERSION = 'della-2026-10-11.1'
BLOCK_ROWS = 16
DEFAULTS = {'density': 0.7, 'epsilon': 0.07, 'lambda': 1.0, 'seed': 42, 'normalize': True}
_POOLS = {}
_LITTLE = sys.byteorder == 'little'
_TWO_M24 = np.float32(2.0 ** -24)
CACHE_ELEMENTS = 1 << 18


def validate(params=None):
    p = {**DEFAULTS, **(params or {})}
    if set(p) - set(DEFAULTS):
        raise ValueError(f'未知的 DELLA 参数：{sorted(set(p) - set(DEFAULTS))}')
    density, epsilon, lam = float(p['density']), float(p['epsilon']), float(p['lambda'])
    if not all(math.isfinite(v) for v in (density, epsilon, lam)):
        raise ValueError('DELLA 参数必须是有限数')
    if not 0 < density <= 1 or epsilon < 0:
        raise ValueError('DELLA density 必须在 (0, 1]，epsilon 不能为负')
    if density < 1 and not (density - epsilon > 0 and density + epsilon <= 1):
        raise ValueError('DELLA 需要 0 < density - epsilon 且 density + epsilon <= 1')
    if isinstance(p['seed'], bool) or not isinstance(p['seed'], int) or p['seed'] < 0:
        raise ValueError('DELLA seed 必须是非负整数')
    if lam <= 0:
        raise ValueError('DELLA lambda 必须为正数')
    return {'density': density, 'epsilon': epsilon, 'lambda': lam, 'seed': int(p['seed']), 'normalize': bool(p['normalize'])}


def keep_probabilities(n, density, epsilon):
    """Keep probability by ascending magnitude rank, float32 [n]."""
    if n == 1 or density >= 1:
        return np.full(n, density, dtype=np.float32)
    rank = np.arange(n, dtype=np.float64)
    return ((density - epsilon) + (2.0 * epsilon / (n - 1)) * rank).astype(np.float32)


def uniforms(seed_key, first_row, last_row, cols, out=None):
    """Uniform [0, 1) float32 draws for rows first_row..last_row-1; identical for any way of cutting the rows.

    One 64-bit draw per value; its top 24 bits m give the float m * 2**-24 (exact in float32).
    """
    if out is None:
        out = np.empty((last_row - first_row, cols), dtype=np.float32)
    block = first_row // BLOCK_ROWS
    while block * BLOCK_ROWS < last_row:
        start = block * BLOCK_ROWS
        seed = int.from_bytes(hashlib.sha256(f'{seed_key}|{block}'.encode('utf-8')).digest()[:16], 'little')
        raw = np.random.PCG64(seed).random_raw(BLOCK_ROWS * cols).reshape(BLOCK_ROWS, cols)
        a, b = max(first_row, start), min(last_row, start + BLOCK_ROWS)
        if _LITTLE:
            # the top 24 bits of a draw are the top 24 bits of its upper 32-bit half: no 64-bit temporaries
            high = raw.view(np.uint32).reshape(BLOCK_ROWS, cols, 2)[a - start:b - start, :, 1]
            np.multiply(high >> np.uint32(8), _TWO_M24, out=out[a - first_row:b - first_row], dtype=np.float32, casting='unsafe')
        else:
            out[a - first_row:b - first_row] = (raw[a - start:b - start] >> np.uint64(40)).astype(np.float32) * _TWO_M24
        block += 1
    return out


def rank_probabilities(delta, density, epsilon):
    """Per-entry keep probability from the row-wise magnitude rank; ties are ordered by column."""
    rows, n = delta.shape
    table = keep_probabilities(n, density, epsilon)
    if n == 1 or density >= 1:
        return np.broadcast_to(table, (rows, n)).copy()
    # |x| of a finite float32 orders like its bit pattern, so (bits << 32 | column) is a unique, ordered key.
    key = np.empty((rows, n), dtype=np.uint64)
    if _LITTLE:
        halves = key.view(np.uint32).reshape(rows, n, 2)           # [..., 0] low half (column), [..., 1] high half
        halves[:, :, 0] = np.arange(n, dtype=np.uint32)
        np.abs(delta, out=halves[:, :, 1].view(np.float32))        # the bits of |x| go straight into the high half
    else:
        key[...] = np.abs(delta).view(np.uint32)
        key <<= np.uint64(32)
        key |= np.arange(n, dtype=np.uint64)[None, :]
    key.sort(axis=1)
    # rank k of row r sits at column (key & 0xFFFFFFFF): one flat scatter of the table
    key &= np.uint64(0xFFFFFFFF)
    key += (np.arange(rows, dtype=np.uint64) * np.uint64(n))[:, None]
    prob = np.empty(rows * n, dtype=np.float32)
    prob[key.view(np.intp)] = table
    return prob.reshape(rows, n)


def magprune(delta, seed_key, first_row, density, epsilon, out=None):
    """Drop and rescale one source's delta rows. Returns float32 [rows, n] (written to `out` when given)."""
    delta = np.ascontiguousarray(delta, dtype=np.float32)
    if delta.ndim != 2:
        raise ValueError('DELLA 只处理二维行块')
    if density >= 1:
        if out is None:
            return delta.copy()
        out[...] = delta
        return out
    prob = rank_probabilities(delta, density, epsilon)
    keep = uniforms(seed_key, first_row, first_row + delta.shape[0], delta.shape[1]) < prob
    with np.errstate(divide='ignore', invalid='ignore'):     # a dropped entry may have probability 0; it is masked below
        out = np.divide(delta, prob, out=prob if out is None else out)
    # dropped entries become +0.0, kept ones keep every bit: AND the bit patterns with an all-ones / all-zeros mask
    mask = keep.astype(np.uint32)
    mask *= np.uint32(0xFFFFFFFF)
    bits = out.view(np.uint32)
    bits &= mask
    return out


def fuse(pruned, weights, normalize=True, lam=1.0, out=None):
    """Sign election by weighted sum (zero counts as positive), then the weighted mean of agreeing entries.

    An entry agrees when it is non-zero and has the elected sign: x * sign > 0. Its contribution x (times its
    weight) is formed as max(x * sign, 0) * sign, which is x itself for an agreeing entry and a zero otherwise;
    a zero of either sign never changes the sum, which starts at +0.0 and cannot become -0.0.
    """
    if len(pruned) != len(weights) or not pruned:
        raise ValueError('DELLA 来源与权重数量不符')
    weights = [np.float32(w) for w in weights]
    unit = all(w == 1 for w in weights)
    num = np.empty_like(pruned[0]) if out is None else out
    num.fill(0); tmp = np.empty_like(pruned[0])
    for x, w in zip(pruned, weights):             # first the weighted sum, for the election
        if w == 1:
            num += x                              # x * 1 is x
        else:
            np.multiply(x, w, out=tmp)
            num += tmp
    sign = np.where(num >= 0, np.float32(1), np.float32(-1))
    num.fill(0)                                   # reuse the buffer
    agree = np.empty(num.shape, dtype=bool); agreeing = []
    count = np.zeros(num.shape, dtype=np.uint8) if unit and len(pruned) < 256 else None
    den = None if count is not None else np.zeros_like(num)
    for x, w in zip(pruned, weights):
        np.multiply(x, sign, out=tmp)
        np.greater(tmp, 0, out=agree)             # non-zero and of the elected sign
        agreeing.append(int(np.count_nonzero(agree)))
        if count is not None:
            count += agree.view(np.uint8)
        else:
            den += np.where(agree, w, np.float32(0))
        np.maximum(tmp, np.float32(0), out=tmp)
        tmp *= sign
        if w != 1:
            tmp *= w
        num += tmp
    if normalize:
        if count is not None:
            np.maximum(count, 1, out=count)       # nothing agrees: the numerator is +0.0 and stays +0.0
            den = count.astype(np.float32)
        else:
            den += (den == 0)
        num /= den
    if lam != 1.0:
        num *= np.float32(lam)
    return num, agreeing


def _pool(threads):
    if threads not in _POOLS:
        _POOLS[threads] = ThreadPoolExecutor(max_workers=threads, thread_name_prefix='della')
    return _POOLS[threads]


def _row_chunks(first_row, rows, parts):
    """About `parts` consecutive row ranges, cut where a block of random draws starts (no block is drawn twice)."""
    if parts <= 1 or rows <= BLOCK_ROWS:
        return [(0, rows)]
    size = -(-rows // parts)
    cuts, a = [], 0
    while a < rows:
        b = a + size
        b -= (first_row + b) % BLOCK_ROWS           # end on a block boundary of the absolute row numbering
        if b <= a:
            b = a + BLOCK_ROWS - (first_row + a) % BLOCK_ROWS
        b = min(b, rows)
        cuts.append((a, b))
        a = b
    return cuts


def _step_rows(cols):
    """Rows handled at a time: whole blocks of random draws, about CACHE_ELEMENTS values, so that the dozen passes
    over one step (keys, sort, draws, masks, election) stay in the CPU cache instead of streaming through RAM."""
    return BLOCK_ROWS * max(1, CACHE_ELEMENTS // (BLOCK_ROWS * max(1, cols)))


def della_rows(deltas, weights, params, seed_keys, first_row, threads=1):
    """Fuse the row blocks `deltas` (one float32 [rows, n] array per source). Returns (float32 block, stats).

    The rows are worked through in cache-sized steps: all sources of a step are pruned and fused before the next
    step. threads > 1 spreads row ranges over a thread pool (sorting, the random draws and the array arithmetic
    release the GIL). The result depends on neither.
    """
    p = validate(params)
    if not (len(deltas) == len(weights) == len(seed_keys)):
        raise ValueError('DELLA 来源、权重与种子键数量不符')
    if any(not math.isfinite(float(w)) or w <= 0 for w in weights):
        raise ValueError('DELLA 来源权重必须为正数')
    deltas = [np.ascontiguousarray(d, dtype=np.float32) for d in deltas]
    rows, n = deltas[0].shape
    if any(d.shape != (rows, n) for d in deltas):
        raise ValueError('DELLA 来源的行块形状不一致')
    members = range(len(deltas))
    seeds = [f"{p['seed']}|{key}" for key in seed_keys]
    out = np.empty((rows, n), dtype=np.float32)
    row_sq = [np.empty(rows, dtype=np.float32) for _ in members]     # per-row sums of squares of every source
    step = _step_rows(n)

    def work(cut):
        a, b = cut
        nonzero, kept, elected = [0] * len(deltas), [0] * len(deltas), [0] * len(deltas)
        scratch = [np.empty((min(step, b - a), n), dtype=np.float32) for _ in members]
        i = a
        while i < b:
            j = min(b, i + step - (first_row + i) % BLOCK_ROWS)       # steps end where a block of draws ends
            parts = []
            for k in members:
                delta = deltas[k][i:j]
                if not np.isfinite(delta).all():
                    raise ValueError('DELLA 输入含 NaN/Inf')
                x = magprune(delta, seeds[k], first_row + i, p['density'], p['epsilon'], out=scratch[k][:j - i])
                row_sq[k][i:j] = np.einsum('ij,ij->i', delta, delta)
                nonzero[k] += int(np.count_nonzero(delta))
                kept[k] += int(np.count_nonzero(x))
                parts.append(x)
            agreeing = fuse(parts, weights, p['normalize'], p['lambda'], out=out[i:j])[1]
            for k in members:
                elected[k] += agreeing[k]
            i = j
        return nonzero, kept, elected

    threads = max(1, int(threads))
    cuts = _row_chunks(first_row, rows, threads)
    done = list(_pool(threads).map(work, cuts)) if threads > 1 and len(cuts) > 1 else [work(cut) for cut in cuts]
    # the per-row sums of squares are added up in one go, so the statistics do not depend on the cutting either
    stats = [{'delta_sq': float(row_sq[k].sum(dtype=np.float64)), 'nonzero': sum(r[0][k] for r in done),
              'kept': sum(r[1][k] for r in done), 'elected': sum(r[2][k] for r in done)} for k in members]
    return out, {'sources': stats, 'elements': int(out.size), 'nonzero': int(np.count_nonzero(out)),
                 'out_sq': float(np.einsum('ij,ij->i', out, out).sum(dtype=np.float64))}


def mean_rows(deltas, supports, weights):
    """Weighted mean of the deltas over the sources that really have each row (output head policy)."""
    num = np.zeros_like(deltas[0], dtype=np.float32); den = np.zeros((deltas[0].shape[0], 1), dtype=np.float32)
    for delta, ok, w in zip(deltas, supports, weights):
        ok = np.asarray(ok, dtype=bool).reshape(-1, 1)
        num += np.where(ok, delta * np.float32(w), np.float32(0))
        den += np.where(ok, np.float32(w), np.float32(0))
    out = np.zeros_like(num)
    np.divide(num, den, out=out, where=den > 0)
    return out


def merge_stats(total, part):
    """Accumulate della_rows statistics over row intervals."""
    if total is None:
        return {'sources': [dict(s) for s in part['sources']], 'elements': part['elements'], 'nonzero': part['nonzero'], 'out_sq': part['out_sq']}
    for a, b in zip(total['sources'], part['sources']):
        for k in a:
            a[k] += b[k]
    for k in ('elements', 'nonzero', 'out_sq'):
        total[k] += part[k]
    return total
'''
load_embedded_module('qmerge_della', MODULE_SOURCE)
del MODULE_SOURCE

In [ ]:
# 内嵌源代码；独立运行无需额外文件。
MODULE_SOURCE = r'''"""Transport-and-Merge delta N from a donor of another architecture onto B (Qwen3.8-27B), on a CPU runtime.

Same method as the multi-source T&M notebook (Qwen38_tm_multisource): per-document-mean activations on one shared
calibration corpus, Pearson costs, entropic OT with an adaptive eps inside every (target layer, donor layer) pair, an
OT over the layer-pair costs, then W~ = Q_out W_donor Q_in^T on the TOPK rows of each target matrix whose activation
is largest, centred, matched to the norm of B's row and mixed in with alpha:

    N[rows] = alpha * (acc[rows] - B[rows]),        F = S + lambda*[D - kappa*(S-B)] + mu*C + nu*N.

What is different here, so that it fits a CPU runtime:
  * the target activations are the ones that notebook already stored in its activation dataset (same calibration
    texts, same per-document means); they are checked and read, not recomputed;
  * the donor's forward runs layer by layer on the CPU in float32, its weights read by HTTP ranges with the next layer
    read ahead; documents of equal token length run together without padding (the per-document means equal those of
    the padded, masked GPU pass); the hidden states are checkpointed so that a new session resumes;
  * the go/no-go statistics (z multiple, sharpness against noise at the same eps, the self-match upper bound) come
    first, and the costly pair OT runs only for modules that pass;
  * inner OT runs only on the (target layer, donor layer) pairs a cheap linear-CKA prefilter keeps (pair_keep per
    target layer plus the depth-aligned donor layer). The layer-level OT then runs over the donor layers that some
    target layer kept, a missing pair counted at the largest cost, as in that notebook;
  * only the TOPK rows that change are transported (the same numbers as transporting every row and then masking).
The donor and B do not share a tokenizer; per-document means make that irrelevant to the statistics. Nothing here is a
claim that the donor's abilities move across: what moves is a small, row-normalised change in the directions the
transport finds. Whether it helps is decided by the proxy search (nu) and by the user's own tests.
"""
import gc
import json
import math
import os
import re
import shutil
import threading
import time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch

import qmerge_core as core
import qmerge_runtime as runtime

TM_VERSION = 'tm-cpu-2026-10-26.1'
ACTS_VERSION = 'tm-donor-acts-2026-10-26.1'      # changes only when the donor activations would come out different
OT_VERSION = 'tm-ot-2026-10-26.1'

DEFAULTS = {
    'enabled': True, 'tag': 'NanbeigeTM', 'repo': 'Nanbeige/Nanbeige4.3-33B', 'revision': '',
    'act_repo': 'YFC-112358/tm-acts-qwen38-multi', 'act_revision': '', 'calib_file': 'calib_2000.json',
    'docs': 2000, 'seq_len': 128,
    'modules': ['self_attn.v_proj', 'self_attn.o_proj', 'mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj'],
    'target_stride': 4, 'source_stride': 2,
    'pair_keep': 2, 'pair_depth_prior': True,
    'eps_delta': 0.01, 'eps_delta_out': 0.01, 'eps_floor': 5e-4, 'eps_cap': 0.10,
    'iters_inner': 200, 'iters_outer': 1000, 'tol': 1e-6,
    'top_j': 2, 'conc_min': 3.0, 'topk': 128, 'alpha': 0.05,
    'inject': 'row', 'inject_min_ratio': 0.05, 'inject_center': True, 'bcast_max': 0.35,
    'z_go': 2.0, 'sharp_margin': 1.5, 'marginal': 1.5, 'up_excess_max': 6.0, 'up_sharp_mult': 10.0,
    'noise_seed': 20260819, 'cka_seed': 0,
    'batch_docs': 16, 'chunk_docs': 500, 'checkpoint_every': 4, 'dtype': 'float32', 'attention': 'sdpa', 'gate_min_hits': 2,
    'act_cache_gib': 12.0, 'chunk_rows': 2048,
    'nu_grid': [0.5, 1.0], 'while_waiting': True, 'on_error': 'raise', 'keep_activations': True, 'dir': '',
}
GATE_PROMPTS = [
    ("The capital of France is", [" Paris", "Paris"]),
    ("1, 2, 3, 4,", [" 5", "5"]),
    ("The opposite of hot is", [" cold", "cold"]),
    ("Water freezes at zero degrees", [" Celsius", " C", "Celsius"]),
]
_TARGET_FILE = re.compile(r'^acts_acts-v2_target_([0-9a-f]{12})_L(\d{3})\.safetensors$')
_LAYER = re.compile(r'(?:^|\.)layers\.(\d+)\.')


def options(settings):
    raw = dict(settings.get('tm_extra') or {})
    unknown = set(raw) - set(DEFAULTS)
    if unknown:
        raise ValueError(f'tm_extra 里有不认识的项：{sorted(unknown)}')
    o = {**DEFAULTS, **raw}
    o['modules'] = list(o['modules'])
    if not o['enabled']:
        return o
    for k in ('repo', 'act_repo', 'calib_file'):
        if not isinstance(o[k], str) or not o[k].strip():
            raise ValueError(f'tm_extra["{k}"] 不能为空')
    for k in ('revision', 'act_revision'):
        if o[k] and not re.fullmatch(r'[0-9a-fA-F]{40,64}', str(o[k])):
            raise ValueError(f'tm_extra["{k}"] 要么留空，要么是完整 commit SHA')
    if not 2 <= int(o['docs']) <= 2000 or int(o['seq_len']) != 128:
        raise ValueError('tm_extra：docs 在 2..2000 之间（目标激活是 2000 篇算的），seq_len 只能是 128（目标激活的序列长）')
    bad = [m for m in o['modules'] if m not in DEFAULTS['modules'] + ['self_attn.k_proj']]
    if bad or not o['modules']:
        raise ValueError(f'tm_extra["modules"] 只能取 {DEFAULTS["modules"] + ["self_attn.k_proj"]}；q_proj 不搬（目标的 q_proj 混着门控通道）')
    if o['inject'] not in ('raw', 'row', 'tensor'):
        raise ValueError('tm_extra["inject"] 只能是 raw / row / tensor')
    if not 0 < float(o['alpha']) <= 0.5:
        raise ValueError('tm_extra["alpha"] 要在 (0, 0.5]')
    grid = [float(x) for x in o['nu_grid']]
    if not grid or any(not math.isfinite(x) or not 0 < x <= 2 for x in grid):
        raise ValueError('tm_extra["nu_grid"] 要给出 (0, 2] 内的系数')
    o['nu_grid'] = sorted(set(grid))
    o['require_all_donors'] = bool(settings.get('require_all_donors', True))
    if o['on_error'] not in ('raise', 'skip'):
        raise ValueError('tm_extra["on_error"] 只能是 raise / skip')
    if o['dtype'] not in ('float32', 'bfloat16'):
        raise ValueError('tm_extra["dtype"] 只能是 float32 / bfloat16')
    return o


def _clean(value):
    """JSON-safe: NaN / inf become None (records are written with allow_nan=False)."""
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, dict):
        return {str(k): _clean(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_clean(v) for v in value]
    if isinstance(value, (np.floating,)):
        return _clean(float(value))
    if isinstance(value, (np.integer,)):
        return int(value)
    return value


def _record(path, identity):
    try:
        return core.load_record(path, identity)
    except (ValueError, OSError, json.JSONDecodeError):
        return None


def _library_versions():
    import importlib.metadata
    out = {}
    for p in ('torch', 'transformers', 'accelerate', 'safetensors'):
        try:
            out[p] = importlib.metadata.version(p)
        except importlib.metadata.PackageNotFoundError:
            out[p] = None
    return out


# ============================================================================ the row-sparse delta N
class RowDelta:
    """N as the pipeline reads it: rows(name, i, j) gives the dense rows i..j of N[name] (float32, zero where N does
    not touch the row), like StoredDelta.rows."""
    def __init__(self, directory, fingerprint):
        from safetensors.torch import load_file
        self.directory = Path(directory)
        self.fingerprint = fingerprint
        data = load_file(str(self.directory / 'tm_delta.safetensors'))
        self.entries = {}
        for key in data:
            if key.endswith('|rows'):
                name = key[:-5]
                rows = data[key].long()
                delta = data[name + '|delta'].float()
                order = torch.argsort(rows)
                self.entries[name] = (rows[order].contiguous(), delta[order].contiguous())
        self.tags = ['N']

    def has(self, name):
        return name in self.entries

    def location(self, name):
        return None

    def rows(self, name, i, j):
        rows, delta = self.entries[name]
        out = torch.zeros(j - i, delta.shape[1], dtype=torch.float32)
        lo = int(torch.searchsorted(rows, torch.tensor(i)))
        hi = int(torch.searchsorted(rows, torch.tensor(j)))
        if hi > lo:
            out[rows[lo:hi] - i] = delta[lo:hi]
        return out

    def norms(self):
        return {name: float(d.norm()) for name, (_, d) in self.entries.items()}


def open_delta(result):
    """The RowDelta of a finished TM result (status 'go'), or None."""
    if not result or result.get('status') != 'go':
        return None
    return RowDelta(result['directory'], result['fingerprint'])


# ============================================================================ numpy OT core (same formulas as the TM notebook)
def zstd(X):
    """Per channel standardisation along the document axis; a constant channel becomes 0 (correlates with nothing)."""
    X = np.asarray(X, dtype=np.float32)
    X = X - X.mean(0, keepdims=True, dtype=np.float64).astype(np.float32)
    s = X.std(0, keepdims=True, dtype=np.float64).astype(np.float32)
    ok = s > 1e-8
    return np.where(ok, X / np.where(ok, s, 1.0), 0.0).astype(np.float32)


def lower_median(x, axis=None):
    """torch.median semantics (the lower of the two middle values), which the TM notebook used."""
    x = np.asarray(x)
    if axis is None:
        x = x.reshape(-1)
        k = (x.size - 1) // 2
        return np.partition(x, k)[k]
    k = (x.shape[axis] - 1) // 2
    return np.take(np.partition(x, k, axis=axis), k, axis=axis)


def cost_matrix(Xn, Yn, chunk=2048):
    """1 - Pearson correlation between every target channel (rows) and every donor channel (columns)."""
    T = Xn.shape[0]
    n, m = Xn.shape[1], Yn.shape[1]
    C = np.empty((n, m), dtype=np.float32)
    for a in range(0, n, chunk):
        b = min(a + chunk, n)
        np.matmul(Xn[:, a:b].T, Yn, out=C[a:b])
        C[a:b] *= np.float32(-1.0 / T)
        C[a:b] += np.float32(1.0)
    return C


def row_shift(C):
    """In place: every row minus its minimum (does not change the optimal plan; keeps exp(-C/eps) from underflowing)."""
    C -= C.min(axis=1, keepdims=True)
    return C


def auto_eps(C0, delta, floor, cap, chunk=2048):
    """eps = gap / ln(m / delta), gap = median over rows of the row median of the shifted cost."""
    m = C0.shape[1]
    meds = np.concatenate([lower_median(C0[a:a + chunk], axis=1) for a in range(0, C0.shape[0], chunk)])
    gap = float(lower_median(meds))
    if not gap > 0:
        gap = float(C0.std())
    eps = gap / math.log(m / max(delta, 1e-9))
    return float(min(max(eps, floor), cap)), gap


def kernel(C0, eps, chunk=2048):
    K = np.empty_like(C0)
    for a in range(0, C0.shape[0], chunk):
        np.multiply(C0[a:a + chunk], np.float32(-1.0 / eps), out=K[a:a + chunk])
        np.exp(K[a:a + chunk], out=K[a:a + chunk])
    return K


def sinkhorn(K, iters, tol):
    n, m = K.shape
    a = np.float32(1.0 / n)
    b = np.float32(1.0 / m)
    u = np.ones(n, dtype=np.float32)
    v = np.ones(m, dtype=np.float32)
    tiny = np.float32(1e-30)
    for it in range(iters):
        u = a / np.maximum(K @ v, tiny)
        v = b / np.maximum(K.T @ u, tiny)
        if it % 25 == 24:
            r = float(np.abs(u * (K @ v) - a).max())
            c = float(np.abs(v * (K.T @ u) - b).max())
            if max(r, c) < tol:
                break
            sc = max(float(np.abs(u).max()), 1e-30)
            u = (u / np.float32(sc)).astype(np.float32)
            v = (v * np.float32(sc)).astype(np.float32)
    return u, v


def sharpness(u, v, K, chunk=2048):
    """Effective support of a row of the plan / m (1 = uniform, 1/m = a single match)."""
    n, m = K.shape
    tot = 0.0
    for a in range(0, n, chunk):
        Q = (u[a:a + chunk, None] * K[a:a + chunk]) * v[None, :]
        P = Q / np.maximum(Q.sum(1, keepdims=True), np.float32(1e-30))
        H = -(P * np.log(np.maximum(P, np.float32(1e-30)))).sum(1, dtype=np.float64)
        tot += float(np.exp(H).sum())
    return tot / n / m


def transport_cost(u, v, K, C0, chunk=2048):
    tot = 0.0
    for a in range(0, K.shape[0], chunk):
        Q = (u[a:a + chunk, None] * K[a:a + chunk]) * v[None, :]
        tot += float((C0[a:a + chunk] * Q).sum(dtype=np.float64))
    return tot


def z_null(m):
    """Expected gap*sqrt(T) without information: the mean of the largest of m standard normals (finite correction)."""
    L = math.log(max(m, 2))
    a = math.sqrt(2 * L)
    return a - (math.log(L) + math.log(4 * math.pi)) / (2 * a)


def inner_ot(Xn, Yn, o, eps=None):
    """Entropic OT between the channels of two standardised activation matrices. eps given: use it (same-eps noise
    check); else adaptive. Returns u, v, eps and the statistics the verdict and the layer OT use."""
    C0 = row_shift(cost_matrix(Xn, Yn, o['chunk_rows']))
    eps_auto, gap = auto_eps(C0, o['eps_delta'], o['eps_floor'], o['eps_cap'], o['chunk_rows'])
    use = eps_auto if eps is None else float(eps)
    K = kernel(C0, use, o['chunk_rows'])
    u, v = sinkhorn(K, int(o['iters_inner']), float(o['tol']))
    shp = sharpness(u, v, K, o['chunk_rows'])
    cl = transport_cost(u, v, K, C0, o['chunk_rows'])
    n, m = C0.shape
    del C0, K
    T = Xn.shape[0]
    return {'u': u, 'v': v, 'eps': use, 'eps_auto': eps_auto, 'gap': gap, 'cl': cl, 'sharp': shp,
            'n': n, 'm': m, 'T': T, 'z': gap * math.sqrt(T), 'znull': z_null(m)}


def outer_ot(costs, TT, ST, o):
    """Layer-level OT over the pair costs {(target layer, donor layer): cost}; missing pairs at the largest cost."""
    L, M = len(TT), len(ST)
    Cm = np.full((L, M), np.nan)
    for x, li in enumerate(TT):
        for y, mj in enumerate(ST):
            if (li, mj) in costs:
                Cm[x, y] = costs[(li, mj)]
    if np.isnan(Cm).any():
        fill = float(np.nanmax(Cm)) if (~np.isnan(Cm)).any() else 1.0
        Cm = np.nan_to_num(Cm, nan=fill)
    Cm0 = Cm - Cm.min(axis=1, keepdims=True)
    eta, gap = auto_eps(Cm0, o['eps_delta_out'], o['eps_floor'], o['eps_cap'])
    K = np.exp(-Cm0 / eta)
    a = np.full(L, 1.0 / L)
    b = np.full(M, 1.0 / M)
    u = np.ones(L)
    v = np.ones(M)
    for it in range(int(o['iters_outer'])):
        u = a / np.maximum(K @ v, 1e-30)
        v = b / np.maximum(K.T @ u, 1e-30)
        if it % 50 == 49 and float(np.abs(u * (K @ v) - a).max()) < 1e-9:
            break
    P = (u[:, None] * K) * v[None, :]
    Pn = P / np.maximum(P.sum(1, keepdims=True), 1e-30)
    H = -(Pn * np.log(np.maximum(Pn, 1e-30))).sum(1)
    return P, float(np.exp(H).mean() / M), eta, gap


def effective_plan(costs_in, costs_out, TT, ST, o):
    """P_eff = sqrt(P_in * P_out) (Eq. 13), each row normalised to 1."""
    p_in, s_in, e_in, _ = outer_ot(costs_in, TT, ST, o)
    p_out, s_out, e_out, _ = outer_ot(costs_out, TT, ST, o)
    Pe = np.sqrt(np.clip(p_in * p_out, 0, None))
    Pe = Pe / np.maximum(Pe.sum(1, keepdims=True), 1e-30)
    return Pe, {'sharp': (s_in + s_out) / 2, 'eta': [e_in, e_out]}


def depth_match_ratio(Pe, TT, ST, target_layers, source_layers, band=0.10):
    Lt = max(target_layers - 1, 1)
    Ls = max(source_layers - 1, 1)
    num = den = 0.0
    for x, li in enumerate(TT):
        for y, mj in enumerate(ST):
            w = float(Pe[x, y])
            den += w
            if abs(li / Lt - mj / Ls) < band:
                num += w
    return num / max(den, 1e-30)


def gram(Z):
    """Normalised Gram matrix of a column-centred activation matrix: <gram(X), gram(Y)> is linear CKA."""
    G = (Z @ Z.T).astype(np.float32)
    return G / max(float(np.linalg.norm(G)), 1e-30)


def keep_pairs(scores, TT, ST, keep, depth_prior, target_layers, source_layers):
    """{target layer: [donor layers]}: the `keep` best by CKA, plus the depth-aligned one."""
    out = {}
    for x, li in enumerate(TT):
        order = np.argsort(-scores[x], kind='stable')
        chosen = [ST[int(y)] for y in order[:keep]]
        if depth_prior:
            frac = li / max(target_layers - 1, 1)
            near = min(ST, key=lambda mj: abs(mj / max(source_layers - 1, 1) - frac))
            if near not in chosen:
                chosen.append(near)
        out[li] = sorted(chosen)
    return out


def s1_share(M):
    """Share of the largest singular value in the energy (1/rows for random rows, near 1 for one broadcast row)."""
    try:
        s = np.linalg.svd(np.asarray(M, dtype=np.float64), compute_uv=False) ** 2
        return float(s[0] / max(s.sum(), 1e-30))
    except np.linalg.LinAlgError:
        return float('nan')


def plan_rows(Xn, Yn, u, v, eps, rows=None, chunk=2048):
    """Rows of the row-normalised plan Q = diag(u) exp(-rowshift(C)/eps) diag(v), recomputed from the activations
    exactly as the inner OT built it; yields (row indices, Q block)."""
    T = Xn.shape[0]
    index = np.arange(Xn.shape[1]) if rows is None else np.asarray(rows)
    for a in range(0, len(index), chunk):
        sel = index[a:a + chunk]
        C = np.float32(1.0) - (Xn[:, sel].T @ Yn) * np.float32(1.0 / T)
        C -= C.min(axis=1, keepdims=True)
        Q = (u[sel, None] * np.exp(C * np.float32(-1.0 / eps))) * v[None, :]
        Q /= np.maximum(Q.sum(1, keepdims=True), np.float32(1e-30))
        yield sel, Q.astype(np.float32)


def transport_rows(rows, WB, plan_out, plan_in, in_A, chunk=2048):
    """(Q_out W_B Q_in^T)[rows]: plan_out / plan_in are (Xn, Yn, u, v, eps) of the output / input side."""
    Xo, Yo, uo, vo, eo = plan_out
    Qo = next(plan_rows(Xo, Yo, uo, vo, eo, rows, chunk=max(len(rows), 1)))[1]
    Z = Qo @ WB                                    # (k, in_B)
    out = np.zeros((len(rows), in_A), dtype=np.float32)
    Xi, Yi, ui, vi, ei = plan_in
    if Xi.shape[1] != in_A:
        raise ValueError('输入侧计划的行数与目标矩阵的列数不同')
    for sel, Qi in plan_rows(Xi, Yi, ui, vi, ei, None, chunk):
        out[:, sel] = Z @ Qi.T
    return out


def inject(acc_rows, WA_rows, o):
    """The injection rule of the TM notebook (v5) on the masked rows: centre, check for a broadcast, scale each row to
    the norm of B's row (or the whole block to B's norm); returns the rows and diagnostics."""
    alpha = float(o['alpha'])
    shrink = float(np.linalg.norm(alpha * WA_rows))
    inj_raw = float(np.linalg.norm(alpha * acc_rows))
    bc0 = s1_share(acc_rows)
    bc1 = bc0
    mode = o['inject']
    acc = acc_rows
    if mode != 'raw':
        cand = acc_rows - acc_rows.mean(0, keepdims=True) if o['inject_center'] else acc_rows
        bc1 = s1_share(cand) if o['inject_center'] else bc0
        if bc1 > float(o['bcast_max']):
            mode = 'raw'
        else:
            acc = cand
            na = np.linalg.norm(acc, axis=1, keepdims=True)
            nw = np.linalg.norm(WA_rows, axis=1, keepdims=True)
            med = max(float(lower_median(na[:, 0])), 1e-30)
            live = na > med * float(o['inject_min_ratio'])
            if mode == 'row':
                scale = np.where(live, nw / np.maximum(na, 1e-30), 1.0)
            else:
                scale = np.full_like(na, float(np.linalg.norm(WA_rows)) / max(float(np.linalg.norm(acc)), 1e-30))
            acc = (acc * scale).astype(np.float32)
    inj = float(np.linalg.norm(alpha * acc))
    return acc.astype(np.float32), {'mode': mode, 'bc0': bc0, 'bc1': bc1, 'inj': inj / max(shrink, 1e-30),
                                    'inj0': inj_raw / max(shrink, 1e-30)}


# ============================================================================ activation files
class ActStore:
    """Per-layer activation files ({layer}|{module}|{suffix}), read lazily with a byte-bounded cache of the
    standardised matrices (float32)."""
    def __init__(self, paths, docs, cache_gib):
        self.paths = dict(paths)            # layer -> file
        self.docs = int(docs)
        self.limit = int(float(cache_gib) * 1024 ** 3)
        self._cache = {}
        self._bytes = 0

    def raw(self, layer, module, suffix):
        """float32 (docs, channels); a layer may be stored as several files of consecutive documents."""
        from safetensors import safe_open
        paths = self.paths[layer] if isinstance(self.paths[layer], (list, tuple)) else [self.paths[layer]]
        key = f'{layer}|{module}|{suffix}'
        parts = []
        for path in paths:
            with safe_open(str(path), framework='np') as f:
                if key not in f.keys():
                    raise KeyError(f'激活文件 {path} 里没有 {key}')
                parts.append(np.asarray(f.get_tensor(key), dtype=np.float32))
        value = parts[0] if len(parts) == 1 else np.concatenate(parts, axis=0)
        if value.shape[0] < self.docs:
            raise ValueError(f'激活 {key} 只有 {value.shape[0]} 行，少于 {self.docs} 篇')
        return value[:self.docs]

    def z(self, layer, module, suffix):
        key = (layer, module, suffix)
        if key in self._cache:
            value = self._cache.pop(key)
            self._cache[key] = value
            return value
        value = zstd(self.raw(layer, module, suffix))
        size = value.nbytes
        while self._cache and self._bytes + size > self.limit:
            old = self._cache.pop(next(iter(self._cache)))
            self._bytes -= old.nbytes
        self._cache[key] = value
        self._bytes += size
        return value

    def clear(self):
        self._cache.clear()
        self._bytes = 0


# ============================================================================ donor forward on the CPU
class DonorForward:
    """The donor model's own decoder layers, run one layer at a time over many sequences (weights of one layer in
    memory). Like the pipeline's LayerwiseScorer: the model's own forward is traced once per batch with stand-ins for
    the decoder layers, which record each layer's arguments (rotary values, mask), then the real layers run."""
    def __init__(self, engine, repo, meta_dir, o):
        from transformers import AutoConfig, AutoModelForCausalLM
        from accelerate import init_empty_weights
        from accelerate.utils import set_module_tensor_to_device
        import inspect
        self.engine, self.repo, self.o = engine, repo, o
        self.dtype = getattr(torch, o['dtype'])
        cfg = AutoConfig.from_pretrained(str(meta_dir), local_files_only=True, trust_remote_code=False)
        if getattr(cfg, 'quantization_config', None):
            raise ValueError('供体配置是量化模型；激活必须用原精度权重来算')
        with init_empty_weights(include_buffers=False):
            model = AutoModelForCausalLM.from_config(cfg, attn_implementation=o['attention'], trust_remote_code=False)
        model.to(self.dtype)
        for p in model.parameters():
            p.requires_grad_(False)
        model.eval()
        self.model = model
        tc = cfg.get_text_config() if hasattr(cfg, 'get_text_config') else cfg
        self.config = tc
        self.num_layers = int(tc.num_hidden_layers)
        self.hidden = int(tc.hidden_size)
        found = [(n, m) for n, m in model.named_modules() if isinstance(m, torch.nn.ModuleList)
                 and re.search(r'(^|\.)layers$', n) and len(m) == self.num_layers]
        if len(found) != 1:
            raise RuntimeError(f'供体：找不到唯一的解码层列表（{len(found)} 个候选）')
        self.prefix, self.layers = found[0]
        self.catalogue = engine._catalogue(repo)
        self.mapping, _ = runtime.match_model_weights(model, self.catalogue)
        pattern = re.compile(re.escape(self.prefix) + r'\.(\d+)\.')
        self.units = {i: [] for i in range(self.num_layers)}
        self.outer = []
        for name in self.mapping:
            m = pattern.match(name)
            if m:
                self.units[int(m[1])].append(name)
            else:
                self.outer.append(name)
        self.embed = [n for n in self.outer if n.endswith('embed_tokens.weight')]
        if len(self.embed) != 1:
            raise RuntimeError('供体：找不到唯一的 embed_tokens')
        self._set = set_module_tensor_to_device
        self._extra = {'clear_cache': False} if 'clear_cache' in inspect.signature(set_module_tensor_to_device).parameters else {}
        self.tuple_output = None
        self._pool = ThreadPoolExecutor(max_workers=1, thread_name_prefix='tm-read')
        self.bytes_read = 0

    # -- weights
    def fetch(self, names):
        out = {}
        for name in names:
            t = self.engine.tensor(self.repo, self.mapping[name])
            self.bytes_read += t.numel() * t.element_size()
            out[name] = t
        return out

    def fetch_async(self, names):
        return self._pool.submit(self.fetch, list(names))

    def load(self, tensors):
        for name, value in tensors.items():
            self._set(self.model, name, 'cpu', value=value, **self._extra)

    def release(self, names):
        for name in names:
            self._set(self.model, name, 'meta', **self._extra)

    def layer_names(self, i):
        return list(self.units[i])

    def layer_params(self, i):
        return sum(math.prod(self.catalogue[self.mapping[n]]['shape']) for n in self.units[i])

    def close(self):
        self._pool.shutdown(wait=False, cancel_futures=True)

    # -- tracing the model's own forward
    def _stubbed(self, make):
        from contextlib import contextmanager

        @contextmanager
        def scope():
            saved = [layer.__dict__.get('forward') for layer in self.layers]
            for i, layer in enumerate(self.layers):
                layer.forward = make(i)
            try:
                yield
            finally:
                for layer, old in zip(self.layers, saved):
                    if old is None:
                        layer.__dict__.pop('forward', None)
                    else:
                        layer.forward = old
        return scope()

    def trace(self, ids):
        """Arguments of every decoder layer for one batch of equal-length sequences (no padding), and the input of the
        first layer. The embedding must be loaded."""
        last = self.num_layers - 1
        for tuple_output in ([False, True] if self.tuple_output is None else [self.tuple_output]):
            calls = []
            args_of = [None] * self.num_layers

            def make(i):
                def stub(*args, **kwargs):
                    calls.append(i)
                    kwargs = dict(kwargs)
                    if args:
                        hidden, rest, named = args[0], tuple(args[1:]), False
                    else:
                        hidden, rest, named = kwargs.pop('hidden_states'), (), True
                    args_of[i] = (rest, kwargs, named, hidden)
                    if i == last:
                        raise runtime._StopForward()
                    return (hidden,) if tuple_output else hidden
                return stub
            with self._stubbed(make):
                try:
                    with torch.no_grad():
                        self.model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False, return_dict=True)
                except runtime._StopForward:
                    pass
            expected = (ids.shape[0], ids.shape[1], self.hidden)
            if calls == list(range(self.num_layers)) and all(a is not None and tuple(a[3].shape) == expected for a in args_of):
                first = args_of[0][3]
                if first.device.type == 'meta' or not first.is_floating_point():
                    raise RuntimeError('供体：第一层的输入不是实际的浮点张量')
                if self.tuple_output is None:
                    self.tuple_output = tuple_output
                return [a[:3] for a in args_of], first.detach().clone()
        raise RuntimeError('供体：解码层的调用方式不是“每层一次、按顺序、隐状态在前”')

    def run_layer(self, i, call, hidden):
        rest, kwargs, named = call
        with torch.no_grad():
            out = self.layers[i](*rest, hidden_states=hidden, **kwargs) if named else self.layers[i](hidden, *rest, **kwargs)
        out = out[0] if isinstance(out, (tuple, list)) else out
        if tuple(out.shape) != tuple(hidden.shape):
            raise RuntimeError('供体：解码层输出的形状与输入不同')
        if not torch.isfinite(out).all():
            raise RuntimeError(f'供体第 {i} 层出现 NaN/Inf')
        return out

    def logits_last(self, hidden, ids):
        """Next-token logits at the last position, through the model's own final norm and head (must be loaded)."""
        last = self.num_layers - 1

        def make(i):
            def stub(*args, **kwargs):
                if i == last:
                    return (hidden,) if self.tuple_output else hidden
                h = args[0] if args else kwargs['hidden_states']
                return (h,) if self.tuple_output else h
            return stub
        import inspect
        kwargs = {'input_ids': ids, 'attention_mask': torch.ones_like(ids), 'use_cache': False, 'return_dict': True}
        if 'logits_to_keep' in inspect.signature(self.model.forward).parameters:
            kwargs['logits_to_keep'] = 1
        with self._stubbed(make):
            with torch.no_grad():
                out = self.model(**kwargs)
        return out.logits[:, -1, :].float()


def _load_tokenizer(meta_dir):
    from transformers import AutoTokenizer
    return AutoTokenizer.from_pretrained(str(meta_dir), local_files_only=True, trust_remote_code=False)


# ============================================================================ orchestration
class _Run:
    """State of one ensure() call."""
    def __init__(self, context, o):
        self.ctx = context
        self.o = o
        self.eng = context['engine']
        self.settings = context['settings']
        self.root = Path(o['dir'] or (Path(context['root']) / 'tm_delta')).expanduser().resolve()
        self.root.mkdir(parents=True, exist_ok=True)
        self.scratch = Path(context['scratch']) / 'tm_delta'
        self.scratch.mkdir(parents=True, exist_ok=True)
        self.api = self.eng.api
        self.token = self.eng.token

    def say(self, text):
        print('[N] ' + text, flush=True)


def _pin_dataset(run):
    o = run.o
    # First use fixes the commit of the activation dataset (act_repo_pin.json); later sessions read that commit.
    lock = run.root / 'act_repo_pin.json'
    old = json.loads(lock.read_text()) if lock.exists() else None
    wanted = o['act_revision'] or (old['sha'] if old and old.get('repo') == o['act_repo'] else None)
    info = run.api.dataset_info(o['act_repo'], revision=wanted, token=run.token)
    sha = info.sha
    if not isinstance(sha, str) or not re.fullmatch(r'[0-9a-fA-F]{40,64}', sha):
        raise ValueError(f'{o["act_repo"]}: 拿不到 commit SHA')
    core.atomic_json(lock, {'repo': o['act_repo'], 'sha': sha})
    return sha, [x.rfilename for x in info.siblings]


def _target_layers(run):
    """Target layer lists as the TM notebook chose them: attention on the full-attention layers, MLP on those plus
    every target_stride-th layer."""
    cfg = run.ctx['B'].config
    tc = cfg.get('text_config', cfg)
    n = int(tc['num_hidden_layers'])
    types = tc.get('layer_types') or ['full_attention'] * n
    if len(types) != n:
        raise ValueError('B 的 layer_types 与层数不符')
    full = [i for i, t in enumerate(types) if 'full' in t]
    rec = sorted(set(full) | set(range(0, n, int(run.o['target_stride']))))
    return n, full, rec


def _target_modules_for(module, full, rec):
    return list(full) if module.startswith('self_attn.') else list(rec)


def _find_target_acts(run, files, act_sha, calib_sha, full, rec):
    """The stored target activations: a set of acts_acts-v2_target_<key>_L###.safetensors files covering every needed
    layer, whose metadata names B, this calibration file, T=2000 and seq 128, and which hold every needed tensor."""
    o = run.o
    groups = {}
    for name in files:
        m = _TARGET_FILE.match(name)
        if m:
            groups.setdefault(m[1], {})[int(m[2])] = name
    need_layers = sorted(set(rec) | set(full))
    brepo = run.ctx['B_repo']
    want_rev = run.ctx['identity']['inputs']['B']['commit']
    reasons = []
    for key in sorted(groups):
        have = groups[key]
        if any(li not in have for li in need_layers):
            reasons.append(f'{key}: 缺 {len([li for li in need_layers if li not in have])} 层')
            continue
        bad = None
        meta_rev = None
        for li in need_layers:
            info = run.api.parse_safetensors_file_metadata(o['act_repo'], have[li], repo_type='dataset', revision=act_sha, token=run.token)
            meta = info.metadata or {}
            if (meta.get('tag') != 'target' or meta.get('repo') != brepo or meta.get('calib_sha') != calib_sha
                    or meta.get('T') != '2000' or meta.get('seq') != str(o['seq_len']) or meta.get('pool') != 'per_document_mean'):
                bad = f'{key}: L{li} 的元数据对不上（{ {k: meta.get(k) for k in ("tag", "repo", "T", "seq", "pool")} }）'
                break
            meta_rev = meta.get('revision')
            names = set(info.tensors)
            for mod in o['modules']:
                if li not in _target_modules_for(mod, full, rec):
                    continue
                for sfx in ('in_mean', 'out_mean', 'out_abs'):
                    if f'{li}|{mod}|{sfx}' not in names:
                        bad = f'{key}: L{li} 缺 {mod}|{sfx}'
                        break
                if bad:
                    break
            if bad:
                break
        if bad:
            reasons.append(bad)
            continue
        return key, {li: have[li] for li in need_layers}, meta_rev
    raise ValueError(f'{o["act_repo"]} 里找不到可用的 target 激活（{brepo}，校准集 sha {calib_sha[:12]}）：' + '；'.join(reasons[:6]))


def _same_weights(run, rev_a, rev_b):
    """True when two commits of B hold byte-identical safetensors files (LFS sha256)."""
    if rev_a == rev_b:
        return True
    brepo = run.ctx['B_repo']
    def lfs(rev):
        info = run.api.model_info(brepo, revision=rev, files_metadata=True, token=run.token)
        return {s.rfilename: (s.lfs.sha256 if getattr(s, 'lfs', None) else None) for s in info.siblings
                if s.rfilename.endswith('.safetensors')}
    a, b = lfs(rev_a), lfs(rev_b)
    return bool(a) and a == b and all(v for v in a.values())


def _download(run, filename, act_sha, directory):
    from huggingface_hub import hf_hub_download
    directory.mkdir(parents=True, exist_ok=True)
    return Path(hf_hub_download(run.o['act_repo'], filename, repo_type='dataset', revision=act_sha, token=run.token,
                                local_dir=str(directory)))


def _pin_donor(run):
    o, eng = run.o, run.eng
    if o['revision']:
        old = eng.revisions.get(o['repo'])
        if old and old != o['revision']:
            raise ValueError(f'{o["repo"]}：tm_extra["revision"] 与已锁定的 commit 不同')
        eng.revisions[o['repo']] = o['revision']
    return eng.pin(o['repo'])


def _inputs(run):
    """Pins and checks: donor repository, its config and tokenizer, the calibration texts and the target activations."""
    o, eng = run.o, run.eng
    donor_commit = _pin_donor(run)
    meta_dir = runtime.download_metadata(eng, o['repo'], run.root / 'donor_metadata')
    cfg = json.loads((meta_dir / 'config.json').read_text())
    dc = cfg.get('text_config', cfg)
    experts = dc.get('num_experts') or dc.get('num_local_experts') or dc.get('n_routed_experts') or 0
    if int(experts) > 1:
        raise ValueError('供体是 MoE：MoE 的 FFN 不能这样搬；本步只支持 dense 供体')
    tcfg = run.ctx['B'].config.get('text_config', run.ctx['B'].config)
    if 'self_attn.k_proj' in o['modules'] and dc.get('head_dim') != tcfg.get('head_dim'):
        print(f'[N] 注意：供体 head_dim {dc.get("head_dim")} 与 B 的 {tcfg.get("head_dim")} 不同，k_proj 的 RoPE 通道配对对不上（见 TM notebook 的 ROPE_NOTE）', flush=True)
    n_d = int(dc['num_hidden_layers'])
    cat = eng._catalogue(o['repo'])
    names = list(cat)
    prefix = None
    for n in names:
        m = re.match(r'^(.*\.layers)\.(\d+)\.', n)
        if m and 'mtp' not in m[1] and 'visual' not in m[1]:
            prefix = m[1]
            break
    if prefix is None:
        raise ValueError('供体：张量名里找不到 .layers.N.')
    src_layers = list(range(0, n_d, int(o['source_stride'])))
    shapes = {}
    for mod in o['modules']:
        seen = set()
        for mj in src_layers:
            key = f'{prefix}.{mj}.{mod}.weight'
            if key not in cat:
                raise ValueError(f'供体缺少 {key}')
            seen.add(tuple(cat[key]['shape']))
        if len(seen) != 1:
            raise ValueError(f'供体的 {mod} 在各层形状不一：{seen}')
        shapes[mod] = list(seen.pop())
    n_t, full, rec = _target_layers(run)
    # calibration texts and target activations
    act_sha, files = _pin_dataset(run)
    if o['calib_file'] not in files:
        raise ValueError(f'{o["act_repo"]} 里没有校准文本 {o["calib_file"]}')
    calib_path = _download(run, o['calib_file'], act_sha, run.root / 'calibration')
    calib_sha = core.sha_file(calib_path)
    texts = json.loads(calib_path.read_text(encoding='utf-8'))
    if not isinstance(texts, list) or len(texts) < 2000 or not all(isinstance(t, str) and t.strip() for t in texts[:2000]):
        raise ValueError('校准文本不是 2000 篇非空文本的列表')
    texts = texts[:int(o['docs'])]
    key, tfiles, meta_rev = _find_target_acts(run, files, act_sha, calib_sha, full, rec)
    b_commit = run.ctx['identity']['inputs']['B']['commit']
    if meta_rev and meta_rev != b_commit and not _same_weights(run, meta_rev, b_commit):
        raise ValueError(f'target 激活是在 B 的 commit {meta_rev[:10]} 上算的，与本次 B（{b_commit[:10]}）的权重文件不同')
    # target matrices that will change: S keys, shapes
    s_shapes = run.ctx['S'].shapes
    targets = {}
    for mod in o['modules']:
        for li in _target_modules_for(mod, full, rec):
            hits = [k for k in s_shapes if re.search(rf'(?:^|\.)layers\.{li}\.{re.escape(mod)}\.weight$', k)
                    and not re.search(r'(^|\.)(mtp|visual|nextn)(\.|$)', k)]
            if len(hits) != 1:
                raise ValueError(f'S 里 layers.{li}.{mod}.weight 不唯一（{len(hits)} 个）')
            targets[(mod, li)] = hits[0]
    return {'donor_commit': donor_commit, 'meta_dir': str(meta_dir), 'donor_layers': n_d, 'donor_prefix': prefix,
            'source_layers': src_layers, 'donor_shapes': shapes, 'target_layers': n_t, 'full': full, 'rec': rec,
            'act_sha': act_sha, 'calib_sha': calib_sha, 'calib_path': str(calib_path), 'texts': texts,
            'target_key': key, 'target_files': tfiles, 'targets': targets}


def _identity(run, inp):
    o = run.o
    return {'version': TM_VERSION, 'donor': {'repo': o['repo'], 'commit': inp['donor_commit']},
            'B': run.ctx['identity']['inputs']['B'], 'act_repo': {'repo': o['act_repo'], 'commit': inp['act_sha']},
            'calib_sha': inp['calib_sha'], 'target_acts': inp['target_key'], 'docs': int(o['docs']), 'seq': int(o['seq_len']),
            'modules': o['modules'], 'source_layers': inp['source_layers'], 'target_rec': inp['rec']}


# ---------------------------------------------------------------------------- gate
def _gate_identity(run, inp):
    o = run.o
    return {'kind': 'tm_gate', 'version': ACTS_VERSION, 'repo': o['repo'], 'commit': inp['donor_commit'],
            'dtype': o['dtype'], 'attention': o['attention'], 'libraries': _library_versions()}


def _gate(run, inp, fwd, tok):
    o = run.o
    identity = _gate_identity(run, inp)
    path = run.root / 'gate.record.json'
    done = _record(path, identity)
    if done is not None and done['hits'] >= int(o['gate_min_hits']):
        run.say(f'前向闸门已通过（{done["hits"]}/{len(GATE_PROMPTS)}）')
        return done
    run.say(f'前向闸门：4 条短提示走完供体全部 {fwd.num_layers} 层（读一遍权重，约 {sum(fwd.layer_params(i) for i in range(fwd.num_layers)) * 2 / 1024**3:.0f} GiB）')
    items = []
    for prompt, expected in GATE_PROMPTS:
        ids = torch.tensor([tok(prompt)['input_ids']], dtype=torch.long)
        want = tok.encode(expected[0], add_special_tokens=False)
        items.append((ids, want[0] if len(want) == 1 else None))
    fwd.load(fwd.fetch(fwd.embed + [n for n in fwd.outer if n not in fwd.embed]))
    traces = [fwd.trace(ids) for ids, _ in items]
    states = [h for _, h in traces]
    future = fwd.fetch_async(fwd.layer_names(0))
    started = time.perf_counter()
    for i in range(fwd.num_layers):
        tensors = future.result()
        if i + 1 < fwd.num_layers:
            future = fwd.fetch_async(fwd.layer_names(i + 1))
        fwd.load(tensors)
        del tensors
        try:
            states = [fwd.run_layer(i, traces[k][0][i], states[k]) for k in range(len(items))]
        finally:
            fwd.release(fwd.layer_names(i))
        if i % 13 == 12 or i == fwd.num_layers - 1:
            run.say(f'  闸门第 {i + 1}/{fwd.num_layers} 层，{(time.perf_counter() - started) / 60:.1f} 分钟')
    hits = 0
    lines = []
    for k, (prompt, expected) in enumerate(GATE_PROMPTS):
        logits = fwd.logits_last(states[k], items[k][0])[0]
        top = int(logits.argmax())
        got = tok.decode([top])
        hit = any(got.strip().lower() == x.strip().lower() for x in expected)
        rank = None
        if items[k][1] is not None:
            rank = int((logits > logits[items[k][1]]).sum()) + 1
        hits += int(hit)
        lines.append({'prompt': prompt, 'top1': got, 'hit': hit, 'expected_rank': rank})
        run.say(f'  {"通过" if hit else "失败"} {prompt!r} -> {got!r}' + (f'（期望 token 排名 #{rank}）' if rank else ''))
    fwd.release([n for n in fwd.outer if n not in fwd.embed])
    result = {'hits': hits, 'prompts': lines}
    if hits < int(o['gate_min_hits']):
        raise RuntimeError(f'供体的流式前向闸门没过（{hits}/{len(GATE_PROMPTS)}）：层参数、位置编码或 mask 的适配有问题，激活不能用。请把上面几行反馈回来。')
    core.save_record(path, identity, result)
    return result


# ---------------------------------------------------------------------------- donor activations
def _acts_identity(run, inp):
    o = run.o
    return {'kind': 'tm_donor_acts', 'version': ACTS_VERSION, 'repo': o['repo'], 'commit': inp['donor_commit'],
            'calib_sha': inp['calib_sha'], 'docs': int(o['docs']), 'seq': int(o['seq_len']), 'modules': sorted(o['modules']),
            'layers': inp['source_layers'], 'dtype': o['dtype'], 'attention': o['attention'], 'libraries': _library_versions(),
            'pool': 'per_document_mean (no padding; equal-length documents batched)'}


def _donor_acts(run, inp, tok, make_forward):
    """Per-document means of the inputs and outputs of the donor's modules at the recorded layers.

    The documents are cut into chunks of chunk_docs; each chunk runs through every layer once (the weights are read
    again per chunk, which costs little next to the compute) and leaves one small file per recorded layer on the record
    directory. A new session resumes at the first unfinished chunk, and inside a chunk from the hidden-state checkpoint
    on the local disk if that is still there. The streaming forward is built (and passes the gate) only when something
    is missing. Returns ({layer: [chunk files]}, summary)."""
    o = run.o
    identity = _acts_identity(run, inp)
    fp = core.digest(identity)
    directory = run.root / 'donor_acts' / fp[:24]
    directory.mkdir(parents=True, exist_ok=True)
    core.atomic_json(directory / 'identity.json', identity)
    T = len(inp['texts'])
    size = max(1, int(o['chunk_docs']))
    chunks = [(a, min(a + size, T)) for a in range(0, T, size)]
    def chunk_dir(a, b):
        return directory / f'docs_{a:04d}-{b:04d}'
    def chunk_files(a, b):
        return {mj: chunk_dir(a, b) / f'L{mj:03d}.safetensors' for mj in inp['source_layers']}
    def chunk_done(a, b):
        rec = _record(chunk_dir(a, b) / 'complete.record.json', {**identity, 'docs_range': [a, b]})
        files = chunk_files(a, b)
        return rec is not None and all(files[mj].is_file() and files[mj].stat().st_size == rec['sizes'][str(mj)] for mj in files)
    todo = [(a, b) for a, b in chunks if not chunk_done(a, b)]
    if todo:
        run.say(f'供体激活：{T} 篇分成 {len(chunks)} 段（每段 {size} 篇），还剩 {len(todo)} 段')
        fwd = make_forward()
        try:
            _gate(run, inp, fwd, tok)
            enc = tok(inp['texts'], truncation=True, max_length=int(o['seq_len']), padding=False)
            lengths = [len(x) for x in enc['input_ids']]
            if min(lengths) < 2:
                raise ValueError('有校准文本分词后不足 2 个 token')
            for n, (a, b) in enumerate(todo, 1):
                run.say(f'供体激活第 {chunks.index((a, b)) + 1}/{len(chunks)} 段（第 {a}–{b - 1} 篇）')
                _extract_chunk(run, inp, fwd, enc['input_ids'], a, b, chunk_dir(a, b), chunk_files(a, b), {**identity, 'docs_range': [a, b]}, fp)
        finally:
            fwd.close()
            del fwd
            gc.collect()
    else:
        run.say('供体激活已经算好，直接使用')
    files = {mj: [chunk_files(a, b)[mj] for a, b in chunks] for mj in inp['source_layers']}
    return files, {'chunks': len(chunks), 'documents': T, 'directory': str(directory)}


def _extract_chunk(run, inp, fwd, input_ids, a, b, cdir, files, identity, fp):
    """Documents a..b-1 through every layer up to the last recorded one."""
    from safetensors.torch import save_file, load_file
    o = run.o
    cdir.mkdir(parents=True, exist_ok=True)
    by_len = {}
    for d in range(a, b):
        by_len.setdefault(len(input_ids[d]), []).append(d)
    batches = []
    bsz = max(1, int(o['batch_docs']))
    for n in sorted(by_len, reverse=True):
        docs = by_len[n]
        for k in range(0, len(docs), bsz):
            part = docs[k:k + bsz]
            batches.append((torch.tensor([d - a for d in part], dtype=torch.long),
                            torch.tensor([input_ids[d] for d in part], dtype=torch.long)))
    T = b - a
    tokens = sum(len(input_ids[d]) for d in range(a, b))
    last = max(inp['source_layers'])
    rec = set(inp['source_layers'])
    # resume point inside the chunk: a hidden-state checkpoint on the local disk (it does not outlive the machine)
    ck_dir = run.scratch / 'donor_hidden'
    ck_dir.mkdir(parents=True, exist_ok=True)
    ck_path = ck_dir / f'{fp[:16]}_{a:04d}-{b:04d}.safetensors'
    ck_meta = ck_path.with_suffix('.json')
    for other in ck_dir.glob('*.safetensors'):
        if other != ck_path:
            other.unlink(missing_ok=True)
            other.with_suffix('.json').unlink(missing_ok=True)
    start = 0
    states = None
    if ck_path.is_file() and ck_meta.is_file():
        try:
            meta = json.loads(ck_meta.read_text())
            if meta.get('identity') == fp and meta.get('batches') == len(batches) and all(files[mj].is_file() for mj in rec if mj <= meta['layer']):
                loaded = load_file(str(ck_path))
                states = [loaded[f'b{k}'] for k in range(len(batches))]
                start = int(meta['layer']) + 1
                run.say(f'  从第 {start} 层继续（本机上的隐状态检查点）')
        except (OSError, ValueError, KeyError, json.JSONDecodeError):
            states = None
            start = 0
    threads, speed = runtime.tune_cpu_threads()
    per_layer = fwd.layer_params(0)
    hours = 2 * per_layer * tokens * (last + 1 - start) / speed / 1e9 / 3600
    run.say(f'  {T} 篇、{tokens} token、{len(batches)} 批；第 {start}–{last} 层（记录 {len(rec)} 层），'
            f'{threads} 线程下 float32 约 {speed:.0f} GFLOPS，估计 {hours:.1f} 小时（另加读一遍权重）')
    fwd.load(fwd.fetch(fwd.embed))
    fresh = states is None
    traces = []
    first_states = []
    for docs, ids in batches:
        calls, first = fwd.trace(ids)
        traces.append(calls)
        if fresh:
            first_states.append(first.to(fwd.dtype))
    if fresh:
        states = first_states
    del first_states
    fwd.release(fwd.embed)
    gc.collect()
    current = {'rows': None}
    started = time.perf_counter()
    future = fwd.fetch_async(fwd.layer_names(start)) if start <= last else None
    for i in range(start, last + 1):
        tensors = future.result()
        future = fwd.fetch_async(fwd.layer_names(i + 1)) if i + 1 <= last else None
        fwd.load(tensors)
        del tensors
        acc = {}
        hooks = []
        if i in rec:
            for mod in o['modules']:
                sub = fwd.layers[i].get_submodule(mod)
                if not isinstance(sub, torch.nn.Linear):
                    raise RuntimeError(f'供体第 {i} 层的 {mod} 不是 Linear')
                acc[mod] = {'in_mean': torch.zeros(T, sub.in_features, dtype=torch.float32),
                            'out_mean': torch.zeros(T, sub.out_features, dtype=torch.float32)}

                def hook(module, args, output, _mod=mod):
                    x = args[0] if isinstance(args, tuple) else args
                    y = output[0] if isinstance(output, (tuple, list)) else output
                    rows = current['rows']
                    acc[_mod]['in_mean'][rows] = x.float().mean(1)
                    acc[_mod]['out_mean'][rows] = y.float().mean(1)
                hooks.append(sub.register_forward_hook(hook))
        try:
            for k, (rows, ids) in enumerate(batches):
                current['rows'] = rows
                states[k] = fwd.run_layer(i, traces[k][i], states[k])
        finally:
            for h in hooks:
                h.remove()
            fwd.release(fwd.layer_names(i))
        if i in rec:
            flat = {}
            for mod, d in acc.items():
                for sfx, value in d.items():
                    big = float(value.abs().max()) > 60000.0
                    flat[f'{i}|{mod}|{sfx}'] = (value if big else value.to(torch.float16)).contiguous()
            meta = {'schema': ACTS_VERSION, 'tag': o['tag'], 'repo': o['repo'], 'revision': inp['donor_commit'],
                    'calib_sha': inp['calib_sha'], 'docs': f'{a}-{b}', 'seq': str(o['seq_len']), 'layer': str(i),
                    'mods': json.dumps(sorted(o['modules'])), 'pool': 'per_document_mean', 'identity': fp}
            tmp = files[i].with_name(files[i].name + '.tmp')
            save_file(flat, str(tmp), metadata=meta)
            tmp.replace(files[i])
            del flat
        del acc
        done_layers = i + 1 - start
        elapsed = time.perf_counter() - started
        if done_layers in (1, 2) or i % 8 == 7 or i == last:
            run.say(f'  第 {i + 1}/{last + 1} 层，本段已用 {elapsed / 3600:.2f} 小时，估计还需 {elapsed / done_layers * (last - i) / 3600:.1f} 小时'
                    f'（累计读权重 {fwd.bytes_read / 1024**3:.0f} GiB）')
        if (i + 1) % max(1, int(o['checkpoint_every'])) == 0 and i < last:
            try:
                tmp = ck_path.with_name(ck_path.name + '.tmp')
                save_file({f'b{k}': s.contiguous() for k, s in enumerate(states)}, str(tmp))
                tmp.replace(ck_path)
                core.atomic_json(ck_meta, {'identity': fp, 'layer': i, 'batches': len(batches)})
            except OSError as error:
                print(f'[N]   隐状态检查点没写成（{error}）；不影响结果，只是中断后要从本段开头重来', flush=True)
        gc.collect()
    del states, traces
    sizes = {str(mj): files[mj].stat().st_size for mj in files}
    core.save_record(cdir / 'complete.record.json', identity,
                     {'sizes': sizes, 'tokens': tokens, 'documents': T, 'batches': len(batches),
                      'hours': (time.perf_counter() - started) / 3600})
    for p in (ck_path, ck_meta):
        p.unlink(missing_ok=True)


# ---------------------------------------------------------------------------- target activations
def _target_store(run, inp):
    """The needed target activation files on a local disk (the record directory when the local disk is short)."""
    infos = run.api.get_paths_info(run.o['act_repo'], list(inp['target_files'].values()), repo_type='dataset',
                                   revision=inp['act_sha'], token=run.token)
    total = sum(int(getattr(x, 'size', 0) or 0) for x in infos)
    local = run.scratch / 'target_acts' / inp['target_key']
    have = sum(p.stat().st_size for p in local.glob('*.safetensors')) if local.is_dir() else 0
    directory = local if shutil.disk_usage(run.scratch).free > (total - have) + 20 * 1024 ** 3 else run.root / 'target_acts' / inp['target_key']
    paths = {}
    missing = [li for li, name in inp['target_files'].items() if not (directory / name).is_file()]
    if missing:
        run.say(f'下载 target 激活 {len(missing)} 个文件（共约 {total / 1024**3:.0f} GiB）到 {directory}')
    for li, name in inp['target_files'].items():
        paths[li] = _download(run, name, inp['act_sha'], directory)
    return ActStore(paths, run.o['docs'], run.o['act_cache_gib'] / 2), directory


# ---------------------------------------------------------------------------- verdict
def _verdict(run, inp, ident, tgt, src):
    o = run.o
    keys = ('eps_delta', 'eps_floor', 'eps_cap', 'iters_inner', 'tol', 'z_go', 'sharp_margin', 'marginal',
            'up_excess_max', 'up_sharp_mult', 'noise_seed')
    identity = {'kind': 'tm_verdict', 'version': OT_VERSION, 'inputs': ident, 'options': {k: o[k] for k in keys}}
    path = run.root / 'verdict.record.json'
    done = _record(path, identity)
    if done is not None:
        return done
    n_t = inp['target_layers']
    n_s = inp['donor_layers']
    ST = inp['source_layers']
    rows = {}
    rng = np.random.default_rng(int(o['noise_seed']))
    for mod in o['modules']:
        TT = _target_modules_for(mod, inp['full'], inp['rec'])
        probes = []
        for frac in (0.25, 0.5, 0.75):
            li = min(TT, key=lambda x: abs(x / max(n_t - 1, 1) - frac))
            mj = min(ST, key=lambda x: abs(x / max(n_s - 1, 1) - frac))
            probes.append((li, mj))
        stats = []
        for li, mj in probes:
            r = inner_ot(tgt.z(li, mod, 'out_mean'), src.z(mj, mod, 'out_mean'), o)
            stats.append(r)
            run.say(f'{mod:16s} L{li:2d}×L{mj:2d}  n={r["n"]}×{r["m"]}  gap={r["gap"]:.3f}  z={r["z"]:.1f}（零信息 {r["znull"]:.1f}）'
                    f'  eps={r["eps"]:.4f}  锐度={r["sharp"]:.5f}')
        mean = {k: float(np.mean([r[k] for r in stats])) for k in ('gap', 'z', 'znull', 'eps', 'sharp', 'cl')}
        # upper bound: the target with itself (the answer is the identity)
        li_up = TT[len(TT) // 2]
        X = tgt.z(li_up, mod, 'out_mean')
        up = inner_ot(X, X, o)
        excess = up['cl'] / max(float(o['eps_delta']) * up['gap'], 1e-9)
        sharp_ratio_up = up['sharp'] * up['m']
        up_ok = excess <= float(o['up_excess_max']) and sharp_ratio_up <= float(o['up_sharp_mult'])
        # lower bound: the target against Gaussian noise of the donor's shape, at the donor pair's eps
        li, mj = probes[1]
        raw = src.raw(mj, mod, 'out_mean')
        R = (rng.standard_normal(raw.shape).astype(np.float32) * np.float32(raw.std()))
        lo = inner_ot(tgt.z(li, mod, 'out_mean'), zstd(R), o, eps=mean['eps'])
        del R, raw
        z_ratio = mean['z'] / mean['znull'] if mean['znull'] > 0 else 0.0
        sharp_ratio = lo['sharp'] / mean['sharp'] if mean['sharp'] > 0 else 0.0
        if not up_ok:
            verdict = 'SKIP'
        elif z_ratio >= float(o['z_go']) and sharp_ratio >= float(o['sharp_margin']):
            verdict = 'GO'
        elif z_ratio >= float(o['marginal']) and sharp_ratio >= float(o['marginal']):
            verdict = 'MARGINAL'
        else:
            verdict = 'NO-GO'
        rows[mod] = {'probes': probes, **mean, 'z_ratio': z_ratio, 'noise_sharp': lo['sharp'], 'sharp_ratio': sharp_ratio,
                     'upper': {'layer': li_up, 'cl': up['cl'], 'gap': up['gap'], 'excess': excess, 'sharp_ratio': sharp_ratio_up, 'ok': up_ok},
                     'verdict': verdict}
        run.say(f'{mod:16s} z 倍数 {z_ratio:.2f}  锐度倍数 {sharp_ratio:.2f}  上界 {"通过" if up_ok else "失败"}（超出 {excess:.1f}x，锐度/理想 {sharp_ratio_up:.1f}x）  → {verdict}')
        tgt.clear()
        src.clear()
        gc.collect()
    result = _clean({'modules': rows, 'go': [m for m, r in rows.items() if r['verdict'] in ('GO', 'MARGINAL')]})
    core.save_record(path, identity, result)
    return result


# ---------------------------------------------------------------------------- pair OT
def _pair_ot(run, inp, ident, tgt, src, mod):
    """Inner OT on the prefiltered pairs of one module, both sides; resumable."""
    import pickle
    o = run.o
    keys = ('eps_delta', 'eps_floor', 'eps_cap', 'iters_inner', 'tol', 'pair_keep', 'pair_depth_prior', 'chunk_rows')
    identity = {'kind': 'tm_pairs', 'version': OT_VERSION, 'inputs': ident, 'module': mod, 'options': {k: o[k] for k in keys}}
    fp = core.digest(identity)
    path = run.root / 'pairs' / f'{mod.replace(".", "-")}_{fp[:16]}.pkl'
    path.parent.mkdir(parents=True, exist_ok=True)
    state = None
    if path.is_file():
        try:
            with path.open('rb') as f:
                state = pickle.load(f)
            if state.get('identity') != fp:
                state = None
        except (OSError, pickle.UnpicklingError, EOFError, AttributeError):
            state = None
    TT = _target_modules_for(mod, inp['full'], inp['rec'])
    ST = inp['source_layers']
    if state is None:
        # linear CKA on both sides decides which donor layers each target layer is matched against
        scores = np.zeros((len(TT), len(ST)))
        for side in ('in_mean', 'out_mean'):
            gs = [gram(src.z(mj, mod, side)) for mj in ST]
            for x, li in enumerate(TT):
                g = gram(tgt.z(li, mod, side))
                for y in range(len(ST)):
                    scores[x, y] += 0.5 * float((g * gs[y]).sum(dtype=np.float64))
            del gs
            tgt.clear()
            src.clear()
        kept = keep_pairs(scores, TT, ST, int(o['pair_keep']), bool(o['pair_depth_prior']), inp['target_layers'], inp['donor_layers'])
        state = {'identity': fp, 'cka': scores.tolist(), 'kept': {int(k): v for k, v in kept.items()}, 'done': {}, 'complete': False}
    if state['complete']:
        return state
    pairs = [(li, mj) for li in TT for mj in state['kept'][li]]
    todo = [p for p in pairs if p not in state['done']]
    run.say(f'{mod}：{len(pairs)} 个层对（共 {len(TT)}×{len(ST)}={len(TT) * len(ST)} 个里由 CKA 留下），还要算 {len(todo)} 个')
    started = time.perf_counter()

    def save():
        tmp = path.with_name(path.name + '.tmp')
        with tmp.open('wb') as f:
            pickle.dump(state, f, protocol=4)
        tmp.replace(path)
    # target layer outer, donor layers inner: each target matrix is standardised once
    count = 0
    for li in TT:
        for mj in state['kept'][li]:
            if (li, mj) in state['done']:
                continue
            got = {}
            for side, sfx in (('in', 'in_mean'), ('out', 'out_mean')):
                r = inner_ot(tgt.z(li, mod, sfx), src.z(mj, mod, sfx), o)
                got[side] = {k: r[k] for k in ('u', 'v', 'eps', 'gap', 'cl', 'sharp', 'n', 'm')}
            state['done'][(li, mj)] = got
            count += 1
            if count % 4 == 0 or count == len(todo):
                save()
                el = time.perf_counter() - started
                run.say(f'  {mod} {len(state["done"])}/{len(pairs)}，本次 {count} 个，估计还需 {el / count * (len(todo) - count) / 60:.0f} 分钟')
    state['complete'] = True
    save()
    tgt.clear()
    src.clear()
    gc.collect()
    return state


# ---------------------------------------------------------------------------- fusion
def _fuse(run, inp, ident, tgt, src, verdict, pair_states):
    from safetensors.numpy import save_file
    o = run.o
    keys = ('alpha', 'topk', 'top_j', 'conc_min', 'inject', 'inject_min_ratio', 'inject_center', 'bcast_max',
            'eps_delta_out', 'eps_floor', 'eps_cap', 'iters_outer', 'chunk_rows')
    identity = {'kind': 'tm_delta', 'version': TM_VERSION, 'inputs': ident, 'options': {k: o[k] for k in keys},
                'verdict': verdict['go'], 'pairs': {m: core.digest(_clean({str(k): v for k, v in s['kept'].items()})) for m, s in pair_states.items()},
                'targets': {f'{m}|{li}': name for (m, li), name in inp['targets'].items() if m in verdict['go']}}
    fp = core.digest(identity)
    directory = run.root / 'delta' / fp[:24]
    record_path = directory / 'tm_delta.record.json'
    done = _record(record_path, identity)
    if done is not None and (directory / 'tm_delta.safetensors').is_file() and core.sha_file(directory / 'tm_delta.safetensors') == done['sha256']:
        return fp, directory, done
    directory.mkdir(parents=True, exist_ok=True)
    out = {}
    diag = {}
    plans = {}
    wb_cache = {}
    started = time.perf_counter()
    for mod in verdict['go']:
        st = pair_states[mod]
        TT = _target_modules_for(mod, inp['full'], inp['rec'])
        used = sorted({mj for li in TT for mj in st['kept'][li]})
        cin = {p: r['in']['cl'] for p, r in st['done'].items()}
        cout = {p: r['out']['cl'] for p, r in st['done'].items()}
        Pe, info = effective_plan(cin, cout, TT, used, o)
        dmr = depth_match_ratio(Pe, TT, used, inp['target_layers'], inp['donor_layers'])
        plans[mod] = {'TT': TT, 'ST': used, 'P': Pe.tolist(), 'dmr': dmr, **info}
        run.say(f'{mod}：层级计划 {Pe.shape}，锐度 {info["sharp"]:.3f}，DMR {dmr:.3f}')
        for x, li in enumerate(TT):
            name = inp['targets'][(mod, li)]
            row = Pe[x]
            conc = float(row.max() / max(row.mean(), 1e-30))
            kept = st['kept'][li]
            weights = sorted(((float(row[used.index(mj)]), mj) for mj in kept), reverse=True)
            pick = weights[:int(o['top_j'])] if conc >= float(o['conc_min']) else weights
            total = sum(w for w, _ in pick)
            pick = [(w / max(total, 1e-30), mj) for w, mj in pick]
            pick = [(w, mj) for w, mj in pick if w >= 1e-4]
            WA = run.ctx['B'].tensor(name).float().numpy()
            score = tgt.raw(li, mod, 'out_abs').mean(0)
            k = min(int(o['topk']), WA.shape[0])
            idx = np.sort(np.argsort(-score, kind='stable')[:k])
            acc = np.zeros((k, WA.shape[1]), dtype=np.float32)
            for w, mj in pick:
                key = (mod, mj)
                if key not in wb_cache:
                    while len(wb_cache) >= 4:
                        wb_cache.pop(next(iter(wb_cache)))
                    wb_cache[key] = run.eng.tensor(run.o['repo'], f'{inp["donor_prefix"]}.{mj}.{mod}.weight').float().numpy()
                WB = wb_cache[key]
                r = st['done'][(li, mj)]
                po = (tgt.z(li, mod, 'out_mean'), src.z(mj, mod, 'out_mean'), r['out']['u'], r['out']['v'], r['out']['eps'])
                pi = (tgt.z(li, mod, 'in_mean'), src.z(mj, mod, 'in_mean'), r['in']['u'], r['in']['v'], r['in']['eps'])
                if WB.shape != (po[1].shape[1], pi[1].shape[1]) or WA.shape != (po[0].shape[1], pi[0].shape[1]):
                    raise ValueError(f'{name}：权重形状与激活通道数对不上（B {WA.shape}，供体 {WB.shape}）')
                acc += np.float32(w) * transport_rows(idx, WB, po, pi, WA.shape[1], int(o['chunk_rows']))
            rows_new, d = inject(acc, WA[idx], o)
            delta = (np.float32(o['alpha']) * (rows_new - WA[idx])).astype(np.float32)
            out[name + '|rows'] = idx.astype(np.int64)
            out[name + '|delta'] = delta
            diag[name] = {**d, 'module': mod, 'layer': li, 'donor_layers': [[w, mj] for w, mj in pick], 'conc': conc,
                          'rows': int(k), 'dW': float(np.linalg.norm(delta) / max(float(np.linalg.norm(WA)), 1e-30))}
            del WA, acc
        run.say(f'{mod}：{len(TT)} 张矩阵已算好，已用 {(time.perf_counter() - started) / 60:.0f} 分钟')
        tgt.clear()
        src.clear()
        wb_cache.clear()
        gc.collect()
    tmp = directory / 'tm_delta.safetensors.tmp'
    save_file(out, str(tmp), metadata={'format': 'np', 'fingerprint': fp})
    tmp.replace(directory / 'tm_delta.safetensors')
    sha = core.sha_file(directory / 'tm_delta.safetensors')
    report = _clean({'tensors': diag, 'plans': plans})
    core.atomic_json(directory / 'tm_report.json', {'verdict': verdict, **report})
    dws = [d['dW'] for d in diag.values()]
    injs = [d['inj'] for d in diag.values() if d['inj'] is not None]
    raws = [n for n, d in diag.items() if d['mode'] == 'raw']
    summary = {'sha256': sha, 'tensors': len(diag), 'mean_dW': float(np.mean(dws)) if dws else 0.0,
               'mean_inject_ratio': float(np.mean(injs)) if injs else None, 'raw_fallback_tensors': len(raws),
               'norm': float(math.sqrt(sum(float(np.square(v).sum(dtype=np.float64)) for k, v in out.items() if k.endswith('|delta'))))}
    core.save_record(record_path, identity, _clean(summary))
    return fp, directory, _clean(summary)


# ---------------------------------------------------------------------------- entry point
def ensure(context):
    """Build (or reuse) N. Returns None when the step is off, else a dict with status 'go' (directory, fingerprint),
    'no-go' or, with on_error='skip', 'failed'."""
    o = options(context['settings'])
    if not o['enabled']:
        return None
    try:
        return _ensure(context, o)
    except Exception as error:
        if o['on_error'] == 'skip' and not o['require_all_donors']:
            print(f'[N] 跨架构增量 N 这次没能算出（{type(error).__name__}: {str(error)[:300]}）；按 on_error="skip"，本次融合不含 N。', flush=True)
            return {'status': 'failed', 'error': f'{type(error).__name__}: {str(error)[:300]}', 'repo': o['repo']}
        raise


_RUN_ONLY = ('enabled', 'while_waiting', 'on_error', 'keep_activations', 'dir', 'nu_grid', 'batch_docs', 'chunk_docs',
             'checkpoint_every', 'act_cache_gib', 'act_revision', 'revision', 'tag')


def _result_options(o):
    """The options a finished N depends on (not how it was run, nor nu)."""
    return {**{k: o[k] for k in DEFAULTS if k not in _RUN_ONLY}, 'require_all_donors': bool(o.get('require_all_donors', True))}


def _cached_result(run):
    """A finished result for these options, this donor commit and this B, without touching the activation dataset
    again (it may be gone by now): the record keeps the dataset commit, calibration hash and target key it used."""
    path = run.root / 'tm_result.record.json'
    if not path.is_file():
        return None
    try:
        envelope = json.loads(path.read_text(encoding='utf-8'))
        identity = envelope['document']['identity']
    except (OSError, ValueError, KeyError, TypeError):
        return None
    o = run.o
    inputs = identity.get('inputs') or {}
    if (identity.get('options') != _result_options(o) or inputs.get('version') != TM_VERSION
            or inputs.get('donor') != {'repo': o['repo'], 'commit': _pin_donor(run)}
            or inputs.get('B') != run.ctx['identity']['inputs']['B']
            or (inputs.get('act_repo') or {}).get('repo') != o['act_repo']
            or (o['act_revision'] and inputs['act_repo'].get('commit') != o['act_revision'])
            or inputs.get('docs') != int(o['docs']) or inputs.get('seq') != int(o['seq_len']) or inputs.get('modules') != o['modules']):
        return None
    done = _record(path, identity)
    if done is None or (done['status'] == 'go' and not (Path(done['directory']) / 'tm_delta.safetensors').is_file()):
        return None
    return done


def _ensure(context, o):
    run = _Run(context, o)
    done = _cached_result(run)
    if done is not None:
        run.say(f'跨架构增量 N（{o["repo"]}）沿用已有结果：{done["status"]}')
        return done
    inp = _inputs(run)
    ident = _identity(run, inp)
    final_identity = {'kind': 'tm_result', 'inputs': ident, 'options': _result_options(o)}
    final_path = run.root / 'tm_result.record.json'
    run.say(f'跨架构增量 N：{o["repo"]} @ {inp["donor_commit"][:10]}，{inp["donor_layers"]} 层（记录 {len(inp["source_layers"])} 层）；'
            f'target 激活 {o["act_repo"]} @ {inp["act_sha"][:10]}（键 {inp["target_key"]}），{o["docs"]} 篇；模块 {o["modules"]}')
    meta_dir = Path(inp['meta_dir'])
    tok = _load_tokenizer(meta_dir)
    acts_dir = run.root / 'donor_acts' / core.digest(_acts_identity(run, inp))[:24]
    files, acts = _donor_acts(run, inp, tok, lambda: DonorForward(run.eng, o['repo'], meta_dir, o))
    gate = _record(run.root / 'gate.record.json', _gate_identity(run, inp))
    tgt, tdir = _target_store(run, inp)
    src = ActStore(files, o['docs'], o['act_cache_gib'] / 2)
    verdict = _verdict(run, inp, ident, tgt, src)
    table = verdict['modules']
    forced = False
    if not verdict['go'] and o['require_all_donors']:
        # Every donor must contribute: build N from the modules whose self-match check passed (all of them if none did),
        # and say so in the result and the model card.
        modules = [m for m, r in table.items() if r['verdict'] != 'SKIP'] or list(o['modules'])
        verdict = {**verdict, 'go': modules}
        forced = True
        run.say(f'没有模块达到融合阈值；按 require_all_donors 仍用 {modules} 构造 N（结果里记为 forced）')
    if not verdict['go']:
        result = {'status': 'no-go', 'repo': o['repo'], 'commit': inp['donor_commit'], 'verdict': table,
                  'reason': '没有模块达到融合阈值（z 倍数与同 eps 噪声锐度倍数）'}
        core.save_record(final_path, final_identity, _clean(result))
        run.say('结论 NO-GO：本次融合不含 N')
        return _clean(result)
    pair_states = {mod: _pair_ot(run, inp, ident, tgt, src, mod) for mod in verdict['go']}
    fp, directory, summary = _fuse(run, inp, ident, tgt, src, verdict, pair_states)
    result = {'status': 'go', 'repo': o['repo'], 'commit': inp['donor_commit'], 'fingerprint': fp, 'directory': str(directory),
              'modules': verdict['go'], 'verdict': table, 'summary': summary, 'gate': gate,
              'act_repo': {'repo': o['act_repo'], 'commit': inp['act_sha'], 'target_key': inp['target_key']},
              'alpha': o['alpha'], 'topk': o['topk'], 'forced': forced}
    core.save_record(final_path, final_identity, _clean(result))
    run.say(f'N 已算好：{summary["tensors"]} 张矩阵，平均 |ΔW|/|W| = {summary["mean_dW"]:.5f}，'
            f'平均 注入/收缩 = {summary["mean_inject_ratio"]}，退回 raw 的张量 {summary["raw_fallback_tensors"]} 张')
    # local copies of the target activations are re-downloadable
    if tdir.is_relative_to(run.scratch):
        shutil.rmtree(tdir, ignore_errors=True)
    if not o['keep_activations']:
        shutil.rmtree(acts_dir, ignore_errors=True)
    return _clean(result)
'''
load_embedded_module('qmerge_tm', MODULE_SOURCE)
del MODULE_SOURCE


In [ ]:
# 内嵌源代码；独立运行无需额外文件。
MODULE_SOURCE = r'''"""Second cross-architecture delta N2 (SenseNova-Loop-9B-Preview) onto B, the same way as N (Nanbeige, qmerge_tm).

    F = S + lambda*[D - kappa*(S-B)] + mu*C + nu*N + nu2*N2,     N2[rows] = alpha * (acc[rows] - B[rows])

Same method, thresholds and injection rule as qmerge_tm, on the same calibration texts and the same stored target
activations (read through qmerge_tm, so both deltas use one activation-dataset commit). What is new here:

  * the donor is a recurrent-depth model: 32 distinct decoder layers, layers [4, 28) form a block that runs twice, so
    one forward executes 56 layers. Every *execution* is its own candidate donor layer for the CKA prefilter and the
    pair / layer OT (its depth is its position among the 56); the weights of an execution are those of its physical
    layer, so one donor matrix can be transported to several target layers. The activations are the actual inputs and
    outputs of each projection at that execution, so the two residual streams of the loop (mHC) need no choice;
  * the donor forward is the author's own remote code (trust_remote_code), pinned to one commit and run only after the
    user has read it and written its digest into the settings (code_sha256). It runs in a child process with its own
    transformers version (the author tested 5.18) installed in a separate directory: the notebook's own transformers
    version, which is part of the identity of every cached score, never changes. Weights stay in their checkpoint
    dtypes in RAM (BF16 backbone about 18 GB, mHC parameters FP32); each decoder layer is cast to float32 just while it
    runs, so the arithmetic is float32 as for Nanbeige;
  * nothing about the architecture is taken on trust: the forward gate checks the execution order it observes, the
    layer types, widths, heads and vocabulary against the model card (settings "expect") and stops on any difference,
    and compares the float32 path with the model's native forward (cosine of the next-token logits) besides the four
    short prompts;
  * k_proj is moved only when the head dimension, the rotary dimension and the M-RoPE layout of the donor equal B's
    and the stored target activations hold k_proj ("k_proj": "auto"). Linear-attention projections are not moved: the
    shared activation dataset has no target activations for B's linear-attention layers (and their recurrence
    parameters have no counterpart), so there is nothing to transport them against.
"""
import gc
import hashlib
import json
import math
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import torch

import qmerge_core as core
import qmerge_runtime as runtime
import qmerge_tm as tm

TM2_VERSION = 'tm2-cpu-2026-10-31.1'
ACTS2_VERSION = 'tm2-donor-acts-2026-10-31.1'
OT2_VERSION = 'tm2-ot-2026-10-31.1'

# Model card of sensenova/SenseNova-Loop-9B-Preview (2026-10-30). Checked against the pinned config and the observed
# forward; any difference stops the step (edit "expect" only after reading the actual config.json).
EXPECT = {'layers': 32, 'loop': [4, 28], 'loop_times': 2, 'executions': 56, 'hidden': 4096, 'ffn': 12288,
          'heads': 16, 'kv_heads': 4, 'head_dim': 256, 'vocab': 248320, 'linear_per_full': 3}
# The OT, verdict and injection options are N's (qmerge_tm.DEFAULTS) unless tm_extra2 overrides them.
OT_KEYS = ('eps_delta', 'eps_delta_out', 'eps_floor', 'eps_cap', 'iters_inner', 'iters_outer', 'tol', 'top_j', 'conc_min',
           'topk', 'alpha', 'inject', 'inject_min_ratio', 'inject_center', 'bcast_max', 'z_go', 'sharp_margin', 'marginal',
           'up_excess_max', 'up_sharp_mult', 'noise_seed', 'pair_keep', 'pair_depth_prior', 'chunk_rows', 'act_cache_gib',
           'gate_min_hits')
SHARED = ('act_repo', 'act_revision', 'calib_file', 'docs', 'seq_len')      # always N's: one corpus, one dataset commit
DEFAULTS = {
    'enabled': True, 'tag': 'SenseNovaTM', 'repo': 'sensenova/SenseNova-Loop-9B-Preview', 'revision': '',
    'code_sha256': '', 'transformers': '5.18.*', 'isolated_env': True,
    'modules': ['self_attn.v_proj', 'self_attn.o_proj', 'mlp.gate_proj', 'mlp.up_proj', 'mlp.down_proj'],
    'k_proj': 'auto', 'module_map': {}, 'layers_modules': [], 'mlp_stride': 2, 'expect': dict(EXPECT),
    'nu_grid': [0.25, 0.5, 1.0], 'batch_docs': 16, 'chunk_docs': 250, 'gate_min_cosine': 0.98,
    'while_waiting': True, 'on_error': 'raise', 'keep_activations': True, 'keep_checkpoint': False, 'dir': '',
}
_RUN_ONLY = ('enabled', 'while_waiting', 'on_error', 'keep_activations', 'keep_checkpoint', 'dir', 'nu_grid',
             'batch_docs', 'chunk_docs', 'act_cache_gib', 'revision', 'tag', 'isolated_env')
_VISION = re.compile(r'(^|\.)(visual|vision_tower|vision_model|vision|vit|mm_projector|multi_modal_projector|merger|image_newline)(\.|$)')
_RISKY = re.compile(r'\b(subprocess|os\.system|os\.popen|socket|urllib|requests|httpx|ftplib|paramiko|pickle\.loads?|'
                    r'marshal|ctypes|eval\(|exec\(|__import__|shutil\.rmtree|os\.remove|os\.unlink|base64\.b64decode|'
                    r'hf_hub_download|snapshot_download|torch\.hub)')


def options(settings):
    raw = dict(settings.get('tm_extra2') or {})
    unknown = set(raw) - set(DEFAULTS) - set(OT_KEYS)
    if unknown & set(SHARED):
        raise ValueError(f'tm_extra2 不能单独设置 {sorted(unknown & set(SHARED))}：N₂ 与 N 共用同一个激活仓和校准语料，改 tm_extra')
    if unknown:
        raise ValueError(f'tm_extra2 里有不认识的项：{sorted(unknown)}')
    n1 = tm.options(settings)
    o = {**{k: tm.DEFAULTS[k] for k in OT_KEYS}, **{k: n1[k] for k in OT_KEYS if k in n1}, **DEFAULTS,
         **{k: n1[k] for k in SHARED}, **raw}
    o['modules'] = list(o['modules'])
    o['expect'] = {**EXPECT, **(raw.get('expect') or {})}
    o['module_map'] = dict(o['module_map'])
    if not o['enabled']:
        return o
    if not isinstance(o['repo'], str) or not o['repo'].strip():
        raise ValueError('tm_extra2["repo"] 不能为空')
    if o['revision'] and not re.fullmatch(r'[0-9a-fA-F]{40,64}', str(o['revision'])):
        raise ValueError('tm_extra2["revision"] 要么留空，要么是完整 commit SHA')
    if o['code_sha256'] and not re.fullmatch(r'[0-9a-f]{64}', str(o['code_sha256'])):
        raise ValueError('tm_extra2["code_sha256"] 是审读过的远程代码包的 SHA-256（64 位小写十六进制），留空则先打印它')
    allowed = tm.DEFAULTS['modules']
    bad = [m for m in o['modules'] if m not in allowed]
    if bad or not o['modules']:
        raise ValueError(f'tm_extra2["modules"] 只能取 {allowed}（k_proj 由 "k_proj" 控制；线性注意力的投影不搬）')
    if o['k_proj'] not in ('auto', True, False):
        raise ValueError('tm_extra2["k_proj"] 只能是 "auto" / True / False')
    if o['inject'] not in ('raw', 'row', 'tensor') or not 0 < float(o['alpha']) <= 0.5:
        raise ValueError('tm_extra2：inject 只能是 raw / row / tensor，alpha 要在 (0, 0.5]')
    grid = [float(x) for x in o['nu_grid']]
    if not grid or any(not math.isfinite(x) or not 0 < x <= 2 for x in grid):
        raise ValueError('tm_extra2["nu_grid"] 要给出 (0, 2] 内的系数（每个供体都要有贡献，不能含 0）')
    o['nu_grid'] = sorted(set(grid))
    if int(o['mlp_stride']) < 1 or int(o['chunk_docs']) < 1 or int(o['batch_docs']) < 1:
        raise ValueError('tm_extra2：mlp_stride、chunk_docs、batch_docs 都要是正整数')
    o['require_all_donors'] = bool(settings.get('require_all_donors', True))
    if o['on_error'] not in ('raise', 'skip'):
        raise ValueError('tm_extra2["on_error"] 只能是 raise / skip')
    return o


# ============================================================================ pure helpers (unit-tested)
def expected_order(expect):
    """Physical layer of every execution: [0..a) + [a..b) * times + [b..L)."""
    a, b = expect['loop']
    return list(range(0, a)) + list(range(a, b)) * int(expect['loop_times']) + list(range(b, int(expect['layers'])))


def schedule(order):
    """[(execution, physical layer, pass)] from the observed order of decoder-layer calls in one forward."""
    seen = {}
    out = []
    for e, p in enumerate(order):
        seen[p] = seen.get(p, 0) + 1
        out.append((e, int(p), seen[p]))
    return out


def full_attention_layers(layer_types, n):
    """Indices of the full-attention layers from config layer_types (or None when the config has none)."""
    if not layer_types:
        return None
    if len(layer_types) != n:
        raise ValueError(f'layer_types 有 {len(layer_types)} 项，与 {n} 层不符')
    return [i for i, t in enumerate(layer_types) if 'full' in str(t).lower() or str(t).lower() in ('attention', 'softmax')]


def check_structure(report, expect):
    """Every difference between what the forward and the config show and the model card; [] when they agree."""
    out = []
    order = report.get('order') or []
    want = expected_order(expect)
    if order != want:
        out.append(f'解码层的执行顺序与模型卡不同：观察到 {len(order)} 次执行 {_runs(order)}，'
                   f'模型卡是 {len(want)} 次 {_runs(want)}（第 {expect["loop"][0]}–{expect["loop"][1] - 1} 层执行 {expect["loop_times"]} 遍）')
    if report.get('layers') != expect['layers']:
        out.append(f'不同的解码层 {report.get("layers")} 个，模型卡 {expect["layers"]} 个')
    for key in ('hidden', 'ffn', 'heads', 'kv_heads', 'head_dim', 'vocab'):
        if report.get(key) is not None and int(report[key]) != int(expect[key]):
            out.append(f'{key} = {report[key]}，模型卡 {expect[key]}')
        elif report.get(key) is None:
            out.append(f'{key}：config 和模块形状里都确定不了')
    full = report.get('full_layers')
    if full is None:
        out.append('分不出哪些层是全注意力（config 没有 layer_types，层里也找不到 k_proj）')
    else:
        n, k = int(expect['layers']), int(expect['linear_per_full']) + 1
        phases = [ph for ph in range(k) if sorted(full) == [i for i in range(n) if i % k == ph]]
        if not phases:
            out.append(f'全注意力层 {full} 不是“每 {k} 层 {k - 1} 层线性注意力 + 1 层全注意力”')
    return out


def _runs(order):
    """Compact text of an execution order: 0-3,4-27,4-27,28-31."""
    if not order:
        return '[]'
    parts, a = [], order[0]
    for x, y in zip(order, order[1:] + [None]):
        if y != x + 1:
            parts.append(f'{a}' if a == x else f'{a}-{x}')
            a = y
    return ','.join(parts)


def donor_candidates(execs, full_layers, modules, mlp_stride):
    """{module: [executions]}: attention modules at every execution of a full-attention layer; MLP modules at every
    mlp_stride-th execution (the same density as Nanbeige's every second layer)."""
    full = set(full_layers)
    out = {}
    for mod in modules:
        if mod.startswith('self_attn.'):
            out[mod] = [e for e, p, _ in execs if p in full]
        else:
            out[mod] = [e for e, _, _ in execs if e % int(mlp_stride) == 0]
        if not out[mod]:
            raise ValueError(f'{mod}：没有可用的供体执行层')
    return out


def rope_compatible(donor, target):
    """(bool, reason): can k_proj channels keep their rotary frequency? Head dim, rotary dim and M-RoPE layout equal."""
    def info(cfg):
        cfg = cfg.get('text_config', cfg)
        rope = dict(cfg.get('rope_scaling') or {}) | dict(cfg.get('rope_parameters') or {})
        hd = cfg.get('head_dim') or (cfg['hidden_size'] // cfg['num_attention_heads'])
        frac = float(rope.get('partial_rotary_factor', cfg.get('partial_rotary_factor', 1.0)))
        return {'head_dim': int(hd), 'rotary_dim': int(round(hd * frac)), 'mrope_section': rope.get('mrope_section'),
                'mrope_interleaved': bool(rope.get('mrope_interleaved', False)), 'type': rope.get('rope_type', rope.get('type', 'default')),
                'theta': rope.get('rope_theta', cfg.get('rope_theta'))}
    a, b = info(donor), info(target)
    for key in ('head_dim', 'rotary_dim', 'mrope_section', 'mrope_interleaved'):
        if a[key] != b[key]:
            return False, f'{key}：供体 {a[key]}，B {b[key]}'
    note = '' if a['theta'] == b['theta'] else f'（rope_theta 不同：供体 {a["theta"]}，B {b["theta"]}；通道配对不变，频率略有差别）'
    return True, '头维、旋转维与 M-RoPE 布局一致' + note


def tokenizer_digest(path):
    """Digest of what decides the token ids: the model (vocab, merges), added tokens, normalizer and pre-tokenizer."""
    data = json.loads(Path(path).read_text(encoding='utf-8'))
    keep = {k: data.get(k) for k in ('model', 'added_tokens', 'normalizer', 'pre_tokenizer')}
    return core.digest(keep)


def code_bundle(directory):
    """(digest, [(file, sha256)]) of the remote code (*.py) in a metadata directory."""
    files = sorted(p for p in Path(directory).rglob('*.py') if '__pycache__' not in p.parts)
    rows = [(str(p.relative_to(directory)), hashlib.sha256(p.read_bytes()).hexdigest()) for p in files]
    return core.digest(rows), rows


def risky_lines(directory, limit=40):
    out = []
    for p in sorted(Path(directory).rglob('*.py')):
        for n, line in enumerate(p.read_text(encoding='utf-8', errors='replace').splitlines(), 1):
            if _RISKY.search(line):
                out.append(f'{p.name}:{n}: {line.strip()[:160]}')
                if len(out) >= limit:
                    return out
    return out


def pick_nu(tried, base_mean, required):
    """The nu to take from a line search [{'nu', 'mean_nll', 'feasible'}]: every donor required -> the best feasible
    (the best overall when none is); otherwise only a strictly lower NLL than without the term. 0 = not taken."""
    if required:
        good = [(t['mean_nll'], t['nu']) for t in tried if t['feasible']] or [(t['mean_nll'], t['nu']) for t in tried]
    else:
        good = [(t['mean_nll'], t['nu']) for t in tried if t['feasible'] and t['mean_nll'] < base_mean]
    return min(good)[1] if good else 0.


# ============================================================================ donor forward (child process)
# Runs as its own script (written next to the activation files) with PYTHONPATH = the isolated transformers directory.
# It imports only torch, numpy, safetensors and transformers: none of the notebook modules.
WORKER_SOURCE = r"""
import json, math, os, sys, time
from pathlib import Path
import numpy as np
import torch

class _Stop(Exception):
    pass

def log(text):
    print(text, flush=True)

def checkpoint_dtypes(directory):
    from safetensors import safe_open
    out = {}
    for f in sorted(Path(directory).glob('*.safetensors')):
        with safe_open(str(f), framework='pt') as h:
            for k in h.keys():
                out[k] = str(h.get_slice(k).get_dtype())
    return out

def match_key(name, keys):
    if name in keys:
        return name
    tails = [k for k in keys if k.endswith('.' + name) or name.endswith('.' + k)]
    if len(tails) == 1:
        return tails[0]
    stripped = name.split('.', 1)[1] if '.' in name else name
    tails = [k for k in keys if k.endswith(stripped)]
    return tails[0] if len(tails) == 1 else None

def restore_fp32(model, directory):
    # Parameters stored as F32 in the checkpoint (mHC) stay float32 even if the loader cast them.
    from safetensors import safe_open
    dtypes = checkpoint_dtypes(directory)
    index = {}
    for f in sorted(Path(directory).glob('*.safetensors')):
        with safe_open(str(f), framework='pt') as h:
            for k in h.keys():
                index[k] = f
    fixed = 0
    for name, p in model.named_parameters():
        key = match_key(name, index)
        if key is None or dtypes.get(key) not in ('F32', 'float32', 'torch.float32') or p.dtype == torch.float32:
            continue
        with safe_open(str(index[key]), framework='pt') as h:
            p.data = h.get_tensor(key).to(torch.float32)
        fixed += 1
    return fixed

def find_layers(model, n=None, paths=()):
    if paths:
        # The decoder layers split over several lists (e.g. before / loop / after): concatenated in this order.
        lists = [model.get_submodule(p) for p in paths]
        return '+'.join(paths), torch.nn.ModuleList([m for l in lists for m in l])
    found = [(name, m) for name, m in model.named_modules() if isinstance(m, torch.nn.ModuleList)
             and name.split('.')[-1] == 'layers' and 'vision' not in name and 'visual' not in name and len(m) > 1]
    if n is not None:
        found = [x for x in found if len(x[1]) == n] or found
    if len(found) != 1:
        raise RuntimeError('找不到唯一的语言模型解码层列表：' + str([(a, len(b)) for a, b in found]))
    return found[0]

class Fp32Layers:
    # Each decoder layer's floating parameters are cast to float32 while it runs and back afterwards; the counter of
    # executions in the current forward tells hooks which execution of a shared layer is running.
    def __init__(self, layers, stop_after=None):
        self.layers, self.stop_after = layers, stop_after
        self.order, self.saved, self.handles, self.active = [], {}, [], True
        for i, layer in enumerate(layers):
            self.handles.append(layer.register_forward_pre_hook(self._pre(i)))
            self.handles.append(layer.register_forward_hook(self._post(i)))

    def reset(self):
        self.order = []

    @property
    def current(self):
        return len(self.order) - 1

    def _pre(self, i):
        def hook(module, args):
            self.order.append(i)
            if self.active:
                for name, p in module.named_parameters():
                    if p.is_floating_point() and p.dtype != torch.float32:
                        self.saved[(i, name)] = p.dtype
                        p.data = p.data.to(torch.float32)
            if args and torch.is_tensor(args[0]) and args[0].is_floating_point() and self.active and args[0].dtype != torch.float32:
                return (args[0].to(torch.float32),) + tuple(args[1:])
        return hook

    def _post(self, i):
        def hook(module, args, output):
            for name, p in module.named_parameters():
                dt = self.saved.pop((i, name), None)
                if dt is not None:
                    p.data = p.data.to(dt)
            if self.stop_after is not None and self.current >= self.stop_after:
                raise _Stop()
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()

class Recorder:
    # Per-document means of the input and output of each wanted Linear at each wanted execution.
    def __init__(self, layers, fp, wanted, module_map, docs):
        self.fp, self.wanted, self.rows = fp, wanted, None
        self.acc, self.handles = {}, []
        for mod, execs in wanted.items():
            name = module_map.get(mod, mod)
            for phys in sorted({p for _, p in execs}):
                sub = layers[phys].get_submodule(name)
                if not isinstance(sub, torch.nn.Linear):
                    raise RuntimeError(f'第 {phys} 层的 {name} 不是 nn.Linear（{type(sub).__name__}）；在 module_map 里指明投影的名字')
                for e, p in execs:
                    if p == phys:
                        self.acc[(e, mod)] = {'in_mean': torch.zeros(docs, sub.in_features), 'out_mean': torch.zeros(docs, sub.out_features)}
                self.handles.append(sub.register_forward_hook(self._hook(mod, phys)))

    def _hook(self, mod, phys):
        def hook(module, args, output):
            key = (self.fp.current, mod)
            slot = self.acc.get(key)
            if slot is None or self.rows is None:
                return
            x = args[0] if isinstance(args, tuple) else args
            y = output[0] if isinstance(output, (tuple, list)) else output
            slot['in_mean'][self.rows] = x.float().mean(1)
            slot['out_mean'][self.rows] = y.float().mean(1)
        return hook

    def remove(self):
        for h in self.handles:
            h.remove()

def describe(model, layers, cfg):
    tc = cfg.get('text_config', cfg)
    def first(*names):
        for n in names:
            if tc.get(n) is not None:
                return tc[n]
        return None
    out = {'layers': len(layers)}
    out['hidden'] = first('hidden_size')
    out['ffn'] = first('intermediate_size')
    out['heads'] = first('num_attention_heads')
    out['kv_heads'] = first('num_key_value_heads')
    out['head_dim'] = first('head_dim') or (out['hidden'] // out['heads'] if out['hidden'] and out['heads'] else None)
    out['vocab'] = first('vocab_size')
    emb = model.get_input_embeddings()
    if emb is not None:
        out['vocab_embedding'] = int(emb.weight.shape[0])
    lin = {}
    for i, layer in enumerate(layers):
        lin[i] = {n: [m.out_features, m.in_features] for n, m in layer.named_modules() if isinstance(m, torch.nn.Linear)}
    out['linear'] = lin
    types = first('layer_types')
    out['layer_types'] = types
    return out

def ids_batches(tok, texts, a, b, seq, bsz):
    enc = tok(texts[a:b], truncation=True, max_length=seq, padding=False)
    by_len = {}
    for d, ids in enumerate(enc['input_ids']):
        if len(ids) < 2:
            raise ValueError('有校准文本分词后不足 2 个 token')
        by_len.setdefault(len(ids), []).append(d)
    out = []
    for n in sorted(by_len, reverse=True):
        docs = by_len[n]
        for k in range(0, len(docs), bsz):
            part = docs[k:k + bsz]
            out.append((torch.tensor(part), torch.tensor([enc['input_ids'][d] for d in part])))
    return out, sum(len(x) for x in enc['input_ids'])

def main(job_path):
    job = json.loads(Path(job_path).read_text())
    torch.set_num_threads(int(job.get('threads') or os.cpu_count() or 1))
    import transformers
    from transformers import AutoConfig, AutoModelForCausalLM, AutoTokenizer
    ck = job['checkpoint']
    cfg = json.loads((Path(ck) / 'config.json').read_text())
    log(f'[N2 worker] transformers {transformers.__version__}，torch {torch.__version__}，{torch.get_num_threads()} 线程')
    tok = AutoTokenizer.from_pretrained(ck, local_files_only=True, trust_remote_code=True)
    import transformers as tf
    model, errors = None, []
    for cls_name in ('AutoModelForCausalLM', 'AutoModelForImageTextToText', 'AutoModelForVision2Seq'):
        cls = getattr(tf, cls_name, None)
        if cls is None:
            continue
        try:
            # Checkpoint dtypes (BF16 backbone; F32 tensors are restored below). Only the text path is run.
            model = cls.from_pretrained(ck, local_files_only=True, trust_remote_code=True, dtype='auto', low_cpu_mem_usage=True)
            break
        except ValueError as exc:
            errors.append(f'{cls_name}: {str(exc)[:200]}')
    if model is None:
        raise RuntimeError('远程代码里没有可用的语言模型类：' + ' | '.join(errors))
    model.eval()
    for p in model.parameters():
        p.requires_grad_(False)
    fixed = restore_fp32(model, ck)
    lname, layers = find_layers(model, job.get('layers'), job.get('layers_modules') or ())
    prefixes = tuple(x + '.' for x in lname.split('+'))
    result = {'transformers': transformers.__version__, 'torch': torch.__version__, 'layers_module': lname,
              'restored_fp32': fixed, **describe(model, layers, cfg)}
    if job['mode'] == 'probe':
        # 1. the native forward (checkpoint dtypes, as the author runs it), with the execution order observed
        order = []
        handles = [layer.register_forward_pre_hook(lambda m, a, i=i: order.append(i)) for i, layer in enumerate(layers)]
        native = []
        with torch.no_grad():
            for prompt in job['prompts']:
                ids = torch.tensor([tok(prompt)['input_ids']])
                order.clear()
                native.append(model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False).logits[0, -1].float())
                result['order'] = list(order)
        for h in handles:
            h.remove()
        if not result['order']:
            raise RuntimeError('解码层不是通过 module 调用的（forward hook 没有触发）；这种远程代码要手写前向')
        # 2. the float32 path used for the activations
        for name, p in model.named_parameters():
            if not name.startswith(prefixes) and p.is_floating_point():
                p.data = p.data.to(torch.float32)
        fp = Fp32Layers(layers)
        items = []
        with torch.no_grad():
            for k, (prompt, expected) in enumerate(zip(job['prompts'], job['expected'])):
                ids = torch.tensor([tok(prompt)['input_ids']])
                fp.reset()
                logits = model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False).logits[0, -1].float()
                cos = float(torch.nn.functional.cosine_similarity(logits, native[k], dim=0))
                top = int(logits.argmax())
                got = tok.decode([top])
                hit = any(got.strip().lower() == x.strip().lower() for x in expected)
                items.append({'prompt': prompt, 'top1': got, 'hit': hit, 'cosine_vs_native': cos,
                              'same_top1_as_native': top == int(native[k].argmax())})
                log(f'[N2 worker] {"通过" if hit else "失败"} {prompt!r} -> {got!r}；与原生前向的余弦 {cos:.5f}')
        result['gate'] = items
    else:
        # extraction of documents a..b
        texts = json.loads(Path(job['calib']).read_text(encoding='utf-8'))[:job['docs']]
        a, b = job['range']
        batches, tokens = ids_batches(tok, texts, a, b, job['seq'], job['batch_docs'])
        for name, p in model.named_parameters():
            if not name.startswith(prefixes) and p.is_floating_point():
                p.data = p.data.to(torch.float32)
        wanted = {mod: [(int(e), int(p)) for e, p in pairs] for mod, pairs in job['wanted'].items()}
        last = max(e for pairs in wanted.values() for e, _ in pairs)
        fp = Fp32Layers(layers, stop_after=last)
        rec = Recorder(layers, fp, wanted, job.get('module_map') or {}, b - a)
        started = time.perf_counter()
        with torch.no_grad():
            for n, (rows, ids) in enumerate(batches, 1):
                fp.reset()
                rec.rows = rows
                try:
                    model(input_ids=ids, attention_mask=torch.ones_like(ids), use_cache=False)
                except _Stop:
                    pass
                if fp.order != job['order'][:len(fp.order)]:
                    raise RuntimeError('这一批的执行顺序与闸门时观察到的不同：' + str(fp.order[:60]))
                if n in (1, 3) or n % 20 == 0 or n == len(batches):
                    el = time.perf_counter() - started
                    log(f'[N2 worker] 第 {a}–{b - 1} 篇：{n}/{len(batches)} 批，{el / 60:.1f} 分钟，估计还需 {el / n * (len(batches) - n) / 60:.0f} 分钟')
        from safetensors.torch import save_file
        out = Path(job['out'])
        out.mkdir(parents=True, exist_ok=True)
        by_exec = {}
        for (e, mod), d in rec.acc.items():
            for sfx, value in d.items():
                if not torch.isfinite(value).all():
                    raise RuntimeError(f'执行 {e} 的 {mod} 出现 NaN/Inf')
                big = float(value.abs().max()) > 60000.0
                by_exec.setdefault(e, {})[f'{e}|{mod}|{sfx}'] = (value if big else value.to(torch.float16)).contiguous()
        sizes = {}
        for e, flat in by_exec.items():
            path = out / f'L{e:03d}.safetensors'
            tmp = path.with_name(path.name + '.tmp')
            save_file(flat, str(tmp), metadata={'schema': job['schema'], 'exec': str(e), 'docs': f'{a}-{b}', 'identity': job['identity']})
            tmp.replace(path)
            sizes[str(e)] = path.stat().st_size
        result.update(sizes=sizes, tokens=tokens, batches=len(batches), hours=(time.perf_counter() - started) / 3600)
    Path(job['result']).write_text(json.dumps(result, ensure_ascii=False))

if __name__ == '__main__':
    main(sys.argv[1])
"""


# ============================================================================ orchestration (notebook side)
class _Run(tm._Run):
    def __init__(self, context, o):
        super().__init__(context, {**o, 'dir': o['dir'] or str(Path(context['root']) / 'tm_delta2')})
        self.scratch = Path(context['scratch']) / 'tm_delta2'
        self.scratch.mkdir(parents=True, exist_ok=True)
        # N's own run object: the activation dataset, calibration texts and target files are N's, at N's commit.
        self.n1 = tm._Run(context, tm.options(context['settings']))

    def say(self, text):
        print('[N2] ' + text, flush=True)


def _pin_donor(run):
    o, eng = run.o, run.eng
    if o['revision']:
        old = eng.revisions.get(o['repo'])
        if old and old != o['revision']:
            raise ValueError(f'{o["repo"]}：tm_extra2["revision"] 与已锁定的 commit 不同')
        eng.revisions[o['repo']] = o['revision']
    return eng.pin(o['repo'])


def _code(run, meta_dir):
    """Remote code at the pinned commit, read and approved by the user before anything executes it."""
    eng, o = run.eng, run.o
    names = [n for n in eng.files(o['repo']) if n.endswith('.py') and '/' not in n.strip('/')]
    for name in names:
        path = Path(meta_dir) / runtime.safe_relative(name)
        content = eng._small_bytes(o['repo'], name)
        if not path.exists() or path.read_bytes() != content:
            path.write_bytes(content)
    if not names:
        raise ValueError(f'{o["repo"]} 在锁定的 commit 里没有 .py 远程代码；trust_remote_code 用不上，检查仓库')
    digest, rows = code_bundle(meta_dir)
    if o['code_sha256'] != digest:
        lines = '\n'.join(f'  {n}  {h[:16]}…' for n, h in rows)
        risky = risky_lines(meta_dir)
        raise RuntimeError(
            f'N₂ 要执行 {o["repo"]} 的远程代码（commit {eng.pin(o["repo"])[:10]}），先审读再放行。代码在 {meta_dir}：\n{lines}\n'
            + (('需要特别看的行（网络、进程、文件删除、动态执行等）：\n  ' + '\n  '.join(risky) + '\n') if risky else
               '没有找到网络、进程、文件删除、动态执行之类的调用。\n')
            + f'读完确认没问题后，在第 1 节把 SETTINGS["tm_extra2"]["code_sha256"] 设成 "{digest}" 再运行。'
              '代码或 commit 一变，这个值就对不上，会再停下来。')
    return digest, rows


def _isolated_env(run):
    """A pip --target directory with the donor's transformers version; the notebook's own transformers is untouched."""
    o = run.o
    spec = str(o['transformers']).strip()
    want = ('transformers' + (spec if spec[:1] in '=<>!~' else '==' + spec)) if spec else 'transformers'
    site = run.scratch / ('site-' + core.digest({'transformers': want, 'python': sys.version_info[:2]})[:12])
    marker = site / '.tm2-ready.json'
    if not marker.is_file():
        run.say(f'为供体前向单独安装 {want}（到 {site}，不影响本内核的 transformers）')
        shutil.rmtree(site, ignore_errors=True)
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--target', str(site), want])
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', '--target', str(site), 'accelerate'])
        for heavy in ('torch', 'numpy', 'safetensors'):
            # never shadow the kernel's own builds of these
            for p in list(site.glob(heavy)) + list(site.glob(heavy + '-*')) + list(site.glob(heavy + '.libs')):
                shutil.rmtree(p, ignore_errors=True)
        probe = subprocess.run([sys.executable, '-c', 'import transformers, accelerate; print(transformers.__version__, accelerate.__version__)'],
                               env={**os.environ, 'PYTHONPATH': str(site)}, capture_output=True, text=True, check=True)
        version, acc = probe.stdout.split()
        core.atomic_json(marker, {'requirement': want, 'transformers': version, 'accelerate': acc})
    return site, json.loads(marker.read_text())


def _worker(run, site, job, label):
    """Run the donor worker as a child process, streaming its output; returns its result JSON."""
    work = run.scratch / 'worker'
    work.mkdir(parents=True, exist_ok=True)
    script = work / 'tm2_donor_worker.py'
    script.write_text(WORKER_SOURCE, encoding='utf-8')
    job = {**job, 'result': str(work / f'{label}.result.json')}
    Path(job['result']).unlink(missing_ok=True)
    path = work / f'{label}.job.json'
    core.atomic_json(path, job)
    env = {**os.environ, 'PYTHONPATH': str(site) if site else os.environ.get('PYTHONPATH', ''), 'HF_HUB_OFFLINE': '1',
           'TRANSFORMERS_OFFLINE': '1', 'TOKENIZERS_PARALLELISM': 'false'}
    proc = subprocess.Popen([sys.executable, '-u', str(script), str(path)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, env=env, cwd=str(work))
    tail = []
    for line in proc.stdout:
        line = line.rstrip()
        tail = (tail + [line])[-40:]
        if line.startswith('[N2 worker]'):
            print(line, flush=True)
    if proc.wait() != 0 or not Path(job['result']).is_file():
        raise RuntimeError('供体前向子进程失败：\n' + '\n'.join(tail[-25:]))
    return json.loads(Path(job['result']).read_text())


def _inputs(run):
    o, eng = run.o, run.eng
    donor_commit = _pin_donor(run)
    meta_dir = runtime.download_metadata(eng, o['repo'], run.root / 'donor_metadata')
    code_digest, code_rows = _code(run, meta_dir)
    cfg = json.loads((meta_dir / 'config.json').read_text())
    dc = cfg.get('text_config', cfg)
    experts = dc.get('num_experts') or dc.get('num_local_experts') or dc.get('n_routed_experts') or 0
    if int(experts) > 1:
        raise ValueError('供体是 MoE：本步只支持 dense 供体')
    cat = {k: v for k, v in eng._catalogue(o['repo']).items() if not _VISION.search(k)}
    prefix = None
    for n in cat:
        m = re.match(r'^(.*\.layers)\.(\d+)\.', n)
        if m and not re.search(r'(^|\.)(mtp|nextn)(\.|$)', m[1]):
            prefix = m[1]
            break
    if prefix is None:
        raise ValueError('供体：张量名里找不到语言模型的 .layers.N.')
    n_t, full, rec = tm._target_layers(run.n1)
    act_sha, files = tm._pin_dataset(run.n1)
    calib_path = tm._download(run.n1, o['calib_file'], act_sha, run.n1.root / 'calibration')
    calib_sha = core.sha_file(calib_path)
    texts = json.loads(calib_path.read_text(encoding='utf-8'))[:int(o['docs'])]
    bcfg = run.ctx['B'].config
    modules = list(o['modules'])
    k_note = 'k_proj 关闭（tm_extra2["k_proj"]=False）'
    if o['k_proj'] in ('auto', True):
        ok, why = rope_compatible(cfg, bcfg)
        if ok:
            modules.append('self_attn.k_proj')
            k_note = 'k_proj 搬：' + why
        elif o['k_proj'] is True:
            raise ValueError('tm_extra2["k_proj"]=True，但 RoPE 对不上：' + why)
        else:
            k_note = 'k_proj 不搬：' + why
    lookup = tm._Run(run.ctx, {**run.n1.o, 'modules': modules})
    try:
        key, tfiles, meta_rev = tm._find_target_acts(lookup, files, act_sha, calib_sha, full, rec)
    except ValueError as error:
        if 'self_attn.k_proj' not in modules or 'k_proj' not in str(error):
            raise
        modules.remove('self_attn.k_proj')
        k_note = 'k_proj 不搬：激活仓里没有 B 的 k_proj target 激活'
        lookup = tm._Run(run.ctx, {**run.n1.o, 'modules': modules})
        key, tfiles, meta_rev = tm._find_target_acts(lookup, files, act_sha, calib_sha, full, rec)
    b_commit = run.ctx['identity']['inputs']['B']['commit']
    if meta_rev and meta_rev != b_commit and not tm._same_weights(run.n1, meta_rev, b_commit):
        raise ValueError(f'target 激活是在 B 的 commit {meta_rev[:10]} 上算的，与本次 B（{b_commit[:10]}）的权重文件不同')
    b_meta = runtime.download_metadata(eng, run.ctx['B_repo'], run.root / 'B_metadata')
    try:
        same_tok = tokenizer_digest(meta_dir / 'tokenizer.json') == tokenizer_digest(b_meta / 'tokenizer.json')
    except (OSError, ValueError):
        same_tok = None
    s_shapes = run.ctx['S'].shapes
    targets = {}
    for mod in modules:
        for li in tm._target_modules_for(mod, full, rec):
            hits = [k for k in s_shapes if re.search(rf'(?:^|\.)layers\.{li}\.{re.escape(mod)}\.weight$', k)
                    and not re.search(r'(^|\.)(mtp|visual|nextn)(\.|$)', k)]
            if len(hits) != 1:
                raise ValueError(f'S 里 layers.{li}.{mod}.weight 不唯一（{len(hits)} 个）')
            targets[(mod, li)] = hits[0]
    return {'donor_commit': donor_commit, 'meta_dir': str(meta_dir), 'code_sha256': code_digest, 'code_files': code_rows,
            'donor_prefix': prefix, 'modules': modules, 'k_note': k_note, 'full': full, 'rec': rec, 'target_layers': n_t,
            'act_sha': act_sha, 'calib_sha': calib_sha, 'calib_path': str(calib_path), 'texts': texts, 'target_key': key,
            'target_files': tfiles, 'targets': targets, 'same_tokenizer': same_tok, 'config': cfg}


def _identity(run, inp):
    o = run.o
    return {'version': TM2_VERSION, 'donor': {'repo': o['repo'], 'commit': inp['donor_commit'], 'code': inp['code_sha256']},
            'B': run.ctx['identity']['inputs']['B'], 'act_repo': {'repo': o['act_repo'], 'commit': inp['act_sha']},
            'calib_sha': inp['calib_sha'], 'target_acts': inp['target_key'], 'docs': int(o['docs']), 'seq': int(o['seq_len']),
            'modules': inp['modules'], 'target_rec': inp['rec'], 'mlp_stride': int(o['mlp_stride']), 'module_map': o['module_map']}


def _probe(run, inp, site, env_info):
    """The gate: native vs float32 forward, four short prompts, and the observed structure against the model card."""
    o = run.o
    identity = {'kind': 'tm2_gate', 'version': ACTS2_VERSION, 'repo': o['repo'], 'commit': inp['donor_commit'],
                'code': inp['code_sha256'], 'env': env_info, 'torch': torch.__version__, 'expect': o['expect']}
    path = run.root / 'gate.record.json'
    done = tm._record(path, identity)
    if done is not None:
        run.say(f'前向闸门已通过（{done["hits"]}/{len(tm.GATE_PROMPTS)}，执行 {len(done["structure"]["order"])} 层）')
        return done
    ckpt = _checkpoint(run, inp)
    run.say('前向闸门：原生前向与 float32 前向各跑 4 条短提示，并核对执行顺序与结构')
    rep = _worker(run, site, {'mode': 'probe', 'checkpoint': str(ckpt), 'layers': int(o['expect']['layers']),
                              'layers_modules': list(o['layers_modules']),
                              'prompts': [p for p, _ in tm.GATE_PROMPTS], 'expected': [e for _, e in tm.GATE_PROMPTS]}, 'probe')
    full = full_attention_layers(rep.get('layer_types'), rep['layers'])
    if full is None:
        name = o['module_map'].get('self_attn.k_proj', 'self_attn.k_proj')
        full = [int(i) for i, mods in rep['linear'].items() if name in mods] or None
    structure = {**{k: rep.get(k) for k in ('order', 'layers', 'hidden', 'ffn', 'heads', 'kv_heads', 'head_dim')},
                 'vocab': rep.get('vocab') or rep.get('vocab_embedding'), 'full_layers': full}
    problems = check_structure(structure, o['expect'])
    gate = rep['gate']
    hits = sum(1 for g in gate if g['hit'])
    worst = min(g['cosine_vs_native'] for g in gate)
    if hits < int(o['gate_min_hits']):
        problems.append(f'4 条短提示只答对 {hits} 条（至少 {o["gate_min_hits"]}）')
    if worst < float(o['gate_min_cosine']) or not all(g['same_top1_as_native'] for g in gate):
        problems.append(f'float32 前向与原生前向不一致（最低余弦 {worst:.4f}，要求 ≥ {o["gate_min_cosine"]}，且 top-1 相同）')
    if problems:
        raise RuntimeError('SenseNova 的前向闸门没过，激活不能用：\n- ' + '\n- '.join(problems) +
                           '\n若 config.json 与模型卡确实不同，读完 config 后在 SETTINGS["tm_extra2"]["expect"] 里改，再运行。')
    result = {'hits': hits, 'gate': gate, 'structure': structure, 'linear': rep['linear'], 'worker': {k: rep.get(k) for k in ('transformers', 'torch', 'restored_fp32', 'layers_module')}}
    core.save_record(path, identity, tm._clean(result))
    return result


def _checkpoint(run, inp):
    """The donor's full checkpoint on the local disk, with the approved remote code next to it."""
    eng, o = run.eng, run.o
    need = sum(eng._sizes[o['repo']].get(n) or 0 for n in {v['file'] for v in eng._catalogue(o['repo']).values()})
    if shutil.disk_usage(run.scratch).free < need + 8 * 1024 ** 3:
        raise RuntimeError(f'本地盘放不下 {o["repo"]} 的权重（约 {need / 1024 ** 3:.0f} GiB，另留 8 GiB）')
    ckpt = runtime.download_checkpoint(eng, o['repo'], run.scratch / 'checkpoint')
    for name, _ in inp['code_files']:
        shutil.copyfile(Path(inp['meta_dir']) / name, Path(ckpt) / name)
    digest, _ = code_bundle(ckpt)
    if digest != inp['code_sha256']:
        raise RuntimeError('本地权重目录里的远程代码与审读过的不同；删掉 ' + str(ckpt) + ' 后重跑')
    return ckpt


def _acts_identity(run, inp, plan, env_info):
    o = run.o
    return {'kind': 'tm2_donor_acts', 'version': ACTS2_VERSION, 'repo': o['repo'], 'commit': inp['donor_commit'],
            'code': inp['code_sha256'], 'calib_sha': inp['calib_sha'], 'docs': int(o['docs']), 'seq': int(o['seq_len']),
            'wanted': {m: v for m, v in sorted(plan['wanted'].items())}, 'module_map': o['module_map'], 'env': env_info,
            'torch': torch.__version__, 'pool': 'per_document_mean (no padding; equal-length documents batched)'}


def _donor_acts(run, inp, plan, site, env_info):
    o = run.o
    identity = _acts_identity(run, inp, plan, env_info)
    fp = core.digest(identity)
    directory = run.root / 'donor_acts' / fp[:24]
    directory.mkdir(parents=True, exist_ok=True)
    core.atomic_json(directory / 'identity.json', identity)
    T = len(inp['texts'])
    size = int(o['chunk_docs'])
    chunks = [(a, min(a + size, T)) for a in range(0, T, size)]
    execs = sorted({e for pairs in plan['wanted'].values() for e, _ in pairs})

    def cdir(a, b):
        return directory / f'docs_{a:04d}-{b:04d}'

    def done(a, b):
        recd = tm._record(cdir(a, b) / 'complete.record.json', {**identity, 'docs_range': [a, b]})
        return recd is not None and all((cdir(a, b) / f'L{e:03d}.safetensors').is_file() and
                                        (cdir(a, b) / f'L{e:03d}.safetensors').stat().st_size == recd['sizes'][str(e)] for e in execs)
    todo = [(a, b) for a, b in chunks if not done(a, b)]
    if todo:
        ckpt = _checkpoint(run, inp)
        threads, speed = runtime.tune_cpu_threads()
        run.say(f'供体激活：{T} 篇分成 {len(chunks)} 段（每段 {size} 篇），还剩 {len(todo)} 段；{threads} 线程，float32 约 {speed:.0f} GFLOPS')
        for a, b in todo:
            run.say(f'供体激活第 {chunks.index((a, b)) + 1}/{len(chunks)} 段（第 {a}–{b - 1} 篇）')
            rep = _worker(run, site, {'mode': 'extract', 'checkpoint': str(ckpt), 'layers': int(o['expect']['layers']),
                                      'layers_modules': list(o['layers_modules']),
                                      'calib': inp['calib_path'], 'docs': int(o['docs']), 'range': [a, b], 'seq': int(o['seq_len']),
                                      'batch_docs': int(o['batch_docs']), 'wanted': plan['wanted'], 'order': plan['order'],
                                      'module_map': o['module_map'], 'schema': ACTS2_VERSION, 'identity': fp, 'threads': threads,
                                      'out': str(cdir(a, b))}, f'extract-{a:04d}')
            core.save_record(cdir(a, b) / 'complete.record.json', {**identity, 'docs_range': [a, b]},
                             {'sizes': rep['sizes'], 'tokens': rep['tokens'], 'batches': rep['batches'], 'hours': rep['hours']})
        if not o['keep_checkpoint']:
            shutil.rmtree(run.scratch / 'checkpoint', ignore_errors=True)
    else:
        run.say('供体激活已经算好，直接使用')
    files = {e: [cdir(a, b) / f'L{e:03d}.safetensors' for a, b in chunks] for e in execs}
    return files, {'chunks': len(chunks), 'documents': T, 'directory': str(directory)}


def _plan(inp, gate, o):
    execs = schedule(gate['structure']['order'])
    cands = donor_candidates(execs, gate['structure']['full_layers'], inp['modules'], int(o['mlp_stride']))
    phys = {e: p for e, p, _ in execs}
    wanted = {mod: [[e, phys[e]] for e in es] for mod, es in cands.items()}
    return {'order': gate['structure']['order'], 'execs': execs, 'phys': phys, 'cands': cands, 'wanted': wanted, 'n_exec': len(execs)}


# ---------------------------------------------------------------------------- verdict / pairs / fusion with executions
def verdict(run, inp, plan, ident, tgt, src, say=None):
    """qmerge_tm's go/no-go per module, the donor side being the module's executions (depth = position among all)."""
    o = run.o
    say = say or run.say
    keys = ('eps_delta', 'eps_floor', 'eps_cap', 'iters_inner', 'tol', 'z_go', 'sharp_margin', 'marginal',
            'up_excess_max', 'up_sharp_mult', 'noise_seed')
    identity = {'kind': 'tm2_verdict', 'version': OT2_VERSION, 'inputs': ident, 'options': {k: o[k] for k in keys}}
    path = Path(run.root) / 'verdict.record.json'
    done = tm._record(path, identity)
    if done is not None:
        return done
    n_t, n_s = inp['target_layers'], plan['n_exec']
    rows = {}
    rng = np.random.default_rng(int(o['noise_seed']))
    for mod in inp['modules']:
        TT = tm._target_modules_for(mod, inp['full'], inp['rec'])
        ST = plan['cands'][mod]
        probes = []
        for frac in (0.25, 0.5, 0.75):
            li = min(TT, key=lambda x: abs(x / max(n_t - 1, 1) - frac))
            mj = min(ST, key=lambda x: abs(x / max(n_s - 1, 1) - frac))
            probes.append((li, mj))
        stats = [tm.inner_ot(tgt.z(li, mod, 'out_mean'), src.z(mj, mod, 'out_mean'), o) for li, mj in probes]
        mean = {k: float(np.mean([r[k] for r in stats])) for k in ('gap', 'z', 'znull', 'eps', 'sharp', 'cl')}
        li_up = TT[len(TT) // 2]
        X = tgt.z(li_up, mod, 'out_mean')
        up = tm.inner_ot(X, X, o)
        excess = up['cl'] / max(float(o['eps_delta']) * up['gap'], 1e-9)
        sharp_ratio_up = up['sharp'] * up['m']
        up_ok = excess <= float(o['up_excess_max']) and sharp_ratio_up <= float(o['up_sharp_mult'])
        li, mj = probes[1]
        raw = src.raw(mj, mod, 'out_mean')
        R = rng.standard_normal(raw.shape).astype(np.float32) * np.float32(raw.std())
        lo = tm.inner_ot(tgt.z(li, mod, 'out_mean'), tm.zstd(R), o, eps=mean['eps'])
        z_ratio = mean['z'] / mean['znull'] if mean['znull'] > 0 else 0.0
        sharp_ratio = lo['sharp'] / mean['sharp'] if mean['sharp'] > 0 else 0.0
        if not up_ok:
            v = 'SKIP'
        elif z_ratio >= float(o['z_go']) and sharp_ratio >= float(o['sharp_margin']):
            v = 'GO'
        elif z_ratio >= float(o['marginal']) and sharp_ratio >= float(o['marginal']):
            v = 'MARGINAL'
        else:
            v = 'NO-GO'
        rows[mod] = {'probes': probes, **mean, 'z_ratio': z_ratio, 'noise_sharp': lo['sharp'], 'sharp_ratio': sharp_ratio,
                     'upper': {'layer': li_up, 'excess': excess, 'sharp_ratio': sharp_ratio_up, 'ok': up_ok}, 'verdict': v}
        say(f'{mod:16s} z 倍数 {z_ratio:.2f}  锐度倍数 {sharp_ratio:.2f}  上界 {"通过" if up_ok else "失败"}  → {v}')
        tgt.clear()
        src.clear()
    result = tm._clean({'modules': rows, 'go': [m for m, r in rows.items() if r['verdict'] in ('GO', 'MARGINAL')]})
    core.save_record(path, identity, result)
    return result


def pair_ot(run, inp, plan, ident, tgt, src, mod, say=None, store=True):
    """Linear-CKA prefilter, then inner OT on both sides of the kept (target layer, donor execution) pairs; resumable
    (state saved every few pairs, as for N)."""
    import pickle
    o = run.o
    say = say or run.say
    keys = ('eps_delta', 'eps_floor', 'eps_cap', 'iters_inner', 'tol', 'pair_keep', 'pair_depth_prior', 'chunk_rows')
    fp = core.digest({'kind': 'tm2_pairs', 'version': OT2_VERSION, 'inputs': ident, 'module': mod, 'cands': plan['cands'][mod],
                      'options': {k: o[k] for k in keys}})
    path = Path(run.root) / 'pairs' / f'{mod.replace(".", "-")}_{fp[:16]}.pkl'
    state = None
    if store and path.is_file():
        try:
            with path.open('rb') as f:
                state = pickle.load(f)
            state = state if state.get('identity') == fp else None
        except (OSError, pickle.UnpicklingError, EOFError, AttributeError):
            state = None
    TT = tm._target_modules_for(mod, inp['full'], inp['rec'])
    ST = plan['cands'][mod]
    if state is None:
        scores = np.zeros((len(TT), len(ST)))
        for side in ('in_mean', 'out_mean'):
            gs = [tm.gram(src.z(mj, mod, side)) for mj in ST]
            for x, li in enumerate(TT):
                g = tm.gram(tgt.z(li, mod, side))
                for y in range(len(ST)):
                    scores[x, y] += 0.5 * float((g * gs[y]).sum(dtype=np.float64))
            tgt.clear()
            src.clear()
        kept = tm.keep_pairs(scores, TT, ST, int(o['pair_keep']), bool(o['pair_depth_prior']), inp['target_layers'], plan['n_exec'])
        state = {'identity': fp, 'cka': scores.tolist(), 'kept': {int(k): v for k, v in kept.items()}, 'done': {}, 'complete': False}

    def save():
        if store:
            path.parent.mkdir(parents=True, exist_ok=True)
            tmp = path.with_name(path.name + '.tmp')
            with tmp.open('wb') as f:
                pickle.dump(state, f, protocol=4)
            tmp.replace(path)
    if state['complete']:
        return state
    todo = [(li, mj) for li in TT for mj in state['kept'][li] if (li, mj) not in state['done']]
    say(f'{mod}：{sum(len(state["kept"][li]) for li in TT)} 个层对（{len(TT)}×{len(ST)} 里由 CKA 留下），还要算 {len(todo)} 个')
    started = time.perf_counter()
    for count, (li, mj) in enumerate(todo, 1):
        got = {}
        for side, sfx in (('in', 'in_mean'), ('out', 'out_mean')):
            r = tm.inner_ot(tgt.z(li, mod, sfx), src.z(mj, mod, sfx), o)
            got[side] = {k: r[k] for k in ('u', 'v', 'eps', 'gap', 'cl', 'sharp', 'n', 'm')}
        state['done'][(li, mj)] = got
        if count % 4 == 0 or count == len(todo):
            save()
            say(f'  {mod} {len(state["done"])} 个层对已算，估计还需 {(time.perf_counter() - started) / count * (len(todo) - count) / 60:.0f} 分钟')
    state['complete'] = True
    save()
    tgt.clear()
    src.clear()
    return state


def fuse(run, inp, plan, verdict_, pair_states, tgt, src, weight_of, target_of, say=None):
    """N2 rows for every target matrix of the GO modules. weight_of(mod, physical layer) gives the donor matrix (one per
    physical layer, shared by both executions); target_of(name) gives B's matrix."""
    o = run.o
    say = say or run.say
    out, diag, plans, cache = {}, {}, {}, {}
    for mod in verdict_['go']:
        st = pair_states[mod]
        TT = tm._target_modules_for(mod, inp['full'], inp['rec'])
        used = sorted({mj for li in TT for mj in st['kept'][li]})
        cin = {p: r['in']['cl'] for p, r in st['done'].items()}
        cout = {p: r['out']['cl'] for p, r in st['done'].items()}
        Pe, info = tm.effective_plan(cin, cout, TT, used, o)
        dmr = tm.depth_match_ratio(Pe, TT, used, inp['target_layers'], plan['n_exec'])
        plans[mod] = {'TT': TT, 'ST': used, 'phys': [plan['phys'][e] for e in used], 'P': Pe.tolist(), 'dmr': dmr, **info}
        for x, li in enumerate(TT):
            name = inp['targets'][(mod, li)]
            row = Pe[x]
            conc = float(row.max() / max(row.mean(), 1e-30))
            weights = sorted(((float(row[used.index(mj)]), mj) for mj in st['kept'][li]), reverse=True)
            pick = weights[:int(o['top_j'])] if conc >= float(o['conc_min']) else weights
            total = sum(w for w, _ in pick)
            pick = [(w / max(total, 1e-30), mj) for w, mj in pick]
            pick = [(w, mj) for w, mj in pick if w >= 1e-4]
            WA = target_of(name)
            score = tgt.raw(li, mod, 'out_abs').mean(0)
            k = min(int(o['topk']), WA.shape[0])
            idx = np.sort(np.argsort(-score, kind='stable')[:k])
            acc = np.zeros((k, WA.shape[1]), dtype=np.float32)
            for w, mj in pick:
                phys = plan['phys'][mj]
                if (mod, phys) not in cache:
                    while len(cache) >= 4:
                        cache.pop(next(iter(cache)))
                    cache[(mod, phys)] = weight_of(mod, phys)
                WB = cache[(mod, phys)]
                r = st['done'][(li, mj)]
                po = (tgt.z(li, mod, 'out_mean'), src.z(mj, mod, 'out_mean'), r['out']['u'], r['out']['v'], r['out']['eps'])
                pi = (tgt.z(li, mod, 'in_mean'), src.z(mj, mod, 'in_mean'), r['in']['u'], r['in']['v'], r['in']['eps'])
                if WB.shape != (po[1].shape[1], pi[1].shape[1]) or WA.shape != (po[0].shape[1], pi[0].shape[1]):
                    raise ValueError(f'{name}：权重形状与激活通道数对不上（B {WA.shape}，供体 {WB.shape}）')
                acc += np.float32(w) * tm.transport_rows(idx, WB, po, pi, WA.shape[1], int(o['chunk_rows']))
            rows_new, d = tm.inject(acc, WA[idx], o)
            delta = (np.float32(o['alpha']) * (rows_new - WA[idx])).astype(np.float32)
            out[name + '|rows'] = idx.astype(np.int64)
            out[name + '|delta'] = delta
            diag[name] = {**d, 'module': mod, 'layer': li, 'donor_executions': [[w, mj, plan['phys'][mj]] for w, mj in pick],
                          'conc': conc, 'rows': int(k), 'dW': float(np.linalg.norm(delta) / max(float(np.linalg.norm(WA)), 1e-30))}
        say(f'{mod}：{len(TT)} 张矩阵已算好；层级计划锐度 {info["sharp"]:.3f}，DMR {dmr:.3f}')
        tgt.clear()
        src.clear()
    return out, diag, plans


# ---------------------------------------------------------------------------- entry point
def ensure(context):
    """Build (or reuse) N2. None when off; else a dict with status 'go' (directory, fingerprint), 'no-go' or 'failed'."""
    o = options(context['settings'])
    if not o['enabled']:
        return None
    try:
        return _ensure(context, o)
    except Exception as error:
        if o['on_error'] == 'skip' and not o['require_all_donors']:
            print(f'[N2] 跨架构增量 N₂ 这次没能算出（{type(error).__name__}: {str(error)[:300]}）；按 on_error="skip"，本次融合不含 N₂。', flush=True)
            return {'status': 'failed', 'error': f'{type(error).__name__}: {str(error)[:300]}', 'repo': o['repo']}
        raise


def _result_options(o):
    return {**{k: o[k] for k in DEFAULTS if k not in _RUN_ONLY}, **{k: o[k] for k in OT_KEYS if k != 'act_cache_gib'},
            'require_all_donors': bool(o.get('require_all_donors', True))}


def open_delta(result):
    return tm.open_delta(result)


def _ensure(context, o):
    from safetensors.numpy import save_file
    run = _Run(context, o)
    final_path = run.root / 'tm_result.record.json'
    inp = _inputs(run)
    ident = _identity(run, inp)
    final_identity = {'kind': 'tm2_result', 'inputs': ident, 'options': _result_options(o)}
    done = tm._record(final_path, final_identity)
    if done is not None and (done['status'] != 'go' or (Path(done['directory']) / 'tm_delta.safetensors').is_file()):
        run.say(f'跨架构增量 N₂（{o["repo"]}）沿用已有结果：{done["status"]}')
        return done
    run.say(f'跨架构增量 N₂：{o["repo"]} @ {inp["donor_commit"][:10]}；远程代码 {inp["code_sha256"][:12]}…（已审读）；'
            f'target 激活 {o["act_repo"]} @ {inp["act_sha"][:10]}（与 N 相同）；模块 {inp["modules"]}；{inp["k_note"]}；'
            + ('tokenizer 与 B 相同：校准文本切出的 token 与 B 一致' if inp['same_tokenizer'] else
               'tokenizer 与 B 不同：校准文本按供体自己的 tokenizer 切分（按篇取均值，统计不受影响）' if inp['same_tokenizer'] is False else
               'tokenizer 无法比较'))
    site, env_info = _isolated_env(run) if o['isolated_env'] else (None, {'transformers': 'kernel'})
    gate = _probe(run, inp, site, env_info)
    plan = _plan(inp, gate, o)
    files, acts = _donor_acts(run, inp, plan, site, env_info)
    tgt, tdir = tm._target_store(tm._Run(context, {**run.n1.o, 'modules': inp['modules']}), inp)
    src = tm.ActStore(files, o['docs'], o['act_cache_gib'] / 2)
    ident = {**ident, 'acts': acts['directory']}
    v = verdict(run, inp, plan, ident, tgt, src)
    forced = False
    if not v['go'] and o['require_all_donors']:
        modules = [m for m, r in v['modules'].items() if r['verdict'] != 'SKIP'] or list(inp['modules'])
        v = {**v, 'go': modules}
        forced = True
        run.say(f'没有模块达到融合阈值；按 require_all_donors 仍用 {modules} 构造 N₂（结果里记为 forced）')
    if not v['go']:
        result = {'status': 'no-go', 'repo': o['repo'], 'commit': inp['donor_commit'], 'verdict': v['modules'],
                  'reason': '没有模块达到融合阈值（z 倍数与同 eps 噪声锐度倍数）'}
        core.save_record(final_path, final_identity, tm._clean(result))
        return tm._clean(result)
    pairs = {mod: pair_ot(run, inp, plan, ident, tgt, src, mod) for mod in v['go']}
    prefix = inp['donor_prefix']

    def weight_of(mod, phys):
        return run.eng.tensor(o['repo'], f'{prefix}.{phys}.{o["module_map"].get(mod, mod)}.weight').float().numpy()

    def target_of(name):
        return run.ctx['B'].tensor(name).float().numpy()
    out, diag, plans = fuse(run, inp, plan, v, pairs, tgt, src, weight_of, target_of)
    fp = core.digest({**final_identity, 'go': v['go']})
    directory = run.root / 'delta' / fp[:24]
    directory.mkdir(parents=True, exist_ok=True)
    tmp = directory / 'tm_delta.safetensors.tmp'
    save_file(out, str(tmp), metadata={'format': 'np', 'fingerprint': fp})
    tmp.replace(directory / 'tm_delta.safetensors')
    core.atomic_json(directory / 'tm_report.json', tm._clean({'verdict': v, 'tensors': diag, 'plans': plans,
                                                             'structure': gate['structure'], 'k_proj': inp['k_note']}))
    dws = [d['dW'] for d in diag.values()]
    injs = [d['inj'] for d in diag.values() if d['inj'] is not None]
    summary = {'sha256': core.sha_file(directory / 'tm_delta.safetensors'), 'tensors': len(diag),
               'mean_dW': float(np.mean(dws)) if dws else 0.0, 'mean_inject_ratio': float(np.mean(injs)) if injs else None,
               'raw_fallback_tensors': sum(1 for d in diag.values() if d['mode'] == 'raw'),
               'norm': float(math.sqrt(sum(float(np.square(val).sum(dtype=np.float64)) for key, val in out.items() if key.endswith('|delta'))))}
    result = {'status': 'go', 'repo': o['repo'], 'commit': inp['donor_commit'], 'fingerprint': fp, 'directory': str(directory),
              'modules': v['go'], 'verdict': v['modules'], 'summary': summary, 'gate': {'hits': gate['hits']},
              'act_repo': {'repo': o['act_repo'], 'commit': inp['act_sha'], 'target_key': inp['target_key']},
              'alpha': o['alpha'], 'topk': o['topk'], 'forced': forced, 'k_proj': inp['k_note'],
              'same_tokenizer': inp['same_tokenizer'], 'code_sha256': inp['code_sha256'], 'executions': plan['n_exec']}
    core.save_record(final_path, final_identity, tm._clean(result))
    run.say(f'N₂ 已算好：{summary["tensors"]} 张矩阵，平均 |ΔW|/|W| = {summary["mean_dW"]:.5f}')
    if not o['keep_activations']:
        shutil.rmtree(Path(acts['directory']), ignore_errors=True)
    return tm._clean(result)
'''
load_embedded_module('qmerge_tm2', MODULE_SOURCE)
del MODULE_SOURCE


In [ ]:
# 内嵌源代码；独立运行无需额外文件。
MODULE_SOURCE = r'''
"""Final donor + fixed auxiliary sources + a searched delta -> the subject S: streaming BF16 candidates, search, component selection, upload.

F = S + lambda_g*(D - kappa*(S-B)) + mu*C + nu*N, where D = (M-B) + sum_k c_k*T_k is computed once with fixed
coefficients and cached (float16), and C (one searched delta) is cached on its own, so each search candidate reads only S, B, D and C.
N is a row-sparse Transport-and-Merge delta from a donor of another architecture (qmerge_tm), built once on the CPU;
its coefficient nu is scored after the global search, on the winner (a line search over nu_grid). A term T_k is one
source's delta against its base (A_k-R_k), or the DELLA fusion of several sources' deltas against B (or a member's own base).
The sources of D are read in parallel, one worker per repository. With the layer-by-layer scoring engine
(checked against the batched forward on the real S first) a search candidate is computed in RAM one decoder
layer at a time and scored without being written. The coefficient search picks F; the component stage then
cuts F - S into orthogonal components, estimates each one's effect on every proxy question's loss from one
forward and backward pass, and builds and scores variants without the components that do not help
(component_stage, qmerge_components). Only the chosen candidate is built as files, its bytes are checked
against the digests of the weights that were scored, and it is uploaded to the output repository.
Everything runs on a CPU runtime. The only measurement is the proxy (response NLL); the model is not
evaluated by generation here. The component stage borrows parts of existing methods (see qmerge_components);
none of them is reproduced. No teacher-correction vector, IRT asset, or optimal coefficient is fabricated.
"""
import copy
import gc
import hashlib
import json
import math
import os
from pathlib import Path
import re
import shutil
import itertools
import threading
import time
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
from safetensors import safe_open
from safetensors.torch import save_file
import qmerge_core as core
import qmerge_runtime as runtime
import qmerge_components as components
import qmerge_gguf as gguf
import qmerge_della as della
import qmerge_tm as tm_delta
import qmerge_tm2 as tm_delta2


EXTRA_TERMS=('nu','nu2')


def validate_recipe(recipe):
    """qmerge_core.validate_recipe, plus nu and nu2: the coefficients of the cross-architecture deltas N (Nanbeige) and
    N2 (SenseNova), outside the lambda bracket (absent means 0). They live here and not in qmerge_core because that
    module's file hash is part of the identity of every cached forward score (S's baseline, the backward check): adding
    a field there would throw them away."""
    extra=[k for k in EXTRA_TERMS if k in recipe]
    for k in extra:
        v=recipe[k]
        if isinstance(v,bool) or not isinstance(v,(int,float)) or not math.isfinite(float(v)) or not 0<=float(v)<=2:
            raise ValueError(f'{k} 必须在 [0,2]')
    if extra:
        core.validate_recipe({k:v for k,v in recipe.items() if k not in EXTRA_TERMS})
        return recipe
    return core.validate_recipe(recipe)


def canonical_recipe(recipe):
    """qmerge_core.canonical_recipe; a nonzero nu / nu2 is kept (a zero one is the recipe without the field)."""
    validate_recipe(recipe)
    r=core.canonical_recipe({k:v for k,v in recipe.items() if k not in EXTRA_TERMS})
    for k in EXTRA_TERMS:
        v=float(recipe.get(k,0) or 0)
        if v:r[k]=v
    return r


def make_scoped_engine_class(legacy):
    class ScopedEngine(legacy.LockedEngine):
        """Same commit may contain BF16 main, FP8 model and draft; scope is identity."""
        def __init__(self,*args,subfolders=None,**kwargs):
            self.subfolders={r:runtime.safe_relative(p.strip('/')) if p.strip('/') else '' for r,p in (subfolders or {}).items()}
            self.local_catalogues={}
            # Headers are cached per set of subfolders; the commit lock is one file for every set, so that adding a
            # source (and with it a subfolder) never re-pins the repositories that were pinned already.
            base_work=Path(kwargs['work']);supplied=dict(kwargs.get('revisions') or {})
            kwargs['work']=base_work/core.digest(self.subfolders)[:20]
            super().__init__(*args,**kwargs)
            stable=base_work/'input_commits.json'
            pins=json.loads(stable.read_text()).get('pins',{}) if stable.exists() else {}
            self.saved_pins={**self.saved_pins,**pins};self.lock_path=stable
            self.revisions={**self.saved_pins,**supplied}
            legacy.atomic_json(stable,{'pins':self.saved_pins})
        def _remember_info(self,repo,info):
            self._infos[repo]=info;prefix=self.subfolders.get(repo,'');prefix=prefix+'/' if prefix else ''
            siblings=[s for s in info.siblings if s.rfilename.startswith(prefix)]
            self._names[repo]=sorted(s.rfilename[len(prefix):] for s in siblings)
            self._sizes[repo]={s.rfilename[len(prefix):]:s.size for s in siblings}
        def _url(self,repo,name):
            prefix=self.subfolders.get(repo,'');filename=(prefix+'/' if prefix else '')+runtime.safe_relative(name)
            return legacy.hf_hub_url(repo_id=repo,filename=filename,revision=self.pin(repo))
        def tensor(self,repo,name):
            if repo not in self.local_catalogues:return super().tensor(repo,name)
            info=self.local_catalogues[repo][name]
            with safe_open(info['file'],framework='pt',device='cpu') as f:return f.get_tensor(name)
        def rows(self,repo,name,i,j):
            if repo not in self.local_catalogues:return super().rows(repo,name,i,j)
            info=self.local_catalogues[repo][name]
            with safe_open(info['file'],framework='pt',device='cpu') as f:
                if not info['shape']:return f.get_tensor(name).float()
                return f.get_slice(name)[i:j].float()
    return ScopedEngine


AUX_DELTA_VERSION='aux-delta-2026-10-18.3'
_SHA=r'[0-9a-fA-F]{40,64}'


def _revision(raw,key,label):
    value=str(raw.get(key,'')).strip()
    if value and not re.fullmatch(_SHA,value):raise ValueError(label+f'：{key} 必须是完整 commit SHA')
    return value


def _subfolder(raw,key,label):
    """'auto' (decided from the repository's file list), '' (the repository root) or a relative folder name."""
    value=str(raw.get(key,'auto')).strip()
    if value!='auto':
        value=value.strip('/')
        if value:
            try:runtime.safe_relative(value)
            except ValueError as error:raise ValueError(label+f'：{key} 不是安全的相对目录名') from error
    return value


def aux_plan(settings):
    """Fixed auxiliary mix, plus the separately searched delta.

    M has weight donor_weight; each extra source k adds (w_k/donor_weight)*T_k to D. T_k is A_k-R_k for a single
    source (R_k = B unless reference_repo is set), or the DELLA fusion of the members' deltas, each against B or
    against that member's own reference_repo. A source with weight 0 is dropped. A required source whose repo is
    still empty is pending: the merge waits for it. A source with defer=True is decided and added to D only once
    nothing is pending (the full run), so that a choice made late (a GGUF replaced by a BF16 release) never
    invalidates a D built while waiting.

    searched_extras: at most one delta C = A - R, stored on its own and added outside the lambda bracket with a
    searched coefficient mu (mu_grid). Returns (donor_weight, ready sources, pending required sources, searched).
    """
    dw=float(settings.get('donor_weight',1.))
    if not math.isfinite(dw) or dw<=0:raise ValueError('donor_weight 必须为正数')
    ready,pending,tags=[],[],set()
    def claim(tag):
        if not re.fullmatch(r'[A-Za-z0-9_]+',tag) or tag in tags or tag in {'S','M','B','base'}:
            raise ValueError('辅料 tag 必须唯一、只含字母数字下划线，且不能是 S/M/B：'+tag)
        tags.add(tag)
    for raw in settings.get('aux_extras',[]):
        tag=str(raw.get('tag',''));claim(tag)
        item={'tag':tag,'repo':str(raw.get('repo','')).strip(),'revision':_revision(raw,'revision',tag),
              'format':str(raw.get('format','safetensors')),'file':str(raw.get('file','')).strip(),
              'subfolder':_subfolder(raw,'subfolder',tag),
              'reference_repo':str(raw.get('reference_repo','')).strip(),'reference_revision':_revision(raw,'reference_revision',tag),
              'reference_subfolder':_subfolder(raw,'reference_subfolder',tag),
              'weight':float(raw.get('weight',0)),'required':bool(raw.get('required',True)),'late':bool(raw.get('late',False)),
              'defer':bool(raw.get('defer',False)),'bf16_repos':[str(x).strip() for x in raw.get('bf16_repos',[]) if str(x).strip()]}
        if item['format'] not in {'safetensors','gguf','della'}:raise ValueError(tag+'：format 只能是 safetensors、gguf 或 della')
        if item['format']=='gguf':
            if (item['file'] and not item['file'].endswith('.gguf')) or item['reference_repo']:
                raise ValueError(tag+'：GGUF 来源的 file 留空（自动选）或填 .gguf 文件名；参考底座只能是 B')
            item['subfolder']=''
        elif item['bf16_repos']:raise ValueError(tag+'：bf16_repos 只用于 GGUF 来源（有 BF16 发布时改读它）')
        if not math.isfinite(item['weight']) or item['weight']<0:raise ValueError(tag+'：weight 必须是非负数')
        if item['format']=='della':
            if item['repo'] or item['file'] or item['reference_repo']:
                raise ValueError(tag+'：DELLA 组不填 repo / file / reference_repo；成员写在 members 里（成员可以各自填 reference_repo）')
            members=[]
            for m in raw.get('members',[]):
                mtag=str(m.get('tag',''));claim(mtag);label=tag+'/'+mtag
                member={'tag':mtag,'repo':str(m.get('repo','')).strip(),'revision':_revision(m,'revision',label),
                        'subfolder':_subfolder(m,'subfolder',label),'weight':float(m.get('weight',1)),
                        'reference_repo':str(m.get('reference_repo','')).strip(),'reference_revision':_revision(m,'reference_revision',label),
                        'reference_subfolder':_subfolder(m,'reference_subfolder',label)}
                if not member['repo']:raise ValueError(label+'：DELLA 成员必须填 repo')
                if not math.isfinite(member['weight']) or member['weight']<=0:raise ValueError(label+'：成员 weight 必须为正数')
                members.append(member)
            if len(members)<2:raise ValueError(tag+'：DELLA 至少需要两个成员')
            item['members']=members;item['della']=della.validate(raw.get('della'))
        if item['weight']==0:continue
        item['coefficient']=item['weight']/dw
        if item['format']!='della' and not item['repo']:
            if item['required']:pending.append(item)
            continue
        ready.append(item)
    searched=[]
    for raw in settings.get('searched_extras',[]):
        tag=str(raw.get('tag',''));claim(tag)
        item={'tag':tag,'repo':str(raw.get('repo','')).strip(),'revision':_revision(raw,'revision',tag),
              'subfolder':_subfolder(raw,'subfolder',tag),'format':'safetensors','file':'',
              'reference_repo':str(raw.get('reference_repo','')).strip(),'reference_revision':_revision(raw,'reference_revision',tag),
              'reference_subfolder':_subfolder(raw,'reference_subfolder',tag),'roles':sorted(set(raw.get('roles',['backbone']))),
              'weight':1.,'coefficient':1.,'required':True,'late':False,'defer':False,'bf16_repos':[]}
        if not item['repo'] or not item['reference_repo']:raise ValueError(tag+'：单独搜索的增量 C = A - R 要填 repo 和 reference_repo')
        if not item['roles'] or set(item['roles'])-{'backbone','head'}:raise ValueError(tag+'：roles 只能含 "backbone"、"head"')
        searched.append(item)
    if len(searched)>1:raise ValueError('searched_extras 目前只支持一项（一个 μ）')
    grid=[float(x) for x in settings.get('mu_grid',[])]
    if searched and (not grid or any(not math.isfinite(x) or not 0<x<=2 for x in grid)):raise ValueError('mu_grid 要给出 (0, 2] 内的系数')
    if require_all_donors(settings):
        lambdas=[float(x) for x in settings.get('lambda_grid',[])]
        if not lambdas or any(not x>0 for x in lambdas):
            raise ValueError('require_all_donors=True：lambda_grid 里不能有 0（λ=0 等于 D 里的供体全部没有贡献）；可以用很小的值，例如 0.025')
    return dw,ready,pending,searched


def require_all_donors(settings):
    """Every donor must contribute to the chosen model (its coefficient may be small, never 0): lambda (D: Final and the
    fixed auxiliary sources), mu (C) and nu (N) are all > 0 in every candidate that can be chosen; S is scored only as
    the reference; the mu=0 ablation is off; the component stage may not drop every decoder component."""
    return bool(settings.get('require_all_donors',True))


def detect_subfolder(names,want='auto'):
    """The folder of a repository that holds its full safetensors model: the root when it has config.json and a
    safetensors index (or model.safetensors), else the one such top-level folder (one named like BF16 first)."""
    if want!='auto':return want
    names=set(names)
    def full(prefix):
        p=prefix+'/' if prefix else ''
        return p+'config.json' in names and (p+'model.safetensors.index.json' in names or p+'model.safetensors' in names)
    if full(''):return ''
    hits=[d for d in sorted({n.split('/',1)[0] for n in names if '/' in n}) if full(d)]
    bf16=[d for d in hits if re.search(r'(?i)bf16',d)]
    if len(bf16)==1:return bf16[0]
    if len(hits)==1:return hits[0]
    raise ValueError('找不到唯一的完整 safetensors 目录（根目录没有 config.json + 索引；候选目录：'+(', '.join(hits) or '无')+'）。请在 SETTINGS 里写明 subfolder')


def _file_list(api,token,repo,revision):
    try:return api.model_info(repo,revision=revision or 'main',token=token or None)
    except Exception as error:
        raise RuntimeError(f'{repo}：读不到文件列表（{type(error).__name__}）。仓库不存在、未发布，或需要先在模型页同意条款（gated）。') from error


def pin_subject(settings,api,token,root):
    """Fix S's commit and subfolder once per record directory (subject_pin.json) and write them into settings, so that
    the data cell, the baseline and the merge all read the same S."""
    path=Path(root)/'subject_pin.json';repo=settings['S_repo']
    want_rev=str(settings.get('S_revision','')).strip();want_sub=_subfolder(settings,'S_subfolder','S')
    saved=json.loads(path.read_text(encoding='utf-8')) if path.exists() else None
    if saved and saved.get('repo')==repo and (not want_rev or saved['revision']==want_rev) and want_sub in ('auto',saved['subfolder']):
        settings.update(S_revision=saved['revision'],S_subfolder=saved['subfolder']);return saved
    if saved and saved.get('repo')!=repo:print(f'主模型换成了 {repo}（之前是 {saved.get("repo")}），重新锁定 commit 和子目录。',flush=True)
    info=_file_list(api,token,repo,want_rev)
    if not re.fullmatch(_SHA,str(info.sha)):raise ValueError(f'{repo}：Hub 没有返回 commit SHA')
    sub=detect_subfolder([s.rfilename for s in info.siblings],want_sub)
    saved={'repo':repo,'revision':str(info.sha),'subfolder':sub}
    core.atomic_json(path,saved);settings.update(S_revision=saved['revision'],S_subfolder=sub)
    print(f'主模型 S：{repo} @ {saved["revision"][:12]}，'+(f'读取 {sub}/ 子目录' if sub else '读取仓库根目录'),flush=True)
    return saved


def resolve_subfolders(root,api,token,wanted):
    """{repo: (subfolder or 'auto', revision)} -> {repo: subfolder}. 'auto' is decided once from the file list and kept in
    source_subfolders.json, so that every session reads the same layout."""
    path=Path(root)/'source_subfolders.json'
    saved=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}
    out={};changed=False
    for repo,(want,revision) in sorted(wanted.items()):
        if want!='auto':out[repo]=want;continue
        if repo in saved:out[repo]=saved[repo]['subfolder'];continue
        info=_file_list(api,token,repo,revision)
        try:sub=detect_subfolder([s.rfilename for s in info.siblings])
        except ValueError as error:raise ValueError(f'{repo}：{error}') from error
        out[repo]=sub;saved[repo]={'subfolder':sub,'seen_at':str(info.sha)};changed=True
        print(f'{repo}：读取'+(f' {sub}/ 子目录' if sub else '仓库根目录'),flush=True)
    if changed:core.atomic_json(path,saved)
    return out


def resolve_choice(root,api,token,item):
    """A GGUF source with bf16_repos: the first of those repositories that holds a full safetensors model is read instead
    (its delta against B, without quantization noise); otherwise the GGUF. Decided the first time it is needed and kept
    in source_choices.json, so that a later release never switches the source under a D that is already built."""
    if item['format']!='gguf' or not item['bf16_repos']:return item
    path=Path(root)/'source_choices.json'
    saved=json.loads(path.read_text(encoding='utf-8')) if path.exists() else {}
    options=item['bf16_repos']+[item['repo']];entry=saved.get(item['tag'])
    if entry is None or entry.get('options')!=options:
        entry=None
        for repo in item['bf16_repos']:
            try:info=api.model_info(repo,token=token or None)
            except Exception:continue
            try:sub=detect_subfolder([s.rfilename for s in info.siblings])
            except ValueError:continue
            entry={'format':'safetensors','repo':repo,'revision':str(info.sha),'subfolder':sub};break
        entry=entry or {'format':'gguf','repo':item['repo']};entry['options']=options
        saved[item['tag']]=entry;core.atomic_json(path,saved)
        print(f'{item["tag"]}：'+(f'已有 BF16 发布 {entry["repo"]}，改读它（不再用 GGUF）' if entry['format']=='safetensors'
                                 else f'还没有 BF16 发布（查过 {", ".join(item["bf16_repos"])}），用 GGUF {item["repo"]}')
              +'。这个选择记在 source_choices.json，之后的会话沿用；要重新选就删掉其中这一项（D 会重算）。',flush=True)
    if entry['format']=='gguf':return item
    return {**item,'format':'safetensors','repo':entry['repo'],'revision':entry['revision'],'subfolder':entry['subfolder'],
            'file':'','reference_repo':'','reference_revision':'','reference_subfolder':'auto','instead_of_gguf':item['repo']}


def setup_inputs(settings,legacy,api,token,extras,searched=()):
    """Pin and align S, M, B, the ready extra sources and the searched delta. Reads headers and configs only."""
    root=Path(settings['work']).expanduser().resolve();root.mkdir(parents=True,exist_ok=True)
    scratch=Path(settings.get('scratch') or root).expanduser().resolve();scratch.mkdir(parents=True,exist_ok=True)
    srepo=settings['S_repo'];mrepo=settings['M_repo'];brepo=settings['B_repo']
    mrev=settings.get('M_revision','');brev=settings.get('B_revision','')
    if not re.fullmatch(_SHA,mrev):raise ValueError('M_revision 必须是已发布 Final 的完整 commit SHA；不猜测同名旧产物')
    if len({srepo,mrepo,brepo})!=3:raise ValueError('S、M、B 必须是三个明确不同的仓库')
    extras=[resolve_choice(root,api,token,copy.deepcopy(item)) for item in extras];searched=[copy.deepcopy(item) for item in searched]
    # A source left without a revision follows "main" once: the engine records the commit it saw
    # (input_engine/input_commits.json) and every later session, on any runtime, reads that commit.
    revisions={mrepo:mrev}
    if settings.get('S_revision'):revisions[srepo]=settings['S_revision']
    if brev:revisions[brepo]=brev
    used={srepo,mrepo,brepo};wanted={}
    def note(label,repo,revision,subfolder):
        if revision:
            if revisions.get(repo,revision)!=revision:raise ValueError(f'{label}：{repo} 在两处填了不同的 revision')
            revisions[repo]=revision
        old=wanted.get(repo)
        if old is None or old[0]=='auto':wanted[repo]=(subfolder,revision or (old[1] if old else ''))
        elif subfolder not in ('auto',old[0]):raise ValueError(f'{label}：{repo} 在两处填了不同的 subfolder')
    def claim(label,repo,revision,subfolder):
        if repo in used:raise ValueError(label+'：仓库与 S/M/B 或其他来源重复（同一个模型只能进一个节点）')
        used.add(repo);note(label,repo,revision,subfolder)
    def reference(label,item,source):
        ref=item.get('reference_repo','')
        if not ref or ref==brepo:item['reference_repo']='';return ''
        if ref in {srepo,mrepo,source}:raise ValueError(label+'：reference_repo 不能是 S、M 或来源本身（留空表示 B）')
        note(label,ref,item.get('reference_revision',''),item.get('reference_subfolder','auto'))
        return ref
    for item in extras:
        if item['format']=='della':
            for m in item['members']:
                label=item['tag']+'/'+m['tag'];claim(label,m['repo'],m['revision'],m['subfolder']);reference(label,m,m['repo'])
            continue
        claim(item['tag'],item['repo'],item['revision'],'' if item['format']=='gguf' else item['subfolder'])
        reference(item['tag'],item,item['repo'])
    for item in searched:
        claim(item['tag'],item['repo'],item['revision'],item['subfolder']);reference(item['tag'],item,item['repo'])
    subfolders=resolve_subfolders(root,api,token,{srepo:(settings.get('S_subfolder','auto'),settings.get('S_revision','')),**wanted})
    Engine=make_scoped_engine_class(legacy)
    # Read-ahead windows only decide how many HTTP requests carry the bytes. With several repositories read at
    # once (parallel_reads) larger windows mean fewer requests against the Hub's per-account request quota.
    eng=Engine(api=api,token=token,work=root/'input_engine',shard_gb=settings['shard_gib'],block_mb=settings['block_mib'],
               revisions=revisions,subfolders={r:s for r,s in subfolders.items() if s},
               readahead_mb=settings.get('readahead_mib',64),readahead_cache_mb=settings.get('readahead_cache_mib',2048))
    donor_manifest=eng.manifest(mrepo);recipe=donor_manifest.get('identity',{}).get('recipe',{})
    if not donor_manifest.get('complete') or recipe.get('stage')!='final-aux-priority-cts' or not recipe.get('prepublication_checks_passed'):
        raise ValueError('M 不是通过原 Final 数值审计的完整供体')
    base_commit=donor_manifest.get('identity',{}).get('sources',{}).get(brepo)
    if not base_commit or (brev and brev!=base_commit):raise ValueError('B 必须是制作此 M 时的准确参考底座及 commit')
    audit_bytes=eng._small_bytes(mrepo,'contribution_audit.json')
    if hashlib.sha256(audit_bytes).hexdigest()!=recipe.get('audit_sha256') or not json.loads(audit_bytes).get('verified'):
        raise ValueError('供体贡献审计缺失、改变或未通过')
    eng.revisions[brepo]=base_commit
    legacy.ROWS_MB=settings.get('rows_mib',64);legacy.TENSOR_KEY_OVERRIDES=settings.get('tensor_key_overrides',{})
    # B and M are held to S's tokenizer pipeline unless the user allows a difference: a tokenizer.json saved
    # again by another library version can spell the same pipeline differently.
    legacy.ALLOW_TOKENIZER_PIPELINE_DIFFERENCES=bool(settings.get('allow_tokenizer_pipeline_differences',False))
    try:
        subject=legacy.RepoNode(eng,srepo)
        base=legacy.RepoNode(eng,brepo,subject,strict=True)
        donor=legacy.RepoNode(eng,mrepo,subject,strict=True)
    except legacy.InputNotReady as error:
        if 'tokenizer pipeline 不同' not in str(error):raise
        raise legacy.InputNotReady(str(error)+'。词表、merges 和每个 token 的 ID 已核对一致，差别只在 normalizer / pre_tokenizer / post_processor / decoder 的写法。'
                                   '成品沿用 S 的 tokenizer，进入融合的只有权重差值；确认无妨后，在 SETTINGS 里设 "allow_tokenizer_pipeline_differences": True 再运行。') from error
    finally:legacy.ALLOW_TOKENIZER_PIPELINE_DIFFERENCES=False
    for node in (subject,base,donor):node.identity['subfolder']=eng.subfolders.get(node.repo,'')
    sc=subject.config.get('text_config',subject.config)
    def same_config(node):
        nc=node.config.get('text_config',node.config)
        for key in ('rope_parameters','rope_scaling','rope_theta','rms_norm_eps','hidden_act'):
            if key in sc and key in nc and sc[key]!=nc[key]:raise ValueError(f'{node.repo}: {key} 不一致，不能仅凭权重形状直接融合')
    for node in (base,donor):same_config(node)
    nodes={}
    def aligned(repo):
        # Only differences enter the merge and the output keeps S's tokenizer, so a tokenizer
        # pipeline difference against S is tolerated here. Core vocabulary, merges and token IDs
        # are still checked against S.
        if repo in nodes:return nodes[repo]
        legacy.ALLOW_TOKENIZER_PIPELINE_DIFFERENCES=True
        try:node=legacy.RepoNode(eng,repo,subject,strict=True)
        finally:legacy.ALLOW_TOKENIZER_PIPELINE_DIFFERENCES=False
        node.identity['subfolder']=eng.subfolders.get(node.repo,'');same_config(node)
        nodes[repo]=node
        return node
    def vocabulary(node):return {k:node.tokenizer[k] for k in ('core','tokens','merges')}
    def pair(label,item):
        """(source node, its reference node or None for B). A source must share its reference's vocabulary exactly,
        or A - R is not a usable delta."""
        node=aligned(item['repo']);ref=aligned(item['reference_repo']) if item.get('reference_repo') else None
        if ref is not None and vocabulary(node)!=vocabulary(ref):
            raise ValueError(label+'：来源与其训练底座的核心词表、merges 或 token ID 不同，A - R 不是可用的增量')
        return node,ref
    sources=[]
    for item in extras:
        if item['format']=='gguf':
            names=eng.files(item['repo'])
            if not item['file']:
                item['file'],considered=gguf.pick_file(names)
                print(f'{item["tag"]}：从 {item["repo"]} 选用 {item["file"]}（可读的候选：{len(considered)} 个）',flush=True)
            gfile=gguf.GGUFFile(eng,item['repo'],item['file'],root/'gguf_headers')
            node=gguf.GGUFSource(gfile,subject,subject.config,settings.get('gguf_policy',{}))
            sources.append({**item,'node':node,'reference':None,'identity':{'source':node.identity,'reference':'B'}})
            continue
        if item['format']=='della':
            members=[]
            for m in item['members']:
                node,ref=pair(item['tag']+'/'+m['tag'],m);members.append({**m,'node':node,'reference':ref})
            identity={'kind':'della','implementation':della.VERSION,'params':item['della'],'reference':'B unless a member names its own',
                      'head_policy':'weighted mean over the members that have the row',
                      'members':[{'tag':m['tag'],'weight':m['weight'],'source':m['node'].identity,
                                  **({'reference':m['reference'].identity} if m['reference'] is not None else {})} for m in members]}
            sources.append({**item,'members':members,'node':None,'reference':None,'identity':identity})
            continue
        node,ref=pair(item['tag'],item)
        sources.append({**item,'node':node,'reference':ref,'identity':{'source':node.identity,'reference':ref.identity if ref is not None else 'B'}})
    searched_terms=[]
    for item in searched:
        node,ref=pair(item['tag'],item)
        searched_terms.append({**item,'node':node,'reference':ref,
                               'identity':{'source':node.identity,'reference':ref.identity if ref is not None else 'B','roles':item['roles']}})
    dw=float(settings.get('donor_weight',1.))
    if settings.get('output_repo') in used|set(wanted):raise ValueError('输出仓库不能覆盖输入')
    identity={'version':core.VERSION,'inputs':{t:runtime.source_identity(eng,r) for t,r in (('S',srepo),('M',mrepo),('B',brepo))},
              'alignment':{t:n.identity for t,n in (('S',subject),('M',donor),('B',base))},
              'donor_manifest_sha256':core.digest(donor_manifest),'donor_selection':recipe.get('a3_selection'),
              'policy':'S protected vision/MTP/embedding/nonmatrix/control; independently scaled head; eta=0',
              'aux_mix':{'version':AUX_DELTA_VERSION,'donor_weight':dw,
                         'formula':'F = S + lambda_g*[D - kappa*(S-B)] + mu*C; D = (M-B) + sum_k (w_k/donor_weight) * g_k * T_k; '
                                   'T_k = A_k-R_k (R_k = B unless given), or DELLA(A_k1-R_k1, ..., A_kn-R_kn) for a della group; '
                                   'g_k = 1 except the GGUF noise shrink; C = A_c - R_c on the listed roles, mu searched',
                         'components':[{k:c[k] for k in ('tag','format','weight','coefficient','identity')} for c in sources],
                         'searched':[{'tag':c['tag'],'identity':c['identity']} for c in searched_terms]}}
    return {'S':subject,'M':donor,'B':base,'sources':sources,'searched':searched_terms,'S_repo':srepo,'M_repo':mrepo,'B_repo':brepo,'engine':eng,
            'root':root,'scratch':scratch,'settings':settings,'identity':identity,'donor_manifest':donor_manifest,
            'aux_root':Path(settings.get('aux_delta_dir') or root/'aux_delta').expanduser().resolve(),
            'searched_root':Path(settings.get('searched_delta_dir') or root/'searched_delta').expanduser().resolve(),
            'num_layers':int(sc['num_hidden_layers']),'tied':bool(sc.get('tie_word_embeddings',subject.config.get('tie_word_embeddings',False)))}


def source_repos(context,record=None):
    """Every repository that entered the merge, in a stable order (the donor of N only when the record uses N)."""
    repos=[context['S_repo'],context['M_repo'],context['B_repo']]
    def add(c):
        repos.append(c['repo'])
        if c.get('reference') is not None:repos.append(c['reference'].repo)
    for c in context['sources']:
        if c['format']=='della':
            for m in c['members']:add(m)
        else:add(c)
    for c in context.get('searched',[]):add(c)
    if record is not None and float(record.get('recipe',{}).get('nu',0) or 0) and context.get('tm'):repos.append(context['tm']['repo'])
    if record is not None and float(record.get('recipe',{}).get('nu2',0) or 0) and context.get('tm2'):repos.append(context['tm2']['repo'])
    return list(dict.fromkeys(repos))


def aux_delta_names(ctx,roles=('backbone','head')):
    s=ctx['S'];eng=ctx['engine']
    return sorted(n for n,shape in s.shapes.items() if eng.is_float(ctx['S_repo'],n) and core.parameter_role(n,tuple(shape)) in set(roles))


def aux_delta_identity(ctx,sources,composed_from=None,kind='aux_delta'):
    ident=ctx['identity']
    out={'kind':kind,'version':AUX_DELTA_VERSION,'gguf_reader':gguf.VERSION,'storage':'float16',
         'inputs':ident['inputs'],'alignment':ident['alignment'],'donor_weight':ident['aux_mix']['donor_weight'],
         'components':[{k:c[k] for k in ('tag','format','weight','coefficient','identity')} for c in sources],
         'composed_from':composed_from,'shard_gib':ctx['settings'].get('shard_gib',1.)}
    if kind!='aux_delta':out.update(donor=False,roles=sorted({r for c in sources for r in c.get('roles',['backbone'])}))
    return out


def delta_spec(ctx,kind='aux_delta'):
    """Where and how a stored delta is built: D (M plus the fixed mix) or C (the searched delta alone)."""
    if kind=='aux_delta':
        return {'kind':kind,'root':ctx['aux_root'],'names':aux_delta_names(ctx),'donor':True,'label':'辅料增量 D','local':'aux_delta_local'}
    roles=sorted({r for c in ctx['searched'] for r in c['roles']})
    return {'kind':kind,'root':ctx['searched_root'],'names':aux_delta_names(ctx,roles),'donor':False,'label':'单独搜索的增量 C','local':'searched_delta_local'}


class StoredDelta:
    """Read-only view of a completed aux-delta cache."""
    def __init__(self,directory,fingerprint,tags):
        self.directory=Path(directory);self.fingerprint=fingerprint;self.tags=set(tags)
        self.index=json.loads((self.directory/'aux_delta.index.json').read_text())['weight_map'];self._layouts={}
    def has(self,name):return name in self.index
    def location(self,name):
        """(file, byte offset, byte length) of one tensor of D, or None: a read-ahead hint."""
        fn=self.index.get(name)
        if fn is None:return None
        if fn not in self._layouts:self._layouts[fn]=runtime.safetensors_layout(self.directory/fn)
        where=self._layouts[fn].get(name)
        return (str(self.directory/fn),*where) if where else None
    def rows(self,name,i,j):
        with safe_open(str(self.directory/self.index[name]),framework='pt',device='cpu') as f:
            return f.get_slice(name)[i:j].float().reshape(j-i,-1)


def _complete_delta(ctx,identity,root=None):
    fp=core.digest(identity);directory=Path(root or ctx['aux_root'])/fp[:24];path=directory/'aux_delta_manifest.json'
    try:done=core.load_record(path,{'kind':'aux_delta_complete','fingerprint':fp})
    except (ValueError,OSError):done=None
    if done is None:return None
    for fn,size in done['files'].items():
        if not (directory/fn).exists() or (directory/fn).stat().st_size!=size:return None
    if ctx['settings'].get('verify_cache_hashes',False):
        for fn,sha in done['sha256'].items():
            if core.sha_file(directory/fn)!=sha:raise ValueError('辅料增量缓存损坏：'+str(directory/fn))
    return {'fingerprint':fp,'directory':directory,'summary':done['summary'],'tags':[c['tag'] for c in identity['components']]}


class ReadPools:
    """One worker thread per input repository (or local store), kept for a whole aux-delta build."""
    def __init__(self,enabled):
        self.enabled=bool(enabled);self.pools={};self.lock=threading.Lock()
    def get(self,key):
        if not self.enabled:return None
        with self.lock:
            pool=self.pools.get(key)
            if pool is None:pool=self.pools[key]=ThreadPoolExecutor(max_workers=1,thread_name_prefix='read')
            return pool
    def close(self):
        with self.lock:
            pools,self.pools=list(self.pools.values()),{}
        for pool in pools:pool.shutdown(wait=True,cancel_futures=True)


class RowFeed:
    """The row intervals of one tensor from several sources, each source read front to back by its own worker.

    A source is read `depth` intervals ahead of the arithmetic, so the reads of different repositories overlap
    each other and the computation. Each repository still sees its requests in the old order, so the engine's
    sequential read-ahead keeps working. Without pools every read happens in the caller, as before. The bytes
    and therefore every result are the same either way.
    """
    def __init__(self,pools,intervals,depth=8):
        self.pools,self.intervals,self.depth=pools,list(intervals),max(1,int(depth))
        self.sources={};self.tasks=[]
    def add(self,key,read,pool_key,retain=False):
        source={'read':read,'pool':self.pools.get(pool_key),'futures':{},'next':0,'consumed':0,'retain':bool(retain)}
        self.sources[key]=source;self._fill(source)
    def _fill(self,source):
        if source['pool'] is None:return
        limit=len(self.intervals) if source['retain'] else min(len(self.intervals),source['consumed']+self.depth)
        while source['next']<limit:
            k=source['next'];i,j=self.intervals[k]
            source['futures'][k]=source['pool'].submit(source['read'],i,j);source['next']+=1
    def get(self,key,k):
        source=self.sources[key]
        if source['pool'] is None:
            if source['retain']:
                if k not in source['futures']:source['futures'][k]=source['read'](*self.intervals[k])
                return source['futures'][k]
            return source['read'](*self.intervals[k])
        future=source['futures'][k] if source['retain'] else source['futures'].pop(k)
        value=future.result()
        if not source['retain']:
            source['consumed']=max(source['consumed'],k+1);self._fill(source)
        return value
    def task(self,pool_key,fn,*args):
        """Run fn(*args) on that pool (inline without pools); returns an object with .result()."""
        pool=self.pools.get(pool_key)
        if pool is None:
            class Done:
                def __init__(self,value):self.value=value
                def result(self):return self.value
            return Done(fn(*args))
        future=pool.submit(fn,*args);self.tasks.append(future);return future
    def close(self):
        for source in self.sources.values():
            for future in source['futures'].values():
                if hasattr(future,'cancel'):future.cancel()
        for future in self.tasks:future.cancel()


def della_threads(settings):
    value=settings.get('della_threads','auto')
    return max(1,os.cpu_count() or 1) if value in (None,'auto') else max(1,int(value))


def _node_key(node):
    return getattr(node,'repo',None) or 'local:'+str(getattr(node,'directory','delta'))


@torch.inference_mode()
def aux_delta_tensor(ctx,sources,name,base_node=None,donor=True):
    """One tensor of D in float32 rows, stored as float16. Rows a source does not really have contribute zero.

    The sources are read ahead in parallel (RowFeed); the terms are added up in the fixed order of `sources`.
    donor=False leaves out M - B (the searched delta C is built this way).
    """
    s=ctx['S'];shape=tuple(s.shapes[name]);role=core.parameter_role(name,shape);head=role=='head'
    intervals=row_intervals(shape,ctx['settings'].get('rows_mib',64));B=ctx['B'];M=ctx['M']
    terms=[c for c in sources if base_node is None or c['tag'] not in base_node.tags]
    def against_base(c):
        if c['format']=='della':return any(m.get('reference') is None for m in c['members'])
        return c['format']=='gguf' or c['reference'] is None
    need_base=(base_node is None and donor) or any(against_base(c) for c in terms)
    threads=della_threads(ctx['settings'])
    has_gguf=any(c['format']=='gguf' for c in terms)
    feed=RowFeed(ctx.get('_read_pools') or ReadPools(False),intervals,ctx['settings'].get('read_ahead_intervals',8))
    index={iv:k for k,iv in enumerate(intervals)}
    def reader(node):return lambda i,j:node.rows(name,i,j)
    try:
        # B is read once for the GGUF layout check and the sum: then it is kept whole, as before
        if need_base:feed.add('B',reader(B),_node_key(B),retain=has_gguf)
        if base_node is not None:feed.add('base',reader(base_node),_node_key(base_node))
        elif donor:feed.add('M',reader(M),_node_key(M))
        for c in terms:
            if c['format']=='della':
                for m in c['members']:
                    feed.add(c['tag']+'/'+m['tag'],reader(m['node']),_node_key(m['node']))
                    if m.get('reference') is not None:feed.add(c['tag']+'/'+m['tag']+'/ref',reader(m['reference']),_node_key(m['reference']))
            elif c['format']!='gguf':
                feed.add(c['tag'],reader(c['node']),_node_key(c['node']))
                if c['reference'] is not None:feed.add(c['tag']+'/ref',reader(c['reference']),_node_key(c['reference']))
        def base_rows(i,j):return feed.get('B',index[(i,j)])
        stats={'tensor':name,'role':role,'gguf':{}};resolved={};fused_stats={}
        pending={c['tag']:feed.task(c['node'].gfile.repo,c['node'].resolve,name,base_rows,intervals,head)
                 for c in terms if c['format']=='gguf'}
        for c in terms:                                    # the report lists the GGUF tensors in the old order
            if c['format']=='gguf':stats['gguf'][c['tag']]=None
        key='base' if base_node is not None else 'M' if donor else None
        sq={**({key:0.} if key else {}),**{c['tag']:0. for c in terms}}
        out=torch.empty(shape,dtype=torch.float16);flat=out.reshape(shape[0],-1)
        width=math.prod(shape[1:]) if shape else 1
        for k,(i,j) in enumerate(intervals):
            bb=feed.get('B',k) if need_base else None
            if base_node is not None:d=feed.get('base',k)
            elif donor:
                d=feed.get('M',k)-bb
                if head:d=torch.where(M.support(name,i,j)&B.support(name,i,j),d,0.)
            else:d=torch.zeros((j-i,width),dtype=torch.float32)
            if key:sq[key]+=float(d.double().square().sum())
            values={}
            # every term on its own first (a GGUF term only waits for its tensor when it is needed) ...
            for c in sorted(terms,key=lambda c:c['format']=='gguf'):
                coef=float(c['coefficient'])
                if c['format']=='gguf':
                    if c['tag'] not in resolved:
                        v,gain,support,record=pending[c['tag']].result()
                        stats['gguf'][c['tag']]=record;resolved[c['tag']]=(v,gain,support)
                    v,gain,support=resolved[c['tag']]
                    if v is None or gain==0:values[c['tag']]=None;continue
                    ok=support[i:j].reshape(-1,1)&B.support(name,i,j)
                    term=torch.where(ok,v[i:j].reshape(j-i,-1)-bb,0.)*(coef*gain)
                elif c['format']=='della':
                    # Each member's delta against B (or its own reference); rows a member does not have are zero and do not vote.
                    base_np=bb.numpy() if bb is not None else None;deltas=[];supports=[]
                    for m in c['members']:
                        ref=m.get('reference')
                        ok=(m['node'].support(name,i,j)&(ref if ref is not None else B).support(name,i,j)).numpy().reshape(-1)
                        delta=feed.get(c['tag']+'/'+m['tag'],k).numpy()-(feed.get(c['tag']+'/'+m['tag']+'/ref',k).numpy() if ref is not None else base_np)
                        if not ok.all():delta[~ok]=0.
                        deltas.append(delta);supports.append(ok)
                    weights=[m['weight'] for m in c['members']]
                    if head:fused=della.mean_rows(deltas,supports,weights)*np.float32(c['della']['lambda'])
                    else:
                        fused,part=della.della_rows(deltas,weights,c['della'],[m['tag']+'|'+name for m in c['members']],i,threads)
                        fused_stats[c['tag']]=della.merge_stats(fused_stats.get(c['tag']),part)
                    term=torch.from_numpy(np.ascontiguousarray(fused,dtype=np.float32))*coef
                elif c['reference'] is None:
                    node=c['node'];ok=node.support(name,i,j)&B.support(name,i,j)
                    term=torch.where(ok,feed.get(c['tag'],k)-bb,0.)*coef
                else:
                    node,ref=c['node'],c['reference'];ok=node.support(name,i,j)&ref.support(name,i,j)
                    term=torch.where(ok,feed.get(c['tag'],k)-feed.get(c['tag']+'/ref',k),0.)*coef
                values[c['tag']]=term
            # ... then added up in the fixed order of the sources
            for c in terms:
                term=values[c['tag']]
                if term is None:continue
                sq[c['tag']]+=float(term.double().square().sum())
                d=d+term
            del values
            if not torch.isfinite(d).all():raise ValueError('辅料增量含 NaN/Inf：'+name)
            half=d.to(torch.float16)
            if not torch.isfinite(half).all():raise ValueError('辅料增量超出 FP16 范围：'+name)
            flat[i:j]=half
        for c in terms:                                    # a GGUF term of a tensor without intervals
            if c['format']=='gguf' and stats['gguf'][c['tag']] is None:stats['gguf'][c['tag']]=pending[c['tag']].result()[3]
    finally:
        feed.close()
    del resolved
    stats['component_sq']=sq
    if fused_stats:stats['della']=fused_stats
    return out,stats



def _summarize_delta(records,sources):
    totals={}
    for r in records:
        for k,v in r['component_sq'].items():totals[k]=totals.get(k,0.)+v
    gguf_summary={};della_summary={}
    for c in sources:
        tag=c['tag']
        if c['format']=='gguf':
            recs=[r['gguf'][tag] for r in records if tag in r.get('gguf',{})]
            if not recs:continue
            ok=[x for x in recs if x['status']=='ok'];layouts={}
            for x in ok:layouts[x['layout']]=layouts.get(x['layout'],0)+1
            delta=sum(x['delta_sq'] for x in ok);noise=sum(x['noise_sq'] for x in ok)
            gains=sorted(x['gain'] for x in ok)
            gguf_summary[tag]={'tensors_used':len(ok),'tensors_excluded':len(recs)-len(ok),'layouts':layouts,
                               'noise_share_of_delta_energy':noise/delta if delta else None,
                               'median_gain':gains[len(gains)//2] if gains else None,'min_gain':gains[0] if gains else None,
                               'excluded':[{'tensor':x['tensor'],'reason':x.get('reason')} for x in recs if x['status']!='ok'][:40]}
        elif c['format']=='della':
            total=None
            for r in records:
                if tag in r.get('della',{}):total=della.merge_stats(total,r['della'][tag])
            if total is None:continue
            members=[{'tag':m['tag'],'weight':m['weight'],'delta_norm':math.sqrt(x['delta_sq']),
                      'kept_share':x['kept']/x['nonzero'] if x['nonzero'] else None,
                      'elected_share_of_kept':x['elected']/x['kept'] if x['kept'] else None}
                     for m,x in zip(c['members'],total['sources'])]
            della_summary[tag]={'params':c['della'],'members':members,'fused_norm':math.sqrt(total['out_sq']),
                                'fused_nonzero_share':total['nonzero']/total['elements'] if total['elements'] else None,
                                'scope':'backbone matrices before the mix coefficient; the output head uses the weighted mean and is not counted here'}
    return {'component_norms':{k:math.sqrt(v) for k,v in totals.items()},'gguf':gguf_summary,'della':della_summary}



def _keep_local_delta(ctx,directory):
    """Whether the shards of a D in the record directory should also be kept on (or copied to) the local disk.

    Every candidate reads all of D. When the record directory is a network drive, the search would otherwise
    copy D back from it right after it was written (localize_delta); the staged shards are the very same bytes.
    """
    want=ctx['settings'].get('aux_delta_local_copy','auto')
    if want=='auto':want=os.stat(directory).st_dev!=os.stat(ctx['scratch']).st_dev
    return bool(want) and ctx['settings'].get('input_cache_mode','auto')!='stream'


def build_aux_delta(ctx,sources,base=None,base_node=None,spec=None):
    """Write D for every modified tensor (float16), resumable per shard; every shard is checked once on its target.

    base: an earlier complete D without the late sources; base_node optionally reads that D from a local copy.
    spec (delta_spec): what is built; by default D. The searched delta C is built the same way, without M - B.
    """
    spec=spec or delta_spec(ctx);title=spec['label']
    identity=aux_delta_identity(ctx,sources,base['fingerprint'] if base else None,spec['kind'])
    done=_complete_delta(ctx,identity,spec['root'])
    if done is not None:return done
    fp=core.digest(identity);directory=Path(spec['root'])/fp[:24];directory.mkdir(parents=True,exist_ok=True)
    core.atomic_json(directory/'aux_delta_identity.json',identity)
    if base_node is None and base:base_node=StoredDelta(base['directory'],base['fingerprint'],base['tags'])
    names=spec['names'];groups=[];group=[];size=0;limit=int(ctx['settings'].get('shard_gib',1)*1024**3)
    for name in names:
        n=math.prod(ctx['S'].shapes[name])*2
        if group and size+n>limit:groups.append(group);group=[];size=0
        group.append(name);size+=n
    if group:groups.append(group)
    staging=ctx['scratch']/'aux_delta_staging';staging.mkdir(parents=True,exist_ok=True)
    todo=[k for k in range(1,len(groups)+1) if not (directory/f'delta-{k:05d}-of-{len(groups):05d}.safetensors').exists()]
    remaining=sum(math.prod(ctx['S'].shapes[n])*2 for k in todo for n in groups[k-1])
    free=shutil.disk_usage(directory).free
    if free<remaining+1024**3:
        raise RuntimeError(f'{title} 还要写约 {remaining/1024**3:.1f} GiB，{directory} 所在盘只剩 {free/1024**3:.1f} GiB。'
                           'Drive 上旧版本的辅料增量（aux_delta/ 下其他指纹的目录）和 qwen38_final_stage 都可以删除后重跑（记得清空回收站）；'
                           '或把 aux_delta_dir / searched_delta_dir 指到别处。')
    local=ctx['scratch']/spec['local']/fp[:24]
    keep_local=_keep_local_delta(ctx,directory) and shutil.disk_usage(ctx['scratch']).free>remaining+8*1024**3
    if keep_local:local.mkdir(parents=True,exist_ok=True)
    records=[];files={};hashes={};mapping={};checked=set()
    def label(c):return c['tag']+('（DELLA：'+'、'.join(m['tag'] for m in c['members'])+'）' if c['format']=='della' else '')
    what=('在已有部分上加入 '+'、'.join(label(c) for c in sources if c['tag'] not in set(base['tags'])) if base
          else ('M + ' if spec['donor'] else '')+' + '.join(label(c) for c in sources))
    parallel=bool(ctx['settings'].get('parallel_reads',True))
    print(f'{title} {fp[:10]}：{what}，共 {len(groups)} 个分片（待算 {len(todo)}），写到 {directory}'
          +(f'；各来源并行读取，DELLA {della_threads(ctx["settings"])} 线程' if parallel else ''),flush=True)
    started=time.perf_counter();built=0;bytes_before=ctx['engine'].bytes_read
    ctx['_read_pools']=ReadPools(parallel)
    try:
        for index,tensors in enumerate(groups,1):
            fn=f'delta-{index:05d}-of-{len(groups):05d}.safetensors';receipt=directory/(fn+'.receipt.json')
            cid={'fingerprint':fp,'index':index,'names':tensors}
            for n in tensors:mapping[n]=fn
            try:old=core.load_record(receipt,cid)
            except (ValueError,OSError):old=None  # a half-synced receipt is recomputed, not trusted
            if old is not None and (directory/fn).exists() and (directory/fn).stat().st_size==old['size']:
                records.extend(old['tensor_stats']);files[fn]=old['size'];hashes[fn]=old['sha256'];continue
            values={};stats=[]
            for name in tensors:
                value,stat=aux_delta_tensor(ctx,sources,name,base_node,spec['donor']);values[name]=value.contiguous();stats.append(stat)
            tmp=staging/(fn+'.tmp');save_file(values,str(tmp),metadata={'format':'pt'})
            del values,value;gc.collect()
            sha=core.sha_file(tmp);nbytes=tmp.stat().st_size
            target=directory/(fn+'.part')
            shutil.copyfile(tmp,target);target.replace(directory/fn)
            # checked on its target right away, while a network drive still holds it in its local cache
            if core.sha_file(directory/fn)!=sha:
                (directory/fn).unlink();raise ValueError(f'{title} 的分片写入后读回不一致，已删除，请重跑本格：{directory/fn}')
            checked.add(fn)
            if keep_local:tmp.replace(local/fn)
            else:tmp.unlink()
            core.save_record(receipt,cid,{'size':nbytes,'sha256':sha,'tensor_stats':stats})
            records.extend(stats);files[fn]=nbytes;hashes[fn]=sha;built+=1
            elapsed=time.perf_counter()-started;read=(ctx['engine'].bytes_read-bytes_before)/1024**3
            print(f'{title} {fp[:10]}：分片 {index}/{len(groups)}，本次已用 {elapsed/60:.0f} 分钟（网络读取 {read:.1f} GiB，'
                  f'{read*1024/max(elapsed,1e-9):.0f} MiB/s），估计还需 {elapsed/built*max(0,len(todo)-built)/60:.0f} 分钟',flush=True)
    finally:
        pools=ctx.pop('_read_pools',None)
        if pools is not None:pools.close()
    for fn,sha in hashes.items():
        if fn in checked:continue                          # shards of an earlier session: checked once, here
        if core.sha_file(directory/fn)!=sha:
            (directory/fn).unlink();raise ValueError(f'{title} 的分片校验失败，已删除，请重跑本格：{directory/fn}')
    core.atomic_json(directory/'aux_delta.index.json',{'metadata':{'fingerprint':fp},'weight_map':mapping})
    summary=_summarize_delta(records,sources)
    if base is not None:
        # The part computed earlier keeps its own figures (norms, GGUF layouts, DELLA shares).
        summary['composed_from']=base['summary']
        earlier=Path(base['directory'])/'aux_delta_report.json'
        if earlier.is_file():shutil.copyfile(earlier,directory/'aux_delta_report.base.json')
    core.atomic_json(directory/'aux_delta_report.json',{'summary':summary,'tensor_stats':records})
    files['aux_delta.index.json']=(directory/'aux_delta.index.json').stat().st_size
    core.save_record(directory/'aux_delta_manifest.json',{'kind':'aux_delta_complete','fingerprint':fp},
                     {'files':files,'sha256':hashes,'summary':summary})
    return {'fingerprint':fp,'directory':directory,'summary':summary,'tags':[c['tag'] for c in sources]}



def _component_keys(identity):
    return sorted(core.digest(c) for c in identity.get('components') or [])


def stored_aux_deltas(ctx,sources):
    """Complete D caches in aux_root built from the same inputs and settings as this run, whose sources are a subset
    of `sources` (a D built earlier, before a source was added). Returns [(missing component count, cache)], fewest
    missing first; 0 missing means that D is this run's D, whichever way it was composed."""
    frame=aux_delta_identity(ctx,[])
    want=set(_component_keys(aux_delta_identity(ctx,sources)))
    found=[]
    if not ctx['aux_root'].is_dir():return found
    for directory in sorted(ctx['aux_root'].iterdir()):
        path=directory/'aux_delta_identity.json'
        if not directory.is_dir() or not path.is_file():continue
        try:identity=json.loads(path.read_text(encoding='utf-8'))
        except (OSError,ValueError):continue
        if not isinstance(identity,dict) or any(identity.get(k)!=v for k,v in frame.items() if k not in ('components','composed_from')):continue
        have=_component_keys(identity)
        if len(set(have))!=len(have) or not set(have)<=want:continue
        try:done=_complete_delta(ctx,identity)
        except (ValueError,OSError):done=None
        if done is None or Path(done['directory'])!=directory:continue
        found.append((len(want)-len(have),done))
    found.sort(key=lambda x:(x[0],str(x[1]['directory'])))
    return found


def existing_aux_delta(ctx,sources):
    """The complete D for exactly these sources when it is already stored (built at once, or composed by adding
    sources to an earlier D); nothing is built."""
    for missing,done in stored_aux_deltas(ctx,sources):
        if missing==0:return done
    return None


def ensure_aux_delta(ctx,sources):
    """Reuse or build D for exactly these sources. A stored D whose sources are a subset of these (for instance one
    built before StartLux or the second DELLA group was added) is reused and only the missing terms are added; the
    one with the most sources is taken."""
    stored=stored_aux_deltas(ctx,sources)
    result=next((done for missing,done in stored if missing==0),None)
    if result is None:
        base=next((done for missing,done in stored if missing>0 and done['tags']),None)
        base_node=None
        if base is not None:
            added=[c['tag'] for c in sources if c['tag'] not in set(base['tags'])]
            print('在已有的辅料增量 D（'+'、'.join(['M']+base['tags'])+'）上补加：'+'、'.join(added),flush=True)
        if base is not None and _keep_local_delta(ctx,Path(base['directory'])):
            # The earlier D is read in full: one verified copy to the local disk instead of reads through the drive.
            need=sum(_delta_sizes(base).values())
            target=core.digest(aux_delta_identity(ctx,sources,base['fingerprint']))[:24]
            if shutil.disk_usage(ctx['scratch']).free>2*need+8*1024**3:
                base_node=localize_delta(ctx,StoredDelta(base['directory'],base['fingerprint'],base['tags']),keep=(target,))
        result=build_aux_delta(ctx,sources,base,base_node)
        if base is not None and ctx['settings'].get('aux_delta_drop_partial',True):
            target=Path(base['directory'])
            if target.parent==ctx['aux_root'] and (target/'aux_delta_manifest.json').exists():
                shutil.rmtree(target);print('已删除被补加取代的旧辅料增量：',target,flush=True)
    for other in sorted(ctx['aux_root'].iterdir()) if ctx['aux_root'].is_dir() else []:
        if other.is_dir() and other!=Path(result['directory']) and (other/'aux_delta_identity.json').is_file():
            size=sum(p.stat().st_size for p in other.glob('*.safetensors'))
            print(f'提示：{other}（{size/1024**3:.1f} GiB）是另一组来源或比例的辅料增量，本次用不到；确认不再需要可以删除。',flush=True)
    return result


def ensure_searched_delta(ctx):
    """C = A - R of the searched term, built once (float16, only the roles it lists) and kept under searched_root;
    None when there is no searched term."""
    if not ctx.get('searched'):return None
    spec=delta_spec(ctx,'searched_delta')
    done=build_aux_delta(ctx,ctx['searched'],spec=spec)
    root=Path(spec['root'])
    for other in sorted(root.iterdir()) if root.is_dir() else []:
        if other.is_dir() and other!=Path(done['directory']) and (other/'aux_delta_identity.json').is_file():
            size=sum(p.stat().st_size for p in other.glob('*.safetensors'))
            print(f'提示：{other}（{size/1024**3:.1f} GiB）是另一组来源的增量 C，本次用不到；确认不再需要可以删除。',flush=True)
    return done


ROW_REPORT_VERSION='row-change-2026-10-22.1'


def della_member_notes(summary):
    """Per DELLA group (also in the part of D built earlier, composed_from): every member's backbone delta norm, and a
    note for a member whose norm is zero (it acts on the output head at most) or more than twice the median of the
    other members (for instance a delta moved across two base generations)."""
    lines=[];notes=[];seen=set()
    def walk(s):
        if not isinstance(s,dict):return
        for tag,group in (s.get('della') or {}).items():
            if tag in seen:continue
            seen.add(tag);members=group.get('members') or []
            norms=[float(m.get('delta_norm') or 0.) for m in members]
            lines.append(f'{tag} 成员主干增量范数：'+'，'.join(f'{m["tag"]} {n:.2f}' for m,n in zip(members,norms)))
            for k,m in enumerate(members):
                others=norms[:k]+norms[k+1:]
                median=float(np.median(others)) if others else 0.
                if norms[k]==0:lines.append(f'  {tag}/{m["tag"]} 的主干增量为 0：它在这个组里只影响输出头（只改 lm_head 的成员就是这样）')
                elif median>0 and norms[k]>2*median:
                    notes.append(f'{tag}/{m["tag"]} 的主干增量范数是组内其他成员中位数的 {norms[k]/median:.1f} 倍；'
                                 '它在符号选举和均值里分量偏重，可以考虑移出 DELLA 组、单独给一个更小的系数')
        walk(s.get('composed_from'))
    walk(summary)
    return lines,notes


def print_della_notes(summary):
    lines,notes=della_member_notes(summary)
    for line in lines:print(line,flush=True)
    for note in notes:print('注意：'+note,flush=True)
    return notes


def row_change_report(ctx):
    """How far lambda*D, mu*C and both together move the rows of S, against the row cap (max_relative_change).

    For every row of every matrix D or C changes: ||D_row||/||S_row||, ||C_row||/||S_row|| and <D_row, C_row>/||S_row||^2,
    from which the share of rows over the cap follows exactly for every lambda of lambda_grid, mu of mu_grid and their
    sums (kappa = 0, all group scales 1: the term -lambda*kappa*(S - B) needs B and is left out). The output head is
    reported for head_grid. A row over the cap is scaled back as a whole, so where lambda*D (or mu*C) alone exceeds it, the
    other term has no room left. Reads S, D and C once; cached per D, C and grids. Returns the summary, or None."""
    D=ctx.get('D');C=ctx.get('C');cfg=ctx['settings']
    if D is None or not cfg.get('row_change_report',True):return None
    cap=row_change_cap(ctx)
    lambdas=sorted({float(x) for x in cfg.get('lambda_grid',[]) if float(x)>0})
    heads=[] if ctx['tied'] else sorted({float(x) for x in cfg.get('head_grid',[]) if float(x)>0})
    mus=sorted({float(x) for x in cfg.get('mu_grid',[]) if float(x)>0}) if C is not None else []
    path=ctx['root']/'row_change_report.json'
    identity={'kind':'row_change','version':ROW_REPORT_VERSION,'D':D.fingerprint,'C':C.fingerprint if C is not None else None,
              'S':ctx['identity']['inputs']['S'],'cap':cap,'lambda_grid':lambdas,'head_grid':heads,'mu_grid':mus}
    try:done=core.load_record(path,identity)
    except (ValueError,OSError):done=None
    if done is None:
        started=time.perf_counter()
        c_roles={r for c in ctx.get('searched',[]) for r in c['roles']}
        c_names=set(C.index) if C is not None else set()
        names=sorted(set(D.index)|c_names)
        print(f'逐行改动报告：读一遍 S、D'+('、C' if C is not None else '')+f'（{len(names)} 个张量），只算一次',flush=True)
        backbone=[];head=[];kinds={}
        for n,name in enumerate(names,1):
            shape=tuple(ctx['S'].shapes[name]);role=core.parameter_role(name,shape)
            if role not in ('backbone','head'):continue
            has_d=D.has(name);has_c=name in c_names and role in c_roles
            parts=[]
            for i,j in row_intervals(shape,cfg.get('rows_mib',64)):
                s=ctx['S'].rows(name,i,j).double().reshape(j-i,-1);ss=(s*s).sum(1).clamp_min(1e-30)
                d=D.rows(name,i,j).double() if has_d else None;c=C.rows(name,i,j).double() if has_c else None
                zero=torch.zeros(j-i,dtype=torch.float64)
                parts.append(torch.stack([(d*d).sum(1)/ss if d is not None else zero,(c*c).sum(1)/ss if c is not None else zero,
                                          (d*c).sum(1)/ss if d is not None and c is not None else zero],1).numpy())
                del s,d,c
            arr=np.concatenate(parts)
            if role=='head':head.append(arr)
            else:
                backbone.append(arr)
                kind='attn' if '.self_attn.' in name else 'linear' if '.linear_attn.' in name else 'mlp' if '.mlp.' in name else 'other'
                kinds.setdefault(kind,[]).append(arr[:,0])
            if n%64==0:print(f'  逐行改动报告：{n}/{len(names)} 个张量，已用 {(time.perf_counter()-started)/60:.0f} 分钟',flush=True)
        B=np.concatenate(backbone) if backbone else np.zeros((0,3))
        rd=np.sqrt(B[:,0]);rc=np.sqrt(B[:,1])
        def summary(x):
            x=x[x>0]
            if not x.size:return None
            return {'rows':int(x.size),'median':float(np.median(x)),'p90':float(np.quantile(x,.9)),'p99':float(np.quantile(x,.99)),'max':float(x.max())}
        def share(r):return float((r>cap).mean()) if cap and r.size else None
        done={'cap':cap,'backbone_rows':int(B.shape[0]),'scope':'kappa=0, group scales 1; shares are over all rows of matrices D or C changes',
              'D_ratio':summary(rd),'D_ratio_by_module':{k:summary(np.sqrt(np.concatenate(v))) for k,v in sorted(kinds.items())},
              'C_ratio':summary(rc) if C is not None else None,
              'share_over_cap_lambda_D':{f'{l:g}':share(l*rd) for l in lambdas},
              'share_over_cap_mu_C':{f'{m:g}':share(m*rc) for m in mus},
              'share_over_cap_lambda_D_plus_mu_C':{f'{l:g}|{m:g}':share(np.sqrt(np.maximum(0.,l*l*B[:,0]+2*l*m*B[:,2]+m*m*B[:,1]))) for l in lambdas for m in mus},
              'lambda_for_1pct_rows':float(cap/np.quantile(rd,.99)) if cap and rd.size and np.quantile(rd,.99)>0 else None,
              'seconds':None}
        if head:
            H=np.sqrt(np.concatenate(head)[:,0]);done['head_D_ratio']=summary(H)
            done['share_over_cap_head']={f'{h:g}':share(h*H) for h in heads}
        done['seconds']=time.perf_counter()-started
        core.save_record(path,identity,done)
    cap=done['cap'];d=done.get('D_ratio') or {}
    print(f'D 相对 S 的逐行大小 ||D||/||S||：中位数 {d.get("median",0):.4f}，p99 {d.get("p99",0):.4f}，最大 {d.get("max",0):.4f}',flush=True)
    if done.get('C_ratio'):
        c=done['C_ratio'];print(f'C 相对 S 的逐行大小 ||C||/||S||：中位数 {c["median"]:.4f}，p99 {c["p99"]:.4f}，最大 {c["max"]:.4f}',flush=True)
    if cap:
        print(f'超过行改动上限 {cap:.0%} 的行（κ=0）：'+'；'.join(f'λ={l}·D {v:.2%}' for l,v in done['share_over_cap_lambda_D'].items())
              +(('；'+'；'.join(f'μ={m}·C {v:.2%}' for m,v in done['share_over_cap_mu_C'].items())) if done['share_over_cap_mu_C'] else ''),flush=True)
        if done['share_over_cap_lambda_D_plus_mu_C']:
            print('λ·D + μ·C 合计超过上限的行：'+'；'.join(f'λ={k.split("|")[0]}, μ={k.split("|")[1]}: {v:.2%}'
                                                    for k,v in done['share_over_cap_lambda_D_plus_mu_C'].items()),flush=True)
        if done.get('share_over_cap_head'):
            print('输出头 λ_head·D 超过上限的行：'+'；'.join(f'λ_head={h}: {v:.2%}' for h,v in done['share_over_cap_head'].items()),flush=True)
        worst=max(done['share_over_cap_lambda_D'].values() or [0.])
        if worst>0.01:
            print(f'注意：最大的 λ 下有 {worst:.1%} 的行仅 λ·D 就超过上限，会被整体缩回，搜索的高 λ 一端被上限削平。λ ≤ '
                  f'{done["lambda_for_1pct_rows"]:.3f} 时只有约 1% 的行超过；可以把 lambda_grid 的上限压到这附近，'
                  '或调高 max_relative_change。改了之后搜索会重做，D 和 C 不用重算。',flush=True)
        both=max(done['share_over_cap_lambda_D_plus_mu_C'].values() or [0.])
        if both>0.05:print(f'注意：λ·D + μ·C 在网格最大处有 {both:.1%} 的行超过上限，那些候选里两项都按比例缩小了。',flush=True)
    return done


def _delta_sizes(delta):
    manifest=core.load_record(Path(delta['directory'])/'aux_delta_manifest.json',{'kind':'aux_delta_complete','fingerprint':delta['fingerprint']})
    return {fn:size for fn,size in manifest['files'].items() if fn.endswith('.safetensors')}


def copy_verified(source,target,chunk=16*1024**2):
    """Copy one file and return the sha256 of the bytes written."""
    target=Path(target);target.parent.mkdir(parents=True,exist_ok=True)
    tmp=target.with_name(target.name+'.part');digest=hashlib.sha256()
    with Path(source).open('rb') as src,tmp.open('wb') as dst:
        for block in iter(lambda:src.read(chunk),b''):
            digest.update(block);dst.write(block)
    tmp.replace(target)
    return digest.hexdigest()


def copy_many(pairs,workers=4,check=None):
    """copy_verified for many files at once: {name: sha256}. A network drive serves several files in parallel
    much faster than one after another. pairs: (name, source, target). With check(name, sha) a file whose copy
    does not pass is copied once more (a network drive can hiccup)."""
    pairs=list(pairs)
    def one(item):
        name,source,target=item
        sha=copy_verified(source,target)
        if check is not None and not check(name,sha):sha=copy_verified(source,target)
        return name,sha
    workers=max(1,min(int(workers),len(pairs) or 1))
    if workers==1:return dict(one(item) for item in pairs)
    with ThreadPoolExecutor(max_workers=workers,thread_name_prefix='copy') as pool:
        return dict(pool.map(one,pairs))


def _tree_bytes(path):
    path=Path(path)
    return sum(p.stat().st_size for p in path.rglob('*') if p.is_file()) if path.is_dir() else 0


def localize_delta(ctx,delta,keep=(),store_name='aux_delta_local'):
    """Copy the aux-delta shards to local scratch, checked against the hashes in its manifest.

    Every candidate reads all of D once; from a network drive that is the slowest part of a trial. Other local
    copies of D are removed, except those named in keep.
    """
    manifest=core.load_record(delta.directory/'aux_delta_manifest.json',{'kind':'aux_delta_complete','fingerprint':delta.fingerprint})
    store=ctx['scratch']/store_name;target=store/delta.fingerprint[:24];target.mkdir(parents=True,exist_ok=True)
    for other in store.iterdir():
        if other!=target and other.name not in keep:shutil.rmtree(other)             # a local copy of some other D
    started=time.perf_counter()
    todo=[fn for fn,size in sorted(manifest['files'].items()) if not ((target/fn).is_file() and (target/fn).stat().st_size==size)]
    shas=copy_many([(fn,delta.directory/fn,target/fn) for fn in todo],ctx['settings'].get('copy_workers',4))
    for fn,sha in shas.items():
        if fn in manifest['sha256'] and sha!=manifest['sha256'][fn]:
            (target/fn).unlink();raise ValueError(f'辅料增量分片读出的内容与记录不符：{delta.directory/fn}。删除该分片及其 receipt 后重跑即可重算。')
    copied=sum(manifest['files'][fn] for fn in todo)
    if copied:print(f'{"辅料增量 D" if store_name=="aux_delta_local" else "单独搜索的增量 C"} 已复制到本地盘：{copied/1024**3:.1f} GiB，用时 {(time.perf_counter()-started)/60:.1f} 分钟',flush=True)
    return StoredDelta(target,delta.fingerprint,delta.tags)



def setup_local(ctx,subject_only=False):
    """Local scratch copies: S always (it is also the baseline); then the aux delta D; then the searched delta C; then B, as far as the disk allows.

    subject_only is for the stage that builds no candidates (waiting for a source or data). With the layer-by-layer
    engine the search builds no candidate files, so no room is kept for one until the winner is built
    (make_room_for_candidate), and B usually fits as well.
    """
    settings,eng=ctx['settings'],ctx['engine'];srepo,brepo=ctx['S_repo'],ctx['B_repo']
    if 'metadata_dir' not in ctx:ctx['metadata_dir']=runtime.download_metadata(eng,srepo,ctx['root']/'metadata')
    if not ctx.get('preflight_done'):
        runtime.preflight_checkpoint(eng,srepo,ctx['metadata_dir'],settings['proxy_options']);ctx['preflight_done']=True
    mode=settings.get('input_cache_mode','auto')
    if mode not in {'auto','stream','local'}:raise ValueError('input_cache_mode 必须为 auto / stream / local')
    target=ctx['scratch']/'source_checkpoints';target.mkdir(parents=True,exist_ok=True)
    def model_bytes(repo):return sum(math.prod(x['shape'])*legacy_bytes(eng,x) for x in eng._catalogue(repo).values())
    def missing(repo):
        have=_tree_bytes(target/('checkpoint-'+core.digest(runtime.source_identity(eng,repo))[:24]))
        return max(0,model_bytes(repo)-have)
    candidate=0 if ctx.get('scoring_engine')=='layerwise' else sum(math.prod(v)*2 for v in ctx['S'].shapes.values());margin=4*1024**3
    free=shutil.disk_usage(target).free;need_s=missing(srepo);to_cache=[srepo];copy_delta=False;copy_c=False
    if subject_only:
        if free<need_s+margin:raise RuntimeError(f'本地盘放不下 S：还需约 {(need_s+margin)/1024**3:.1f} GiB，只剩 {free/1024**3:.1f} GiB')
    else:
        # A candidate left from an earlier build is replaced by the next one (build_candidate), so its space counts.
        free+=0 if settings.get('keep_search_candidates',False) else _tree_bytes(ctx['scratch']/'candidates')
        budget=free-need_s-candidate-margin
        if budget<0:raise RuntimeError(f'本地盘还需约 {(need_s+candidate+margin)/1024**3:.1f} GiB（S 缓存'+(' + 一个候选' if candidate else '')+f'），只剩 {free/1024**3:.1f} GiB')
        def need_local(delta,store):
            if delta is None or mode=='stream' or not _keep_local_delta(ctx,delta.directory):return None
            return max(0,sum(p.stat().st_size for p in delta.directory.glob('*.safetensors'))-_tree_bytes(ctx['scratch']/store/delta.fingerprint[:24]))
        need_d=need_local(ctx.get('D'),'aux_delta_local');need_c=need_local(ctx.get('C'),'searched_delta_local')
        need_b=missing(brepo)
        if mode=='local' and budget<(need_d or 0)+(need_c or 0)+need_b:
            raise RuntimeError(f'input_cache_mode=local：本地盘还差 {((need_d or 0)+(need_c or 0)+need_b-budget)/1024**3:.1f} GiB 才放得下 S、D、C、B'
                               +('和一个候选' if candidate else ''))
        if need_d is not None and budget>=need_d:copy_delta=True;budget-=need_d
        copy_c=need_c is not None and budget>=need_c
        if copy_c:budget-=need_c
        if mode!='stream' and budget>=need_b:to_cache.append(brepo)
        print('本地缓存：S'+('、辅料增量 D' if copy_delta else '')+('、增量 C' if copy_c else '；C 从记录目录读取' if need_c is not None else '')
              +('、B' if brepo in to_cache else '；B 按范围读取'),flush=True)
    for repo in to_cache:
        if repo in eng.local_catalogues:continue
        directory=runtime.download_checkpoint(eng,repo,target)
        eng.local_catalogues[repo]=runtime.checkpoint_catalogue(directory)
        if repo==srepo:ctx['baseline_dir']=directory
    if copy_delta:ctx['D']=localize_delta(ctx,ctx['D'])
    if not subject_only and copy_c:ctx['C']=localize_delta(ctx,ctx['C'],store_name='searched_delta_local')
    ctx['input_cache_mode']=mode
    return ctx


def legacy_bytes(eng,entry):
    sizes={'F64':8,'F32':4,'F16':2,'BF16':2,'I64':8,'I32':4,'I16':2,'I8':1,'U8':1,'BOOL':1}
    return sizes[entry['dtype']]


def row_intervals(shape,rows_mib=64,arrays=12):
    rows=shape[0] if shape else 1;width=math.prod(shape[1:]) if shape else 1
    step=max(1,int(rows_mib*1024**2/max(4*arrays*width,1)))
    return [(i,min(i+step,rows)) for i in range(0,rows,step)]


def layer_group(name,num_layers):
    m=re.search(r'(?:^|\.)layers\.(\d+)\.',name)
    if m is None:return 0
    return 0 if int(m[1])<num_layers/2 else 1


def searched_mu(context,name,role,recipe):
    """The coefficient of C on this tensor: the recipe's mu where C covers the tensor's role, else 0."""
    mu=float(recipe.get('mu',0.))
    if not mu:return 0.
    if context.get('C') is None:raise ValueError('配方含 mu，但没有单独搜索的增量 C')
    roles={r for c in context.get('searched',[]) for r in c['roles']}
    if role not in roles or not context['C'].has(name):return 0.
    return mu


def searched_nu(context,name,role,recipe):
    """The coefficient of N on this tensor: the recipe's nu where N has rows of this matrix, else 0."""
    nu=float(recipe.get('nu',0.))
    if not nu:return 0.
    if context.get('N') is None:raise ValueError('配方含 nu，但没有跨架构增量 N')
    if role!='backbone' or not context['N'].has(name):return 0.
    return nu


def searched_nu2(context,name,role,recipe):
    """The coefficient of N2 on this tensor: the recipe's nu2 where N2 has rows of this matrix, else 0."""
    nu2=float(recipe.get('nu2',0.))
    if not nu2:return 0.
    if context.get('N2') is None:raise ValueError('配方含 nu2，但没有跨架构增量 N₂')
    if role!='backbone' or not context['N2'].has(name):return 0.
    return nu2


@torch.inference_mode()
def candidate_tensor(context,name,recipe,stats=True):
    """One tensor of the candidate of `recipe`, as build_candidate writes it. stats=False skips the statistics
    of the manifest (the search scores candidates in RAM and needs only the values, which are the same)."""
    validate_recipe(recipe);s=context['S'];shape=s.shapes[name];eng=context['engine']
    role=core.parameter_role(name,shape)
    if not eng.is_float(context['S_repo'],name) or role=='protected':
        value=eng.tensor(context['S_repo'],name)
        if value.is_floating_point() and not torch.isfinite(value).all():raise ValueError('主体保护/不变张量含 NaN/Inf：'+name)
        return value,{'role':'protected','update_sq':0.,'bf16_changed':0,'elements':value.numel()}
    lam=recipe['lambda_head'] if role=='head' else recipe['lambda_backbone']
    if recipe.get('rho_late'):raise ValueError('本版辅料比例固定在 D 里，配方不再含 rho_late')
    mu=searched_mu(context,name,role,recipe);nu=searched_nu(context,name,role,recipe);nu2=searched_nu2(context,name,role,recipe)
    if context['tied'] and role=='head' and (lam!=0 or mu!=0):raise ValueError('S 的 embedding/head 绑定，embedding 保护时不能独立修改 head')
    if lam==0 and mu==0 and nu==0 and nu2==0:
        value=eng.tensor(context['S_repo'],name)
        if value.is_floating_point() and not torch.isfinite(value).all():raise ValueError('主体保护/不变张量含 NaN/Inf：'+name)
        return value,{'role':role,'update_sq':0.,'bf16_changed':0,'elements':value.numel()}
    output=torch.empty(shape,dtype=torch.bfloat16);flat=output.reshape(shape[0],-1)
    group=layer_group(name,context['num_layers']);low,high=recipe['group_scales'][2*group:2*group+2]
    squares=[];changed=0;capped=0;kappa=recipe['kappa'];cap=row_change_cap(context)
    for i,j in row_intervals(shape,context['settings'].get('rows_mib',64)):
        sb=s.rows(name,i,j);update=None
        if lam:
            delta=context['D'].rows(name,i,j)
            if kappa:
                sub=sb-context['B'].rows(name,i,j)
                if role=='head':
                    # RepoNode substitutes S for vocabulary rows absent from M or B; such rows have no
                    # donor direction to interpolate towards, exactly as before.
                    sub=torch.where(context['M'].support(name,i,j)&context['B'].support(name,i,j),sub,0.)
                delta=delta-kappa*sub
            if role=='backbone':delta=core.magnitude_scale(delta,recipe['high_fraction'],low,high)
            update=lam*delta
        if mu:
            # The searched delta C enters outside the lambda bracket: F = S + lambda*[D - kappa*(S-B)] + mu*C.
            term=mu*context['C'].rows(name,i,j)
            update=term if update is None else update+term
        if nu:
            # The cross-architecture delta N (rows of B transported from the donor), also outside the bracket.
            term=nu*context['N'].rows(name,i,j)
            update=term if update is None else update+term
        if nu2:
            # The second cross-architecture delta N2 (SenseNova), the same kind of term with its own coefficient.
            term=nu2*context['N2'].rows(name,i,j)
            update=term if update is None else update+term
        if cap:
            # No row may move away from S by more than `cap` of its own norm (max_relative_change).
            update,n=components.cap_rows(update,sb,cap);capped+=n
        q=(sb+update).to(torch.bfloat16)
        if not torch.isfinite(q).all():raise ValueError('候选 BF16 溢出：'+name)
        flat[i:j]=q
        if stats:squares.append(float((q.float()-sb).double().square().sum()));changed+=int((q.float()!=sb).sum())
    return output,{'role':role,'update_sq':math.fsum(squares),'bf16_changed':changed,'elements':output.numel(),'rows_capped':capped}


class RecipeWeights:
    """The weights of a candidate recipe, computed tensor by tensor in RAM, with exactly the values that
    build_candidate would write to disk. Lets the search score a candidate without building its files."""
    def __init__(self,context,recipe):
        self.context,self.recipe=context,canonical_recipe(recipe)
        self.catalogue=context['engine'].local_catalogues.get(context['S_repo']) or {}
    def tensor(self,key):
        return candidate_tensor(self.context,key,self.recipe,stats=False)[0]
    def ranges(self,keys):
        """The local byte ranges these tensors are computed from (S; D, C and, with kappa, B where they are used):
        read-ahead hints for the layer-by-layer scorer."""
        context=self.context;eng=context['engine'];recipe=self.recipe;out=[]
        s=eng.local_catalogues.get(context['S_repo']) or {};b=eng.local_catalogues.get(context['B_repo']) or {}
        for key in keys:
            info=s.get(key)
            if info is None or 'offset' not in info:continue
            out.append((info['file'],info['offset'],info['nbytes']))
            role=core.parameter_role(key,tuple(info['shape']))
            if role=='protected':continue
            if searched_mu(context,key,role,recipe):
                where=context['C'].location(key) if hasattr(context['C'],'location') else None
                if where:out.append(where)
            if not (recipe['lambda_head'] if role=='head' else recipe['lambda_backbone']):continue
            where=context['D'].location(key) if hasattr(context['D'],'location') else None
            if where:out.append(where)
            if recipe['kappa'] and 'offset' in b.get(key,{}):out.append((b[key]['file'],b[key]['offset'],b[key]['nbytes']))
        return out
    def close(self):
        pass


def row_change_cap(context):
    """max_relative_change: how far any row of a changed matrix may move away from S, as a fraction of that row's norm
    in S (None or 0: no limit)."""
    cap=context['settings'].get('max_relative_change',0.1)
    if cap in (None,0,0.):return None
    cap=float(cap)
    if not 0<cap<=1:raise ValueError('max_relative_change 必须在 0 和 1 之间（或 None 表示不限制）')
    return cap


def candidate_storage_settings(context):
    return {'rows_mib':context['settings'].get('rows_mib',64),
            'shard_gib':context['settings'].get('shard_gib',1.),
            'aux_delta':context['D'].fingerprint,'max_relative_change':row_change_cap(context),
            **({'searched_delta':context['C'].fingerprint} if context.get('C') is not None else {}),
            **({'tm_delta':context['N'].fingerprint} if context.get('N') is not None else {}),
            **({'tm2_delta':context['N2'].fingerprint} if context.get('N2') is not None else {})}


def build_candidate(context,recipe,pruning=None):
    """Write the candidate of `recipe` (with the pieces of a component-stage pruning spec taken out) as files."""
    recipe=canonical_recipe(recipe)
    identity={'inputs':context['identity'],'recipe':recipe,'version':core.VERSION,'pipeline_code_sha256':core.sha_file(__file__),
              'storage':'BF16 modified tensors, original dtype for protected/unchanged tensors',
              **candidate_storage_settings(context),
              'torch':str(torch.__version__)}
    pruned=None
    if pruning:
        identity.update(pruning=pruning,components_code_sha256=core.sha_file(components.__file__))
        pruned=ComponentWeights(context,recipe,pruning)
    fp=core.digest(identity);directory=context['scratch']/'candidates'/fp;directory.mkdir(parents=True,exist_ok=True)
    if not context['settings'].get('keep_search_candidates',False):
        # One candidate at a time on the local disk: an earlier one (another recipe, or a previous selection) goes first.
        for other in directory.parent.iterdir():
            if other!=directory and other.is_dir() and ((other/'merge_manifest.json').exists() or any(other.glob('*.receipt.json'))):
                shutil.rmtree(other)
    names=sorted(context['S'].shapes);groups=[];group=[];size=0;limit=int(context['settings'].get('shard_gib',1)*1024**3)
    for name in names:
        n=math.prod(context['S'].shapes[name])*max(2,torch.empty((),dtype=context['engine'].dtype(context['S_repo'],name)).element_size())
        if group and size+n>limit:groups.append(group);group=[];size=0
        group.append(name);size+=n
    if group:groups.append(group)
    records=[];mapping={}
    for index,tensors in enumerate(groups,1):
        fn=f'model-{index:05d}-of-{len(groups):05d}.safetensors';receipt=directory/(fn+'.receipt.json')
        cid={'fingerprint':fp,'index':index,'names':tensors};old=core.load_record(receipt,cid)
        for n in tensors:mapping[n]=fn
        if old is not None:
            path=directory/fn
            if not path.exists() or path.stat().st_size!=old['size'] or runtime.file_sha256(path)!=old['sha256']:raise ValueError('候选分片断点损坏：'+str(path))
            records.extend(old['tensor_stats']);continue
        expected=sum(math.prod(context['S'].shapes[n])*4 for n in tensors)
        if shutil.disk_usage(directory).free<expected+64*1024**2:raise RuntimeError('候选本地分片空间不足')
        if expected+256*1024**2>runtime.effective_ram()*.85:raise RuntimeError('一个输出分片/单个大张量超过保守 RAM 预算；降低 shard_gib 或增加 RAM')
        values={};stats=[]
        for name in tensors:
            value,stat=pruned.tensor_and_stats(name) if pruned is not None else candidate_tensor(context,name,recipe)
            values[name]=value.contiguous();stats.append({'tensor':name,**stat})
        tmp=directory/(fn+'.tmp');save_file(values,str(tmp),metadata={'format':'pt'});tmp.replace(directory/fn)
        del values,value;gc.collect()
        record={'size':(directory/fn).stat().st_size,'sha256':core.sha_file(directory/fn),'tensor_stats':stats}
        runtime.remember_sha256(directory/fn,record['sha256'])   # written and hashed just now; not read again for the fingerprint
        core.save_record(receipt,cid,record);records.extend(stats)
        print(f'构造候选 {fp[:10]}: 分片 {index}/{len(groups)}',flush=True)
    for name in runtime.METADATA_EXACT:
        src=Path(context['metadata_dir'])/name
        if src.is_file():shutil.copy2(src,directory/name)
    templates=Path(context['metadata_dir'])/'chat_templates'
    if templates.exists():shutil.copytree(templates,directory/'chat_templates',dirs_exist_ok=True)
    # Keep S's architecture and special-token pipeline. No draft or inference-speed promises.
    core.atomic_json(directory/'model.safetensors.index.json',{'metadata':{},'weight_map':mapping})
    manifest={'fingerprint':fp,'identity':identity,'complete':True,'source_subject_preserved_metadata':True,
              'quality_status':'candidate_not_accepted','tensor_stats':records,'formula':'S + lambda_g*(D-kappa*(S-B)) + mu*C + nu*N + nu2*N2, D = (M-B) + sum_k c_k*T_k (see aux_mix), N, N2 = cross-architecture T&M rows (see tm_delta, tm2_delta); eta=0',
              'tm_delta':None if not recipe.get('nu') else {'nu':recipe['nu'],'fingerprint':context['N'].fingerprint,
                                                            'repo':context['tm']['repo'],'commit':context['tm']['commit']},
              'tm2_delta':None if not recipe.get('nu2') else {'nu2':recipe['nu2'],'fingerprint':context['N2'].fingerprint,
                                                              'repo':context['tm2']['repo'],'commit':context['tm2']['commit']},'aux_mix':context['identity']['aux_mix'],'aux_delta':context['D'].fingerprint,
              'selection':'global coefficient search, then component selection: decoder pieces of F - S given coefficients in '
                          '[0, 1 + max_boost] from their first-order proxy effect, the head kept or dropped from its exact endpoint effect; '
                          'plans chosen by internal cross-fitted real proxy NLL with a shared activation decomposition (see selection.json); '
                          'every changed row within max_relative_change of S',
              'component_pruning':None if not pruning else {'coefficients':pruning['components'],'decomposition':pruning['decomposition'],
                                                            'tensors':len(pruning['pieces']),'bands':pruning['bands'],'split':pruning.get('split')},
              'max_relative_change':row_change_cap(context),
              'protected_policy':'subject vision/MTP/input embeddings/nonmatrix/control parameters'}
    core.atomic_json(directory/'merge_manifest.json',manifest)
    return directory,manifest


def layer_scorer(context):
    """The layer-by-layer scorer of this session (its empty model is built once)."""
    if context.get('_layer_scorer') is None:
        cat=context['engine'].local_catalogues.get(context['S_repo'])
        if not cat:raise ValueError('分层评分需要 S 的本地副本')
        context['_layer_scorer']=runtime.LayerwiseScorer(context['metadata_dir'],context['settings']['proxy_options'],cat)
    return context['_layer_scorer']


def choose_scoring_engine(context,rows):
    """'layerwise' or 'batched' for this session, decided once.

    proxy_options['engine']: 'auto' (default) checks the layer-by-layer engine against the ordinary batched
    forward on the two shortest items and the longest one, with the real S and this runtime's libraries; it is
    used only when every NLL agrees within 2e-3. The verdict is kept in the record directory. 'layerwise' / 'batched' force a choice.
    """
    if context.get('scoring_engine'):return context['scoring_engine']
    options=context['settings']['proxy_options'];want=options.get('engine','auto')
    if want not in {'auto','layerwise','batched'}:raise ValueError('proxy_options.engine 只能是 auto / layerwise / batched')
    if want!='auto':
        context['scoring_engine']=want;return want
    directory=context['baseline_dir'];fingerprint,_=runtime.checkpoint_fingerprint(directory)
    base={k:v for k,v in options.items() if k!='engine'}
    # Both engines live in qmerge_runtime, whose code is part of runtime_versions(): a change elsewhere does not
    # call for the check again.
    identity={'kind':'scoring_engine_check','engine':runtime.LAYERWISE_VERSION,'subject':fingerprint,'options':base,
              'runtime':runtime.runtime_versions()}
    path=context['root']/'scoring_engine_check.json'
    try:verdict=core.load_record(path,identity)
    except (ValueError,OSError):verdict=None
    if verdict is None:
        tok=runtime.load_tokenizer(context['metadata_dir'])
        lengths={r['id']:core.pack_reference(tok,r,base['template_kwargs'],base['max_tokens'],base['max_response_tokens'])[0].shape[1] for r in rows}
        ordered=sorted(rows,key=lambda r:(lengths[r['id']],r['id']))
        subset=ordered[:2]+[r for r in ordered[-1:] if r not in ordered[:2]]     # the two shortest and the longest
        print(f'分层评分引擎：先在 S 上与普通前向对照 {len(subset)} 条题（最短的两条和最长的一条；只做一次）',flush=True)
        reference=runtime.score_checkpoint(directory,subset,context['root']/'engine_check',{'tag':'S','check':'batched'},
                                           context['metadata_dir'],{**base,'batch_size':len(subset),'estimate_time':False})
        worst=None;reason=''
        try:
            layered=runtime.score_layerwise(layer_scorer(context),runtime.CheckpointWeights(directory),subset,context['root']/'engine_check',
                                            {'tag':'S','check':'layerwise'},{'checkpoint':fingerprint},context['metadata_dir'],
                                            {**base,'estimate_time':False})
            pairs=[(a['nll'],b['nll']) for a,b in zip(reference['records'],layered['records'])]
            worst=max(abs(a-b) for a,b in pairs)
            ok=all(math.isfinite(b) for _,b in pairs) and worst<=2e-3
            reason=f'NLL 最大差 {worst:.1e}'+('' if ok else '，超过 2e-3')
        except Exception as error:   # any failure of the new engine means: use the batched forward
            ok=False;reason=f'{type(error).__name__}: {str(error)[:300]}'
        verdict={'engine':'layerwise' if ok else 'batched','max_nll_difference':worst,'reason':reason,
                 'reference_nll':[r['nll'] for r in reference['records']]}
        core.save_record(path,identity,verdict)
    if verdict['engine']=='layerwise':print(f'分层评分引擎与普通前向一致（{verdict["reason"]}）：搜索中的候选在内存中逐层算出并评分，不再写入磁盘。',flush=True)
    else:print(f'分层评分引擎没有通过对照（{verdict["reason"]}），本次改用原来的分批前向评分。请把这行反馈回来。',flush=True)
    context['scoring_engine']=verdict['engine']
    return verdict['engine']


def layer_identity(context,recipe,index):
    """What the weights of decoder layer `index` of a candidate depend on (the subject itself for S)."""
    mu=float(recipe.get('mu',0.)) if recipe is not None else 0.
    nu=float(recipe.get('nu',0.)) if recipe is not None else 0.
    nu2=float(recipe.get('nu2',0.)) if recipe is not None else 0.
    if recipe is None or (recipe['lambda_backbone']==0 and mu==0 and nu==0 and nu2==0):return 'S'
    extra={'nu':nu,'tm_delta':context['N'].fingerprint} if nu else {}
    if nu2:extra.update(nu2=nu2,tm2_delta=context['N2'].fingerprint)
    if recipe['lambda_backbone']==0:
        # A C-only (or N-only) decoder is independent of kappa, D/B and the backbone group scales, including before canonicalization.
        out={'max_relative_change':row_change_cap(context),**extra}
        if mu:out.update(mu=mu,searched_delta=context['C'].fingerprint)
        return out
    group=0 if index<context['num_layers']/2 else 1
    low,high=recipe['group_scales'][2*group:2*group+2]
    out={'lambda':recipe['lambda_backbone'],'kappa':recipe['kappa'],'scale':[low,high] if low!=high else low,
         'high_fraction':recipe['high_fraction'] if low!=high else None,'aux_delta':context['D'].fingerprint,
         'B':context['identity']['inputs']['B'] if recipe['kappa'] else None,'max_relative_change':row_change_cap(context)}
    if mu:out.update(mu=mu,searched_delta=context['C'].fingerprint)
    out.update(extra)
    return out


def layer_memo(context,rows,recipe):
    """Hidden states after the last layer of each depth group, shared between candidates that agree up to there:
    a candidate that differs from an earlier one only in the output head, or only in the second depth group,
    starts from the stored states instead of from the first layer."""
    if not context['settings'].get('layer_memo',True):return None
    common={'kind':'layer_memo','subject':context['identity']['inputs']['S'],'dataset':core.digest(rows),
            'options':context['settings']['proxy_options'],'runtime':runtime.runtime_versions(),'engine':runtime.LAYERWISE_VERSION,
            'pipeline_code_sha256':core.sha_file(__file__),'components_code_sha256':core.sha_file(components.__file__)}
    layers=context['num_layers'];mid=math.ceil(layers/2)
    keys={}
    for index in sorted({mid-1,layers-1}):
        if index<0:continue
        keys[index]=core.digest({**common,'layers':[layer_identity(context,recipe,k) for k in range(index+1)]})
    return runtime.LayerMemo(context['scratch']/'layer_memo',keys)


def make_room_for_candidate(context):
    """Before the winner is built: free the local disk the search used (layer memo, then the local copy of B)."""
    need=sum(math.prod(v)*2 for v in context['S'].shapes.values())+4*1024**3
    directory=context['scratch'];shutil.rmtree(directory/'layer_memo',ignore_errors=True)
    if shutil.disk_usage(directory).free>=need:return
    eng=context['engine'];brepo=context['B_repo']
    if brepo in eng.local_catalogues:
        local=directory/'source_checkpoints'/('checkpoint-'+core.digest(runtime.source_identity(eng,brepo))[:24])
        eng.local_catalogues.pop(brepo,None);shutil.rmtree(local,ignore_errors=True)
        print('为构造入围候选腾出本地盘：删除 B 的本地副本，构造时 B 按范围读取',flush=True)
    if shutil.disk_usage(directory).free<need:
        raise RuntimeError(f'本地盘放不下入围候选：需要约 {need/1024**3:.1f} GiB，只剩 {shutil.disk_usage(directory).free/1024**3:.1f} GiB')


def coarse_bank(lambdas,kappas,heads,tied=False,lates=(0.,),mus=(0.,)):
    bank={}
    for lam,k,head,late,mu in itertools.product(lambdas,kappas,([0.] if tied else heads),lates,mus):
        r=canonical_recipe({'lambda_backbone':float(lam),'lambda_head':float(head),'kappa':float(k),
                                 'group_scales':[1.,1.,1.,1.],'high_fraction':.2,'rho_late':float(late),'mu':float(mu)})
        bank[core.digest(r)]=r
    baseline=canonical_recipe({'lambda_backbone':0.,'lambda_head':0.,'kappa':0.,'group_scales':[1.,1.,1.,1.],'high_fraction':.2})
    bank[core.digest(baseline)]=baseline
    return list(bank.values())


def is_baseline(recipe):return (recipe['lambda_backbone']==0 and recipe['lambda_head']==0 and not recipe.get('mu',0) and not recipe.get('nu',0)
                               and not recipe.get('nu2',0))



def baseline_recipe():
    return canonical_recipe({'lambda_backbone':0.,'lambda_head':0.,'kappa':0.,'group_scales':[1.,1.,1.,1.],'high_fraction':.2})


def subject_identity(context):
    """Identity of S alone. Its scores do not depend on the donor or the auxiliary mix, so they are
    computed once and stay valid when a source is added later."""
    return {'tag':'S','inputs':{'S':context['identity']['inputs']['S']}}


def trial_identity(context,recipe,rows):
    common={'kind':'proxy_search_trial','recipe':recipe,'dataset':core.digest(rows),'options':context['settings']['proxy_options'],
            'runtime':runtime.runtime_versions(),'version':core.VERSION,'engine':context.get('scoring_engine','batched')}
    if is_baseline(recipe):return {**common,'inputs':subject_identity(context)['inputs']}
    return {**common,'inputs':context['identity'],'candidate_storage':candidate_storage_settings(context),'pipeline_code_sha256':core.sha_file(__file__)}



def evaluate_trial(context,recipe,rows):
    ident=trial_identity(context,recipe,rows);path=context['root']/'search_trials'/(core.digest(ident)+'.json')
    cached=core.load_record(path,ident)
    if cached is not None:return cached
    if context.get('scoring_engine')=='layerwise':
        result=_layerwise_trial(context,recipe,rows)
        core.save_record(path,ident,result)
        return result
    directory=context['baseline_dir'] if is_baseline(recipe) else build_candidate(context,recipe)[0]
    try:
        model=subject_identity(context) if is_baseline(recipe) else {'tag':core.digest(recipe)[:12],'recipe':recipe,'inputs':context['identity']}
        report=runtime.score_checkpoint(directory,rows,context['root']/'proxy_outputs',model,context['metadata_dir'],context['settings']['proxy_options'])
        result={'recipe':recipe,'mean_nll':report['mean_nll'],'losses':[r['nll'] for r in report['records']],
                'category_nll':core.category_means(rows,[r['nll'] for r in report['records']]),
                'checkpoint_fingerprint':report['context']['weights'],'evidence_kind':'forward_proxy'}
        core.save_record(path,ident,result)
        return result
    finally:
        if not is_baseline(recipe) and not context['settings'].get('keep_search_candidates',False):
            expected=context['scratch']/'candidates'
            if directory.parent!=expected or not (directory/'merge_manifest.json').exists():raise ValueError('拒绝删除非本程序候选目录')
            shutil.rmtree(directory)


def _layerwise_trial(context,recipe,rows):
    """One search trial with the layer-by-layer engine: the candidate's weights are computed in RAM layer by layer
    (RecipeWeights, the same values build_candidate writes) and never written to disk. Their tensor digests are
    kept, so that the winner, once built, can be shown to be exactly the weights that were scored (S's digests
    too: a candidate that changes only the output head resumes from S's stored hidden states and their digests)."""
    baseline=is_baseline(recipe)
    if baseline:
        fingerprint,_=runtime.checkpoint_fingerprint(context['baseline_dir'])
        weights=runtime.CheckpointWeights(context['baseline_dir']);wid={'checkpoint':fingerprint};model=subject_identity(context)
    else:
        weights=RecipeWeights(context,recipe);model={'tag':core.digest(recipe)[:12],'recipe':recipe,'inputs':context['identity']}
        wid={'recipe':recipe,'inputs':context['identity'],'candidate_storage':candidate_storage_settings(context),
             'pipeline_code_sha256':core.sha_file(__file__)}
    try:
        report=runtime.score_layerwise(layer_scorer(context),weights,rows,context['root']/'proxy_outputs',model,wid,context['metadata_dir'],
                                       context['settings']['proxy_options'],layer_memo(context,rows,None if baseline else recipe),
                                       want_digests=True,label='S' if baseline else core.digest(recipe)[:10])
    finally:
        weights.close()
    losses=[r['nll'] for r in report['records']]
    return {'recipe':recipe,'mean_nll':report['mean_nll'],'losses':losses,'category_nll':core.category_means(rows,losses),
            'checkpoint_fingerprint':wid.get('checkpoint'),'weights_digest':report['weights_digest'],
            'tensor_digests':report['tensor_digests'],'evidence_kind':'forward_proxy','engine':'layerwise'}


def verify_built_candidate(path,tensor_digests,subject_dir=None):
    """Every tensor of the built candidate files has exactly the bytes that the layer-by-layer trial scored; a
    tensor the forward pass never uses (the vision tower) must have exactly S's bytes (subject_dir).
    Returns the first tensor that differs, else None."""
    weights=runtime.CheckpointWeights(path);subject=runtime.CheckpointWeights(subject_dir) if subject_dir else None
    try:
        for key in sorted(weights.catalogue):
            expected=tensor_digests.get(key)
            if expected is None:
                if subject is None or key not in subject.catalogue:return key+'（评分时没有记录）'
                expected=runtime.tensor_digest(subject.tensor(key))
            if runtime.tensor_digest(weights.tensor(key))!=expected:return key
    finally:
        weights.close()
        if subject is not None:subject.close()
    return None


def proxy_feasible(result,baseline,max_drop):
    return all(result['category_nll'][c]<=baseline['category_nll'][c]+max_drop for c in baseline['category_nll'])



def search_vector(recipe,cfg):
    """Recipe coordinates for the surrogate, each scaled to [0, 1] over its own search range.

    Unscaled, lambda (up to 0.2) and the head coefficient (up to 0.05) are far below the kernel length
    and the surrogate can only tell kappa apart.
    """
    def span(key):return max([abs(float(v)) for v in cfg.get(key,[])]+[0.]) or 1.
    return [recipe['lambda_backbone']/span('lambda_grid'),recipe['lambda_head']/span('head_grid'),recipe['kappa']/span('kappa_grid'),
            *[v/span('group_values') for v in recipe['group_scales']],float(recipe.get('mu',0.))/span('mu_grid')]


def kappa_profile(trials,subject_nll):
    """The best proxy NLL among the scored candidates for each kappa (lambda_backbone > 0 only), and a note when the
    proxy keeps improving as kappa grows. kappa scales -(S - B) inside F - S, i.e. how much of S's own fine-tune is
    taken back; reference answers written by people (scored with thinking off) may simply prefer a model closer to B,
    so a proxy gain that grows with kappa calls for a check with thinking-mode generation."""
    best={}
    for t in trials:
        r=t['recipe']
        if r['lambda_backbone']<=0:continue
        k=float(r['kappa'])
        if k not in best or t['mean_nll']<best[k]['mean_nll']:best[k]={'mean_nll':t['mean_nll'],'recipe':r}
    kappas=sorted(best);values=[best[k]['mean_nll'] for k in kappas]
    profile={'by_kappa':[{'kappa':k,'best_mean_nll':best[k]['mean_nll'],'gain_vs_subject':subject_nll-best[k]['mean_nll'],
                          'lambda_backbone':best[k]['recipe']['lambda_backbone']} for k in kappas],'note':None}
    if len(kappas)>=2 and all(b<a for a,b in zip(values,values[1:])):
        profile['note']=('代理 NLL 随 κ 增大单调下降：κ 越大，F 越多地撤回 S 自己的微调（κ(S−B) 这一项）。不开思考的参考答案可能本来就更接近 B，'
                         '这种改善未必说明模型变好，请用开思考的生成测试核实；也可以把 kappa_grid 收窄到 [0, 0.5] 再比较。')
    return profile


def subject_subspace_diagnostics(stats,energy,coefficients=None):
    """How the first-order effects of the components split between the part of F - S inside the input subspace of S's
    own fine-tune (pieces '.in') and the part outside it ('.out'). energy: squared Frobenius norm per component (None for
    the head). coefficients: the chosen plan's, to split its predicted change as well. A gain concentrated in '.in',
    out of proportion to its share of the delta, means the proxy rewards changing what S itself learned."""
    side=lambda c:c.rsplit('.',1)[-1] if c!='head' and c.rsplit('.',1)[-1] in ('in','out') else None
    names=[c for c in stats if side(c)]
    if not names:return None
    sums={k:{'energy':0.,'helpful':0.,'harmful':0.,'predicted':0.} for k in ('in','out')}
    for c in names:
        k=side(c);e=stats[c]['effect'];sums[k]['energy']+=energy.get(c) or 0.
        if e<0:sums[k]['helpful']-=e
        else:sums[k]['harmful']+=e
        if coefficients and c in coefficients:sums[k]['predicted']+=(coefficients[c]-1.)*e
    def share(key):
        total=sums['in'][key]+sums['out'][key]
        return sums['in'][key]/total if total>0 else None
    out={'inside':sums['in'],'outside':sums['out'],'energy_share_inside':share('energy'),'helpful_share_inside':share('helpful'),
         'harmful_share_inside':share('harmful'),'note':None}
    if coefficients:
        total=sums['in']['predicted']+sums['out']['predicted']
        out['chosen_predicted_change_inside']=sums['in']['predicted'];out['chosen_predicted_change_outside']=sums['out']['predicted']
        out['chosen_predicted_share_inside']=sums['in']['predicted']/total if total<0 and sums['in']['predicted']<0 else None
    e,h=out['energy_share_inside'],out['helpful_share_inside']
    if e is not None and h is not None and h>=0.5 and h>=e+0.3:
        out['note']=(f'S 自己微调子空间之内（.in）的分量只占增量能量的 {e:.0%}，却占一阶有益效果的 {h:.0%}：代理指标的改善有相当部分来自改动 S '
                     '自己学到的方向。请用开思考的生成测试核实，不要只看代理 NLL。')
    return out


def paired_head_recipe(recipe,head_grid,tied):
    """The other value of a two-point head grid on exactly the same canonical recipe, or None."""
    heads=sorted({float(x) for x in head_grid})
    if tied or len(heads)!=2:return None
    other=next((h for h in heads if h!=float(recipe['lambda_head'])),None)
    if other is None:return None
    paired=canonical_recipe({**recipe,'lambda_head':other})
    return None if is_baseline(paired) or paired==recipe else paired


def evaluate_mu_ablation(context,best,records,rows):
    """Score the winner without C, outside the GP budget; adopt it only when feasible and strictly better."""
    cfg=context['settings'];original=best
    if require_all_donors(cfg):return best,None          # mu=0 would leave C without any contribution
    if not cfg.get('mu_ablation',True) or not float(best['recipe'].get('mu',0.)):return best,None
    recipe=canonical_recipe({**best['recipe'],'mu':0.})
    trial=next((r for r in records if r['recipe']==recipe),None)
    if trial is None:
        trial=evaluate_trial(context,recipe,rows);records.append(trial)
    feasible=proxy_feasible(trial,records[0],cfg['proxy_max_category_drop'])
    benefit=trial['mean_nll']-original['mean_nll']       # positive: C lowers NLL; negative: removing C is better
    # An exact-S tie has already been settled by the baseline-first winner ordering before this function is called.
    take=feasible and trial['mean_nll']<original['mean_nll']
    report={'with_mu':{'recipe':original['recipe'],'mean_nll':original['mean_nll']},
            'without_mu':{'recipe':recipe,'mean_nll':trial['mean_nll']},
            'without_mu_feasible':feasible,'c_benefit_nll':benefit,'selected_without_mu':bool(take)}
    print(f'μ=0 配对消融：有 C {original["mean_nll"]:.5f}，无 C {trial["mean_nll"]:.5f}；'
          f'C 的 NLL 改善 {benefit:+.5f}'+('' if feasible else '；无 C 候选有一类超过回退上限')+
          ('；采用无 C 候选' if take else '；保留有 C 候选'),flush=True)
    return (trial if take else original),report


def evaluate_nu_line(context,best,records,rows):
    """After the global search: the winner with N added at each nu of tm_extra['nu_grid'] (outside the GP budget). A
    candidate with N replaces the winner only when it is feasible and its proxy NLL is strictly lower."""
    if context.get('N') is None:return best,None
    cfg=context['settings'];grid=tm_delta.options(cfg)['nu_grid'];base=best;tried=[]
    for nu in grid:
        recipe=canonical_recipe({**base['recipe'],'nu':float(nu)})
        trial=next((r for r in records if r['recipe']==recipe),None)
        if trial is None:
            trial=evaluate_trial(context,recipe,rows);records.append(trial)
        feasible=proxy_feasible(trial,records[0],cfg['proxy_max_category_drop'])
        tried.append({'nu':float(nu),'mean_nll':trial['mean_nll'],'feasible':feasible})
        print(f'ν 线搜索：ν={nu:g}（其余同获胜配方）实际前向 NLL {trial["mean_nll"]:.5f}（不加 N：{base["mean_nll"]:.5f}）'
              +('' if feasible else '；有一类超过回退上限'),flush=True)
    if require_all_donors(cfg):
        # N must contribute: the best nu of the grid is taken even when it does not beat the winner without N.
        good=[(t['mean_nll'],t['nu']) for t in tried if t['feasible']] or [(t['mean_nll'],t['nu']) for t in tried]
        if not any(t['feasible'] for t in tried):print('注意：每个 ν 都有一类比 S 差太多；按 require_all_donors 仍取 NLL 最低的 ν',flush=True)
    else:
        good=[(t['mean_nll'],t['nu']) for t in tried if t['feasible'] and t['mean_nll']<base['mean_nll']]
    choice=min(good)[1] if good else 0.
    chosen=base if not choice else next(r for r in records if r['recipe']==canonical_recipe({**base['recipe'],'nu':choice}))
    report={'without_N':{'recipe':base['recipe'],'mean_nll':base['mean_nll']},'tried':tried,'selected_nu':choice,
            'N':{'fingerprint':context['N'].fingerprint,'repo':context['tm']['repo'],'modules':context['tm'].get('modules')}}
    print(f'ν 线搜索结果：'+(f'采用 ν={choice:g}' if choice else '不加 N（没有 ν 让代理 NLL 更低）')
          +('（require_all_donors：N 必须有贡献，不和“不加 N”比）' if require_all_donors(cfg) else ''),flush=True)
    report['required']=require_all_donors(cfg)
    return chosen,report


def evaluate_nu2_line(context,best,records,rows):
    """After the nu line: the winner (with its nu) plus N2 at each nu2 of tm_extra2['nu_grid']. Coordinate-wise rather than
    a joint nu x nu2 grid: 3 + 3 scores instead of 9 (about 3-6 hours less on the CPU), and N and N2 are row-sparse, small
    terms on mostly different rows, so their interaction is second order. Same choice rule as the nu line."""
    if context.get('N2') is None:return best,None
    cfg=context['settings'];grid=tm_delta2.options(cfg)['nu_grid'];base=best;tried=[]
    for nu2 in grid:
        recipe=canonical_recipe({**base['recipe'],'nu2':float(nu2)})
        trial=next((r for r in records if r['recipe']==recipe),None)
        if trial is None:
            trial=evaluate_trial(context,recipe,rows);records.append(trial)
        feasible=proxy_feasible(trial,records[0],cfg['proxy_max_category_drop'])
        tried.append({'nu':float(nu2),'mean_nll':trial['mean_nll'],'feasible':feasible})
        print(f'ν₂ 线搜索：ν₂={nu2:g}（其余同获胜配方，ν={base["recipe"].get("nu",0):g}）实际前向 NLL {trial["mean_nll"]:.5f}'
              f'（不加 N₂：{base["mean_nll"]:.5f}）'+('' if feasible else '；有一类超过回退上限'),flush=True)
    required=require_all_donors(cfg)
    if required and not any(t['feasible'] for t in tried):print('注意：每个 ν₂ 都有一类比 S 差太多；按 require_all_donors 仍取 NLL 最低的 ν₂',flush=True)
    choice=tm_delta2.pick_nu(tried,base['mean_nll'],required)
    chosen=base if not choice else next(r for r in records if r['recipe']==canonical_recipe({**base['recipe'],'nu2':choice}))
    report={'without_N2':{'recipe':base['recipe'],'mean_nll':base['mean_nll']},'tried':tried,'selected_nu2':choice,'required':required,
            'N2':{'fingerprint':context['N2'].fingerprint,'repo':context['tm2']['repo'],'modules':context['tm2'].get('modules')}}
    print('ν₂ 线搜索结果：'+(f'采用 ν₂={choice:g}' if choice else '不加 N₂（没有 ν₂ 让代理 NLL 更低）')
          +('（require_all_donors：N₂ 必须有贡献）' if required else ''),flush=True)
    return chosen,report


def run_search(context,rows):
    """GP proposes primary recipes within global_trials. A two-point head grid gets an immediate paired head score,
    reusing the same decoder memo and joining subsequent GP fits; an initial winner with mu>0 then gets a recorded
    mu=0 ablation. Component selection follows (choose_candidate)."""
    cfg=context['settings'];core.check_coverage(rows,cfg['search_min_items'],cfg['search_min_per_category'])
    mus=[float(x) for x in cfg.get('mu_grid',[])] if context.get('C') is not None else [0.]
    bank=coarse_bank(cfg['lambda_grid'],cfg['kappa_grid'],cfg['head_grid'],context['tied'],mus=mus)
    remaining=[x for x in bank if not is_baseline(x)];rng=np.random.default_rng(cfg.get('seed',42));rng.shuffle(remaining)
    budget=min(int(cfg['global_trials']),len(remaining))
    how='在内存中逐层算出 BF16 权重（不写盘）并前向评分' if context.get('scoring_engine')=='layerwise' else '构造一遍 BF16 权重并前向评分'
    then='；然后对选中的候选做分量筛选' if cfg.get('component_selection',True) else ''
    pair_enabled=bool(cfg.get('paired_head_trials',True) and not context['tied'] and len({float(x) for x in cfg['head_grid']})==2)
    ablate=bool(cfg.get('mu_ablation',True) and context.get('C') is not None and not require_all_donors(cfg))
    extras=('；另一 head 值就地补评、不占主候选预算' if pair_enabled else '')+('；初始获胜配方含 C 时再做 μ=0 消融' if ablate else '')
    if context.get('N') is not None:extras+=f'；最后在获胜配方上评 ν∈{{{", ".join(f"{x:g}" for x in tm_delta.options(cfg)["nu_grid"])}}}（加跨架构增量 N）'
    if context.get('N2') is not None:extras+=f'；再在其上评 ν₂∈{{{", ".join(f"{x:g}" for x in tm_delta2.options(cfg)["nu_grid"])}}}（加 N₂）'
    print(f'搜索计划：S 对照 1 次 + 最多 {budget} 个 GP 主候选；每个候选{how} {len(rows)} 条{extras}{then}',flush=True)
    records=[evaluate_trial(context,next(x for x in bank if is_baseline(x)),rows)]
    vector=lambda recipe:search_vector(recipe,cfg);main_count=0;paired_count=0
    while remaining and main_count<budget:
        if main_count<3:index=0
        else:
            pred,std=core.gp_predict([vector(r['recipe']) for r in records],[r['mean_nll'] for r in records],[vector(r) for r in remaining])
            index=int(np.argmin(pred-cfg.get('surrogate_exploration',.5)*std))
        recipe=remaining.pop(index);main=evaluate_trial(context,recipe,rows);records.append(main);main_count+=1
        r=main['recipe']
        print(f'全局搜索 {main_count}/{budget}（λ={r["lambda_backbone"]:g}，κ={r["kappa"]:g}，head={r["lambda_head"]:g}，μ={r.get("mu",0):g}）'
              f'实际前向 NLL: {main["mean_nll"]:.5f}（S: {records[0]["mean_nll"]:.5f}）',flush=True)
        paired=paired_head_recipe(recipe,cfg['head_grid'],context['tied']) if pair_enabled else None
        if paired is not None:
            key=core.digest(paired);index=next((i for i,r in enumerate(remaining) if core.digest(r)==key),None)
            if index is not None:
                remaining.pop(index);extra=evaluate_trial(context,paired,rows);records.append(extra);paired_count+=1
                print(f'  head 补评（head={paired["lambda_head"]:g}，其余不变）：NLL {extra["mean_nll"]:.5f}；复用同一主干的隐状态',flush=True)
        core.atomic_json(context['root']/'search_trace.json',{'phase':'global','main_trials':main_count,'paired_head_trials':paired_count,'completed':records})
    feasible=[r for r in records if proxy_feasible(r,records[0],cfg['proxy_max_category_drop'])]
    if require_all_donors(cfg):
        # S is the reference only: the winner is the best candidate in which every donor term has a nonzero coefficient.
        pool=[r for r in records if not is_baseline(r['recipe'])]
        good=[r for r in pool if proxy_feasible(r,records[0],cfg['proxy_max_category_drop'])]
        if not good:print('注意：没有哪个候选在每一类上都不比 S 差太多；按 require_all_donors 仍取 NLL 最低的候选（S 只作对照）',flush=True)
        best=min(good or pool,key=lambda x:x['mean_nll'])
    else:
        best=min(feasible,key=lambda x:(x['mean_nll'],not is_baseline(x['recipe'])))
    best,ablation=evaluate_mu_ablation(context,best,records,rows)
    best,nu_line=evaluate_nu_line(context,best,records,rows)
    best,nu2_line=evaluate_nu2_line(context,best,records,rows)
    core.atomic_json(context['root']/'search_trace.json',{'phase':'nu2_line' if nu2_line is not None else 'nu_line' if nu_line is not None
                        else 'mu_ablation' if ablation is not None else 'complete',
                        'main_trials':main_count,'paired_head_trials':paired_count,'completed':records})
    plain=[t for t in records[1:] if not t['recipe'].get('nu') and not t['recipe'].get('nu2')]   # the diagnostics below are about the global search only
    profile=kappa_profile(plain,records[0]['mean_nll'])
    if profile['by_kappa']:
        print('按 κ 分组的最低代理 NLL：'+'，'.join(f'κ={x["kappa"]:g}: {x["best_mean_nll"]:.5f}' for x in profile['by_kappa'])
              +f'（S: {records[0]["mean_nll"]:.5f}）',flush=True)
    if profile['note']:print('注意：'+profile['note'],flush=True)
    by_mu={}
    for t in plain:
        m=float(t['recipe'].get('mu',0.))
        if m not in by_mu or t['mean_nll']<by_mu[m]:by_mu[m]=t['mean_nll']
    if len(by_mu)>1:print('按 μ 分组的最低代理 NLL：'+'，'.join(f'μ={m:g}: {v:.5f}' for m,v in sorted(by_mu.items())),flush=True)
    scope='GP primary recipes'
    if paired_count:scope+=' plus paired head scores'
    if ablation is not None:scope+=' plus one mu=0 ablation'
    if nu_line is not None:scope+=f' plus {len(nu_line["tried"])} nu scores of the winner with N'
    if nu2_line is not None:scope+=f' plus {len(nu2_line["tried"])} nu2 scores with N2'
    output={'winner':best,'baseline':records[0],'trials':records,'kappa_profile':profile,'main_trials':main_count,
            'paired_head_trials':paired_count,'mu_ablation':ablation,'nu_line':nu_line,'nu2_line':nu2_line,
            'mu_profile':[{'mu':m,'best_mean_nll':v} for m,v in sorted(by_mu.items())],
            'scope':scope+'; actual response NLL; no generation evaluation'}
    core.atomic_json(context['root']/'search_result.json',output)
    return output


def score_subject(context,search):
    """S's forward baseline on the search split: the search's own first trial, cached under S's identity."""
    a=evaluate_trial(context,baseline_recipe(),search)
    return {'search_mean_nll':a['mean_nll']}


# ---------------------------------------------------------------- component selection (replaces the local search)
_LAYER=re.compile(r'(?:^|\.)layers\.(\d+)\.')
RULES=('lenient','point','strict')
COMPONENT_DEFAULTS={'rows':'search+proxy_validation','blocks':8,'bands':[4,16,64],'subject_rank':32,'whiten':True,'input_tokens':8192,
                    'damp':1e-2,'plans':['step:0.5','step:1','strict'],'shrink':1.0,'max_boost':0.1,'confidence':0.9,
                    'crossfit':True,'adopt_probability':0.8,'preflight':True,'bootstrap':2000,'seed':0,'segment':'auto'}


def plan_order(plan):
    """Gentler plans first: F, lenient, the steps by size, point, strict."""
    if plan=='global':return (0,0.)
    if plan.startswith('step:'):return (2,float(plan[5:]))
    return ({'lenient':1,'point':3,'strict':4}[plan],0.)


def plan_label(plan):
    if plan=='global':return 'F（不改分量）'
    if plan.startswith('step:'):return '步长 '+plan[5:]
    return {'lenient':'宽松','point':'点估计','strict':'严格'}[plan]


def component_options(cfg):
    """The component stage's settings with their defaults, checked."""
    given=dict(cfg.get('component_options') or {})
    if set(given)-set(COMPONENT_DEFAULTS):raise ValueError('component_options 里有不认识的项：'+', '.join(sorted(set(given)-set(COMPONENT_DEFAULTS))))
    opts={**COMPONENT_DEFAULTS,**given}
    if opts['rows'] not in {'search','search+proxy_validation'}:raise ValueError('component_options["rows"] 只能是 "search" 或 "search+proxy_validation"')
    opts['bands']=components.check_bands(opts['bands'])
    plans=[]
    for plan in opts['plans']:
        plan=str(plan)
        if plan.startswith('step:'):
            try:t=float(plan[5:])
            except ValueError:t=0.
            if not 0<t<=100:raise ValueError('component_options["plans"]：步长方案写成 "step:正数"，例如 "step:1"')
            plan=f'step:{t:g}'
        elif plan not in RULES:raise ValueError('component_options["plans"] 只能含 "lenient"、"point"、"strict" 和 "step:正数"')
        if plan in plans:raise ValueError('component_options["plans"] 有重复：'+plan)
        plans.append(plan)
    if not plans:raise ValueError('component_options["plans"] 至少要有一个方案')
    opts['plans']=sorted(plans,key=plan_order)
    if not 0.5<float(opts['confidence'])<1:raise ValueError('component_options["confidence"] 必须在 0.5 和 1 之间')
    if not 0<=float(opts['adopt_probability'])<1:raise ValueError('component_options["adopt_probability"] 必须在 0 和 1 之间')
    if not 0<=float(opts['max_boost'])<=1:raise ValueError('component_options["max_boost"] 必须在 0 和 1 之间')
    if not 0<=float(opts['shrink'])<=5:raise ValueError('component_options["shrink"] 必须在 0 和 5 之间')
    if int(opts['subject_rank'])<0:raise ValueError('component_options["subject_rank"] 不能是负数（0 表示不按 S 的子空间再切）')
    for key in ('crossfit','preflight','whiten'):
        if not isinstance(opts[key],bool):raise ValueError(f'component_options["{key}"] 是 True 或 False')
    if int(opts['blocks'])<1 or int(opts['bootstrap'])<100:raise ValueError('component_options：blocks 至少为 1，bootstrap 至少 100 次')
    if opts['segment'] not in (None,'auto') and int(opts['segment'])<1:raise ValueError('component_options["segment"] 是 "auto" 或正整数')
    return opts


def component_extra_rows(settings,search):
    """The questions the component stage uses besides the search set: the proxy validation set, without repeats."""
    if not settings.get('component_selection',True) or component_options(settings)['rows']=='search':return []
    path=settings.get('proxy_validation_file','')
    if not path:
        print('没有 proxy_validation_file：分量筛选只用搜索集的题。',flush=True)
        return []
    seen={r['id'] for r in search}
    return [r for r in core.read_jsonl(path) if r['id'] not in seen]


def component_targets(context,recipe):
    """{key: 'matrix' | 'head'}: the tensors the recipe changes. A decoder-layer matrix is cut into pieces; the output
    head is one piece."""
    out={}
    for key,shape in context['S'].shapes.items():
        role=core.parameter_role(key,tuple(shape))
        if role=='backbone' and (recipe['lambda_backbone']>0 or searched_mu(context,key,role,recipe) or searched_nu(context,key,role,recipe)) and _LAYER.search(key):out[key]='matrix'
        elif role=='head' and recipe['lambda_head']>0:out[key]='head'
    return out


def component_name(key,piece,num_layers,blocks):
    """The component a piece belongs to: depth block x module type x piece (e.g. L08-15.mlp.d1-4.in), or the head."""
    if piece=='all':return 'head'
    block=int(_LAYER.search(key)[1])*blocks//num_layers
    layers=[j for j in range(num_layers) if j*blocks//num_layers==block]
    kind='attn' if '.self_attn.' in key else 'linear' if '.linear_attn.' in key else 'mlp' if '.mlp.' in key else 'other'
    return f'L{layers[0]:02d}-{layers[-1]:02d}.{kind}.{piece}'


def component_effects(grads,num_layers,blocks):
    """Per component: every question's first-order loss change per unit of its coefficient (a NumPy array; keeping it
    rather than dropping it changes the loss by about that much), its pieces as (key, piece), and the squared
    Frobenius norm of those pieces (None for the output head)."""
    n=len(grads['questions']);effects={};members={};energy={}
    for key in sorted(grads['scores']):
        info=grads['decomposition'].get(key)
        for piece,values in sorted(grads['scores'][key].items()):
            if len(values)!=n:raise ValueError(f'分量梯度不完整：{key} {piece}')
            name=component_name(key,piece,num_layers,blocks)
            effects.setdefault(name,np.zeros(n));effects[name]+=np.asarray(values,dtype=float)
            members.setdefault(name,[]).append((key,piece))
            if piece=='all':energy[name]=None
            else:energy[name]=energy.get(name,0.)+float(info['energy'][grads['pieces'].index(piece)])
    return effects,members,energy


def gain_statistics(rows,effects,weights=None,repeats=2000,seed=0):
    """Per component: the first-order change of the proxy objective (category-weighted mean NLL, as in the search)
    from keeping it rather than dropping it (negative: keeping it helps), its stratified-bootstrap standard error and
    95% interval over the questions, and the bootstrap probabilities that keeping it helps (p_help) or hurts (p_harm)."""
    names=sorted(effects);E=np.stack([effects[c] for c in names],axis=1)
    cats=sorted({r['category'] for r in rows});w=np.array([float((weights or {}).get(c,1.)) for c in cats]);w/=w.sum()
    rng=np.random.default_rng(seed);point=np.zeros(len(names));boot=np.zeros((int(repeats),len(names)))
    for c,wc in zip(cats,w):
        idx=np.array([i for i,r in enumerate(rows) if r['category']==c])
        point+=wc*E[idx].mean(axis=0)
        for start in range(0,int(repeats),64):
            n=min(64,int(repeats)-start);sample=rng.choice(idx,size=(n,len(idx)),replace=True)
            boot[start:start+n]+=wc*E[sample].mean(axis=1)
    low,high=np.quantile(boot,[.025,.975],axis=0);se=boot.std(axis=0)
    return {c:{'effect':float(point[j]),'se':float(se[j]),'ci95':[float(low[j]),float(high[j])],'p_help':float((boot[:,j]<0).mean()),
               'p_harm':float((boot[:,j]>0).mean())} for j,c in enumerate(names)}


def change_statistics(rows,changes,weights=None,repeats=2000,seed=0):
    """Category-weighted mean of per-question changes (negative: lower NLL), its stratified-bootstrap 95% interval and
    the bootstrap probability that it is negative."""
    s=gain_statistics(rows,{'change':np.asarray(changes,dtype=float)},weights,repeats,seed)['change']
    return {'change':s['effect'],'ci95':s['ci95'],'p_improve':s['p_help']}


def component_coefficients(stats,plan,opts):
    """{component: coefficient} of one plan, only those that are not 1 (1 keeps a component as it is in F, 0 drops it).

    lenient / point / strict drop components (coefficient 0): those that hurt with probability >= confidence / those
    whose estimated effect is harmful at all / all but those that help with probability >= confidence.
    step:t takes one step along the first-order gradient of the decoder components: their effects are first shrunk
    towards 0 by `shrink` bootstrap standard errors, then c = 1 - t * effect / largest |decoder effect|, clipped to
    [0, 1 + max_boost]. The head is different: its stored effect is the exact endpoint difference L(1)-L(0), not a
    derivative, so it is independently kept (1) or dropped (0) by its sign and never sets the decoder step scale or
    gets amplified. Coefficients are rounded to multiples of 1/1024, which are exact in float32.
    """
    if plan in RULES:
        conf=float(opts['confidence'])
        if plan=='lenient':drop=[c for c,s in stats.items() if s['p_harm']>=conf]
        elif plan=='point':drop=[c for c,s in stats.items() if s['effect']>0]
        else:drop=[c for c,s in stats.items() if s['p_help']<conf]
        return {c:0. for c in sorted(drop)}
    t=float(plan[5:]);k=float(opts['shrink']);ceiling=math.floor((1.+float(opts['max_boost']))*1024)/1024
    shrunk={c:math.copysign(max(0.,abs(s['effect'])-k*s['se']),s['effect']) for c,s in stats.items() if c!='head'}
    top=max((abs(v) for v in shrunk.values()),default=0.)
    out={}
    # A positive endpoint difference means the F head hurts relative to S's head on the same final hidden states.
    if stats.get('head',{}).get('effect',0.)>0:out['head']=0.
    if not top:return out
    for c in sorted(shrunk):
        if not shrunk[c]:continue
        value=min(ceiling,max(0.,round((1.-t*shrunk[c]/top)*1024)/1024))
        if value!=1.:out[c]=value
    return out


def describe_coefficients(coefficients):
    """'dropped / scaled down / amplified' counts of a plan."""
    values=list(coefficients.values())
    return {'dropped':sum(v==0 for v in values),'reduced':sum(0<v<1 for v in values),'amplified':sum(v>1 for v in values)}


def component_folds(rows,search_count,seed):
    """Two cross-fitting folds (0 / 1 per row), balanced within every category and source set (search / proxy
    validation): each such group's questions alternate between the folds in a seeded order, and a group of odd size
    gives its extra question to the other fold than the previous odd group did."""
    groups={}
    for i,r in enumerate(rows):groups.setdefault((r['category'],i<search_count),[]).append(i)
    fold=[0]*len(rows);toggle=0
    for key in sorted(groups):
        order=sorted(groups[key],key=lambda i:hashlib.sha256(f'{int(seed)}|{rows[i]["id"]}'.encode()).hexdigest())
        for n,i in enumerate(order):fold[i]=(n+toggle)%2
        if len(order)%2:toggle^=1
    return fold


def component_store(context,decomposition):
    """Where the decomposition (and, while it runs, the gradient pass) of one base candidate is kept."""
    return context['root']/'component_store'/decomposition[:24]


def pruning_spec(grads,members,coefficients):
    """What ComponentWeights and build_candidate need to apply a plan: the coefficient of every piece that is not 1, by
    tensor, and the decomposition files that hold the pieces."""
    pieces={}
    for name,c in coefficients.items():
        for key,piece in members[name]:pieces.setdefault(key,{})[piece]=c
    return {'decomposition':grads['id'],'bands':grads['bands'],'split':bool(grads.get('split')),
            'components':{k:coefficients[k] for k in sorted(coefficients)},
            'pieces':{k:dict(sorted(v.items())) for k,v in sorted(pieces.items())},
            'files':{k:grads['decomposition'][k]['file'] for k in sorted(pieces) if k in grads['decomposition']}}


class ComponentWeights(RecipeWeights):
    """The weights of a recipe with a coefficient on some pieces of its delta (a plan of the component stage), tensor by
    tensor in RAM, with exactly the values build_candidate writes. A tensor whose pieces all keep coefficient 1 is the
    recipe's own. Every changed row is held to max_relative_change of S's row, like the candidates of the search."""
    def __init__(self,context,recipe,pruning):
        super().__init__(context,recipe)
        self.pruning=pruning;self.store=component_store(context,pruning['decomposition']);self.cap=row_change_cap(context)
        missing=sorted({f for f in pruning['files'].values() if not (self.store/f).is_file()})
        if missing:raise FileNotFoundError(f'分量筛选的分解文件不在 {self.store}（{missing[:3]}）；上传完成之前不能删除这个目录。')
    def tensor(self,key):
        return self.tensor_and_stats(key,stats=False)[0]
    def tensor_and_stats(self,key,stats=True):
        coef=self.pruning['pieces'].get(key)
        if not coef:return candidate_tensor(self.context,key,self.recipe,stats)
        context=self.context;S=context['engine'].tensor(context['S_repo'],key)
        F,_=candidate_tensor(context,key,self.recipe,stats=False);capped=0
        if set(coef)=={'all'}:                          # the output head is one piece: S + c (F - S)
            c=float(coef['all'])
            if c==0:value=S
            else:value,capped=self._finish(S,F,c)
        else:
            part=components.read_decomposition(self.store/self.pruning['files'][key],{key})[key]
            delta=components.combine(F,S,part,self.pruning['bands'],coef)
            if delta is None:value=F
            else:value,capped=self._finish(S,delta,None)
            del part,delta
        del F
        if not stats:return value,None
        squares=0.;changed=0
        flat_v=value.reshape(value.shape[0],-1);flat_s=S.reshape(S.shape[0],-1)
        for i,j in row_intervals(tuple(value.shape),context['settings'].get('rows_mib',64)):
            d=(flat_v[i:j].float()-flat_s[i:j].float()).reshape(-1);squares+=float(torch.dot(d,d));changed+=int((d!=0).sum())
        return value,{'role':core.parameter_role(key,tuple(value.shape)),'update_sq':squares,'bf16_changed':changed,
                      'elements':value.numel(),'rows_capped':capped,'coefficients':coef}
    def _finish(self,S,delta,c):
        """(S + delta, or with c S + c (delta - S) for the head given as F, in bfloat16; rows held to the cap), built row
        block by row block; and the number of rows the cap scaled."""
        out=torch.empty(S.shape,dtype=torch.bfloat16);flat=out.reshape(S.shape[0],-1);capped=0
        flat_s=S.reshape(S.shape[0],-1);flat_d=delta.reshape(delta.shape[0],-1)
        for i,j in row_intervals(tuple(S.shape),self.context['settings'].get('rows_mib',64)):
            s=flat_s[i:j].float()
            d=(flat_d[i:j].float()-s)*c if c is not None else flat_d[i:j]
            if self.cap:
                d,n=components.cap_rows(d,s,self.cap);capped+=n
            flat[i:j]=(s+d).to(torch.bfloat16)
        return out,capped


class SourceTensors:
    """The tensors of one input repository (local copy or ranged reads), by checkpoint key."""
    def __init__(self,context,repo):
        self.engine,self.repo=context['engine'],repo
    def tensor(self,key):
        return self.engine.tensor(self.repo,key)
    def close(self):
        pass


def component_gradients(context,base,rows,opts):
    """Every question's first-order loss change from each piece of the base candidate's delta (qmerge_components),
    with the decomposition kept in component_store. Cached; an interrupted pass resumes where it stopped."""
    keep=('bands','subject_rank','whiten','input_tokens','damp','seed')
    identity={'kind':'component_gradients','version':components.COMPONENTS_VERSION,'recipe':base['recipe'],
              'weights':base['weights_digest'],'inputs':core.digest(context['identity']),'dataset':core.digest(rows),
              'options':{k:opts[k] for k in keep},'proxy_options':context['settings']['proxy_options'],
              'runtime':runtime.runtime_versions(),'components_code_sha256':core.sha_file(components.__file__)}
    key=core.digest(identity);store=component_store(context,key);path=store/'gradients.record.json'
    try:done=core.load_record(path,identity)
    except ValueError:done=None
    if done is not None:return {**done,'id':key}
    if store.parent.is_dir():
        for other in store.parent.iterdir():
            if other!=store and other.is_dir():
                shutil.rmtree(other);print('已删除另一个候选或另一组设置的分量筛选中间结果：',other,flush=True)
    options=context['settings']['proxy_options'];tok=runtime.load_tokenizer(context['metadata_dir']);items=[]
    for r in rows:
        ids,p,_=core.pack_reference(tok,r,options['template_kwargs'],options['max_tokens'],options['max_response_tokens'])
        items.append((r['id'],ids,p))
    targets=component_targets(context,base['recipe'])
    if not targets:raise ValueError('这个配方没有改变任何可以拆分的张量')
    threads,speed=runtime.tune_cpu_threads()
    params=sum(math.prod(context['S'].shapes[k]) for k in targets);tokens=sum(int(x.shape[1]) for _,x,_ in items)
    hours=2*params*tokens/speed/1e9/3600
    print(f'分量筛选：{len(rows)} 条题、{tokens} token。一次前向约 {hours:.1f} 小时（{threads} 线程、float32 约 {speed:.0f} GFLOPS），'
          f'梯度这一步约为 5 次前向的计算量（前向 1 次、分段重算 1 次、逐题反向 3 次），估计 {5*hours:.0f} 小时，跨会话自动续跑',flush=True)
    weights=RecipeWeights(context,base['recipe']);subject=runtime.CheckpointWeights(context['baseline_dir'])
    source=SourceTensors(context,context['B_repo']) if int(opts['subject_rank'])>0 else None
    try:
        result=components.gradient_components(layer_scorer(context),weights,subject,items,targets,store,opts,
                                              label=core.digest(base['recipe'])[:10],expected=base['tensor_digests'],base=source)
    finally:
        weights.close();subject.close()
    core.save_record(path,identity,result)
    return {**result,'id':key}


def evaluate_variant(context,base,pruning,rows,label):
    """The real forward NLL of a variant of the base candidate (a plan's coefficients on its components), on these
    rows. Computed in RAM layer by layer like a search candidate (never written); cached."""
    recipe=base['recipe']
    wid={'recipe':recipe,'inputs':context['identity'],'candidate_storage':candidate_storage_settings(context),'pruning':pruning,
         'pipeline_code_sha256':core.sha_file(__file__),'components_code_sha256':core.sha_file(components.__file__)}
    ident={'kind':'component_variant','dataset':core.digest(rows),'options':context['settings']['proxy_options'],
           'runtime':runtime.runtime_versions(),'version':core.VERSION,'weights':wid}
    path=context['root']/'component_trials'/(core.digest(ident)+'.json')
    try:cached=core.load_record(path,ident)
    except ValueError:cached=None
    if cached is not None:return cached
    weights=ComponentWeights(context,recipe,pruning)
    try:
        report=runtime.score_layerwise(layer_scorer(context),weights,rows,context['root']/'proxy_outputs',
                                       {'tag':label,'recipe':recipe,'pruning':core.digest(pruning),'inputs':context['identity']},wid,
                                       context['metadata_dir'],context['settings']['proxy_options'],None,want_digests=True,label=label)
    finally:
        weights.close()
    losses=[r['nll'] for r in report['records']]
    result={'mean_nll':report['mean_nll'],'losses':losses,'category_nll':core.category_means(rows,losses),
            'weights_digest':report['weights_digest'],'tensor_digests':report['tensor_digests'],'engine':'layerwise'}
    core.save_record(path,ident,result)
    return result


def subject_losses(context,search,extra):
    """S's NLL on the component rows: the search set's from the search itself, the extra rows scored once (cached)."""
    losses=list(evaluate_trial(context,baseline_recipe(),search)['losses'])
    if extra:losses+=list(evaluate_trial(context,baseline_recipe(),extra)['losses'])
    return losses


def component_base(context,search_result):
    """The candidate the component stage starts from: the search winner; when S itself scored best, the best of the
    other candidates (part of its delta may still help)."""
    winner=search_result['winner']
    if not is_baseline(winner['recipe']):return winner
    others=[t for t in search_result['trials'] if not is_baseline(t['recipe'])]
    if not others:return None
    feasible=[t for t in others if proxy_feasible(t,search_result['baseline'],context['settings']['proxy_max_category_drop'])]
    return min(feasible or others,key=lambda t:t['mean_nll'])


def backward_check_identity(context,fingerprint):
    return {'kind':'backward_check','version':components.BACKWARD_CHECK_VERSION,'subject':fingerprint,
            'options':context['settings']['proxy_options'],'runtime':runtime.runtime_versions(),
            'components_code_sha256':core.sha_file(components.__file__)}


def backward_summary(record):
    """The short form of a backward-check record for the run status."""
    if record is None:return None
    out={'ok':bool(record.get('ok')),'question':record.get('question')}
    for key in ('error','problems'):
        if record.get(key):out[key]=record[key]
    out['checks']=[{k:c.get(k) for k in ('kind','direction','predicted','actual','ok')} for c in record.get('checks',[])]
    return out


def check_backward(context,rows):
    """components.backward_check on the real S, once per S, options and libraries; the record is kept in the record
    directory. Returns it, or None when it is switched off (backward_check=False, component_selection=False) or the
    batched scoring engine is in use (the component stage is skipped then anyway).

    A failure of the gradient path itself is recorded as ok=False, and the component stage is then skipped before its
    long passes. OSError and MemoryError are not recorded: they say nothing about the path, and the next run tries again.
    """
    cfg=context['settings']
    if not (cfg.get('backward_check',True) and cfg.get('component_selection',True)):return None
    if context.get('scoring_engine')!='layerwise':return None
    fingerprint,_=runtime.checkpoint_fingerprint(context['baseline_dir'])
    identity=backward_check_identity(context,fingerprint);path=context['root']/'backward_check.json'
    try:record=core.load_record(path,identity)
    except ValueError:record=None
    if record is not None:return record
    options=cfg['proxy_options'];tok=runtime.load_tokenizer(context['metadata_dir'])
    packed={r['id']:core.pack_reference(tok,r,options['template_kwargs'],options['max_tokens'],options['max_response_tokens']) for r in rows}
    pool=[r for r in rows if packed[r['id']][2]['response_tokens']>=8] or list(rows)
    row=min(pool,key=lambda r:(packed[r['id']][0].shape[1],r['id']))
    # S's NLL for this question from the scoring engine (cached by the search or by the baseline run).
    trial=evaluate_trial(context,baseline_recipe(),rows)
    expected=trial['losses'][[r['id'] for r in rows].index(row['id'])]
    directions={}
    delta=context.get('D')
    if delta is not None:
        def merge_step(key,delta=delta):
            # A step the size the merge itself takes: lambda = 0.1 of the aux delta D of this matrix.
            if not delta.has(key):return None
            shape=tuple(context['S'].shapes[key])
            return 0.1*delta.rows(key,0,shape[0]).reshape(shape)
        directions['0.1xD']=merge_step
    print(f'反向自检：在真实 S 上，用题 {row["id"]}（{packed[row["id"]][0].shape[1]} token）检查分量筛选要用的梯度'
          '（最后的线性注意力层和全注意力层各一个矩阵；约 15–40 分钟，只做一次）',flush=True)
    weights=runtime.CheckpointWeights(context['baseline_dir'])
    try:
        record=components.backward_check(layer_scorer(context),weights,(row['id'],packed[row['id']][0],packed[row['id']][1]),
                                         directions,expected_nll=expected)
    except (OSError,MemoryError):
        raise
    except Exception as error:
        if runtime._out_of_memory(error):raise
        record={'version':components.BACKWARD_CHECK_VERSION,'ok':False,'question':row['id'],'checks':[],
                'error':f'{type(error).__name__}: {str(error)[:400]}'}
    finally:
        weights.close()
    core.save_record(path,identity,record)
    if record['ok']:
        print('反向自检通过：梯度与真实前向的中心差分吻合。之后的分量筛选可以照常进行。',flush=True)
    else:
        why=record.get('error') or '；'.join(record.get('problems') or [])
        bad=[c for c in record.get('checks',[]) if c.get('decisive',True) and not c.get('ok')]
        if bad:why=(why+'；' if why else '')+'；'.join(f'{c["kind"]} 沿 {c["direction"]}：实际 {c.get("actual")}，预测 {c.get("predicted")}' for c in bad)
        print(f'反向自检没有通过（{why}）。之后的分量筛选会直接跳过，用全局搜索选中的候选。请把这行和 backward_check.json 反馈回来；'
              '确认这项检查本身有误时，可设 SETTINGS["backward_check"]=False 跳过它。',flush=True)
    return record


def component_stage(context,base,search,extra):
    """Component selection, in place of the old NPS-style local search.

    The delta of the base candidate F (the search's choice) against S is cut into orthogonal pieces: per matrix,
    bands of its singular directions whitened by the matrix's actual inputs, and the rest, each cut again into the part
    inside and outside the input subspace of S's own fine-tune S - B; the pieces are pooled into components (depth
    block x module type x piece, and the output head). One forward and one backward pass of F over the component rows
    give every question's first-order loss change per unit of each component's coefficient, and a stratified bootstrap
    over the questions gives each effect's standard error and its probability of helping or hurting.

    A plan turns those into coefficients: the rules lenient / point / strict drop components (coefficient 0), and
    step:t moves decoder coefficients one step along the shrunk first-order gradient, within [0, 1 + max_boost]. The
    head instead uses its exact endpoint difference for an independent keep/drop decision and never sets the decoder
    step scale. With crossfit, the rows are split into two folds balanced by category and source set: a fold's plan is
    made without the other fold's loss/gradient statistics and scored by real forward NLL on that other fold, then
    vice versa. All questions still share the teacher-forced activation covariance decomposition, so the folds are not
    independent; this is an internal cross-fitted estimate, with plan-selection and global-search bias. The plan with
    the largest cross-fitted improvement (category limits against S as in the search, improvement probability >=
    adopt_probability) is then made from all rows and scored on all rows; it is kept only if it is also better than F
    there. Without crossfit, every plan from all rows is scored on all rows and the best is kept. Returns
    {'status': 'done' | 'skipped' | 'failed', ...}.
    """
    cfg=context['settings'];opts=component_options(cfg);rows=list(search)+list(extra);n=len(rows)
    if context.get('scoring_engine')!='layerwise':
        return {'status':'skipped','reason':'分量筛选要用逐层评分引擎，这次用的是分批前向'}
    if not base.get('tensor_digests') or not base.get('weights_digest'):
        return {'status':'skipped','reason':'全局搜索选中的候选没有记录权重摘要'}
    # The gradient path, checked on the real S before the hours of forward and backward passes below (cached when the
    # waiting run already did it).
    try:check=check_backward(context,search)
    except (OSError,MemoryError):raise
    except Exception as error:
        # Only the setup around the check failed (the check itself records its own failures): go on, the in-stage
        # preflight still guards the long passes.
        if cfg.get('component_on_error','fallback')=='raise':raise
        print(f'反向自检没能开始（{type(error).__name__}: {str(error)[:200]}）；继续分量筛选，由其中的梯度自检把关。',flush=True)
        check=None
    if check is not None and not check.get('ok'):
        reason='反向自检没有通过（见 backward_check.json）'
        if cfg.get('component_on_error','fallback')=='raise':raise RuntimeError(reason)
        print(reason+'：跳过分量筛选，用全局搜索选中的候选。',flush=True)
        return {'status':'skipped','reason':reason,'backward_check':backward_summary(check)}
    started=time.perf_counter();weights=cfg['proxy_options'].get('category_weights');max_drop=cfg['proxy_max_category_drop']
    sets=['search']+(['proxy_validation'] if extra else [])
    print(f'分量筛选：从全局搜索选中的候选（{json.dumps(base["recipe"],sort_keys=True)}）出发，用 {n} 条题'
          f'（搜索集 {len(search)}'+(f' + 代理验证集 {len(extra)}' if extra else '')+'）',flush=True)
    try:
        s_losses=subject_losses(context,search,extra)
        grads=component_gradients(context,base,rows,opts)
        if grads['questions']!=[r['id'] for r in rows]:raise ValueError('分量梯度的题目与本次不符')
        missing=[k for k,v in component_targets(context,base['recipe']).items() if k not in grads['scores']]
        if missing:raise ValueError('有张量没有分量梯度：'+', '.join(missing[:3]))
        f_losses=[float(x) for x in grads['nll']]
        # The pass's own forward must reproduce the search's scores of F on the search rows.
        worst=max(abs(a-b) for a,b in zip(f_losses[:len(search)],base['losses']))
        if not worst<=1e-3:raise RuntimeError(f'梯度这一步前向得到的 F 的 NLL 与搜索时不一致（最大差 {worst:.1e}）')
        subject={'mean_nll':core.weighted_mean(rows,s_losses,weights),'category_nll':core.category_means(rows,s_losses),'losses':s_losses}
        full={'name':'global','mean_nll':core.weighted_mean(rows,f_losses,weights),'category_nll':core.category_means(rows,f_losses),
              'losses':f_losses,'coefficients':{},'predicted_change':0.,'tensor_digests':base['tensor_digests']}
        effects,members,energy=component_effects(grads,context['num_layers'],int(opts['blocks']))
        count=len(effects);repeats=int(opts['bootstrap']);seed=int(opts['seed'])
        stats=gain_statistics(rows,effects,weights,repeats,seed)
        plans={p:component_coefficients(stats,p,opts) for p in opts['plans']}
        # require_all_donors: every donor term must keep some contribution after the component stage. D (its decoder part)
        # and C touch every decoder matrix the recipe changes; N only its own rows. A plan that sets to 0 every component
        # holding a term's matrices is not a candidate.
        decoder=[c for c in effects if c!='head']
        groups={'D/C':set(decoder)}
        if float(base['recipe'].get('nu',0) or 0) and context.get('N') is not None:
            groups['N']={c for c in decoder if any(context['N'].has(key) for key,_ in members[c])}
        if float(base['recipe'].get('nu2',0) or 0) and context.get('N2') is not None:
            groups['N2']={c for c in decoder if any(context['N2'].has(key) for key,_ in members[c])}
        def drops_decoder(coefficients):
            if not require_all_donors(cfg):return False
            return any(names and all(float(coefficients.get(c,1.))==0. for c in names) for names in groups.values())
        for p in [p for p in opts['plans'] if drops_decoder(plans[p])]:
            print(f'  {plan_label(p)}会让某个供体项（'+'、'.join(g for g,names in groups.items() if names and all(float(plans[p].get(c,1.))==0. for c in names))
                  +'）完全没有贡献；按 require_all_donors 不考虑这个方案',flush=True)
        print(f'分量筛选：{count} 个分量；按全部题，'+'；'.join(
            f'{plan_label(p)}去掉 {d["dropped"]}、缩小 {d["reduced"]}、放大 {d["amplified"]} 个'
            for p in opts['plans'] for d in [describe_coefficients(plans[p])]),flush=True)
        tag=core.digest(base['recipe'])[:6];scored={}
        def losses_of(coefficients,idx,label):
            """The real NLL on rows idx of F with these component coefficients (F itself and S need no scoring), and the
            variant's record when it had to be scored."""
            if not coefficients:return [f_losses[i] for i in idx],None
            if len(coefficients)==count and not any(coefficients.values()):return [s_losses[i] for i in idx],None
            key=(tuple(sorted(coefficients.items())),tuple(idx))
            if key not in scored:
                spec=pruning_spec(grads,members,coefficients)
                scored[key]=({**evaluate_variant(context,base,spec,[rows[i] for i in idx],label),'pruning':spec})
            return scored[key]['losses'],scored[key]
        def predicted(coefficients,i):
            return sum((c-1.)*effects[name][i] for name,c in coefficients.items())
        crossfit={};fold=None
        if opts['crossfit']:
            fold=component_folds(rows,len(search),seed);halves=[[i for i in range(n) if fold[i]==f] for f in (0,1)]
            print(f'内部交叉拟合：两折各 {len(halves[0])} / {len(halves[1])} 条题；每折方案不用另一折的损失/梯度统计，在另一折用真实 NLL 评分，再反过来'
                  f'（全部题共享激活协方差分解，折不完全独立；最多 {2*len(opts["plans"])} 次半份评分，相同或等于 F / S 的不重复评；'
                  '之后选中的方案再在全部题上评一次）',flush=True)
            for p in opts['plans']:
                diffs=np.zeros(n);forecast=np.zeros(n);sizes=[]
                for f in (0,1):
                    own,held=halves[f],halves[1-f]
                    part=gain_statistics([rows[i] for i in own],{c:e[own] for c,e in effects.items()},weights,repeats,seed+1+f)
                    coefficients=component_coefficients(part,p,opts);sizes.append(describe_coefficients(coefficients))
                    losses,_=losses_of(coefficients,held,f'{tag}-{p}-cf{f+1}')
                    for i,value in zip(held,losses):
                        diffs[i]=value-f_losses[i];forecast[i]=predicted(coefficients,i)
                category=core.category_means(rows,diffs.tolist())
                crossfit[p]={**change_statistics(rows,diffs,weights,repeats,seed),
                             'predicted_change':core.weighted_mean(rows,forecast.tolist(),weights),'category_change':category,
                             'feasible':all(full['category_nll'][c]+category[c]<=subject['category_nll'][c]+max_drop for c in category),
                             'half_plans':sizes,'full_plan':describe_coefficients(plans[p]),'per_question':diffs.tolist()}
                cf=crossfit[p]
                print(f'  {plan_label(p)}：在折外题上比 F {cf["change"]:+.5f}（95% 区间 {cf["ci95"][0]:+.5f} ~ {cf["ci95"][1]:+.5f}，'
                      f'变好的概率 {cf["p_improve"]:.2f}；一阶预测 {cf["predicted_change"]:+.5f}）'+('' if cf['feasible'] else '，有一类比 S 差太多'),flush=True)
            usable=[p for p in opts['plans'] if plans[p] and crossfit[p]['feasible'] and crossfit[p]['change']<0
                    and not (len(plans[p])==count and not any(plans[p].values())) and not drops_decoder(plans[p])
                    and crossfit[p]['p_improve']>=float(opts['adopt_probability'])]
            picks=sorted(usable,key=lambda p:(crossfit[p]['change'],plan_order(p)))[:1]
            if not picks:print('交叉拟合：没有哪个方案在折外题上比 F 好（或够把握），保留 F 的全部增量',flush=True)
        else:
            picks=[]
            for p in opts['plans']:
                if plans[p] and not (len(plans[p])==count and not any(plans[p].values())) and not drops_decoder(plans[p]) and all(plans[p]!=plans[q] for q in picks):picks.append(p)
        variants=[full]
        for p in picks:
            losses,result=losses_of(plans[p],list(range(n)),f'{tag}-{p}')
            variants.append({'name':p,'mean_nll':result['mean_nll'],'category_nll':result['category_nll'],'losses':losses,
                             'coefficients':plans[p],'predicted_change':sum((c-1.)*stats[name]['effect'] for name,c in plans[p].items()),
                             'pruning':result['pruning'],'tensor_digests':result['tensor_digests']})
        for v in variants:
            v['feasible']=proxy_feasible(v,subject,max_drop);v['change']=v['mean_nll']-full['mean_nll']
            d=describe_coefficients(v['coefficients'])
            print(f'  {plan_label(v["name"])}：去掉 {d["dropped"]}、缩小 {d["reduced"]}、放大 {d["amplified"]} 个分量，全部 {n} 题上 NLL {v["mean_nll"]:.5f}'
                  f'（比 F {v["change"]:+.5f}，一阶预测 {v["predicted_change"]:+.5f}；S {subject["mean_nll"]:.5f}）'+('' if v['feasible'] else '，有一类比 S 差太多'),flush=True)
        if opts['crossfit']:
            # The plan was chosen by internal cross-fitting, with a shared all-row activation decomposition. Made from
            # all questions, it must also be better than F on them, or the transfer failed and F is kept.
            better=[v for v in variants[1:] if v['feasible'] and v['change']<0]
            if variants[1:] and not better:print('按全部题定出的系数反而不比 F 好，保留 F 的全部增量',flush=True)
            chosen=better[0] if better else full
        else:
            chosen=min([v for v in variants if v['feasible']] or variants,key=lambda v:(v['mean_nll'],plan_order(v['name'])))
        heldout=None
        if extra:
            # The global search did not use these rows, but this component stage did. These remain internal proxy
            # comparisons, not an untouched final validation; cross-fitting also shares the activation decomposition.
            pv=list(range(len(search),n));f_vs_s=np.asarray([f_losses[i]-s_losses[i] for i in pv])
            heldout={'rows':len(pv),'F_vs_S':change_statistics(list(extra),f_vs_s,weights,repeats,seed)}
            if chosen['name'] in crossfit:
                d=np.asarray([crossfit[chosen['name']]['per_question'][i] for i in pv])
                heldout['chosen_vs_S']=change_statistics(list(extra),f_vs_s+d,weights,repeats,seed)
        total=sum(x for x in energy.values() if x is not None) or 1.
        table=[]
        for c in sorted(stats):
            layers,kind,piece=c.split('.',2) if c!='head' else ('','head','all')
            table.append({'component':c,'layers':layers,'module':kind,'piece':piece,'pieces':len(members[c]),
                          'energy_share':None if energy[c] is None else energy[c]/total,**stats[c],
                          'coefficients':{p:plans[p].get(c,1.) for p in opts['plans']}})
        brief={p:{k:v for k,v in cf.items() if k!='per_question'} for p,cf in crossfit.items()}
        diagnostics=subject_subspace_diagnostics(stats,energy,chosen['coefficients'])
        if diagnostics is not None:
            e,h=diagnostics['energy_share_inside'],diagnostics['helpful_share_inside']
            print('S 自己微调子空间之内（.in）的分量：占增量能量 '+('-' if e is None else f'{e:.0%}')+'，占一阶有益效果 '
                  +('-' if h is None else f'{h:.0%}'),flush=True)
            if diagnostics['note']:print('注意：'+diagnostics['note'],flush=True)
        report={'version':components.COMPONENTS_VERSION,'base_recipe':base['recipe'],'decomposition':grads['id'],
                'rows':{'total':n,'search':len(search),'proxy_validation':len(extra),
                        'by_category':{c:sum(r['category']==c for r in rows) for c in sorted({r['category'] for r in rows})},
                        'fold_sizes':None if fold is None else [fold.count(0),fold.count(1)]},
                'options':opts,'max_relative_change':row_change_cap(context),'preflight':grads.get('preflight'),
                'subject':{k:subject[k] for k in ('mean_nll','category_nll')},'crossfit':brief,'heldout':heldout,
                'variants':[{'name':v['name'],'mean_nll':v['mean_nll'],'change':v['change'],'predicted_change':v['predicted_change'],
                             'category_nll':v['category_nll'],'feasible':v['feasible'],**describe_coefficients(v['coefficients'])} for v in variants],
                'chosen':chosen['name'],'chosen_coefficients':chosen['coefficients'],'components':table,'seconds':time.perf_counter()-started,
                'subject_subspace':diagnostics,
                'meaning':'effect: first-order change of the proxy objective (category-weighted mean NLL) per unit of the component\'s '
                          'coefficient at F (keeping it instead of dropping it); negative means it helps. The head row is measured exactly '
                          '(S\'s head on F\'s final hidden states). coefficients: 1 keeps a component as in F, 0 drops it; the head is only '
                          'kept or dropped, while decoder coefficients are bounded by 1 + max_boost. crossfit: for each plan, the real NLL '
                          'change against F of coefficients made without the other fold\'s loss/gradient statistics and scored on that fold, '
                          'pooled over both folds (negative is better). All rows share the activation covariance decomposition, so this is '
                          'an internal cross-fitted estimate, not independent validation. heldout: internal comparisons against S on proxy '
                          'validation rows, which the global coefficient search did not use but this component stage did. variants: plans '
                          'from all rows, scored on all rows. '
                          'subject_subspace: the components inside / outside the input subspace of S\'s own fine-tune S - B, their share '
                          'of the delta\'s energy and of the first-order helpful effect; a helpful share far above the energy share means '
                          'the proxy gain comes from changing what S itself learned.'}
        core.atomic_json(context['root']/'component_selection.json',report)
        d=describe_coefficients(chosen['coefficients'])
        print(f'分量筛选选中 {plan_label(chosen["name"])}'+(f'（去掉 {d["dropped"]}、缩小 {d["reduced"]}、放大 {d["amplified"]} 个分量）' if chosen['coefficients'] else '')+
              f'，用时 {(time.perf_counter()-started)/3600:.1f} 小时。逐个分量的估计见 component_selection.json',flush=True)
        summary={'status':'done','chosen_variant':chosen['name'],'rows':n,'components':count,'coefficients':chosen['coefficients'],
                 'changed':describe_coefficients(chosen['coefficients']),'crossfit':brief,'heldout':heldout,
                 'preflight':grads.get('preflight'),'subject_subspace':diagnostics,
                 'variants':[{k:v[k] for k in ('name','mean_nll','change','predicted_change','feasible')}|describe_coefficients(v['coefficients']) for v in variants]}
        return {**summary,'chosen':chosen,'subject':subject,'sets':sets}
    except (OSError,MemoryError,components.WeightsChanged):
        raise
    except Exception as error:
        if cfg.get('component_on_error','fallback')=='raise' or runtime._out_of_memory(error):raise
        reason=f'{type(error).__name__}: {str(error)[:300]}'
        print(f'分量筛选没有完成（{reason}）。这次用全局搜索选中的候选，不改任何分量。请把这行反馈回来；'
              '想让它报错停下而不是跳过，设 SETTINGS["component_on_error"]="raise"。',flush=True)
        return {'status':'failed','reason':reason}


# ---- the chosen candidate on the record directory (only when it is not uploaded)
def persist_candidate(context,directory):
    """Copy the chosen candidate to the record directory, so that it outlives this session's local disk.
    Returns its weights fingerprint."""
    directory=Path(directory).resolve();weights,hashes=runtime.checkpoint_fingerprint(directory)
    store=context['root']/'selected_candidate';target=store/weights[:24];identity={'kind':'persisted_candidate','weights':weights}
    names=dict(hashes)
    if (directory/'merge_manifest.json').is_file():names['merge_manifest.json']=core.sha_file(directory/'merge_manifest.json')
    sizes={n:(directory/n).stat().st_size for n in names}
    def present(n):return (target/n).is_file() and (target/n).stat().st_size==sizes[n]
    try:done=core.load_record(target/'persist_receipt.json',identity)
    except (ValueError,OSError):done=None
    if done is not None and done['sizes']==sizes and all(present(n) for n in sizes):return weights
    if store.is_dir():
        for other in store.iterdir():                       # one saved candidate at a time
            if other!=target and other.is_dir() and (other/'persist_receipt.json').is_file():
                shutil.rmtree(other);print('已删除被新候选取代的旧候选副本：',other,flush=True)
    target.mkdir(parents=True,exist_ok=True)
    need=sum(sizes[n] for n in sizes if not present(n));free=shutil.disk_usage(target).free
    if free<need+1024**3:
        raise RuntimeError(f'把候选存到 {target} 还需约 {need/1024**3:.1f} GiB，该盘只剩 {free/1024**3:.1f} GiB。'
                           '腾出空间后重跑本格；搜索记录都在，只需重建这一个候选。')
    print(f'保存候选到 {target}（{need/1024**3:.1f} GiB）',flush=True);started=time.perf_counter()
    shas=copy_many([(n,directory/n,target/n) for n in sorted(sizes) if not present(n)],context['settings'].get('copy_workers',4))
    for n,sha in shas.items():
        if sha!=names[n]:
            (target/n).unlink();raise ValueError('候选文件在保存过程中改变：'+n)
    core.save_record(target/'persist_receipt.json',identity,{'sizes':sizes,'sha256':names})
    print(f'候选已保存，用时 {(time.perf_counter()-started)/60:.1f} 分钟',flush=True)
    return weights


def selection_identity(context,datasets):
    return {'kind':'search_selection','inputs':core.digest(context['identity']),'datasets':datasets}


def load_selection(context,datasets):
    try:return core.load_record(context['root']/'selection_record.json',selection_identity(context,datasets))
    except ValueError:return None


def selection_settings(context):
    """Everything on our side that decides which candidate is built."""
    cfg=context['settings']
    keys=('lambda_grid','kappa_grid','head_grid','mu_grid','global_trials','surrogate_exploration','seed',
          'search_min_items','search_min_per_category','proxy_max_category_drop','upload_when_no_gain',
          'component_selection','component_on_error','backward_check','paired_head_trials','mu_ablation','require_all_donors')
    tm={'nu_grid':tm_delta.options(cfg)['nu_grid'] if context.get('N') is not None else None,
        'status':(context.get('tm') or {}).get('status')}
    tm2={'nu_grid':tm_delta2.options(cfg)['nu_grid'] if context.get('N2') is not None else None,
         'status':(context.get('tm2') or {}).get('status')}
    return core.digest({'settings':{k:cfg.get(k) for k in keys},'tm':tm,'tm2':tm2,'component_options':component_options(cfg),
                        'proxy_options':cfg.get('_proxy_options_as_configured',cfg['proxy_options']),
                        'storage':candidate_storage_settings(context),'pipeline_code_sha256':core.sha_file(__file__),
                        'components_code_sha256':core.sha_file(components.__file__)})


def choose_candidate(context,search_result,search,extra=()):
    """The candidate to build: the search's choice, refined by the component stage. When nothing scores better than
    S on the proxy, the best candidate is still chosen if upload_when_no_gain, so that it can be judged elsewhere."""
    cfg=context['settings'];baseline=search_result['baseline'];base=component_base(context,search_result)
    if base is None:
        return {'is_baseline':True,'recipe':baseline['recipe'],'reason':'no candidate besides S was scored'}
    stage={'status':'off','reason':'component_selection=False'}
    if cfg.get('component_selection',True):stage=component_stage(context,base,search,list(extra))
    if stage['status']=='done':chosen,subject,sets=stage['chosen'],stage['subject'],stage['sets']
    else:
        chosen={'name':'global','mean_nll':base['mean_nll'],'category_nll':base['category_nll'],'losses':base['losses'],
                'tensor_digests':base.get('tensor_digests')}
        subject,sets=baseline,['search']
    summary={k:v for k,v in stage.items() if k not in ('chosen','subject','sets')}
    beats=chosen['mean_nll']<subject['mean_nll'] and proxy_feasible(chosen,subject,cfg['proxy_max_category_drop'])
    if not beats:
        if not cfg.get('upload_when_no_gain',True):
            return {'is_baseline':True,'recipe':base['recipe'],'reason':'no candidate scored better than S on the proxy','components':summary}
        print(f'没有候选的代理 NLL 比 S 低（S {subject["mean_nll"]:.5f}，选中的候选 {chosen["mean_nll"]:.5f}）。'
              '按 upload_when_no_gain=True 仍然构造并上传它，由你自己的测试来判断。',flush=True)
    pruning=chosen.get('pruning')
    return {'is_baseline':False,'recipe':base['recipe'],'variant':chosen['name'],'pruning':pruning,
            'proxy_mean_nll':chosen['mean_nll'],'subject_proxy_mean_nll':subject['mean_nll'],
            'gain_nll':subject['mean_nll']-chosen['mean_nll'],'beats_subject':beats,
            'category_nll':chosen['category_nll'],'subject_category_nll':subject['category_nll'],
            'items':len(chosen['losses']),'proxy_sets':sets,
            'search_mean_nll':base['mean_nll'],'subject_search_mean_nll':baseline['mean_nll'],'components':summary,
            'kappa_profile':search_result.get('kappa_profile'),'mu_ablation':search_result.get('mu_ablation'),
            'nu_line':search_result.get('nu_line'),'tm':tm_summary(context),
            'nu2_line':search_result.get('nu2_line'),'tm2':tm_summary(context,'tm2'),
            'engine':base.get('engine','batched'),'tensor_digests':chosen.get('tensor_digests'),
            'checkpoint_fingerprint':None if pruning else base.get('checkpoint_fingerprint')}


def tm_summary(context,key='tm'):
    """What the selection record and the model card say about a cross-architecture step (key 'tm': N, 'tm2': N2)."""
    tm=context.get(key)
    if not tm:return None
    out={k:tm.get(k) for k in ('status','repo','commit','modules','alpha','topk','reason','error','k_proj','same_tokenizer','executions')
         if k in tm or k in ('status','repo','commit','modules','alpha','topk','reason','error')}
    out['verdict']={m:{k:v.get(k) for k in ('verdict','z_ratio','sharp_ratio')} for m,v in (tm.get('verdict') or {}).items()}
    if tm.get('summary'):out['summary']={k:tm['summary'].get(k) for k in ('tensors','mean_dW','mean_inject_ratio','raw_fallback_tensors','norm')}
    if tm.get('fingerprint'):out['fingerprint']=tm['fingerprint']
    return out


def build_chosen(context,record):
    """Build the chosen candidate on local disk, checked to be exactly the weights that were scored."""
    if 'baseline_dir' not in context:setup_local(context)
    if context.get('scoring_engine')=='layerwise':make_room_for_candidate(context)
    path,manifest=build_candidate(context,record['recipe'],record.get('pruning'))
    if record.get('tensor_digests'):
        wrong=verify_built_candidate(path,record['tensor_digests'],context['baseline_dir'])
        if wrong is not None:raise ValueError('构造出的候选不等于搜索时评分的权重：'+wrong)
    elif record.get('checkpoint_fingerprint') and runtime.checkpoint_fingerprint(path)[0]!=record['checkpoint_fingerprint']:
        raise ValueError('构造出的候选不等于搜索时评分的权重')
    return path,manifest


def upload_identity(context,record):
    """The published weights and selection metadata; a failed -> successful retry must not reuse stale metadata."""
    return {'kind':'candidate_upload','repo':context['settings'].get('output_repo',''),'inputs':core.digest(context['identity']),
            'recipe':record['recipe'],'pruning':core.digest(record['pruning']) if record.get('pruning') else None,
            'components':core.digest(record.get('components')),'selection':record.get('settings_sha256')}


def recorded_upload(context,record):
    """The upload of this selection on record (nothing is rebuilt then), or None."""
    try:done=core.load_record(context['root']/'upload_receipt.json',upload_identity(context,record))
    except ValueError:done=None
    if done is None:return None
    eng=context['engine']
    try:sha=eng.api.model_info(done['repo'],token=eng.token).sha
    except Exception:return None                         # gone or unreachable: upload (again); the repository check below decides
    if sha!=done['commit']:
        print(f'注意：{done["repo"]} 在上次上传之后又有了新的提交（{str(sha)[:10]}）。本次不再上传；要重新上传就换一个新的 output_repo。',flush=True)
    return done


def _model_card(context,record,weights):
    def member(m):return m['tag']+(f' (against {m["reference"].repo})' if m.get('reference') is not None else '')
    mix=', '.join(f"{c['tag']} {float(c['coefficient']):g}"+(' (DELLA of '+', '.join(member(m) for m in c['members'])+')' if c.get('members') else '')
                  +(f' (against {c["reference"].repo})' if c.get('reference') is not None else '')
                  +(' (Q8_0/F16/BF16 GGUF)' if c['format']=='gguf' else '') for c in context['sources'])
    subj=context['S_repo'];name=subj.split('/')[-1]
    searched=context.get('searched') or []
    mu=float(record['recipe'].get('mu',0.))
    cterm=(f'C = {searched[0]["repo"]} - {searched[0]["reference"].repo if searched[0]["reference"] is not None else "B"} '
           f'({"/".join(searched[0]["roles"])} matrices), added outside the bracket with mu = {mu:g} (searched over '
           f'{", ".join(f"{x:g}" for x in context["settings"].get("mu_grid",[]))}). ') if searched else ''
    ablation=record.get('mu_ablation')
    if ablation:
        cterm+=(f'Paired mu=0 ablation NLL: with C {ablation["with_mu"]["mean_nll"]:.5f}, without C '
                f'{ablation["without_mu"]["mean_nll"]:.5f}; C benefit {ablation["c_benefit_nll"]:+.5f}; '
                f'the {"without-C" if ablation["selected_without_mu"] else "with-C"} candidate was retained. ')
    nu=float(record['recipe'].get('nu',0.) or 0.);tm=context.get('tm') or {}
    nterm=''
    if nu:
        mods=', '.join(tm.get('modules') or [])
        nterm=(f'N is a cross-architecture Transport-and-Merge delta from {tm.get("repo")} (commit {str(tm.get("commit"))[:10]}): '
               'per-document activation statistics on one calibration corpus give OT plans between the channels of matching '
               'layers (layers matched by a layer-level OT over the pair costs, the pairs prefiltered by linear CKA); on the '
               f'{tm.get("topk")} highest-activation rows of each changed matrix ({mods}), the transported donor rows Q_out W Q_in^T, '
               f'centred and scaled to the norm of the base row, are mixed in with alpha = {tm.get("alpha")}. N is added outside the '
               f'bracket with nu = {nu:g} (tried {", ".join(str(t["nu"]) for t in (record.get("nu_line") or {}).get("tried",[]))} on the winner). '
               'It carries directions found by the transport, not the donor\'s abilities; tm_transport_report.json has the go/no-go '
               'statistics and the per-matrix diagnostics. ')
    if tm.get('forced') and nu:
        nterm+=('No module passed the go/no-go thresholds; N was built anyway from the modules that passed the self-match check, '
                'because every donor is required to contribute (require_all_donors). ')
    if require_all_donors(context['settings']):
        nterm+=('Every donor term has a nonzero coefficient by construction (require_all_donors): the subject itself was scored only '
                'as the reference, and no mu=0 / nu=0 candidate could be chosen. ')
    elif (record.get('nu_line') or {}).get('tried'):
        nterm=(f'A cross-architecture delta N from {tm.get("repo")} was also scored on the winner '
               f'(nu in {", ".join(str(t["nu"]) for t in record["nu_line"]["tried"])}) and not taken. ')
    nu2=float(record['recipe'].get('nu2',0.) or 0.);tm2=context.get('tm2') or {}
    if nu2:
        nterm+=(f'N2 is a second cross-architecture Transport-and-Merge delta, from {tm2.get("repo")} (commit {str(tm2.get("commit"))[:10]}), '
                'built the same way on the same calibration texts and target activations. That donor is a recurrent-depth model '
                f'(a block of shared layers runs twice, {tm2.get("executions")} layer executions per forward); every execution was a '
                'separate candidate donor layer, so one donor matrix may feed several target layers. '
                f'Modules: {", ".join(tm2.get("modules") or [])}; {tm2.get("k_proj") or ""}. N2 is added with nu2 = {nu2:g} '
                f'(tried {", ".join(str(t["nu"]) for t in (record.get("nu2_line") or {}).get("tried",[]))} on the winner with N). '
                'tm2_transport_report.json has its statistics. ')
        if tm2.get('forced'):
            nterm+='No module of N2 passed the go/no-go thresholds; it was built anyway because every donor must contribute. '
    gain=record['gain_nll'];stage=record.get('components') or {};pruning=record.get('pruning')
    sets=' + '.join({'search':'search set','proxy_validation':'proxy validation set'}.get(s,s) for s in record.get('proxy_sets',['search']))
    opts=component_options(context['settings']);conf=opts['confidence'];boost=opts['max_boost'];cap=row_change_cap(context)
    def plan_text(plan):
        if plan.startswith('step:'):
            return (f'one step of size {plan[5:]} along the first-order gradient for decoder components, with effects first '
                    f'shrunk by {opts["shrink"]:g} bootstrap standard errors; the head uses its exact endpoint difference '
                    'for an independent keep/drop decision and does not set the decoder step scale')
        return {'lenient':f'drop what hurts with probability >= {conf:g}','point':'drop what has a harmful estimated effect',
                'strict':f'keep only what helps with probability >= {conf:g}'}.get(plan,plan)
    def interval(x):return f'{x["change"]:+.4f} (95% interval {x["ci95"][0]:+.4f} to {x["ci95"][1]:+.4f})'
    crossfit=(stage.get('crossfit') or {}).get(record.get('variant'))
    if pruning:
        d=describe_coefficients(pruning['components'])
        selection=(f'Component selection changed the coefficients of {len(pruning["components"])} of {stage.get("components","?")} components '
                   f'of F - S ({d["dropped"]} dropped, {d["reduced"]} scaled down, {d["amplified"]} amplified, by at most {boost:.0%}); '
                   f'plan: {plan_text(record["variant"])}. Every decoder-layer matrix delta was cut into bands of singular directions '
                   '(whitened by the inputs the matrix actually receives) plus the rest, each split into the part inside and outside the '
                   'input subspace of the subject\'s own fine-tune, and pooled by depth block, module type and piece; each component\'s '
                   'first-order effect on every proxy question\'s loss came from one forward and backward pass (checked against the real '
                   'loss on two questions first), its uncertainty from a bootstrap over the questions. ')
        if crossfit:
            selection+=('The plan was chosen by two-fold internal cross-fitting: for each fold, coefficients made without the other half\'s '
                        f'loss/gradient statistics were scored with real forward NLL on that other half, changing it by {interval(crossfit)} '
                        'against F. All questions share the teacher-forced activation covariance decomposition, so the folds are not '
                        'independent; plan selection and the earlier global search also bias this estimate. ')
        selection+='The coefficients are listed in selection.json, the estimates in component_selection.json.\n\n'
    elif stage.get('status')=='done':
        selection=('Component selection was run and kept the full delta: no plan for changing component coefficients improved the proxy NLL '
                   +('in internal cross-fitted scoring (the folds share the activation covariance decomposition) '
                     if stage.get('crossfit') else '')+'(component_selection.json).\n\n')
    elif stage.get('status') in ('failed','skipped'):
        selection=f'Component selection did not run ({stage.get("reason","")}); the search winner is used as it is.\n\n'
    else:selection=''
    limit=(f'No row of any changed weight matrix differs from the subject\'s by more than {cap:.0%} of that row\'s norm '
           '(rows beyond that were scaled back).\n\n') if cap else ''
    heldout=stage.get('heldout') or {}
    fair=''
    if heldout.get('F_vs_S'):
        fair=(f'On the {heldout["rows"]} proxy validation items, which the global coefficient search did not use, the search winner F changed '
              f'the mean NLL by {interval(heldout["F_vs_S"])} against the subject')
        if heldout.get('chosen_vs_S'):fair+=f'; with the chosen component coefficients (cross-fitted), by {interval(heldout["chosen_vs_S"])}'
        fair+=('. These are internal proxy comparisons, not an untouched final validation: the component stage also used these items, and '
               'any cross-fitted estimate shares the activation covariance decomposition across items. The figure on items used for choosing '
               'is optimistic.\n\n')
    # The proxy scores reference answers written by people, with thinking off; it may favour taking back part of S's own
    # fine-tune. Say so when the search or the component stage showed signs of it.
    signs=[]
    profile=record.get('kappa_profile') or {}
    if profile.get('note'):signs.append('the proxy NLL kept falling as kappa (the share of S - B taken back) grew')
    sub=stage.get('subject_subspace') or {}
    if sub.get('note'):
        signs.append(f'components inside the input subspace of the subject\'s own fine-tune carried {sub["helpful_share_inside"]:.0%} '
                     f'of the first-order proxy gain while holding {sub["energy_share_inside"]:.0%} of the delta')
    bias=('**Proxy bias warning.** '+'; '.join(signs)+'. Part of the proxy gain may come from undoing what the subject learned; '
          'check thinking-mode generation (accuracy, thinking length, truncation) against the subject before relying on it.\n\n') if signs else ''
    return (f'# {name} + Final donor + auxiliary sources\n\n'
            'BF16 main model, built as F = S + lambda_g * [D - kappa * (S - B)]'+(' + mu * C' if searched else '')+(' + nu * N' if nu else '')+' with D = (M - B) + sum_k c_k * T_k: '
            f'S is the subject {subj}, M the Final donor, B their base (Qwen/Qwen3.8-27B). Auxiliary terms (coefficient c_k): {mix}. '
            +cterm+nterm+
            'Single finetunes enter as deltas against their own base (B unless stated); a source read from a GGUF file has its '
            'quantization noise estimated and shrunk per tensor; each DELLA group is pruned by magnitude rank, sign-elected and '
            'averaged before it enters. Deltas computed against an older base (Qwen3.5 / Qwen3.6) are transplanted onto Qwen3.8 '
            'as they are.\n\n'
            f'Recipe: `{json.dumps(record["recipe"],sort_keys=True)}`. Weights fingerprint: `{weights}`.\n\n'+selection+limit+bias+
            '**Not evaluated.** The coefficients and components were chosen by forward NLL on a small public proxy set '
            f'({sets}, {record["items"]} items): mean response NLL {record["proxy_mean_nll"]:.4f} against {record["subject_proxy_mean_nll"]:.4f} '
            f'for the subject ({"better by" if gain>0 else "worse by"} {abs(gain):.4f}) on those items, which were also used to choose '
            'the model. '+fair.rstrip('\n')+(' ' if fair else '')+'That proxy is the only measurement made here; no generation, benchmark or '
            'safety evaluation was run. Evaluate before use. See selection.json and merge_manifest.json.\n\n'
            'Metadata and protected tensors (embedding, vision tower, MTP head if the subject has one, norms and other non-matrix '
            'weights) are the subject\'s own, unchanged. No DFlash draft is included. Some sources are uncensored fine-tunes; refusal '
            'behaviour can differ from the subject\'s. The component selection adapts parts of existing methods (activation-whitened '
            'SVD, task singular vectors, gradient x delta sensitivity, layer-wise merging coefficients, cross-fitting) and reproduces none '
            'of them. Review all upstream license terms (source_licenses.json, source_notices/).\n')


def git_blob_sha1(path):
    """The Git blob id used by Hub metadata for non-LFS files, computed without loading the file at once."""
    path=Path(path);h=hashlib.sha1();h.update(f'blob {path.stat().st_size}\0'.encode())
    with path.open('rb') as stream:
        for chunk in iter(lambda:stream.read(8*1024**2),b''):h.update(chunk)
    return h.hexdigest()


def publication_matches(directory,files,existing):
    """Exact complete publication, including metadata and extras; only Hub-managed .gitattributes is exempt."""
    directory=Path(directory);files=set(files)-{'.gitattributes'}
    if set(existing)-{'.gitattributes'}!=files or not any(n.endswith('.safetensors') for n in files):return False
    for name in files:
        item=existing[name];lfs=getattr(item,'lfs',None)
        if lfs:
            expected=lfs.get('sha256') if isinstance(lfs,dict) else getattr(lfs,'sha256',None)
            actual=core.sha_file(directory/name)
        else:
            expected=getattr(item,'blob_id',None);actual=git_blob_sha1(directory/name)
        if not expected or actual!=expected:return False
    return True


def upload_candidate(context,directory,record):
    """Upload the built candidate to output_repo: a new (or empty) repository, private unless settings say otherwise.
    Nothing here evaluates the model, and the model card says so."""
    from huggingface_hub import CommitOperationAdd
    cfg=context['settings'];repo=cfg.get('output_repo','');inputs=source_repos(context,record)
    if not repo or repo in inputs:raise ValueError('output_repo 未填写，或与某个输入仓库相同')
    eng=context['engine'];api=eng.api;directory=Path(directory)
    weights,_=runtime.checkpoint_fingerprint(directory)
    receipt=context['root']/'upload_receipt.json';identity=upload_identity(context,record)
    api.create_repo(repo_id=repo,repo_type='model',private=cfg.get('private',True),exist_ok=True,token=eng.token)
    files={Path(x['file']).relative_to(directory).as_posix() for x in runtime.checkpoint_catalogue(directory).values()}
    files|={p.name for p in directory.iterdir() if p.name in runtime.METADATA_EXACT or p.name in {'model.safetensors.index.json','merge_manifest.json'}}
    if (directory/'chat_templates').exists():files|={p.relative_to(directory).as_posix() for p in (directory/'chat_templates').rglob('*') if p.is_file()}
    # Upstream notices go along; evaluation prompts and outputs do not.
    licenses={}
    for source in inputs:
        for name in eng.files(source):
            if ('/' not in name and name.lower().startswith(('license','notice'))) or name.startswith('source_notices/'):
                dest='source_notices/'+source.replace('/','__')+'/'+name;path=directory/dest;path.parent.mkdir(parents=True,exist_ok=True)
                path.write_bytes(eng._small_bytes(source,name));files.add(dest)
        if 'source_licenses.json' in eng.files(source):
            dest='source_notices/'+source.replace('/','__')+'/upstream_source_licenses.json';path=directory/dest
            path.parent.mkdir(parents=True,exist_ok=True);path.write_bytes(eng._small_bytes(source,'source_licenses.json'));files.add(dest)
        if eng.subfolders.get(source):
            # A subfolder-scoped checkpoint may keep its license at the repository root.
            for item in eng._infos[source].siblings:
                raw=item.rfilename
                if '/' not in raw and raw.lower().startswith(('license','notice')):
                    dest='source_notices/'+source.replace('/','__')+'/repository_root/'+runtime.safe_relative(raw)
                    path=directory/dest;path.parent.mkdir(parents=True,exist_ok=True)
                    path.write_bytes(eng._output_bytes(source,raw,eng.pin(source)));files.add(dest)
        licenses[source]={'commit':eng.pin(source),'subfolder':eng.subfolders.get(source,''),'terms_must_be_reviewed':True}
    core.atomic_json(directory/'source_licenses.json',licenses);files.add('source_licenses.json')
    evidence={k:v for k,v in record.items() if k not in {'tensor_digests','checkpoint_fingerprint','settings_sha256'}}
    core.atomic_json(directory/'selection.json',{**evidence,'weights':weights,'inputs':context['identity'],
                                                 'scope':'proxy (response NLL on public items) only; no generation evaluation'})
    files.add('selection.json')
    report=context['root']/'component_selection.json'
    if (record.get('components') or {}).get('status')=='done' and report.is_file():
        estimates=json.loads(report.read_text(encoding='utf-8'))
        if estimates.get('base_recipe')==record['recipe'] and (not record.get('pruning') or estimates.get('decomposition')==record['pruning']['decomposition']):
            shutil.copy2(report,directory/'component_selection.json');files.add('component_selection.json')
    if float(record['recipe'].get('nu',0) or 0) and context.get('tm'):
        report=Path(context['tm']['directory'])/'tm_report.json'
        if report.is_file():shutil.copy2(report,directory/'tm_transport_report.json');files.add('tm_transport_report.json')
    if float(record['recipe'].get('nu2',0) or 0) and context.get('tm2'):
        report=Path(context['tm2']['directory'])/'tm_report.json'
        if report.is_file():shutil.copy2(report,directory/'tm2_transport_report.json');files.add('tm2_transport_report.json')
    (directory/'README.md').write_text(_model_card(context,record,weights),encoding='utf-8');files.add('README.md')
    # Use a fresh remote snapshot only after every publication file exists and the complete local manifest is known.
    info=api.model_info(repo,files_metadata=True,token=eng.token);existing={s.rfilename:s for s in info.siblings}
    if set(existing)-{'.gitattributes'}:
        # Receipt lost on an earlier run: only an exact, complete publication may be reused as a success.
        if not publication_matches(directory,files,existing):
            raise ValueError(f'{repo} 里已经有别的或不完整的内容。本 notebook 不覆盖已有仓库：请在 output_repo 填一个新的仓库名。')
        result={'uploaded':True,'repo':repo,'commit':info.sha,'weights':weights,'private':bool(cfg.get('private',True))}
        core.save_record(receipt,identity,result);core.atomic_json(context['root']/'uploaded.json',result)
        print(f'{repo} 里已有完整且一致的这份候选（上一次运行上传的），不再重复上传。',flush=True)
        return result
    total=sum((directory/n).stat().st_size for n in files)
    print(f'上传到 {repo}（{"私有" if cfg.get("private",True) else "公开"}）：{len(files)} 个文件，{total/1024**3:.1f} GiB',flush=True)
    started=time.perf_counter()
    commit=api.create_commit(repo_id=repo,repo_type='model',parent_commit=info.sha,token=eng.token,
        operations=[CommitOperationAdd(path_in_repo=n,path_or_fileobj=str(directory/n)) for n in sorted(files)],
        commit_message='Upload aux-mix merge (search and component selection, not evaluated)')
    result={'uploaded':True,'repo':repo,'commit':commit.oid,'weights':weights,'private':bool(cfg.get('private',True))}
    core.save_record(receipt,identity,result);core.atomic_json(context['root']/'uploaded.json',result)
    print(f'上传完成，用时 {(time.perf_counter()-started)/60:.0f} 分钟：https://huggingface.co/{repo}',flush=True)
    return result


def selection_retryable(record):
    """Fallback choices that must not become permanent merely because the settings stayed unchanged."""
    return (record.get('components') or {}).get('status') in {'failed','skipped'}


def attach_tm_delta(context):
    """Build or reuse the cross-architecture delta N (qmerge_tm) and put it in the context (None when off or NO-GO)."""
    context['tm']=tm_delta.ensure(context);context['N']=tm_delta.open_delta(context['tm'])
    if context['N'] is not None:
        print(f'跨架构增量 N：{context["tm"]["repo"]}，{len(context["N"].entries)} 张矩阵，模块 {context["tm"]["modules"]}；'
              f'在获胜配方上评 ν∈{tm_delta.options(context["settings"])["nu_grid"]}',flush=True)
    elif context['tm'] is not None:
        print(f'跨架构增量 N：{context["tm"]["status"]}（{context["tm"].get("reason") or context["tm"].get("error") or ""}）；本次不含 N',flush=True)
    context['tm2']=tm_delta2.ensure(context);context['N2']=tm_delta2.open_delta(context['tm2'])
    if context['N2'] is not None:
        print(f'跨架构增量 N₂：{context["tm2"]["repo"]}，{len(context["N2"].entries)} 张矩阵，模块 {context["tm2"]["modules"]}；'
              f'在获胜配方上评 ν₂∈{tm_delta2.options(context["settings"])["nu_grid"]}',flush=True)
    elif context['tm2'] is not None:
        print(f'跨架构增量 N₂：{context["tm2"]["status"]}（{context["tm2"].get("reason") or context["tm2"].get("error") or ""}）；本次不含 N₂',flush=True)
    return context['tm']


def prepare_stage(context,search,extra,datasets):
    """D, then the search and the component stage, then which candidate to build. Successful or explicitly disabled
    choices are recorded; failed/skipped component fallbacks remain retryable and keep their lower-level checkpoints."""
    done=ensure_aux_delta(context,context['sources'])
    context['D']=StoredDelta(done['directory'],done['fingerprint'],done['tags'])
    print('辅料增量 D：',done['directory'],flush=True)
    print(json.dumps(done['summary'],ensure_ascii=False,indent=2),flush=True)
    print_della_notes(done['summary'])
    searched=ensure_searched_delta(context)
    if searched is not None:
        context['C']=StoredDelta(searched['directory'],searched['fingerprint'],searched['tags'])
        print('单独搜索的增量 C：',searched['directory'],'，范数：',json.dumps(searched['summary']['component_norms'],ensure_ascii=False),flush=True)
    attach_tm_delta(context)
    key=selection_settings(context);record=load_selection(context,datasets);selection_path=context['root']/'selection_record.json'
    if record is not None and record.get('settings_sha256')==key and not selection_retryable(record):
        print('沿用已记录的选择结果：输入、数据和设置都没有变。',flush=True)
        return record
    if record is not None and selection_retryable(record):selection_path.unlink(missing_ok=True)
    setup_local(context,subject_only=True)
    choose_scoring_engine(context,search)
    setup_local(context)
    row_change_report(context)
    record=choose_candidate(context,run_search(context,search),search,extra)
    record['settings_sha256']=key
    if selection_retryable(record):
        # The current run may still build/upload the fallback, but a later run must be allowed to retry the stage.
        selection_path.unlink(missing_ok=True)
        print('本次分量筛选以 failed/skipped 回退结束：不冻结选择记录；下次重跑会利用已有断点重试。',flush=True)
    else:
        core.save_record(selection_path,selection_identity(context,datasets),record)
    core.atomic_json(context['root']/'selected_candidate.json',{k:v for k,v in record.items() if k!='tensor_digests'})
    return record


def resolve_runtime(settings):
    """Fill in the device-dependent options. device 'auto': the GPU when there is one. dtype 'auto': float32 on
    CPU (fast on every CPU; bfloat16 falls back to a slow generic path on CPUs without AVX512), bfloat16 on GPU."""
    info=runtime.describe_runtime();settings=copy.deepcopy(settings)
    settings['_proxy_options_as_configured']=copy.deepcopy(settings['proxy_options'])
    options=settings['proxy_options']
    if options.get('device','auto')=='auto':options['device']='cuda:0' if info['kind']=='gpu' else 'cpu'
    if options.get('dtype','auto')=='auto':options['dtype']='float32' if options['device']=='cpu' else 'bfloat16'
    return settings,info


def run_pipeline(settings,legacy,api,token):
    """D, search, build the chosen candidate, upload it. All CPU and disk work; no evaluation besides the search."""
    root=Path(settings['work']).expanduser().resolve();root.mkdir(parents=True,exist_ok=True)
    if not settings.get('run'):return {'status':'skipped','heavy_computation_started':False}
    settings,info=resolve_runtime(settings)
    install=settings.get('install_inference_dependencies',True);lock_kind=settings.get('version_lock_kind')
    print(f'运行时：{info["cores"]} 核 CPU、{info["ram_gib"]} GiB 内存、'+(f'{info["gpu"]}（{info["gpu_gib"]} GiB 显存）' if info['gpu'] else '无 GPU'),flush=True)
    # A machine with a GPU is billed as one, and nothing here needs a GPU: nothing starts there unless asked for.
    if (info['kind']=='gpu' or info.get('gpu_unusable')) and not settings.get('allow_prepare_on_gpu'):
        message=('这个 notebook 的计算（算 D、搜索、构造并上传候选）都是 CPU 和磁盘的活，在 GPU 运行时上做只是按 GPU 的价格等待，所以没有开始。'
                 '请把运行时换成 CPU（高 RAM）后从头运行；确实要在这台机器上做，设 SETTINGS["allow_prepare_on_gpu"]=True。')
        print(message,flush=True)
        return {'status':'needs_cpu_runtime','message':message,'heavy_computation_started':False}
    search_path=settings.get('search_file','')
    pin_subject(settings,api,token,root)
    dw,ready,pending,searched=aux_plan(settings)
    if pending or not search_path:
        waiting={'status':'waiting_for_late_donor' if pending else 'waiting_for_inputs',
                 'pending_sources':[c['tag'] for c in pending],'missing_data':[] if search_path else ['search_file']}
        if pending:print('等待尚未发布的辅料：',', '.join(c['tag'] for c in pending),'（仓名留空）。',flush=True)
        if not search_path:print('还缺搜索数据：search_file',flush=True)
        if not settings.get('prepare_while_waiting',True):
            return {**waiting,'heavy_computation_started':False}
        # Useful work that does not depend on what is missing: D over the sources already released (a deferred source
        # is decided and added in the full run), the searched delta C, and S's own forward baseline on the search data.
        deferred=[c['tag'] for c in ready if c.get('defer')]
        if deferred:print('留到正式融合时再定并补加进 D 的来源：',', '.join(deferred),flush=True)
        ctx=setup_inputs(settings,legacy,api,token,[c for c in ready if not c.get('defer')],searched)
        done=ensure_aux_delta(ctx,ctx['sources'])
        print('已就绪的辅料增量 D：',done['directory'],flush=True)
        print(json.dumps(done['summary'],ensure_ascii=False,indent=2),flush=True)
        della_notes=print_della_notes(done['summary'])
        result={**waiting,'heavy_computation_started':True,'deferred_sources':deferred,
                'prepared_aux_delta':{'fingerprint':done['fingerprint'],'directory':str(done['directory']),
                                      'sources':['M']+done['tags'],'summary':done['summary']},
                'della_notes':della_notes}
        built=ensure_searched_delta(ctx)
        if built is not None:
            result['prepared_searched_delta']={'fingerprint':built['fingerprint'],'directory':str(built['directory']),
                                               'norms':built['summary']['component_norms']}
            print('单独搜索的增量 C 已算好：',built['directory'],flush=True)
        if settings.get('baseline_while_waiting',True) and search_path:
            runtime.ensure_inference_dependencies(root,install,lock_kind)
            rows=core.read_jsonl(search_path)
            setup_local(ctx,subject_only=True)
            choose_scoring_engine(ctx,rows)
            result['subject_baseline']=score_subject(ctx,rows)
            print('S 在搜索集上的前向基线已算好并缓存：',result['subject_baseline'],flush=True)
            extra=component_extra_rows(settings,rows)
            if extra:
                # S's scores on the extra rows of the component stage do not depend on the sources either.
                result['subject_baseline_component_rows']={'items':len(extra),'mean_nll':score_subject(ctx,extra)['search_mean_nll']}
                print(f'S 在分量筛选另外用的 {len(extra)} 条题上的前向基线也已算好',flush=True)
            # The backward pass of the component stage first runs only after the search; check it on the real S now.
            ctx['D']=StoredDelta(done['directory'],done['fingerprint'],done['tags'])
            try:
                result['backward_check']=backward_summary(check_backward(ctx,rows))
            except (OSError,MemoryError) as error:
                print(f'反向自检这次没能完成（{type(error).__name__}: {str(error)[:200]}），下次运行会重做；不影响已算好的 D 和基线。',flush=True)
            # How many rows lambda*D and mu*C push past the row cap (S is on the local disk now). D here lacks the deferred
            # sources; the full run reports again for the complete D.
            if built is not None:ctx['C']=StoredDelta(built['directory'],built['fingerprint'],built['tags'])
            try:
                report=row_change_report(ctx)
                if report is not None:
                    result['row_change']={k:report.get(k) for k in ('cap','D_ratio','C_ratio','share_over_cap_lambda_D','share_over_cap_mu_C',
                                                                     'share_over_cap_lambda_D_plus_mu_C','lambda_for_1pct_rows')}
                    result['row_change']['note']='D 还不含 '+'、'.join(deferred+[c['tag'] for c in pending])+'；正式融合时会按完整的 D 重新报告' if deferred or pending else None
            except (OSError,MemoryError) as error:
                print(f'逐行改动报告这次没能完成（{type(error).__name__}: {str(error)[:200]}），正式融合时会再做。',flush=True)
        tm_cfg=tm_delta.options(settings)
        if tm_cfg['enabled'] and tm_cfg['while_waiting']:
            # The cross-architecture delta N does not depend on anything that is missing either. It is the longest step
            # (the donor's forward on the CPU), so it comes last; an error here leaves the rest of the waiting work standing.
            runtime.ensure_inference_dependencies(root,install,lock_kind)
            try:
                tm=tm_delta.ensure(ctx)
                result['tm_delta']=None if tm is None else {k:tm.get(k) for k in ('status','repo','commit','modules','fingerprint','directory','summary','reason','error')}
            except Exception as error:
                print(f'跨架构增量 N 这次没能完成（{type(error).__name__}: {str(error)[:300]}）；下次运行会从断点继续。不影响已算好的 D、C 和基线。',flush=True)
                result['tm_delta']={'status':'failed','error':f'{type(error).__name__}: {str(error)[:300]}'}
        tm2_cfg=tm_delta2.options(settings)
        if tm2_cfg['enabled'] and tm2_cfg['while_waiting']:
            # The second cross-architecture delta N2 (SenseNova) after N: its own module, record directory and checkpoints.
            runtime.ensure_inference_dependencies(root,install,lock_kind)
            try:
                tm2=tm_delta2.ensure(ctx)
                result['tm2_delta']=None if tm2 is None else {k:tm2.get(k) for k in ('status','repo','commit','modules','fingerprint','directory','summary','reason','error','k_proj')}
            except Exception as error:
                # The code-review stop and the forward gate land here too: their message says what to do.
                print(f'跨架构增量 N₂ 这次没能完成（{type(error).__name__}）：\n{str(error)[:3000]}\n下次运行会从断点继续。不影响已算好的 D、C、S 基线和 N。',flush=True)
                result['tm2_delta']={'status':'failed','error':f'{type(error).__name__}: {str(error)[:300]}'}
        return result
    search=core.read_jsonl(search_path)
    core.check_coverage(search,settings['search_min_items'],settings['search_min_per_category'])
    extra=component_extra_rows(settings,search)
    runtime.ensure_inference_dependencies(root,install,lock_kind)
    context=setup_inputs(settings,legacy,api,token,ready,searched)
    old=core.load_record(root/'inputs.json',{'kind':'merge_inputs'})
    if old is not None and old!=context['identity']:raise ValueError('此工作目录已绑定另一组输入；请使用新的独立目录')
    core.save_record(root/'inputs.json',{'kind':'merge_inputs'},context['identity'])
    datasets={'search':core.digest(search)}
    if extra:datasets['component_extra']=core.digest(extra)
    record=prepare_stage(context,search,extra,datasets)
    if record['is_baseline']:
        message='没有候选的代理 NLL 比 S 低；按 upload_when_no_gain=False，不构造也不上传。'
        print(message,flush=True)
        return {'status':'subject_best','message':message,'recipe':record['recipe']}
    summary={k:record[k] for k in ('recipe','variant','proxy_mean_nll','subject_proxy_mean_nll','gain_nll','beats_subject','items','proxy_sets')}
    summary['components']={k:v for k,v in (record.get('components') or {}).items() if k not in ('coefficients','preflight')}
    summary['recipe_mu']=record['recipe'].get('mu',0.)
    summary['recipe_nu']=record['recipe'].get('nu',0.)
    summary['tm']=record.get('tm')
    summary['recipe_nu2']=record['recipe'].get('nu2',0.)
    summary['tm2']=record.get('tm2')
    notes=[(record.get('kappa_profile') or {}).get('note'),((record.get('components') or {}).get('subject_subspace') or {}).get('note')]
    summary['proxy_bias_notes']=[x for x in notes if x]
    if settings.get('upload',True):
        uploaded=recorded_upload(context,record)
        if uploaded is None:
            path,_=build_chosen(context,record)
            uploaded=upload_candidate(context,path,record)
        else:print(f'这份候选已上传到 {uploaded["repo"]}（commit {uploaded["commit"][:10]}），不再重复构造和上传。',flush=True)
        result={'status':'uploaded','repo':uploaded['repo'],'commit':uploaded['commit'],'private':uploaded['private'],
                'weights':uploaded['weights'],'selection':summary}
    else:
        path,_=build_chosen(context,record)
        weights=persist_candidate(context,path)
        result={'status':'candidate_saved','selection':summary,'weights':weights,
                'candidate_saved_copy':str(root/'selected_candidate'/weights[:24])}
    core.atomic_json(root/'result.json',result)
    return result
'''
load_embedded_module('qmerge_pipeline', MODULE_SOURCE)
del MODULE_SOURCE

In [ ]:
# 内嵌源代码；独立运行无需额外文件。
MODULE_SOURCE = r'''
"""Build the four EfficientThink datasets from pinned public Hugging Face evaluation sets.

Six categories, each drawn from one benchmark at a fixed commit:
  math_word        openai/gsm8k (main/test)                 number
  math_competition HuggingFaceH4/MATH-500 (integer answers) number
  knowledge_mc     TIGER-Lab/MMLU-Pro (test, 10 options)    option letter
  chinese_mc       ceval/ceval-exam (val, selected subjects) option letter
  code_reasoning   cruxeval-org/cruxeval (output prediction) Python literal
  logic            lukaemon/bbh (selected closed-answer tasks)

Every item is used at most once across the four splits. References are the datasets' own
answers; the builder checks them for internal consistency (GSM8K calculator annotations are
re-evaluated, MATH boxed answers must equal the answer field, option letters must exist, ...)
and drops items that fail. Public benchmarks may be in the models' training data: the splits
compare S with merges of S on equal footing, but absolute scores can be inflated.
"""
import ast
import hashlib
import json
import math
from pathlib import Path
import re

VERSION = 'evaldata-2026-10-11.1'   # unchanged in e3: default sizes give the same files
SIZES = {'search_file': 8, 'proxy_validation_file': 8, 'validation_file': 16, 'final_test_file': 16}  # per category
PROXY_SPLITS = ('search_file', 'proxy_validation_file')
CEVAL_SUBJECTS = (
    'advanced_mathematics', 'basic_medicine', 'business_administration', 'chinese_language_and_literature',
    'clinical_medicine', 'college_chemistry', 'college_economics', 'college_physics', 'college_programming',
    'computer_architecture', 'computer_network', 'discrete_mathematics', 'electrical_engineer',
    'high_school_biology', 'high_school_chemistry', 'high_school_chinese', 'high_school_geography',
    'high_school_history', 'high_school_mathematics', 'high_school_physics', 'law', 'logic',
    'metrology_engineer', 'middle_school_chemistry', 'middle_school_mathematics', 'middle_school_physics',
    'operating_system', 'probability_and_statistics')
BBH_TASKS = ('boolean_expressions', 'date_understanding', 'formal_fallacies', 'logical_deduction_five_objects',
             'multistep_arithmetic_two', 'navigate', 'object_counting', 'temporal_sequences',
             'tracking_shuffled_objects_five_objects', 'web_of_lies')
SOURCES = {
    'math_word': {'repo': 'openai/gsm8k', 'revision': '740312add88f781978c0658806c59bc2815b9866',
                  'files': ['main/test-00000-of-00001.parquet']},
    'math_competition': {'repo': 'HuggingFaceH4/MATH-500', 'revision': '6e4ed1a2a79af7d8630a6b768ec859cb5af4d3be',
                         'files': ['test.jsonl']},
    'knowledge_mc': {'repo': 'TIGER-Lab/MMLU-Pro', 'revision': 'b189ec765aa7ed75c8acfea42df31fdae71f97be',
                     'files': ['data/test-00000-of-00001.parquet']},
    'chinese_mc': {'repo': 'ceval/ceval-exam', 'revision': '617524a00b307ff6f9933702f724131fe12ca7ce',
                   'files': [f'{s}/val-00000-of-00001.parquet' for s in CEVAL_SUBJECTS]},
    'code_reasoning': {'repo': 'cruxeval-org/cruxeval', 'revision': 'b96af0450242eb4da433032b90998f25588a5d0f',
                       'files': ['test.jsonl']},
    'logic': {'repo': 'lukaemon/bbh', 'revision': '982bb89fd79532a8ac676a61fc42eb1aeec63f99',
              'files': [f'{t}/test-00000-of-00001.parquet' for t in BBH_TASKS]},
}

GEN_EN = 'Think it through, then reply with only the final answer in the form <answer>...</answer> and nothing else.'
GEN_ZH = '请先思考，然后只输出最终答案，格式为 <answer>...</answer>，不要输出其他任何内容。'
GEN_KIND = {
    'number': ' The final answer must be a plain number: digits only, no units, commas or words.',
    'letter': ' The final answer is the letter of the correct option, for example <answer>C</answer>.',
    'literal': ' The final answer is the return value written as a Python literal, exactly as repr() would print it.',
    'word': ' The final answer is exactly the answer word or option label the question asks for.',
    'letter_zh': '最终答案是正确选项的字母，例如 <answer>B</answer>。',
}
PROXY_MATH = 'Briefly show your working, then give the final answer as <answer>...</answer>.'
PROXY_EN = 'Give the final answer as <answer>...</answer>.'
PROXY_MC = 'State the correct option, then give its letter as <answer>...</answer>.'
PROXY_ZH = '指出正确选项，并用 <answer>...</answer> 给出它的字母。'
LETTERS = 'ABCDEFGHIJ'


# ---------- reference checks ----------
_OPS = {ast.Add: lambda a, b: a + b, ast.Sub: lambda a, b: a - b, ast.Mult: lambda a, b: a * b,
        ast.Div: lambda a, b: a / b, ast.FloorDiv: lambda a, b: a // b, ast.Mod: lambda a, b: a % b,
        ast.Pow: lambda a, b: a ** b}


def safe_arith(text):
    """Evaluate +-*/%** arithmetic on number literals only; anything else raises ValueError."""
    def ev(node):
        if isinstance(node, ast.Expression):
            return ev(node.body)
        if isinstance(node, ast.Constant) and type(node.value) in (int, float):
            return node.value
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, (ast.USub, ast.UAdd)):
            v = ev(node.operand)
            return -v if isinstance(node.op, ast.USub) else v
        if isinstance(node, ast.BinOp) and type(node.op) in _OPS:
            a, b = ev(node.left), ev(node.right)
            if isinstance(node.op, ast.Pow) and (abs(b) > 64 or abs(a) > 1e6):
                raise ValueError('power too large')
            return _OPS[type(node.op)](a, b)
        raise ValueError('not arithmetic')
    try:
        return float(ev(ast.parse(text.replace(',', ''), mode='eval')))
    except (SyntaxError, ZeroDivisionError, OverflowError, TypeError) as error:
        raise ValueError(str(error)) from error


def _close(value, text):
    target = float(text)
    if abs(value - target) <= 1e-6 * max(1.0, abs(target)):
        return True
    decimals = len(text.split('.')[1]) if '.' in text else 0
    return round(value, decimals) == target or math.floor(value * 10 ** decimals) / 10 ** decimals == target


def gsm8k_item(row):
    question, answer = str(row['question']).strip(), str(row['answer'])
    if '####' not in answer:
        return None
    body, final = answer.rsplit('####', 1)
    final = final.strip().replace(',', '')
    if not re.fullmatch(r'-?\d+', final):
        return None
    for note in re.findall(r'<<([^<>]*)>>', body):
        if '=' not in note:
            return None
        lhs, rhs = note.rsplit('=', 1)
        try:
            if not _close(safe_arith(lhs), rhs.strip().replace(',', '')):
                return None
        except ValueError:
            return None
    steps = re.sub(r'<<[^<>]*>>', '', body).strip()
    return {'key': hashlib.sha256(question.encode()).hexdigest()[:16], 'group': 'gsm8k', 'question': question,
            'lang': 'en', 'kind': 'number', 'expected': int(final), 'proxy_instruction': PROXY_MATH,
            'response': f'{steps}\n<answer>{final}</answer>',
            'verification': 'every calculator annotation <<a=b>> re-evaluated; final answer is an integer'}


def last_boxed(text):
    start = text.rfind('\\boxed{')
    if start < 0:
        return None
    i, depth = start + len('\\boxed{'), 1
    for j in range(i, len(text)):
        depth += {'{': 1, '}': -1}.get(text[j], 0)
        if depth == 0:
            return text[i:j]
    return None


def math500_item(row):
    answer = str(row['answer']).strip()
    if not re.fullmatch(r'-?\d+', answer):
        return None
    boxed = last_boxed(str(row['solution']))
    if boxed is None or re.sub(r'[\s{}]|\\[!,;]', '', boxed) != answer:
        return None
    return {'key': str(row['unique_id']), 'group': str(row.get('subject', 'math')), 'question': str(row['problem']).strip(),
            'lang': 'en', 'kind': 'number', 'expected': int(answer), 'proxy_instruction': PROXY_MATH,
            'response': f'{str(row["solution"]).strip()}\n<answer>{answer}</answer>',
            'verification': 'last \\boxed{} in the solution equals the integer answer field'}


def mmlu_pro_item(row):
    options = [str(o) for o in list(row['options']) if str(o).strip() and str(o).strip() != 'N/A']
    idx = int(row['answer_index'])
    if not 4 <= len(options) <= 10 or not 0 <= idx < len(options) or str(row['answer']) != LETTERS[idx]:
        return None
    if len(options) != len(list(row['options'])):
        return None  # padded option lists would shift letters
    letter = LETTERS[idx]
    question = str(row['question']).strip() + '\n' + '\n'.join(f'{LETTERS[k]}. {o}' for k, o in enumerate(options))
    return {'key': str(row['question_id']), 'group': str(row.get('category', 'mmlu')), 'question': question,
            'lang': 'en', 'kind': 'letter', 'expected': [letter, f'({letter})'], 'proxy_instruction': PROXY_MC,
            'response': f'{letter}. {options[idx]}\n<answer>{letter}</answer>',
            'verification': 'answer letter equals answer_index and names an existing option'}


def ceval_item(row, subject):
    letter = str(row['answer']).strip()
    options = [str(row.get(k, '')).strip() for k in 'ABCD']
    if letter not in 'ABCD' or len(letter) != 1 or not all(options):
        return None
    question = str(row['question']).strip() + '\n' + '\n'.join(f'{k}. {o}' for k, o in zip('ABCD', options))
    return {'key': f'{subject}/{row["id"]}', 'group': subject, 'question': question, 'lang': 'zh', 'kind': 'letter_zh',
            'expected': [letter, f'({letter})'], 'proxy_instruction': PROXY_ZH,
            'response': f'{letter}. {options["ABCD".index(letter)]}\n<answer>{letter}</answer>',
            'verification': 'answer is one of A-D and all four options are non-empty'}


def _quote_variant(literal):
    if re.fullmatch(r"'[^'\"\\]*'", literal):
        return '"' + literal[1:-1] + '"'
    return None


def cruxeval_item(row):
    output, code, args = str(row['output']).strip(), str(row['code']).rstrip(), str(row['input']).strip()
    try:
        ast.literal_eval(output)
        ast.parse(code)
    except (ValueError, SyntaxError, MemoryError, RecursionError):
        return None
    expected = [output] + [v for v in (_quote_variant(output),) if v]
    question = f'```python\n{code}\n```\nWhat does `f({args})` return?'
    return {'key': str(row['id']), 'group': 'cruxeval', 'question': question, 'lang': 'en', 'kind': 'literal',
            'expected': expected, 'proxy_instruction': PROXY_EN, 'response': f'<answer>{output}</answer>',
            'verification': 'expected output parses as a Python literal and the code parses (not executed here)'}


def bbh_item(row, task, index):
    text, target = str(row['input']).strip(), str(row['target']).strip()
    if re.fullmatch(r'\([A-R]\)', target):
        if target not in text:
            return None
        kind, expected = 'letter', [target, target[1]]
    elif re.fullmatch(r'-?\d+', target):
        kind, expected = 'number', int(target)
    elif re.fullmatch(r'[A-Za-z]+', target):
        kind, expected = 'word', sorted({target, target.lower(), target.capitalize()})
    else:
        return None
    return {'key': f'{task}/{index}', 'group': task, 'question': text, 'lang': 'en', 'kind': kind, 'expected': expected,
            'proxy_instruction': PROXY_EN, 'response': f'<answer>{target}</answer>',
            'verification': 'closed-form target; option targets appear among the listed options'}


# ---------- loading ----------
def read_rows(path):
    path = Path(path)
    if path.suffix == '.jsonl':
        return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
    import pandas as pd
    return pd.read_parquet(path).to_dict('records')


def candidates(category, fetch, reader=read_rows):
    src = SOURCES[category]
    items = []
    for filename in src['files']:
        rows = reader(fetch(src['repo'], src['revision'], filename))
        group = filename.split('/')[0]
        for index, row in enumerate(rows):
            if category == 'math_word':
                item = gsm8k_item(row)
            elif category == 'math_competition':
                item = math500_item(row)
            elif category == 'knowledge_mc':
                item = mmlu_pro_item(row)
            elif category == 'chinese_mc':
                item = ceval_item(row, group)
            elif category == 'code_reasoning':
                item = cruxeval_item(row)
            else:
                item = bbh_item(row, group, index)
            if item is not None:
                item['source'] = f'{src["repo"]}@{src["revision"][:12]}:{filename}#{item["key"]}'
                items.append(item)
    return items


# ---------- prompts, budgets, sampling ----------
def generation_prompt(item):
    if item['lang'] == 'zh':
        return f'{item["question"]}\n\n{GEN_ZH}{GEN_KIND[item["kind"]]}'
    return f'{item["question"]}\n\n{GEN_EN}{GEN_KIND[item["kind"]]}'


def proxy_prompt(item):
    return f'{item["question"]}\n\n{item["proxy_instruction"]}'


def _approx_tokens(text):
    cjk = sum(1 for ch in text if '　' <= ch <= '鿿' or '＀' <= ch <= '￯')
    return cjk * 1.5 + (len(text) - cjk) / 2.5


def within_budget(item, count_tokens=None, template_margin=64):
    count = count_tokens or _approx_tokens
    p, r = count(proxy_prompt(item)), count(item['response'])
    g = count(generation_prompt(item))
    # proxy_options: max_tokens 1024, max_response_tokens 512; generation max_input_tokens 2048
    return p + r + template_margin <= 1024 and r <= 480 and g + template_margin <= 2048


def _rank(seed, category, key):
    return hashlib.sha256(f'{seed}|{category}|{key}'.encode()).hexdigest()


def select(category, items, seed, count_tokens=None, sizes=None):
    """Deduplicate, keep items within budget, then interleave groups in a seeded order."""
    seen, pool = set(), []
    for item in items:
        norm = ' '.join(item['question'].split())
        if norm in seen:
            continue
        seen.add(norm)
        if within_budget(item, count_tokens):
            pool.append(item)
    groups = {}
    for item in sorted(pool, key=lambda x: _rank(seed, category, x['key'])):
        groups.setdefault(item['group'], []).append(item)
    order = sorted(groups, key=lambda g: _rank(seed, category, 'group:' + g))
    picked, need = [], sum((sizes or SIZES).values())
    while len(picked) < need and any(groups.values()):
        for g in order:
            if groups[g] and len(picked) < need:
                picked.append(groups[g].pop(0))
    if len(picked) < need:
        raise ValueError(f'{category}: 只有 {len(picked)} 条合格题目，需要 {need} 条')
    return picked


def rows_for(category, picked, sizes=None):
    out, start = {}, 0
    for split, n in (sizes or SIZES).items():
        out[split] = []
        for k, item in enumerate(picked[start:start + n]):
            base = {'id': f'{split.split("_file")[0]}-{category}-{k:02d}', 'category': category,
                    'source': item['source'], 'verification': item['verification']}
            if split in PROXY_SPLITS:
                out[split].append({**base, 'messages': [{'role': 'user', 'content': proxy_prompt(item)}],
                                   'response': item['response'], 'reference_verified': True})
            else:
                out[split].append({**base, 'messages': [{'role': 'user', 'content': generation_prompt(item)}],
                                   'metric': 'number' if item['kind'] == 'number' else 'exact',
                                   'expected': item['expected']})
        start += n
    return out


def _sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def check_sizes(sizes):
    """Items per category for each of the four splits; None means the defaults."""
    if sizes is None:
        return dict(SIZES)
    if list(sizes) != list(SIZES) or any(type(v) is not int or v < 1 for v in sizes.values()):
        raise ValueError(f'EVAL_SIZES 需要按顺序给出 {list(SIZES)} 四个正整数')
    return dict(sizes)


def build(out_root, fetch, core, count_tokens=None, tokenizer_identity=None, seed=20261011, reader=read_rows, sizes=None):
    """Write the four JSONL files under out_root/<identity digest>/ and return their paths.

    An existing complete build with the same identity is reused unchanged, so the final test
    stays the same across sessions. A different identity gets its own directory.
    """
    sizes = check_sizes(sizes)
    identity = {'version': VERSION, 'seed': seed, 'sizes_per_category': sizes,
                'sources': SOURCES, 'tokenizer': tokenizer_identity if count_tokens else 'approximate'}
    digest = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    directory = Path(out_root) / digest[:16]
    manifest_path = directory / 'eval_data_manifest.json'
    if manifest_path.exists():
        try:
            manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
            if manifest.get('identity') == identity and all(
                    (directory / name).exists() and _sha(directory / name) == sha for name, sha in manifest['sha256'].items()):
                return {'paths': {k: str(directory / v) for k, v in manifest['files'].items()},
                        'manifest': manifest, 'reused': True}
        except (OSError, ValueError, KeyError):
            pass
    splits = {split: [] for split in sizes}
    counts = {}
    for category in SOURCES:
        items = candidates(category, fetch, reader)
        picked = select(category, items, seed, count_tokens, sizes)
        counts[category] = {'candidates': len(items), 'groups': sorted({p['group'] for p in picked})}
        for split, rows in rows_for(category, picked, sizes).items():
            splits[split].extend(rows)
    directory.mkdir(parents=True, exist_ok=True)
    files, hashes = {}, {}
    for split, rows in splits.items():
        name = split.replace('_file', '') + '.jsonl'
        tmp = directory / (name + '.tmp')
        tmp.write_text(''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in rows), encoding='utf-8')
        tmp.replace(directory / name)
        files[split], hashes[name] = name, _sha(directory / name)
    # The pipeline's own validators: formats, coverage and cross-split disjointness.
    loaded = {s: core.read_jsonl(directory / files[s], 'proxy' if s in PROXY_SPLITS else 'generation') for s in sizes}
    core.check_disjoint({s.replace('_file', ''): rows for s, rows in loaded.items()})
    manifest = {'identity': identity, 'files': files, 'sha256': hashes, 'categories': counts,
                'items': {s: len(r) for s, r in loaded.items()}}
    tmp = manifest_path.with_name(manifest_path.name + '.tmp')
    tmp.write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + '\n', encoding='utf-8')
    tmp.replace(manifest_path)
    return {'paths': {k: str(directory / v) for k, v in files.items()}, 'manifest': manifest, 'reused': False}


def tokenizer_counter(tokenizer_path):
    """Exact token counts with the subject's tokenizer.json; None if the library is unavailable."""
    try:
        from tokenizers import Tokenizer
    except ImportError:
        return None
    tok = Tokenizer.from_file(str(tokenizer_path))
    return lambda text: len(tok.encode(text, add_special_tokens=False).ids)
'''
load_embedded_module('qmerge_evaldata', MODULE_SOURCE)
del MODULE_SOURCE

## 4. 评测数据

`AUTO_EVAL_DATA=True` 且搜索数据空着时：按固定 commit 下载 6 个公开评测集（共约 20MB），抽题写到 `DRIVE_DIR/eval_data/`，并把路径填进 `SETTINGS`。已经建好的会直接沿用。这一版用其中的搜索集和代理验证集（后者给分量筛选用）；来源和规则见第 1 节。


In [ ]:
# 先锁定主模型 S 的 commit 和子目录（记在 DRIVE_DIR/subject_pin.json），数据长度检查和之后的融合都用同一个 S。
qmerge_pipeline.pin_subject(SETTINGS, api, TOK, EFFICIENT_WORK)

if AUTO_EVAL_DATA and not all(SETTINGS.get(k) for k in _DATA_KEYS):
    from huggingface_hub import hf_hub_download
    _ds_cache = SCRATCH / "hf_datasets"
    def _fetch(repo, revision, filename):
        return hf_hub_download(repo, filename, repo_type="dataset", revision=revision,
                               token=TOK or None, cache_dir=str(_ds_cache))
    _tok_file = (SETTINGS["S_subfolder"].strip("/") + "/" if SETTINGS["S_subfolder"] else "") + "tokenizer.json"
    _count = qmerge_evaldata.tokenizer_counter(hf_hub_download(
        SETTINGS["S_repo"], _tok_file, revision=SETTINGS["S_revision"], token=TOK or None, cache_dir=str(_ds_cache)))
    if _count is None:
        print("没有 tokenizers 库：长度改按字符数保守估计。")
    EVAL_DATA = qmerge_evaldata.build(
        EFFICIENT_WORK / "eval_data", _fetch, qmerge_core, count_tokens=_count,
        tokenizer_identity={"repo": SETTINGS["S_repo"], "revision": SETTINGS["S_revision"], "file": _tok_file},
        sizes=EVAL_SIZES)
    for _k, _v in EVAL_DATA["paths"].items():
        if not SETTINGS.get(_k):
            SETTINGS[_k] = _v
    print("评测数据" + ("（沿用已建好的）" if EVAL_DATA["reused"] else "（新建）") + "：", EVAL_DATA["manifest"]["items"])
    for _k in _DATA_KEYS:
        print(f"  {_k}: {SETTINGS[_k]}")
else:
    print("评测数据：使用 SETTINGS 中已填写的路径。")

## 5. 融合、搜索、分量筛选与上传

全部在 CPU 运行时上做。断开后从头运行全部单元格，会从断点继续：

1. StartLux 未发布：算 D 已确定的部分（DavidAU 留到正式融合时再定）和增量 C，下载 S、做架构预检、算 S 在搜索集和代理验证集上的前向基线，在真实 S 上做反向自检，报告 λ·D、μ·C 超过行改动上限的比例（`row_change` 一项），再算跨架构增量 N（结果里的 `tm_delta`；状态 `go` / `no-go`，出错时是 `failed`，下次运行从断点继续），然后返回 `waiting_for_late_donor`（结果里的 `backward_check`）。设 `prepare_while_waiting=False` 则直接返回等待状态。
2. 条件都满足后：决定 DavidAU 读 BF16 还是 GGUF → 在 D 上补加 StartLux 和 DavidAU → 把 S、D、C（放得下再加 B）放到本地盘，打印 D、C 相对 S 的逐行大小和超过行改动上限的比例 → 沿用（或算完）跨架构增量 N → 选评分引擎（第一次在 S 上对照，见第 1 节 `proxy_options["engine"]`）→ 全局搜索（最多 12 个 GP 主候选 + 可能的 head 补评 + 初始 winner 的 μ=0 消融，`global_trials` 只计主候选；搜索 λ、κ、输出头系数和 μ），每个候选的权重都真实算出并前向评分（逐层评分时在内存中逐层算出，不写盘），之后打印按 κ、按 μ 分组的最低代理 NLL → 有 N 时在获胜配方上评 `nu_grid` 里的 ν（ν 线搜索） → 分量筛选（替代原来的 NPS 式局部搜索）：先确认反向自检通过，对选中的候选做一次前向，在两道题上核对梯度，再逐题反向，估计每个分量的增益；三种方案（沿梯度的两种步长和严格方案）先用两折交叉拟合在折外题上比较（这是内部代理估计：定本折系数不用另一折的损失 / 梯度，但所有题共享教师强制激活协方差分解，不是完全独立验证或终测），选中的方案再用全部题核对；最后打印 S 自己微调子空间之内的分量占有益效果的比例。
3. 构造选中的候选（或变体），核对它与评分时的权重逐字节相同 → 上传到 `output_repo`，返回 `uploaded`。

分量筛选以 `failed` / `skipped` 回退时，本次仍按全局搜索结果继续，但不把这个回退选择保存为可复用的选择记录；下次从头运行会重试，已经完成的搜索评分和分量断点仍可复用。

`search_file` 用参考 `response` 格式。不调用付费 API，不在 notebook 里执行模型生成的代码。这里不做生成评测，上传的模型请用你自己的测试评测。

状态一览：

| 状态 | 含义 |
|---|---|
| `waiting_for_late_donor` / `waiting_for_inputs` | 在等 StartLux 或搜索数据；能提前做的已经做了 |
| `uploaded` | 已上传；`selection` 里是配方、选中的变体和代理 NLL，`proxy_bias_notes` 里是代理偏差的提示（如果有） |
| `candidate_saved` | `upload=False`：候选存到了 `DRIVE_DIR/selected_candidate/` |
| `subject_best` | 搜索和分量筛选都没有找到代理 NLL 比 S 低的候选，而且 `upload_when_no_gain=False`，没有上传 |
| `needs_cpu_runtime` | 当前是 GPU 运行时（或机器有 GPU 但 torch 用不了），没有开始，换成 CPU 运行时 |


In [ ]:
RESULT = qmerge_pipeline.run_pipeline(SETTINGS, qmerge_legacy, api, TOK)
qmerge_core.atomic_json(EFFICIENT_WORK / "unified_run_status.json", RESULT)
print(json.dumps(RESULT, ensure_ascii=False, indent=2))
_NEXT = {
    "waiting_for_late_donor": "等 StartLux 发布后填 STARTLUX_REPO，在 CPU 运行时从头运行。不含 StartLux 和 DavidAU 的 D、增量 C、S 的基线和反向自检已经做好（自检结果见上面的 backward_check）；跨架构增量 N 的状态见上面的 tm_delta（failed 时从头运行会从断点继续）。正式运行时会先看 DavidAU 有没有发布 BF16，再把它和 StartLux 一起补加进 D。",
    "waiting_for_inputs": "补齐搜索数据（search_file）后从头运行。",
    "uploaded": "BF16 已上传；继续运行第 6 节导出并上传独立 FP8 仓库。用你自己的测试部分评测这个仓库，务必包括开思考的生成（分数、思考长度、截断率都和 S 比）；确认后可在 Hugging Face 上改成公开。",
    "candidate_saved": "候选已存到 DRIVE_DIR/selected_candidate/。要上传就设 SETTINGS['upload']=True 再运行（搜索过的候选不会重新评分）。",
    "subject_best": "搜索和分量筛选都没有找到代理 NLL 比 S 低的候选。仍想上传最好的候选，就设 SETTINGS['upload_when_no_gain']=True 再运行（评过分的候选不会重新评分）。",
    "needs_cpu_runtime": "把运行时换成 CPU（高 RAM）后从头运行。",
}
print("\n下一步：", _NEXT.get(RESULT.get("status"), "见上面的状态。"))
_CHECK = RESULT.get("backward_check") or (RESULT.get("selection", {}).get("components", {}) or {}).get("backward_check")
if _CHECK and not _CHECK.get("ok"):
    print("\n反向自检没有通过：分量筛选会被跳过（或已被跳过），只用全局搜索的结果。详情见 DRIVE_DIR/backward_check.json，请反馈回来。")
_TM = RESULT.get("tm_delta") or RESULT.get("selection", {}).get("tm")
if _TM:
    print(f"\n跨架构增量 N（{_TM.get('repo')}）：{_TM.get('status')}"
          + (f"；进入融合的模块 {_TM.get('modules')}" if _TM.get("status") == "go" else "")
          + (f"；{_TM.get('reason') or _TM.get('error')}" if _TM.get("reason") or _TM.get("error") else "")
          + (f"；成品 ν = {RESULT['selection'].get('recipe_nu', 0)}" if RESULT.get("selection") else ""))
for _note in RESULT.get("selection", {}).get("proxy_bias_notes", []):
    print("\n注意：" + _note)


## 6. 成品转 FP8，并额外上传一个仓库

默认开启。第 5 节返回 `uploaded` 后，本节把**已经上传的 BF16 成品**按固定 commit 读回，逐分片转换，并上传到独立仓库：

`YFC-112358/Qwen3.8-27B-pi-AuxMix-v1-FP8`（或配置的 `fp8_repo`）。新仓库沿用 `private`，原 BF16 仓库不变。

- **格式**：fine-grained FP8，E4M3FN 权重、128×128 分块、FP32 `weight_scale_inv`、动态激活量化配置；无需校准数据。
- **范围**：语言解码层的 MLP 和普通注意力投影。线性注意力、视觉模块、MTP / NextN 草稿头、embedding、输出头、norm，以及维度不能整除 128 的矩阵保留原精度；跳过项写进 `modules_to_not_convert`。这是混合精度 FP8 成品。
- **资源**：在 CPU 上逐分片处理，不把整个 27B 模型装进内存，也不改现有依赖版本。默认另外预留约 40 GiB Drive 空间。固定 commit 的 BF16 权重会重新下载一遍（约 55GB 网络流量），每转换完一片就删除本节的临时下载副本；本地额外空间主要是一片 BF16 的大小。转换和额外上传的耗时未计入前面的估计。
- **续跑**：保留已完成的 FP8 分片及 SHA-256 记录。重跑时校验并跳过完整分片；损坏分片重建。若上传已经成功且全部远端文件一致，则直接返回已有 commit；目标仓库有其他内容时停止，要求换仓名。
- **状态**：BF16 未完成时跳过；`upload=False` 时也跳过。FP8 出错不影响 BF16 成品，修正后重跑本节即可。若重启会话，先运行配置、环境和模块加载单元格，再直接运行本节，可从 `unified_run_status.json` 恢复 BF16 上传结果，无需重做搜索。
- **验证边界**：执行的是格式转换和文件校验，不做完整模型推理/生成评测。FP8 推理仍需支持该模型架构、此量化格式的后端与硬件；CPU 导出不代表支持 CPU FP8 推理。原 BF16 的代理评分不能直接当作 FP8 评分。

配置中的 `fp8_enabled=False` 可关闭这一步；`fp8_repo` 留空时自动使用原仓库名加 `-FP8`。


In [ ]:
# FP8 独立模块；不改动原融合、搜索模块的代码指纹。
FP8_MODULE_SOURCE = r'''
"""CPU streaming export to fine-grained FP8; separate from merge/search caches."""
import gc
import hashlib
import json
from pathlib import Path
import re
import shutil

import torch
from safetensors import safe_open
from safetensors.torch import save_file

VERSION = 'auxmix-fp8-2026-10-22.1'
BLOCK = 128
# Hybrid linear-attention kernels can access .weight directly; retain their original dtype.
# MTP / NextN draft-head layers match the pattern too (mtp.layers.0....) and are kept in their original dtype as well.
TARGET = re.compile(r'(?:^|\.)layers\.\d+\.(?:mlp\.(?:gate_proj|up_proj|down_proj)|self_attn\.(?:q_proj|k_proj|v_proj|o_proj))\.weight$')


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def atomic_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')
    temp.replace(path)


def file_hash(path, git_blob=False):
    path = Path(path)
    h = hashlib.sha1() if git_blob else hashlib.sha256()
    if git_blob:
        h.update(f'blob {path.stat().st_size}\0'.encode())
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(8 * 1024**2), b''):
            h.update(chunk)
    return h.hexdigest()


def safe_name(name):
    path = Path(name)
    if not name or path.is_absolute() or '..' in path.parts or '\\' in name or name.startswith('.'):
        raise ValueError('不安全的模型文件名：' + name)
    return name


def quantize_weight(weight):
    """E4M3FN, float32 dequantization scales: original ~= fp8 * weight_scale_inv."""
    if weight.ndim != 2 or any(d % BLOCK for d in weight.shape) or not weight.is_floating_point():
        raise ValueError('FP8 只转换两个维度都能被 128 整除的浮点矩阵')
    rows, cols = weight.shape
    quantized = torch.empty((rows, cols), dtype=torch.float8_e4m3fn, device='cpu')
    scales = torch.empty((rows // BLOCK, cols // BLOCK), dtype=torch.float32)
    for row in range(0, rows, BLOCK):
        block = weight[row:row + BLOCK].float().reshape(BLOCK, cols // BLOCK, BLOCK)
        if not torch.isfinite(block).all():
            raise ValueError('源权重含 NaN/Inf，停止 FP8 导出')
        maxima = block.abs().amax(dim=(0, 2))
        scale = torch.where(maxima > 0, (maxima / 448.0).clamp_min(torch.finfo(torch.float32).tiny), torch.ones_like(maxima))
        quantized[row:row + BLOCK] = (block / scale[None, :, None]).clamp(-448, 448).to(torch.float8_e4m3fn).reshape(BLOCK, cols)
        scales[row // BLOCK] = scale
    return quantized, scales


def export_checkpoint(fetch, files, directory, source, release=None):
    """fetch(name) returns a pinned local source file; only one source shard is needed at a time."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    files = {safe_name(n) for n in files if n != '.gitattributes'}
    config = read_json(fetch('config.json'))
    if config.get('quantization_config') or config.get('text_config', {}).get('quantization_config'):
        raise ValueError('FP8 输入必须是未量化的最终 BF16 成品')
    if 'model.safetensors.index.json' in files:
        source_map = read_json(fetch('model.safetensors.index.json'))['weight_map']
        shards = sorted({safe_name(n) for n in source_map.values()})
    elif 'model.safetensors' in files:
        source_map, shards = None, ['model.safetensors']
    else:
        raise ValueError('成品缺少标准 safetensors 主权重或索引')
    if not set(shards).issubset(files):
        raise ValueError('源模型索引引用了不存在的分片')
    identity = {'version': VERSION, 'source': source, 'block': [BLOCK, BLOCK], 'torch': str(torch.__version__)}
    state_dir = directory / '.fp8_state'
    state_dir.mkdir(exist_ok=True)
    identity_path = state_dir / 'identity.json'
    if identity_path.exists() and read_json(identity_path) != identity:
        raise ValueError('FP8 目录属于其他输入或导出版本，请更换 fp8_dir')
    atomic_json(identity_path, identity)
    weight_map, ignored, quantized_names, seen = {}, set(), [], set()
    total_size = 0
    for number, name in enumerate(shards, 1):
        output = directory / name
        receipt = state_dir / (hashlib.sha256(name.encode()).hexdigest() + '.json')
        record = read_json(receipt) if receipt.exists() else None
        valid = record and output.exists() and file_hash(output) == record['sha256']
        if not valid:
            print(f'FP8 [{number}/{len(shards)}] {name}', flush=True)
            path = Path(fetch(name))
            if shutil.disk_usage(directory).free < path.stat().st_size + 256 * 1024**2:
                raise OSError('FP8 输出目录剩余空间不足以安全写入下一个分片')
            tensors, skip, converted, original_keys = {}, [], [], []
            with safe_open(path, framework='pt', device='cpu') as reader:
                for key in reader.keys():
                    if source_map is not None and source_map.get(key) != name:
                        raise ValueError('源索引与分片不一致：' + key)
                    original_keys.append(key)
                    value = reader.get_tensor(key)
                    eligible = (TARGET.search(key) and not any(x in key.split('.') for x in ('visual', 'vision', 'vision_tower', 'mtp', 'nextn'))
                                and value.ndim == 2 and all(d % BLOCK == 0 for d in value.shape)
                                and value.dtype in (torch.float16, torch.bfloat16, torch.float32))
                    if eligible:
                        tensors[key], tensors[key[:-6] + 'weight_scale_inv'] = quantize_weight(value)
                        converted.append(key)
                    else:
                        tensors[key] = value
                        if key.endswith('.weight'):
                            skip.append(key[:-7])
                del value
            output.parent.mkdir(parents=True, exist_ok=True)
            temp = output.with_name(output.name + '.tmp')
            save_file(tensors, temp, metadata={'format': 'pt'})
            temp.replace(output)
            record = {'sha256': file_hash(output), 'keys': sorted(tensors), 'original_keys': original_keys,
                      'ignored': skip, 'quantized': converted,
                      'bytes': sum(t.numel() * t.element_size() for t in tensors.values())}
            atomic_json(receipt, record)
            del tensors
            gc.collect()
            if release is not None:
                release(name, path)
        else:
            print(f'FP8 [{number}/{len(shards)}] 已校验，沿用 {name}', flush=True)
        if seen.intersection(record['original_keys']):
            raise ValueError('源分片含重复张量')
        seen.update(record['original_keys'])
        weight_map.update({key: name for key in record['keys']})
        ignored.update(record['ignored'])
        quantized_names.extend(record['quantized'])
        total_size += record['bytes']
    if source_map is not None and seen != set(source_map):
        raise ValueError('源分片缺少索引中的张量')
    if not quantized_names:
        raise ValueError('没有匹配到可转换的注意力/MLP 矩阵，不上传伪 FP8 模型')
    config['quantization_config'] = {'quant_method': 'fp8', 'fmt': 'e4m3', 'activation_scheme': 'dynamic',
                                   'weight_block_size': [BLOCK, BLOCK], 'modules_to_not_convert': sorted(ignored | {'lm_head'})}
    atomic_json(directory / 'config.json', config)
    atomic_json(directory / 'model.safetensors.index.json', {'metadata': {'total_size': total_size}, 'weight_map': weight_map})
    # Keep tokenizer, chat templates, processor, merge evidence and all upstream notices.
    extras = sorted(n for n in files if n not in {'config.json', 'model.safetensors.index.json', 'README.md'}
                    and not n.endswith(('.safetensors', '.bin', '.pt', '.pth')))
    for name in extras:
        target = directory / name
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(fetch(name), target)
    original_card = Path(fetch('README.md')).read_text(encoding='utf-8') if 'README.md' in files else ''
    note = ('\n## FP8 export\n\nFine-grained FP8 E4M3FN, 128x128 weight blocks, float32 weight_scale_inv, '
            'dynamic activation quantization. Decoder MLP and full-attention projections only; embeddings, lm_head, '
            'vision, linear attention, the MTP draft head and non-tileable matrices retain their original precision.\n\n'
            f'BF16 source: {source.get("repo", "")} @ {source.get("commit", "")}. '
            'No calibration dataset is used. BF16 proxy scores below are not FP8 quality measurements. '
            'FP8 inference needs a compatible model implementation, FP8 backend and supported hardware; '
            'CPU export does not imply CPU FP8 inference support. No full-model FP8 inference/evaluation was run by this export.\n\n')
    if original_card.startswith('---\n') and '\n---\n' in original_card[4:]:
        end = original_card.index('\n---\n', 4) + 5
        card = original_card[:end] + note + original_card[end:]
    else:
        card = '# AuxMix FP8\n' + note + original_card
    (directory / 'README.md').write_text(card, encoding='utf-8')
    published_files = sorted(set(shards) | set(extras) | {'config.json', 'model.safetensors.index.json', 'README.md'})
    manifest = {'identity': identity, 'quantized_tensors': len(quantized_names), 'quantized_weights': quantized_names,
                'files': {n: file_hash(directory / n) for n in published_files}}
    atomic_json(directory / 'fp8_manifest.json', manifest)
    return manifest


def publish(directory, repo, private, api, token):
    from huggingface_hub import CommitOperationAdd
    directory = Path(directory)
    manifest = read_json(directory / 'fp8_manifest.json')
    files = set(manifest['files']) | {'fp8_manifest.json'}
    api.create_repo(repo_id=repo, repo_type='model', private=private, exist_ok=True, token=token)
    info = api.model_info(repo, files_metadata=True, token=token)
    existing = {s.rfilename: s for s in info.siblings if s.rfilename != '.gitattributes'}
    if existing:
        same = set(existing) == files
        for name in sorted(files) if same else []:
            item = existing[name]
            lfs = getattr(item, 'lfs', None)
            expected = (lfs.get('sha256') if isinstance(lfs, dict) else getattr(lfs, 'sha256', None)) if lfs else getattr(item, 'blob_id', None)
            if file_hash(directory / name, git_blob=not bool(lfs)) != expected:
                same = False
                break
        if not same:
            raise ValueError(f'{repo} 已有其他内容；请将 fp8_repo 改为新的空仓库。')
        commit = info.sha
    else:
        commit = api.create_commit(repo_id=repo, repo_type='model', token=token, parent_commit=info.sha,
            operations=[CommitOperationAdd(path_in_repo=n, path_or_fileobj=str(directory / n)) for n in sorted(files)],
            commit_message='Add block FP8 export of completed BF16 AuxMix model').oid
    return {'status': 'uploaded', 'repo': repo, 'commit': commit, 'private': bool(info.private),
            'source': manifest['identity']['source'], 'directory': str(directory),
            'quantized_tensors': manifest['quantized_tensors']}


def run_fp8(settings, result, api, token):
    if not settings.get('fp8_enabled', True):
        return {'status': 'disabled'}
    if result.get('status') != 'uploaded':
        return {'status': 'waiting_for_bf16', 'bf16_status': result.get('status')}
    repo = (settings.get('fp8_repo') or result['repo'] + '-FP8').strip()
    forbidden = {result['repo'], settings.get('output_repo'), settings.get('S_repo'), settings.get('M_repo'), settings.get('B_repo')}
    for extra in list(settings.get('aux_extras', [])) + list(settings.get('searched_extras', [])):
        forbidden.update({extra.get('repo'), extra.get('reference_repo')})
        forbidden.update(extra.get('bf16_repos', []))
        for m in extra.get('members', []):
            forbidden.update({m.get('repo'), m.get('reference_repo')})
    forbidden.discard(None)
    forbidden.discard('')
    if repo in forbidden:
        raise ValueError('fp8_repo 必须是独立的新仓库，不能等于 BF16 或任何输入仓库')
    source = {k: result[k] for k in ('repo', 'commit', 'weights')}
    key = hashlib.sha256(json.dumps({'source': source, 'version': VERSION, 'torch': str(torch.__version__)}, sort_keys=True).encode()).hexdigest()[:24]
    output = Path(settings.get('fp8_dir') or Path(settings['work']) / 'fp8') / key
    cache = Path(settings['scratch']) / 'fp8_source' / key
    from huggingface_hub import hf_hub_download
    def fetch(name):
        return Path(hf_hub_download(repo_id=source['repo'], filename=safe_name(name), revision=source['commit'],
                                   token=token, local_dir=str(cache)))
    def release(name, path):
        # Only our dedicated download copy is removed; BF16 repo and merge files stay intact.
        path.unlink()
    info = api.model_info(source['repo'], revision=source['commit'], token=token)
    print(f'FP8 来源：{source["repo"]} @ {source["commit"]}；目标：{repo}', flush=True)
    manifest = export_checkpoint(fetch, [s.rfilename for s in info.siblings], output, source, release=release)
    uploaded = publish(output, repo, bool(settings.get('private', True)), api, token)
    atomic_json(Path(settings['work']) / 'fp8_uploaded.json', uploaded)
    print(f'FP8 已上传：https://huggingface.co/{repo}（{manifest["quantized_tensors"]} 个 FP8 矩阵）', flush=True)
    return uploaded

'''
load_embedded_module('qmerge_fp8', FP8_MODULE_SOURCE)


In [ ]:
# 重启会话后可只运行初始化 + 本节，从记录恢复已上传的 BF16 成品。
_bf16_result = globals().get("RESULT")
if _bf16_result is None:
    _status_path = EFFICIENT_WORK / "unified_run_status.json"
    _bf16_result = json.loads(_status_path.read_text(encoding="utf-8")) if _status_path.exists() else {"status": "not_started"}
if _bf16_result.get("status") == "uploaded" and _bf16_result.get("repo") != SETTINGS["output_repo"]:
    raise ValueError("BF16 上传记录与当前 output_repo 不一致，请先运行第 5 节。")
try:
    FP8_RESULT = qmerge_fp8.run_fp8(SETTINGS, _bf16_result, api, TOK)
except Exception as error:
    FP8_RESULT = {"status": "failed", "bf16_repo": _bf16_result.get("repo"),
                  "error_type": type(error).__name__}
    qmerge_core.atomic_json(EFFICIENT_WORK / "fp8_run_status.json", FP8_RESULT)
    print("FP8 阶段失败；BF16 成品与上传记录不受影响。修正后重跑本节，已完成分片会校验后继续使用。")
    raise
qmerge_core.atomic_json(EFFICIENT_WORK / "fp8_run_status.json", FP8_RESULT)
print(json.dumps(FP8_RESULT, ensure_ascii=False, indent=2))


## 7. 查看结果与续跑

- `DRIVE_DIR/aux_delta/<指纹>/aux_delta_report.json`：各来源在全模型上的增量范数；DELLA 组每个成员的增量范数、保留比例和符号当选比例；Winery 每个张量选了哪种存储顺序、与 B 的相对差、噪声占比和 g，以及被排除的张量和原因。补加 StartLux 后，之前各来源的统计在 `composed_from` 里。
- `DRIVE_DIR/unified_run_status.json`、`result.json`：本次运行的状态和结果。
- `DRIVE_DIR/tm_delta/`：跨架构增量 N。`tm_result.record.json` 是结论（`go` / `no-go`）和判定表；`gate.record.json` 是前向闸门；`verdict.record.json` 是每个模块的 z 倍数、锐度倍数、上界标定；`pairs/` 是每个模块 CKA 留下的层对和层对 OT（续跑用）；`donor_acts/` 是供体激活（约 11GB，改 alpha / topk / inject 只需重做 OT 与融合）；`delta/<指纹>/` 里是 N 和 `tm_report.json`。
- `DRIVE_DIR/tm_delta2/`：跨架构增量 N₂（SenseNova）。`donor_metadata/` 里是锁定 commit 的配置、tokenizer 和远程代码（审读用）；`gate.record.json` 是前向闸门和观察到的结构（执行顺序、全注意力层、宽度）；其余文件与 `tm_delta/` 相同，`tm_report.json` 里每张矩阵还记着用了哪些执行、对应哪个物理层。
- `DRIVE_DIR/row_change_report.json`：λ·D、μ·C、二者之和以及输出头在每个网格值下超过行改动上限的行的比例，D 和 C 的逐行 ‖·‖/‖S‖ 分布（按模块）。等待阶段算的是不含 StartLux 和 DavidAU 的 D。
- `DRIVE_DIR/backward_check.json`：反向自检的结果：用的哪道题，每个检查的矩阵、方向、一阶预测和真实中心差分、是否通过；不通过时分量筛选会被跳过。
- `DRIVE_DIR/search_result.json`：`trials` 是 S 对照、GP 主候选、head 补评及实际评过的 μ=0 候选的配方与代理 NLL；`main_trials` / `paired_head_trials` 分别是主候选和二值非 tied head 补评的计数，`global_trials` 预算只计前者；`nu_line` 是有 N 时在获胜配方上评过的 ν、各自的 NLL 和选中的 ν（没有 N 时为 `null`）；`nu2_line` 是有 N₂ 时在“获胜配方 + ν·N”上评过的 ν₂ 和选中的 ν₂；`mu_ablation` 未触发时为 `null`，触发后含 `with_mu` / `without_mu`（各有 `recipe`、`mean_nll`）、`without_mu_feasible`、`c_benefit_nll` 和 `selected_without_mu`；`kappa_profile` 和 `mu_profile` 分别是按 κ、μ 分组的最低代理 NLL。`selected_candidate.json`：本次选中的配方、变体、去掉的分量和代理 NLL。
- `DRIVE_DIR/component_selection.json`：分量筛选的结果。`components` 里每个分量一行：层段、模块、奇异方向段和在 S 子空间之内 / 外、占增量的份额（Frobenius 范数平方）、一阶效果（输出头为精确端点差；负数表示留着有益）、标准误、95% 区间、有益 / 有害概率、各方案给它的系数；`preflight` 里是梯度自检的结果；`crossfit` 里是每种方案在折外题上相对 F 的真实变化（95% 区间、变好的概率）和一阶预测，这是内部代理估计：定本折系数不用另一折的损失 / 梯度，但所有题共享教师强制激活协方差分解，不是完全独立验证或最终验证；`heldout` 里是代理验证集（全局搜索没用过）上 F 和选中模型各自相对 S 的变化；`variants` 里是用全部题挑出的方案在全部题上的结果；`subject_subspace` 里是 S 自己微调子空间之内 / 之外的分量各占增量能量和一阶有益效果的多少。上传的仓库里也有一份。
- `DRIVE_DIR/component_store/`：分量筛选的中间结果。梯度这一步中断后重跑会接着算；各矩阵的分解（`decomp-*.safetensors`，约 3GB）在上传之前不能删，上传完成后整个目录可以删除。
- `DRIVE_DIR/scoring_engine_check.json`：逐层评分与分批前向在 S 上的对照结论（用了哪个引擎、NLL 差多少）。想换引擎可以在 `proxy_options["engine"]` 里强制指定；换了以后搜索要重做，S 的基线也会按新引擎重算。
- `DRIVE_DIR/fp8_run_status.json` / `fp8_uploaded.json`：FP8 阶段状态、独立仓库及 commit；不改写 BF16 的上传记录。
- `DRIVE_DIR/fp8/<指纹>/`：FP8 成品、`fp8_manifest.json`、隐藏目录 `.fp8_state/` 中的逐分片续跑记录。中断后重跑第 6 节会校验并沿用完整分片；上传成功后如果不再需要本地副本，可以删除这一目录。
- `DRIVE_DIR/uploaded.json`：上传到的仓库和 commit。仓库里的 `selection.json` 和 `merge_manifest.json` 记着配方、去掉的分量、来源版本和公式，`README.md` 写明没有做生成评测。
- 默认传成私有仓库（`private=True`）。公开之前请核对各来源的许可证（仓库里的 `source_licenses.json` 和 `source_notices/`）；ThinkingCap 是 PolyForm Small Business 许可；StartLux 发布后也要看它的许可证（同一家已发布的模型有用 cc-by-nc-4.0 的），很可能限制商用，成品的许可证要取各来源里最严的。

续跑：断开后从头运行全部单元格即可。D、搜索记录和分量筛选的进度都在 Drive 上；本地的 S / D 副本会重新下载或复制。分量筛选以 `failed` / `skipped` 回退时，本次仍写 `selected_candidate.json` 并可继续构造、上传全局搜索候选，但不保存可复用的 `selection_record.json`；下次重跑会重试，已经完成的搜索评分和分量断点仍可复用。上传回执身份包含分量选择结果，失败后重试成功不会沿用旧回执；若选择元数据已不同，需要换新的 `output_repo`。上传中断后需要继续上传时只会重建这一个候选；若上传回执丢失且目标仓库已有内容，会在全部发布文件生成后重新读取远端状态，只有完整发布清单（Hub 自动管理的 `.gitattributes` 除外）及逐项哈希全部一致才认定为已上传，不完整或不同则拒绝覆盖。

改了比例、来源或版本，D 会得到新的指纹并重新计算；`inputs.json` 会拒绝在同一个记录目录里混用两组输入，这时请换一个新的 `DRIVE_DIR`。改了搜索设置会选出新的候选，要上传就换一个新的 `output_repo`（本 notebook 不覆盖已有内容的仓库）。不要同时对同一个目录启动多个运行。
